In [ ]:
# ============================================================
# CELL 1 — CLEAN FEDCTGAD-X RESEARCH ENVIRONMENT
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import shutil
from pathlib import Path

DRIVE_PROJECT = Path("/content/drive/MyDrive/fedctgad_project")
WORK_ROOT = Path("/content/fedctgad_x_v2")

print("Drive project exists:", DRIVE_PROJECT.exists())

if not DRIVE_PROJECT.exists():
    raise FileNotFoundError(
        f"Could not find {DRIVE_PROJECT}. "
        "Check the project location in Google Drive."
    )

# Fresh working directory
if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)

WORK_ROOT.mkdir(parents=True, exist_ok=True)

zip_candidates = list(DRIVE_PROJECT.glob("*.zip"))

print("\nZIP files found:")
for p in zip_candidates:
    print(" -", p)

if not zip_candidates:
    raise FileNotFoundError(
        "No ZIP file found inside fedctgad_project."
    )

ZIP_PATH = zip_candidates[0]

print("\nUsing:", ZIP_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive project exists: True

ZIP files found:
 - /content/drive/MyDrive/fedctgad_project/fedctgad.zip

Using: /content/drive/MyDrive/fedctgad_project/fedctgad.zip


In [ ]:
# ============================================================
# CELL 2 — EXTRACT EXISTING IMPLEMENTATION
# ============================================================

import zipfile

EXTRACT_ROOT = WORK_ROOT / "project"

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_ROOT)

# Find directory containing run_simulation.py
run_files = list(EXTRACT_ROOT.rglob("run_simulation.py"))

if not run_files:
    raise FileNotFoundError(
        "run_simulation.py was not found after extracting the project."
    )

PROJECT_ROOT = run_files[0].parent

print("PROJECT_ROOT =", PROJECT_ROOT)

print("\nProject files:")
for p in sorted(PROJECT_ROOT.rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(PROJECT_ROOT))

PROJECT_ROOT = /content/fedctgad_x_v2/project/fedctgad

Project files:
  README.md
  attacks/poison.py
  attacks/ttl_spoof.py
  client/client.py
  configs/default.yaml
  data/partition.py
  data/slicing.py
  eval/metrics.py
  model/encoder.py
  model/losses.py
  run_simulation.py
  server/aggregate.py


In [ ]:
# ============================================================
# CELL 3 — DEPENDENCIES
# ============================================================

!pip install -q torch-geometric pyyaml scikit-learn pandas numpy matplotlib seaborn

import sys
import os
import random
import numpy as np
import pandas as pd
import torch

sys.path.insert(0, str(PROJECT_ROOT))

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)

try:
    import torch_geometric
    print("PyG:", torch_geometric.__version__)
except Exception as e:
    print("PyG import error:", e)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 45.2 MB/s eta 0:00:00
Python: 3.13.15
PyTorch: 2.11.0+cu128
PyG: 2.8.0.post1


In [ ]:
# ============================================================
# CELL 4 — HARDWARE CHECK
# ============================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )
else:
    print("WARNING: GPU not available.")

Device: cuda
GPU: Tesla T4
GPU memory: 14.56 GB


In [ ]:
# ============================================================
# CELL 5 — DATASET DISCOVERY
# ============================================================

DATA_ROOT = DRIVE_PROJECT / "data"

if not DATA_ROOT.exists():
    raise FileNotFoundError(f"Data directory not found: {DATA_ROOT}")

csv_files = sorted(DATA_ROOT.rglob("*.csv"))

print(f"CSV files found: {len(csv_files)}\n")

for p in csv_files:
    print(p)

CSV files found: 1

/content/drive/MyDrive/fedctgad_project/data/UNSW-NB15_1.csv


In [ ]:
# ============================================================
# CELL 6 — RAW DATASET AUDIT
# ============================================================

UNSW_FILES = [
    p for p in csv_files
    if "UNSW" in p.name.upper()
]

if not UNSW_FILES:
    raise FileNotFoundError(
        "No UNSW CSV was found. "
        "Show me the output of CELL 5 before continuing."
    )

print("UNSW files:")
for p in UNSW_FILES:
    print(" -", p)

UNSW_PATH = UNSW_FILES[0]

print("\nUsing:")
print(UNSW_PATH)

# Read only a small sample first
raw_sample = pd.read_csv(
    UNSW_PATH,
    header=None,
    nrows=5,
    low_memory=False
)

print("\nShape of sample:", raw_sample.shape)
display(raw_sample.head())

UNSW files:
 - /content/drive/MyDrive/fedctgad_project/data/UNSW-NB15_1.csv

Using:
/content/drive/MyDrive/fedctgad_project/data/UNSW-NB15_1.csv

Shape of sample: (5, 49)


,0,1,2,3,4,5,6,7,8,9,...,39,40,41,42,43,44,45,46,47,48
0,59.166.0.0,1390,149.171.126.6,53,udp,CON,0.001055,132,164,31,...,0,3,7,1,3,1,1,1,NaN,0
1,59.166.0.0,33661,149.171.126.9,1024,udp,CON,0.036133,528,304,31,...,0,2,4,2,3,1,1,2,NaN,0
2,59.166.0.6,1464,149.171.126.7,53,udp,CON,0.001119,146,178,31,...,0,12,8,1,2,2,1,1,NaN,0
3,59.166.0.5,3593,149.171.126.5,53,udp,CON,0.001209,132,164,31,...,0,6,9,1,1,1,1,1,NaN,0
4,59.166.0.3,49664,149.171.126.0,53,udp,CON,0.001169,146,178,31,...,0,7,9,1,1,1,1,1,NaN,0


In [ ]:
# ============================================================
# CELL 7 — UNSW-NB15 LOADER
# ============================================================

UNSW_COLUMNS = [
    "srcip", "sport", "dstip", "dsport", "proto", "state", "dur",
    "sbytes", "dbytes", "sttl", "dttl", "sloss", "dloss", "service",
    "sload", "dload", "spkts", "dpkts", "swin", "dwin", "stcpb",
    "dtcpb", "smeansz", "dmeansz", "trans_depth", "res_bdy_len",
    "sjit", "djit", "stime", "ltime", "sintpkt", "dintpkt",
    "tcprtt", "synack", "ackdat", "is_sm_ips_ports", "ct_state_ttl",
    "ct_flw_http_mthd", "is_ftp_login", "ct_ftp_cmd", "ct_srv_src",
    "ct_srv_dst", "ct_dst_ltm", "ct_src_ltm", "ct_src_dport_ltm",
    "ct_dst_sport_ltm", "ct_dst_src_ltm", "attack_cat", "label"
]

assert len(UNSW_COLUMNS) == 49

NROWS = 200_000

df = pd.read_csv(
    UNSW_PATH,
    header=None,
    names=UNSW_COLUMNS,
    nrows=NROWS,
    low_memory=False
)

# Basic cleaning
df["attack_cat"] = (
    df["attack_cat"]
    .fillna("Normal")
    .astype(str)
    .str.strip()
)

# Rename to match our implementation
df = df.rename(columns={
    "srcip": "src_ip",
    "dstip": "dst_ip",
    "stime": "ts_start",
    "dur": "duration"
})

# Numeric conversion
numeric_cols = [
    "ts_start", "duration",
    "sbytes", "dbytes",
    "spkts", "dpkts",
    "sttl", "dttl",
    "sload", "dload"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(
    subset=["src_ip", "dst_ip", "ts_start", "duration"]
).copy()

df["duration"] = df["duration"].clip(lower=0)

df = df.sort_values("ts_start").reset_index(drop=True)

print("Dataset shape:", df.shape)

print("\nAttack-family distribution:")
display(
    df["attack_cat"]
    .value_counts(dropna=False)
    .to_frame("count")
)

print("\nTime range:")
print(
    df["ts_start"].min(),
    "→",
    df["ts_start"].max()
)

print(
    "Duration:",
    round(
        (df["ts_start"].max() - df["ts_start"].min()) / 60,
        2
    ),
    "minutes"
)

Dataset shape: (200000, 49)

Attack-family distribution:


,count
attack_cat,
Normal,177785
Generic,7522
Exploits,5409
Fuzzers,5051
Reconnaissance,1759
DoS,1167
Backdoors,534
Analysis,526
Shellcode,223



Time range:
1421927377 → 1421935026
Duration: 127.48 minutes


In [ ]:
# ============================================================
# CELL 8 — FROZEN BASELINE CONFIGURATION
# ============================================================

BASELINE = {
    "dataset": "UNSW-NB15",
    "nrows": NROWS,

    # CTGAD-X baseline configuration
    "delta_t": 30.0,
    "seq_len": 5,

    # zero-day protocol
    "held_out_family": "Exploits",

    # sequence purity thresholds
    "normal_max_attack_fraction": 0.05,
    "attack_min_attack_fraction": 0.10,

    # training split
    "train_fraction": 0.70,
    "val_fraction": 0.10,
    "test_fraction": 0.20,

    # model
    "hidden_dim": 60,
    "struct_heads": 4,
    "temporal_heads": 4,
    "proj_dim": 80,

    # contrastive learning
    "temperature": 0.10,
    "learning_rate": 1e-3,

    # anomaly score
    "k_neighbors": 15,

    "seed": 42
}

BASELINE

{'dataset': 'UNSW-NB15',
 'nrows': 200000,
 'delta_t': 30.0,
 'seq_len': 5,
 'held_out_family': 'Exploits',
 'normal_max_attack_fraction': 0.05,
 'attack_min_attack_fraction': 0.1,
 'train_fraction': 0.7,
 'val_fraction': 0.1,
 'test_fraction': 0.2,
 'hidden_dim': 60,
 'struct_heads': 4,
 'temporal_heads': 4,
 'proj_dim': 80,
 'temperature': 0.1,
 'learning_rate': 0.001,
 'k_neighbors': 15,
 'seed': 42}

In [ ]:
# ============================================================
# CELL 9 — GLOBAL TEMPORAL INDEX
# ============================================================

DT = BASELINE["delta_t"]

GLOBAL_T0 = float(df["ts_start"].min())

df["t_rel"] = df["ts_start"] - GLOBAL_T0

df["window_start_idx"] = np.floor(
    df["t_rel"] / DT
).astype(int)

df["window_end_idx"] = np.floor(
    (df["t_rel"] + df["duration"].clip(lower=1e-6)) / DT
).astype(int)

print("Global t0:", GLOBAL_T0)
print("Window size:", DT, "seconds")
print("Number of start windows:", df["window_start_idx"].nunique())
print(
    "Window index range:",
    df["window_start_idx"].min(),
    "→",
    df["window_end_idx"].max()
)

Global t0: 1421927377.0
Window size: 30.0 seconds
Number of start windows: 255
Window index range: 0 → 254


In [ ]:
# ============================================================
# CELL 10 — GLOBAL WINDOW LABEL METADATA
# ============================================================

from collections import defaultdict

window_total = defaultdict(float)
window_attack = defaultdict(float)
window_heldout = defaultdict(float)
window_families = defaultdict(set)

HELD_OUT = BASELINE["held_out_family"]

for row in df.itertuples(index=False):

    start = float(row.ts_start - GLOBAL_T0)
    dur = max(float(row.duration), 1e-6)
    end = start + dur

    first_w = int(np.floor(start / DT))
    last_w = int(np.floor(end / DT))

    for w in range(first_w, last_w + 1):

        w_start = w * DT
        w_end = w_start + DT

        overlap = max(
            0.0,
            min(end, w_end) - max(start, w_start)
        )

        if overlap <= 0:
            continue

        # Weight by temporal overlap.
        # This keeps labeling consistent with the causal slicing idea.
        weight = overlap / dur

        window_total[w] += weight

        if row.attack_cat != "Normal":
            window_attack[w] += weight
            window_families[w].add(row.attack_cat)

        if row.attack_cat == HELD_OUT:
            window_heldout[w] += weight


def attack_fraction(w):
    return window_attack[w] / max(window_total[w], 1e-12)


def heldout_fraction(w):
    return window_heldout[w] / max(window_total[w], 1e-12)


window_ids = sorted(window_total.keys())

print("Aligned windows:", len(window_ids))

attack_fracs = np.array(
    [attack_fraction(w) for w in window_ids]
)

heldout_fracs = np.array(
    [heldout_fraction(w) for w in window_ids]
)

print(
    "Attack fraction 10/50/90%:",
    np.percentile(attack_fracs, [10, 50, 90]).round(4)
)

print(
    "Held-out fraction 10/50/90%:",
    np.percentile(heldout_fracs, [10, 50, 90]).round(4)
)

print(
    "Windows with >=10% attack:",
    int((attack_fracs >= 0.10).sum())
)

print(
    f"Windows with >=5% {HELD_OUT}:",
    int((heldout_fracs >= 0.05).sum())
)

Aligned windows: 255
Attack fraction 10/50/90%: [0.0396 0.0922 0.1662]
Held-out fraction 10/50/90%: [0.0135 0.0237 0.0357]
Windows with >=10% attack: 112
Windows with >=5% Exploits: 8


In [ ]:
# ============================================================
# CELL 11 — WRITE CORRECT GRAPH LOADER
# ============================================================

loader_code = r'''
import numpy as np
import torch
from torch_geometric.data import Data, Batch


FEATURE_COLS = [
    "sbytes",
    "dbytes",
    "spkts",
    "dpkts",
    "sload",
    "dload",
    "sttl",
    "dttl",
]


def snapshot_to_data(snap, scaler):
    """
    Convert one GraphSnapshot into a PyG Data object.

    Feature order must remain exactly:
    sbytes, dbytes, spkts, dpkts, sload, dload, sttl, dttl
    """

    if len(snap.node_ids) == 0:
        raise ValueError("Empty snapshot.")

    x = np.stack(
        [snap.node_features[n] for n in snap.node_ids],
        axis=0
    ).astype(np.float32)

    x = scaler.transform(x).astype(np.float32)

    if len(snap.edges) > 0:
        edge_index = torch.tensor(
            np.asarray(snap.edges, dtype=np.int64).T,
            dtype=torch.long
        )
    else:
        # self-loops guarantee GATv2 has valid message passing
        edge_index = torch.arange(
            len(snap.node_ids),
            dtype=torch.long
        ).repeat(2, 1)

    return Data(
        x=torch.tensor(x, dtype=torch.float32),
        edge_index=edge_index,
    )


def sequence_to_data(sequence, scaler):
    return [
        snapshot_to_data(snap, scaler)
        for snap in sequence
    ]


def batch_sequences(sequences, scaler):
    """
    Convert a list of graph sequences into the representation expected by
    CTGAD_Encoder:
        [
          Batch(graphs_at_t0),
          Batch(graphs_at_t1),
          ...
        ]
    """

    if not sequences:
        return []

    seq_len = len(sequences[0])

    output = []

    for t in range(seq_len):
        graphs_t = [
            snapshot_to_data(seq[t], scaler)
            for seq in sequences
        ]

        output.append(
            Batch.from_data_list(graphs_t)
        )

    return output


def to_batches(sequences, scaler, batch_size=16):
    """
    Split sequences into mini-batches while preserving temporal alignment.
    """

    batches = []

    for i in range(0, len(sequences), batch_size):
        batches.append(
            batch_sequences(
                sequences[i:i + batch_size],
                scaler
            )
        )

    return batches
'''

loader_path = PROJECT_ROOT / "data" / "loaders.py"
loader_path.parent.mkdir(parents=True, exist_ok=True)
loader_path.write_text(loader_code)

print("Written:", loader_path)

Written: /content/fedctgad_x_v2/project/fedctgad/data/loaders.py


In [ ]:
# ============================================================
# CELL 12 — PATCH CLIENT IMPORT
# ============================================================

client_path = PROJECT_ROOT / "client" / "client.py"

src = client_path.read_text()

old = """    def local_train_epoch(self, sequence_batches: list) -> float:
        \"\"\"One local epoch of NT-Xent training on this client's normal-only shard.\"\"\"
"""

new = """    def local_train_epoch(self, sequence_batches: list) -> float:
        \"\"\"One local epoch of NT-Xent training on this client's normal-only shard.\"\"\"
        from model.losses import nt_xent_loss
"""

if "from model.losses import nt_xent_loss" not in src:
    if old not in src:
        raise RuntimeError(
            "Expected local_train_epoch definition was not found."
        )

    src = src.replace(old, new)
    client_path.write_text(src)

print(
    "nt_xent import present:",
    "from model.losses import nt_xent_loss" in client_path.read_text()
)

nt_xent import present: True


In [ ]:
# ============================================================
# CELL 13 — CLEAN IMPORTS
# ============================================================

import importlib
import sys

# Make sure project is first on path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data.slicing import slice_flows_to_windows, snapshots_to_sequences
from data.loaders import snapshot_to_data, sequence_to_data, batch_sequences, to_batches

from model.encoder import CTGAD_Encoder, ProjectionHead
from model.losses import nt_xent_loss

from client.client import FederatedClient

print("All CTGAD-X components imported successfully.")

All CTGAD-X components imported successfully.


In [ ]:
# ============================================================
# CELL 14 — PATCH CAUSAL GRAPH SLICER
# CTGAD-X: 11-DIMENSIONAL NODE FEATURES
# ============================================================

from pathlib import Path

SLICER_PATH = PROJECT_ROOT / "data" / "slicing.py"

slicer_code = r'''
"""
Causality-aware temporal graph construction.

CTGAD-X-compatible node features:
    sbytes, dbytes,
    spkts, dpkts,
    sloss, dloss,
    sload, dload,
    sttl, dttl,
    dur

Each flow [ts, ts+dur) contributes only to the temporal
windows it actually overlaps.
"""

from __future__ import annotations

import numpy as np
import pandas as pd

from dataclasses import dataclass, field
from typing import Dict, List, Tuple


ADDITIVE_COLS = [
    "sbytes",
    "dbytes",
    "spkts",
    "dpkts",
    "sloss",
    "dloss",
]

STATE_COLS = [
    "sttl",
    "dttl",
]

FEATURE_COLS = [
    "sbytes",
    "dbytes",
    "spkts",
    "dpkts",
    "sloss",
    "dloss",
    "sload",
    "dload",
    "sttl",
    "dttl",
    "dur",
]


@dataclass
class GraphSnapshot:

    window_idx: int
    t_start: float
    t_end: float

    node_ids: List[str] = field(default_factory=list)

    # edges represented using integer node indices
    edges: List[Tuple[int, int]] = field(default_factory=list)

    # node_id -> 11-dimensional feature vector
    node_features: Dict[str, np.ndarray] = field(default_factory=dict)


def compute_overlap(
    t_s: float,
    dur: float,
    w_start: float,
    delta_t: float
) -> float:

    t_e = t_s + dur
    w_end = w_start + delta_t

    return max(
        0.0,
        min(t_e, w_end) - max(t_s, w_start)
    )


def slice_flows_to_windows(
    flows: pd.DataFrame,
    delta_t: float = 30.0,
    ts_col: str = "ts_start",
    dur_col: str = "duration",
    src_col: str = "src_ip",
    dst_col: str = "dst_ip",
) -> List[GraphSnapshot]:

    if flows.empty:
        return []

    flows = flows.copy()

    # IMPORTANT:
    # The temporal origin is computed ONCE from the complete
    # dataset passed to this function.
    t_min = float(flows[ts_col].min())

    flows[dur_col] = pd.to_numeric(
        flows[dur_col],
        errors="coerce"
    ).fillna(0.0).clip(lower=0.0)

    t_max = float(
        (flows[ts_col] + flows[dur_col]).max()
    )

    n_windows = int(
        np.ceil((t_max - t_min) / delta_t)
    ) + 1

    snapshots = [
        GraphSnapshot(
            window_idx=i,
            t_start=t_min + i * delta_t,
            t_end=t_min + (i + 1) * delta_t
        )
        for i in range(n_windows)
    ]

    # (window_idx, node_id) -> accumulators
    accum = {}

    # window -> set of (src, dst)
    edge_set = {
        i: set()
        for i in range(n_windows)
    }

    for _, flow in flows.iterrows():

        absolute_ts = float(flow[ts_col])

        t_s = absolute_ts - t_min

        dur = max(
            float(flow[dur_col]),
            1e-9
        )

        t_e = t_s + dur

        first_w = int(
            np.floor(t_s / delta_t)
        )

        last_w = int(
            np.floor(t_e / delta_t)
        )

        first_w = max(first_w, 0)
        last_w = min(last_w, n_windows - 1)

        src = str(flow[src_col])
        dst = str(flow[dst_col])

        for w in range(first_w, last_w + 1):

            w_start = w * delta_t

            overlap = compute_overlap(
                t_s,
                dur,
                w_start,
                delta_t
            )

            if overlap <= 0:
                continue

            # Fraction of flow duration represented
            # inside this window.
            frac = overlap / dur

            edge_set[w].add(
                (src, dst)
            )

            for node in (src, dst):

                key = (w, node)

                if key not in accum:

                    accum[key] = {
                        c: 0.0
                        for c in ADDITIVE_COLS
                    }

                    accum[key].update({
                        "sttl_sum": 0.0,
                        "dttl_sum": 0.0,
                        "state_count": 0,
                    })

                a = accum[key]

                # ------------------------------------------------
                # Additive counters
                # ------------------------------------------------

                for c in ADDITIVE_COLS:

                    value = pd.to_numeric(
                        flow.get(c, 0.0),
                        errors="coerce"
                    )

                    if pd.isna(value):
                        value = 0.0

                    a[c] += float(value) * frac

                # ------------------------------------------------
                # TTL state features
                # ------------------------------------------------

                sttl = pd.to_numeric(
                    flow.get("sttl", 0.0),
                    errors="coerce"
                )

                dttl = pd.to_numeric(
                    flow.get("dttl", 0.0),
                    errors="coerce"
                )

                if pd.isna(sttl):
                    sttl = 0.0

                if pd.isna(dttl):
                    dttl = 0.0

                a["sttl_sum"] += float(sttl)
                a["dttl_sum"] += float(dttl)
                a["state_count"] += 1

    # ------------------------------------------------------------
    # Construct node features
    # ------------------------------------------------------------

    for (w, node), a in accum.items():

        snap = snapshots[w]

        if node not in snap.node_ids:
            snap.node_ids.append(node)

        count = max(
            int(a["state_count"]),
            1
        )

        sbytes = a["sbytes"]
        dbytes = a["dbytes"]

        # Recompute rates from allocated counters.
        # sload/dload in UNSW-NB15 are represented as
        # bits per second.
        sload = (sbytes * 8.0) / max(
            delta_t,
            1e-9
        )

        dload = (dbytes * 8.0) / max(
            delta_t,
            1e-9
        )

        # Total temporal contribution represented
        # by this node in this window.
        node_duration = min(
            delta_t,
            max(
                0.0,
                a["state_count"]
            )
        )

        feature_vector = np.array([
            sbytes,
            dbytes,
            a["spkts"],
            a["dpkts"],
            a["sloss"],
            a["dloss"],
            sload,
            dload,
            a["sttl_sum"] / count,
            a["dttl_sum"] / count,
            node_duration,
        ], dtype=np.float32)

        snap.node_features[node] = feature_vector

    # ------------------------------------------------------------
    # Convert string node IDs to integer edge indices
    # ------------------------------------------------------------

    for w in range(n_windows):

        snap = snapshots[w]

        id_map = {
            node_id: idx
            for idx, node_id in enumerate(
                snap.node_ids
            )
        }

        snap.edges = [
            (
                id_map[src],
                id_map[dst]
            )
            for src, dst in edge_set[w]
            if src in id_map and dst in id_map
        ]

    # Keep empty windows out of graph computation.
    # IMPORTANT: sequence construction later checks the
    # actual window_idx values so temporal gaps are not hidden.

    return [
        s
        for s in snapshots
        if len(s.node_ids) > 0
    ]


def snapshots_to_sequences(
    snapshots: List[GraphSnapshot],
    seq_len: int = 5,
    stride: int = 1
):

    snapshots = sorted(
        snapshots,
        key=lambda x: x.window_idx
    )

    sequences = []

    for i in range(
        0,
        len(snapshots) - seq_len + 1,
        stride
    ):

        q = snapshots[
            i:i + seq_len
        ]

        # ---------------------------------------------
        # CRITICAL TEMPORAL-CONTIGUITY CHECK
        # ---------------------------------------------

        indices = [
            s.window_idx
            for s in q
        ]

        if any(
            indices[j + 1] != indices[j] + 1
            for j in range(len(indices) - 1)
        ):
            continue

        sequences.append(q)

    return sequences
'''

SLICER_PATH.write_text(slicer_code)

print("Patched:", SLICER_PATH)
print("CTGAD-X node dimension:", len([
    "sbytes","dbytes","spkts","dpkts",
    "sloss","dloss","sload","dload",
    "sttl","dttl","dur"
]))

Patched: /content/fedctgad_x_v2/project/fedctgad/data/slicing.py
CTGAD-X node dimension: 11


In [ ]:
# ============================================================
# CELL 15 — 11-D FEATURE GRAPH LOADER
# ============================================================

LOADER_PATH = PROJECT_ROOT / "data" / "loaders.py"

loader_code = r'''
import numpy as np
import torch

from torch_geometric.data import Data, Batch


FEATURE_COLS = [
    "sbytes",
    "dbytes",
    "spkts",
    "dpkts",
    "sloss",
    "dloss",
    "sload",
    "dload",
    "sttl",
    "dttl",
    "dur",
]


def snapshot_to_data(
    snap,
    scaler
):

    if len(snap.node_ids) == 0:
        raise ValueError(
            "Cannot convert empty graph snapshot."
        )

    x = np.stack(
        [
            snap.node_features[node]
            for node in snap.node_ids
        ],
        axis=0
    ).astype(np.float32)

    x = scaler.transform(
        x
    ).astype(np.float32)

    if len(snap.edges) > 0:

        edge_index = torch.tensor(
            np.asarray(
                snap.edges,
                dtype=np.int64
            ).T,
            dtype=torch.long
        )

    else:

        # Self-loops for isolated nodes.
        edge_index = torch.arange(
            len(snap.node_ids),
            dtype=torch.long
        ).repeat(2, 1)

    return Data(
        x=torch.tensor(
            x,
            dtype=torch.float32
        ),
        edge_index=edge_index
    )


def batch_sequences(
    sequences,
    scaler
):

    if not sequences:
        return []

    seq_len = len(
        sequences[0]
    )

    output = []

    for t in range(seq_len):

        graphs_t = [
            snapshot_to_data(
                seq[t],
                scaler
            )
            for seq in sequences
        ]

        output.append(
            Batch.from_data_list(
                graphs_t
            )
        )

    return output


def to_batches(
    sequences,
    scaler,
    batch_size=16
):

    batches = []

    for i in range(
        0,
        len(sequences),
        batch_size
    ):

        batches.append(
            batch_sequences(
                sequences[
                    i:i + batch_size
                ],
                scaler
            )
        )

    return batches
'''

LOADER_PATH.write_text(loader_code)

print("Patched:", LOADER_PATH)

Patched: /content/fedctgad_x_v2/project/fedctgad/data/loaders.py


In [ ]:
# ============================================================
# CELL 16 — MODEL CONFIGURATION
# ============================================================

import yaml

CONFIG_PATH = PROJECT_ROOT / "configs" / "default.yaml"

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

cfg["data"]["delta_t"] = 30.0
cfg["data"]["seq_len"] = 5

cfg["model"]["in_dim"] = 11
cfg["model"]["hidden_dim"] = 60
cfg["model"]["struct_heads"] = 4
cfg["model"]["temporal_heads"] = 4
cfg["model"]["proj_dim"] = 80

cfg["model"]["temperature"] = 0.1
cfg["model"]["k_neighbors"] = 15

cfg["partition"]["held_out_family"] = "Exploits"

print("Model configuration:")
print(yaml.dump(cfg["model"], sort_keys=False))

assert cfg["model"]["in_dim"] == 11
assert cfg["model"]["hidden_dim"] == 60
assert cfg["model"]["proj_dim"] == 80

Model configuration:
in_dim: 11
hidden_dim: 60
struct_heads: 4
temporal_heads: 4
proj_dim: 80
lr: 0.001
temperature: 0.1
k_neighbors: 15



In [ ]:
# ============================================================
# CELL 17 — RELOAD PATCHED MODULES
# ============================================================

import sys
import importlib

for p in [
    str(PROJECT_ROOT)
]:
    if p not in sys.path:
        sys.path.insert(0, p)

import data.slicing
import data.loaders
import model.encoder

importlib.reload(
    data.slicing
)

importlib.reload(
    data.loaders
)

importlib.reload(
    model.encoder
)

from data.slicing import (
    slice_flows_to_windows,
    snapshots_to_sequences
)

from data.loaders import (
    snapshot_to_data,
    batch_sequences,
    to_batches
)

from model.encoder import (
    CTGAD_Encoder,
    ProjectionHead
)

print("Patched modules reloaded successfully.")

Patched modules reloaded successfully.


In [ ]:
# ============================================================
# CELL 18 — GLOBAL CAUSAL TEMPORAL GRAPH CONSTRUCTION
# ============================================================

DT = 30.0
SEQ_LEN = 5

# IMPORTANT:
# Slice ALL traffic together.
snaps = slice_flows_to_windows(
    df,
    delta_t=DT
)

print("Graph snapshots:", len(snaps))

print(
    "First window:",
    snaps[0].window_idx,
    snaps[0].t_start,
    snaps[0].t_end
)

print(
    "Last window:",
    snaps[-1].window_idx,
    snaps[-1].t_start,
    snaps[-1].t_end
)

# Check feature dimensionality
feature_dims = sorted(
    set(
        len(v)
        for s in snaps
        for v in s.node_features.values()
    )
)

print(
    "Node feature dimensions:",
    feature_dims
)

assert feature_dims == [11], (
    f"Expected 11-dimensional node features, "
    f"got {feature_dims}"
)

# Build only temporally contiguous sequences.
seqs = snapshots_to_sequences(
    snaps,
    seq_len=SEQ_LEN,
    stride=1
)

print(
    "Contiguous temporal sequences:",
    len(seqs)
)

# Verify contiguity
bad_sequences = 0

for q in seqs:

    idx = [
        s.window_idx
        for s in q
    ]

    if any(
        idx[i + 1] != idx[i] + 1
        for i in range(len(idx) - 1)
    ):
        bad_sequences += 1

print(
    "Non-contiguous sequences:",
    bad_sequences
)

assert bad_sequences == 0

Graph snapshots: 255
First window: 0 1421927377.0 1421927407.0
Last window: 254 1421934997.0 1421935027.0
Node feature dimensions: [11]
Contiguous temporal sequences: 251
Non-contiguous sequences: 0


In [ ]:
# ============================================================
# CELL 19 — GRAPH INTEGRITY AUDIT
# ============================================================

node_counts = np.array([
    len(s.node_ids)
    for s in snaps
])

edge_counts = np.array([
    len(s.edges)
    for s in snaps
])

print("NODE COUNT")
print(
    "min / median / max:",
    node_counts.min(),
    np.median(node_counts),
    node_counts.max()
)

print("\nEDGE COUNT")
print(
    "min / median / max:",
    edge_counts.min(),
    np.median(edge_counts),
    edge_counts.max()
)

print("\nFEATURE CHECK")

sample_snap = snaps[
    min(10, len(snaps)-1)
]

sample_node = sample_snap.node_ids[0]

print(
    "Sample node:",
    sample_node
)

print(
    "Feature vector:",
    sample_snap.node_features[
        sample_node
    ]
)

print(
    "Feature shape:",
    sample_snap.node_features[
        sample_node
    ].shape
)

assert (
    sample_snap.node_features[
        sample_node
    ].shape == (11,)
)

NODE COUNT
min / median / max: 4 39.0 42

EDGE COUNT
min / median / max: 3 128.0 147

FEATURE CHECK
Sample node: 10.40.85.30
Feature vector: [213.7471   154.83516    4.980841   4.980841   0.         0.
  56.99923   41.289375   0.         0.         4.      ]
Feature shape: (11,)


In [ ]:
# ============================================================
# CELL 20 — TEMPORAL SEQUENCE LABELING
# ============================================================

from collections import defaultdict

HELD_OUT = "Exploits"

window_total = defaultdict(float)
window_attack = defaultdict(float)
window_heldout = defaultdict(float)

t0 = float(
    df["ts_start"].min()
)

for row in df.itertuples(
    index=False
):

    start = float(
        row.ts_start - t0
    )

    dur = max(
        float(row.duration),
        1e-9
    )

    end = start + dur

    first_w = int(
        np.floor(start / DT)
    )

    last_w = int(
        np.floor(end / DT)
    )

    for w in range(
        first_w,
        last_w + 1
    ):

        w_start = w * DT
        w_end = w_start + DT

        overlap = max(
            0.0,
            min(end, w_end)
            - max(start, w_start)
        )

        if overlap <= 0:
            continue

        weight = overlap / dur

        window_total[w] += weight

        if row.attack_cat != "Normal":
            window_attack[w] += weight

        if row.attack_cat == HELD_OUT:
            window_heldout[w] += weight


def get_fraction(mapping, w):

    return mapping[w] / max(
        window_total[w],
        1e-12
    )


sequence_records = []

for q in seqs:

    ws = [
        s.window_idx
        for s in q
    ]

    attack_fracs = np.array([
        get_fraction(
            window_attack,
            w
        )
        for w in ws
    ])

    heldout_fracs = np.array([
        get_fraction(
            window_heldout,
            w
        )
        for w in ws
    ])

    sequence_records.append({
        "sequence": q,
        "window_start": ws[0],
        "window_end": ws[-1],
        "attack_mean": float(
            attack_fracs.mean()
        ),
        "attack_max": float(
            attack_fracs.max()
        ),
        "heldout_mean": float(
            heldout_fracs.mean()
        ),
        "heldout_max": float(
            heldout_fracs.max()
        )
    })

print(
    "Total sequences:",
    len(sequence_records)
)

print("\nExample:")
display(
    pd.DataFrame(
        [
            {
                k: v
                for k, v in r.items()
                if k != "sequence"
            }
            for r in sequence_records[:10]
        ]
    )
)

Total sequences: 251

Example:


,window_start,window_end,attack_mean,attack_max,heldout_mean,heldout_max
0,0,4,0.037615,0.082207,0.023137,0.053751
1,1,5,0.047982,0.082207,0.028765,0.053751
2,2,6,0.040389,0.051833,0.022859,0.028138
3,3,7,0.048675,0.060087,0.028700,0.039700
4,4,8,0.055701,0.076520,0.032769,0.045176
5,5,9,0.073272,0.133680,0.033130,0.045176
6,6,10,0.085230,0.133680,0.033642,0.045176
7,7,11,0.088570,0.133680,0.033672,0.045176
8,8,12,0.095465,0.133680,0.030800,0.045176
9,9,13,0.101686,0.133680,0.025699,0.030696


In [ ]:
# ============================================================
# CELL 21 — ZERO-DAY PROTOCOL
# ============================================================

NORMAL_MAX = 0.05
ATTACK_MIN = 0.10
ZERO_DAY_MIN = 0.05

normal_records = []
seen_records = []
zero_day_records = []
ambiguous_records = []

for r in sequence_records:

    attack_mean = r["attack_mean"]
    heldout_mean = r["heldout_mean"]

    if attack_mean <= NORMAL_MAX:

        normal_records.append(r)

    elif (
        attack_mean >= ATTACK_MIN
        and heldout_mean >= ZERO_DAY_MIN
    ):

        zero_day_records.append(r)

    elif (
        attack_mean >= ATTACK_MIN
        and heldout_mean <= 0.01
    ):

        seen_records.append(r)

    else:

        ambiguous_records.append(r)


print(
    "NORMAL:",
    len(normal_records)
)

print(
    "SEEN ATTACK:",
    len(seen_records)
)

print(
    "ZERO-DAY:",
    len(zero_day_records)
)

print(
    "AMBIGUOUS / DROPPED:",
    len(ambiguous_records)
)

NORMAL: 31
SEEN ATTACK: 0
ZERO-DAY: 7
AMBIGUOUS / DROPPED: 213


In [ ]:
# ============================================================
# CELL 22 — ZERO-DAY AVAILABILITY AUDIT
# ============================================================

if zero_day_records:

    zd_mean = np.array([
        r["heldout_mean"]
        for r in zero_day_records
    ])

    zd_max = np.array([
        r["heldout_max"]
        for r in zero_day_records
    ])

    print(
        "Zero-day sequence count:",
        len(zero_day_records)
    )

    print(
        "Held-out mean fraction:",
        np.percentile(
            zd_mean,
            [0, 25, 50, 75, 100]
        ).round(4)
    )

    print(
        "Held-out max fraction:",
        np.percentile(
            zd_max,
            [0, 25, 50, 75, 100]
        ).round(4)
    )

else:

    print(
        "WARNING: No clean zero-day sequences "
        "in the current 200k-row subset."
    )

    print(
        "We will need the full dataset for the final experiment."
    )

Zero-day sequence count: 7
Held-out mean fraction: [0.0673 0.0828 0.1009 0.1166 0.125 ]
Held-out max fraction: [0.1308 0.1472 0.1516 0.1516 0.1516]


In [ ]:
# ============================================================
# CELL 23 — TRUE NORMAL-ONLY TRAINING STREAM
# ============================================================

# IMPORTANT:
# We do NOT redefine timestamps.
# The same absolute timestamps/global temporal origin are retained.

normal_df = df[
    df["attack_cat"].eq("Normal")
].copy()

normal_df = normal_df.sort_values(
    "ts_start"
).reset_index(drop=True)

print("All-flow rows:", len(df))
print("Normal-flow rows:", len(normal_df))

print(
    "Normal percentage:",
    round(
        100 * len(normal_df) / len(df),
        2
    ),
    "%"
)

print(
    "\nNormal time range:"
)

print(
    normal_df["ts_start"].min(),
    "→",
    normal_df["ts_start"].max()
)

All-flow rows: 200000
Normal-flow rows: 177785
Normal percentage: 88.89 %

Normal time range:
1421927377 → 1421935026


In [ ]:
# ============================================================
# CELL 24 — NORMAL-ONLY TEMPORAL GRAPH STREAM
# ============================================================

normal_snaps = slice_flows_to_windows(
    normal_df,
    delta_t=DT
)

print(
    "Normal graph snapshots:",
    len(normal_snaps)
)

normal_seqs = snapshots_to_sequences(
    normal_snaps,
    seq_len=SEQ_LEN,
    stride=1
)

print(
    "Normal contiguous sequences:",
    len(normal_seqs)
)

# Integrity check
bad_normal = 0

for q in normal_seqs:

    idx = [
        s.window_idx
        for s in q
    ]

    if any(
        idx[i + 1] != idx[i] + 1
        for i in range(len(idx) - 1)
    ):
        bad_normal += 1

print(
    "Non-contiguous normal sequences:",
    bad_normal
)

assert bad_normal == 0

Normal graph snapshots: 255
Normal contiguous sequences: 251
Non-contiguous normal sequences: 0


In [ ]:
# ============================================================
# CELL 25 — EXPLICIT GLOBAL TEMPORAL ORIGIN
# ============================================================

slicer_path = PROJECT_ROOT / "data" / "slicing.py"

src = slicer_path.read_text()

src = src.replace(
    "    dst_col: str = \"dst_ip\",\n) -> List[GraphSnapshot]:",
    """    dst_col: str = "dst_ip",
    global_t0: float | None = None,
) -> List[GraphSnapshot]:"""
)

src = src.replace(
    """    # IMPORTANT:
    # The temporal origin is computed ONCE from the complete
    # dataset passed to this function.
    t_min = float(flows[ts_col].min())
""",
    """    # Use an externally supplied global temporal origin whenever
    # the caller needs multiple streams to share exactly the same
    # temporal coordinate system.
    if global_t0 is None:
        t_min = float(flows[ts_col].min())
    else:
        t_min = float(global_t0)
"""
)

slicer_path.write_text(src)

print("Updated temporal slicer.")

Updated temporal slicer.


In [ ]:
# ============================================================
# CELL 26 — REBUILD WITH SHARED GLOBAL TIME ORIGIN
# ============================================================

import importlib
import data.slicing

importlib.reload(data.slicing)

from data.slicing import (
    slice_flows_to_windows,
    snapshots_to_sequences
)

# ------------------------------------------------------------
# ALL traffic
# ------------------------------------------------------------

snaps = slice_flows_to_windows(
    df,
    delta_t=DT,
    global_t0=GLOBAL_T0
)

seqs = snapshots_to_sequences(
    snaps,
    seq_len=SEQ_LEN,
    stride=1
)

# ------------------------------------------------------------
# NORMAL traffic
# ------------------------------------------------------------

normal_snaps = slice_flows_to_windows(
    normal_df,
    delta_t=DT,
    global_t0=GLOBAL_T0
)

normal_seqs = snapshots_to_sequences(
    normal_snaps,
    seq_len=SEQ_LEN,
    stride=1
)

print("ALL")
print("snapshots :", len(snaps))
print("sequences :", len(seqs))

print("\nNORMAL")
print("snapshots :", len(normal_snaps))
print("sequences :", len(normal_seqs))

print("\nFirst global window:")
print(
    snaps[0].window_idx,
    snaps[0].t_start,
    snaps[0].t_end
)

print("\nFirst normal window:")
print(
    normal_snaps[0].window_idx,
    normal_snaps[0].t_start,
    normal_snaps[0].t_end
)

ALL
snapshots : 255
sequences : 251

NORMAL
snapshots : 255
sequences : 251

First global window:
0 1421927377.0 1421927407.0

First normal window:
0 1421927377.0 1421927407.0


In [ ]:
# ============================================================
# CELL 27 — TEMPORAL ALIGNMENT VALIDATION
# ============================================================

all_indices = {
    s.window_idx
    for s in snaps
}

normal_indices = {
    s.window_idx
    for s in normal_snaps
}

common = sorted(
    all_indices.intersection(
        normal_indices
    )
)

print(
    "Common temporal windows:",
    len(common)
)

print(
    "First 10 common windows:",
    common[:10]
)

# Verify timestamps are identical
alignment_errors = []

all_lookup = {
    s.window_idx: s
    for s in snaps
}

normal_lookup = {
    s.window_idx: s
    for s in normal_snaps
}

for w in common:

    a = all_lookup[w]
    n = normal_lookup[w]

    if (
        a.t_start != n.t_start
        or
        a.t_end != n.t_end
    ):
        alignment_errors.append(w)

print(
    "Timestamp alignment errors:",
    len(alignment_errors)
)

assert len(alignment_errors) == 0

Common temporal windows: 255
First 10 common windows: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Timestamp alignment errors: 0


In [ ]:
# ============================================================
# CELL 28 — NORMAL TRAINING DATA AUDIT
# ============================================================

print(
    "Normal sequences available:",
    len(normal_seqs)
)

normal_seq_starts = [
    q[0].window_idx
    for q in normal_seqs
]

print(
    "Normal sequence start range:",
    min(normal_seq_starts),
    "→",
    max(normal_seq_starts)
)

# Number of unique temporal windows represented
normal_windows_used = set()

for q in normal_seqs:

    for s in q:
        normal_windows_used.add(
            s.window_idx
        )

print(
    "Unique normal windows represented:",
    len(normal_windows_used)
)

Normal sequences available: 251
Normal sequence start range: 0 → 250
Unique normal windows represented: 255


In [ ]:
# ============================================================
# CELL 29 — CHRONOLOGICAL NORMAL SPLIT
# ============================================================

normal_seqs = sorted(
    normal_seqs,
    key=lambda q: q[0].window_idx
)

n = len(normal_seqs)

n_train = int(
    0.70 * n
)

n_val = int(
    0.10 * n
)

normal_train_seqs = normal_seqs[
    :n_train
]

normal_val_seqs = normal_seqs[
    n_train:n_train + n_val
]

normal_test_seqs = normal_seqs[
    n_train + n_val:
]

print(
    "Total normal sequences:",
    n
)

print(
    "Train:",
    len(normal_train_seqs)
)

print(
    "Validation:",
    len(normal_val_seqs)
)

print(
    "Normal test:",
    len(normal_test_seqs)
)

print(
    "\nTemporal ranges:"
)

for name, data in [
    ("TRAIN", normal_train_seqs),
    ("VAL", normal_val_seqs),
    ("TEST", normal_test_seqs),
]:

    if data:

        print(
            name,
            ":",
            data[0][0].window_idx,
            "→",
            data[-1][-1].window_idx
        )

Total normal sequences: 251
Train: 175
Validation: 25
Normal test: 51

Temporal ranges:
TRAIN : 0 → 178
VAL : 175 → 203
TEST : 200 → 254


In [ ]:
# ============================================================
# CELL 30 — TEMPORAL SPLIT LEAKAGE CHECK
# ============================================================

def sequence_window_set(sequences):

    s = set()

    for q in sequences:

        for snap in q:

            s.add(
                snap.window_idx
            )

    return s


train_windows = sequence_window_set(
    normal_train_seqs
)

val_windows = sequence_window_set(
    normal_val_seqs
)

test_windows = sequence_window_set(
    normal_test_seqs
)

print(
    "Train ∩ Val:",
    len(train_windows & val_windows)
)

print(
    "Train ∩ Test:",
    len(train_windows & test_windows)
)

print(
    "Val ∩ Test:",
    len(val_windows & test_windows)
)

Train ∩ Val: 4
Train ∩ Test: 0
Val ∩ Test: 4


In [ ]:
# ============================================================
# CELL 31 — LEAKAGE-SAFE CHRONOLOGICAL SPLIT
# ============================================================

# We split the underlying temporal windows FIRST,
# then construct sequences entirely inside each partition.

normal_window_ids = sorted(
    {
        s.window_idx
        for s in normal_snaps
    }
)

n_windows = len(
    normal_window_ids
)

cut_train = int(
    0.70 * n_windows
)

cut_val = int(
    0.80 * n_windows
)

train_window_ids = set(
    normal_window_ids[:cut_train]
)

val_window_ids = set(
    normal_window_ids[cut_train:cut_val]
)

test_window_ids = set(
    normal_window_ids[cut_val:]
)

normal_lookup = {
    s.window_idx: s
    for s in normal_snaps
}


def make_sequences_from_window_ids(
    window_ids,
    seq_len
):

    ids = sorted(window_ids)

    result = []

    for i in range(
        len(ids) - seq_len + 1
    ):

        candidate = ids[
            i:i + seq_len
        ]

        # Must be truly consecutive
        if any(
            candidate[j + 1]
            != candidate[j] + 1
            for j in range(
                seq_len - 1
            )
        ):
            continue

        result.append(
            [
                normal_lookup[w]
                for w in candidate
            ]
        )

    return result


normal_train_seqs = make_sequences_from_window_ids(
    train_window_ids,
    SEQ_LEN
)

normal_val_seqs = make_sequences_from_window_ids(
    val_window_ids,
    SEQ_LEN
)

normal_test_seqs = make_sequences_from_window_ids(
    test_window_ids,
    SEQ_LEN
)

print(
    "Leakage-safe normal split:"
)

print(
    "Train:",
    len(normal_train_seqs)
)

print(
    "Validation:",
    len(normal_val_seqs)
)

print(
    "Test:",
    len(normal_test_seqs)
)

Leakage-safe normal split:
Train: 174
Validation: 22
Test: 47


In [ ]:
# ============================================================
# CELL 32 — FINAL LEAKAGE CHECK
# ============================================================

train_windows = sequence_window_set(
    normal_train_seqs
)

val_windows = sequence_window_set(
    normal_val_seqs
)

test_windows = sequence_window_set(
    normal_test_seqs
)

print(
    "Train ∩ Val:",
    len(train_windows & val_windows)
)

print(
    "Train ∩ Test:",
    len(train_windows & test_windows)
)

print(
    "Val ∩ Test:",
    len(val_windows & test_windows)
)

assert len(train_windows & val_windows) == 0
assert len(train_windows & test_windows) == 0
assert len(val_windows & test_windows) == 0

print(
    "\n✓ NO TEMPORAL WINDOW LEAKAGE"
)

Train ∩ Val: 0
Train ∩ Test: 0
Val ∩ Test: 0

✓ NO TEMPORAL WINDOW LEAKAGE


In [ ]:
# ============================================================
# CELL 33 — FIX TEMPORAL DURATION FEATURE
# ============================================================

slicer_path = PROJECT_ROOT / "data" / "slicing.py"

src = slicer_path.read_text()

# Replace state accumulator
src = src.replace(
'''                    accum[key] = {
                        c: 0.0
                        for c in ADDITIVE_COLS
                    }

                    accum[key].update({
                        "sttl_sum": 0.0,
                        "dttl_sum": 0.0,
                        "state_count": 0,
                    })
''',
'''                    accum[key] = {
                        c: 0.0
                        for c in ADDITIVE_COLS
                    }

                    accum[key].update({
                        "sttl_sum": 0.0,
                        "dttl_sum": 0.0,
                        "state_count": 0,
                        "duration": 0.0,
                    })
'''
)

# Add overlap duration
src = src.replace(
'''            a["sttl_sum"] += float(sttl)
            a["dttl_sum"] += float(dttl)
            a["state_count"] += 1
''',
'''            a["sttl_sum"] += float(sttl)
            a["dttl_sum"] += float(dttl)
            a["state_count"] += 1

            # Actual temporal overlap represented by this
            # flow-node contribution in this window.
            a["duration"] += overlap
'''
)

# Replace the old node_duration calculation
src = src.replace(
'''        # Total temporal contribution represented
        # by this node in this window.
        node_duration = min(
            delta_t,
            max(
                0.0,
                a["state_count"]
            )
        )
''',
'''        # Total duration represented by this node
        # in this temporal window.
        node_duration = a["duration"]
'''
)

slicer_path.write_text(src)

print("Duration accounting patched.")

Duration accounting patched.


In [ ]:
# ============================================================
# CELL 34 — REBUILD GRAPH STREAMS
# ============================================================

import importlib
import data.slicing

importlib.reload(data.slicing)

from data.slicing import (
    slice_flows_to_windows,
    snapshots_to_sequences
)

# Global stream
snaps = slice_flows_to_windows(
    df,
    delta_t=DT,
    global_t0=GLOBAL_T0
)

seqs = snapshots_to_sequences(
    snaps,
    seq_len=SEQ_LEN,
    stride=1
)

# Normal stream
normal_snaps = slice_flows_to_windows(
    normal_df,
    delta_t=DT,
    global_t0=GLOBAL_T0
)

normal_seqs = snapshots_to_sequences(
    normal_snaps,
    seq_len=SEQ_LEN,
    stride=1
)

print("All snapshots:", len(snaps))
print("All sequences:", len(seqs))

print("Normal snapshots:", len(normal_snaps))
print("Normal sequences:", len(normal_seqs))

All snapshots: 255
All sequences: 251
Normal snapshots: 255
Normal sequences: 251


In [ ]:
# ============================================================
# CELL 35 — FEATURE SANITY CHECK
# ============================================================

sample_snap = normal_snaps[10]

sample_node = sample_snap.node_ids[0]

x = sample_snap.node_features[sample_node]

print("Sample node:", sample_node)

print("\n11-dimensional feature vector:")
for i, value in enumerate(x):
    print(f"{i+1:02d}: {value:.6f}")

print("\nFeature shape:", x.shape)

assert x.shape == (11,)

# Duration must be non-negative
all_durations = np.array([
    s.node_features[n][10]
    for s in normal_snaps
    for n in s.node_ids
])

print(
    "\nDuration min / median / max:",
    all_durations.min(),
    np.median(all_durations),
    all_durations.max()
)

assert np.all(all_durations >= 0)

Sample node: 10.40.85.30

11-dimensional feature vector:
01: 213.747101
02: 154.835159
03: 4.980841
04: 4.980841
05: 0.000000
06: 0.000000
07: 56.999229
08: 41.289375
09: 0.000000
10: 0.000000
11: 0.000000

Feature shape: (11,)

Duration min / median / max: 0.0 0.0 0.0


In [ ]:
# ============================================================
# CELL 36 — FINAL NORMAL TRAIN / VAL / TEST SPLIT
# ============================================================

normal_window_ids = sorted({
    s.window_idx
    for s in normal_snaps
})

n_windows = len(normal_window_ids)

cut_train = int(0.70 * n_windows)
cut_val = int(0.80 * n_windows)

train_window_ids = set(
    normal_window_ids[:cut_train]
)

val_window_ids = set(
    normal_window_ids[cut_train:cut_val]
)

test_window_ids = set(
    normal_window_ids[cut_val:]
)

normal_lookup = {
    s.window_idx: s
    for s in normal_snaps
}


def make_sequences_from_window_ids(
    window_ids,
    seq_len
):

    ids = sorted(window_ids)

    result = []

    for i in range(
        len(ids) - seq_len + 1
    ):

        candidate = ids[
            i:i + seq_len
        ]

        if any(
            candidate[j + 1]
            != candidate[j] + 1
            for j in range(seq_len - 1)
        ):
            continue

        result.append([
            normal_lookup[w]
            for w in candidate
        ])

    return result


normal_train_seqs = make_sequences_from_window_ids(
    train_window_ids,
    SEQ_LEN
)

normal_val_seqs = make_sequences_from_window_ids(
    val_window_ids,
    SEQ_LEN
)

normal_test_seqs = make_sequences_from_window_ids(
    test_window_ids,
    SEQ_LEN
)

print("TRAIN:", len(normal_train_seqs))
print("VAL  :", len(normal_val_seqs))
print("TEST :", len(normal_test_seqs))

TRAIN: 174
VAL  : 22
TEST : 47


In [ ]:
# ============================================================
# CELL 37 — FINAL DATA LEAKAGE TEST
# ============================================================

def sequence_window_set(sequences):

    return {
        snap.window_idx
        for seq in sequences
        for snap in seq
    }


train_w = sequence_window_set(
    normal_train_seqs
)

val_w = sequence_window_set(
    normal_val_seqs
)

test_w = sequence_window_set(
    normal_test_seqs
)

print("Train ∩ Val :", len(train_w & val_w))
print("Train ∩ Test:", len(train_w & test_w))
print("Val ∩ Test  :", len(val_w & test_w))

assert not (train_w & val_w)
assert not (train_w & test_w)
assert not (val_w & test_w)

print("\n✓ CLEAN CHRONOLOGICAL SPLIT")

Train ∩ Val : 0
Train ∩ Test: 0
Val ∩ Test  : 0

✓ CLEAN CHRONOLOGICAL SPLIT


In [ ]:
# ============================================================
# CELL 38 — TRAIN-ONLY FEATURE SCALER
# ============================================================

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

train_vectors = []

for seq in normal_train_seqs:

    for snap in seq:

        for node in snap.node_ids:

            train_vectors.append(
                snap.node_features[node]
            )

train_vectors = np.asarray(
    train_vectors,
    dtype=np.float32
)

print(
    "Training node vectors:",
    train_vectors.shape
)

assert train_vectors.shape[1] == 11

scaler.fit(
    train_vectors
)

print("\nScaler means:")
print(
    np.round(
        scaler.mean_,
        4
    )
)

print("\nScaler standard deviations:")
print(
    np.round(
        scaler.scale_,
        4
    )
)

Training node vectors: (24845, 11)

Scaler means:
[2.17571196e+05 2.12868797e+06 1.89453870e+03 2.46751570e+03
 2.82099900e+02 9.45996000e+02 5.80189856e+04 5.67650125e+05
 4.24213000e+01 3.31708000e+01 0.00000000e+00]

Scaler standard deviations:
[1.77863868e+05 2.57362610e+06 1.52363240e+03 2.34662780e+03
 2.07201700e+02 9.87928200e+02 4.74303647e+04 6.86300295e+05
 5.90483000e+01 5.06050000e+01 1.00000000e+00]


In [ ]:
# ============================================================
# CELL 39 — SCALER SANITY CHECK
# ============================================================

val_vectors = []

for seq in normal_val_seqs:

    for snap in seq:

        for node in snap.node_ids:

            val_vectors.append(
                snap.node_features[node]
            )

val_vectors = np.asarray(
    val_vectors,
    dtype=np.float32
)

test_vectors = []

for seq in normal_test_seqs:

    for snap in seq:

        for node in snap.node_ids:

            test_vectors.append(
                snap.node_features[node]
            )

test_vectors = np.asarray(
    test_vectors,
    dtype=np.float32
)

print(
    "Train vectors:",
    train_vectors.shape
)

print(
    "Val vectors:",
    val_vectors.shape
)

print(
    "Test vectors:",
    test_vectors.shape
)

# IMPORTANT:
# We only call transform on val/test.
# scaler.fit() was called ONLY on train.

train_scaled = scaler.transform(
    train_vectors
)

print(
    "\nScaled training mean:",
    np.round(
        train_scaled.mean(axis=0),
        4
    )
)

print(
    "Scaled training std:",
    np.round(
        train_scaled.std(axis=0),
        4
    )
)

Train vectors: (24845, 11)
Val vectors: (3405, 11)
Test vectors: (6265, 11)

Scaled training mean: [ 0.  0.  0.  0. -0. -0.  0.  0. -0.  0.  0.]
Scaled training std: [1.     1.     1.     1.     1.     1.     1.     1.     0.9999 0.9999
 0.    ]


In [ ]:
# ============================================================
# CELL 40 — BUILD PYTORCH GEOMETRIC BATCHES
# ============================================================

import importlib
import data.loaders

importlib.reload(
    data.loaders
)

from data.loaders import (
    batch_sequences,
    to_batches,
    snapshot_to_data
)

BATCH_SIZE = 16

train_batches = to_batches(
    normal_train_seqs,
    scaler,
    batch_size=BATCH_SIZE
)

val_batches = to_batches(
    normal_val_seqs,
    scaler,
    batch_size=BATCH_SIZE
)

test_batches = to_batches(
    normal_test_seqs,
    scaler,
    batch_size=BATCH_SIZE
)

print(
    "Train batches:",
    len(train_batches)
)

print(
    "Validation batches:",
    len(val_batches)
)

print(
    "Test batches:",
    len(test_batches)
)

Train batches: 11
Validation batches: 2
Test batches: 3


In [ ]:
# ============================================================
# CELL 41 — PYTORCH GEOMETRIC BATCH SANITY CHECK
# ============================================================

sample_batch = train_batches[0]

print(
    "Sequence length:",
    len(sample_batch)
)

for t, data_t in enumerate(
    sample_batch
):

    print(
        f"\nTime step {t}:"
    )

    print(
        " nodes:",
        data_t.num_nodes
    )

    print(
        " edges:",
        data_t.num_edges
    )

    print(
        " features:",
        tuple(data_t.x.shape)
    )

    print(
        " graphs:",
        data_t.num_graphs
    )

    print(
        " feature mean:",
        data_t.x.mean().item()
    )

    print(
        " feature std:",
        data_t.x.std().item()
    )

assert len(sample_batch) == SEQ_LEN
assert sample_batch[0].x.shape[1] == 11
assert sample_batch[0].num_graphs <= BATCH_SIZE

Sequence length: 5

Time step 0:
 nodes: 413
 edges: 1597
 features: (413, 11)
 graphs: 16
 feature mean: -0.07837784290313721
 feature std: 0.8583112955093384

Time step 1:
 nodes: 438
 edges: 1698
 features: (438, 11)
 graphs: 16
 feature mean: -0.06672510504722595
 feature std: 0.8641149997711182

Time step 2:
 nodes: 441
 edges: 1697
 features: (441, 11)
 graphs: 16
 feature mean: -0.03653289005160332
 feature std: 0.8619979619979858

Time step 3:
 nodes: 444
 edges: 1695
 features: (444, 11)
 graphs: 16
 feature mean: -0.07153217494487762
 feature std: 0.8201724886894226

Time step 4:
 nodes: 449
 edges: 1698
 features: (449, 11)
 graphs: 16
 feature mean: -0.07977201044559479
 feature std: 0.8285731077194214


In [ ]:
# ============================================================
# CELL 42 — CTGAD-X MODEL
# ============================================================

import torch

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

encoder = CTGAD_Encoder(
    in_dim=11,
    hidden_dim=60,
    seq_len=5,
    struct_heads=4,
    temporal_heads=4
).to(DEVICE)

projection_head = ProjectionHead(
    in_dim=60,
    hidden_dim=60,
    out_dim=80
).to(DEVICE)

print(
    "Device:",
    DEVICE
)

print(
    "Encoder:",
    encoder
)

print(
    "\nProjection head:",
    projection_head
)

Device: cuda
Encoder: CTGAD_Encoder(
  (structural): StructuralEncoder(
    (gat): GATv2Conv(11, 60, heads=4)
  )
  (temporal): TemporalEncoder(
    (attn): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=60, out_features=60, bias=True)
    )
    (norm): LayerNorm((60,), eps=1e-05, elementwise_affine=True)
  )
)

Projection head: ProjectionHead(
  (net): Sequential(
    (0): Linear(in_features=60, out_features=60, bias=True)
    (1): ReLU()
    (2): Linear(in_features=60, out_features=80, bias=True)
  )
)


In [ ]:
# ============================================================
# CELL 43 — PARAMETER COUNT
# ============================================================

def count_parameters(model):

    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


encoder_params = count_parameters(
    encoder
)

projection_params = count_parameters(
    projection_head
)

print(
    "Encoder parameters:",
    f"{encoder_params:,}"
)

print(
    "Projection parameters:",
    f"{projection_params:,}"
)

print(
    "Total training parameters:",
    f"{encoder_params + projection_params:,}"
)

Encoder parameters: 20,820
Projection parameters: 8,540
Total training parameters: 29,360


In [ ]:
# ============================================================
# CELL 44 — FORWARD PASS TEST
# ============================================================

encoder.eval()
projection_head.eval()

with torch.no_grad():

    z = encoder(
        [
            batch.to(DEVICE)
            for batch in sample_batch
        ]
    )

    p = projection_head(z)

print(
    "Encoder output:",
    tuple(z.shape)
)

print(
    "Projection output:",
    tuple(p.shape)
)

assert z.ndim == 2
assert z.shape[1] == 60

assert p.ndim == 2
assert p.shape[1] == 80

print(
    "\n✓ CTGAD-X FORWARD PASS VALID"
)

Encoder output: (16, 60)
Projection output: (16, 80)

✓ CTGAD-X FORWARD PASS VALID


In [ ]:
# ============================================================
# CELL 45 — INSPECT ACTUAL SLICER IMPLEMENTATION
# ============================================================

print(slicer_path.read_text())


"""
Causality-aware temporal graph construction.

CTGAD-X-compatible node features:
    sbytes, dbytes,
    spkts, dpkts,
    sloss, dloss,
    sload, dload,
    sttl, dttl,
    dur

Each flow [ts, ts+dur) contributes only to the temporal
windows it actually overlaps.
"""

from __future__ import annotations

import numpy as np
import pandas as pd

from dataclasses import dataclass, field
from typing import Dict, List, Tuple


ADDITIVE_COLS = [
    "sbytes",
    "dbytes",
    "spkts",
    "dpkts",
    "sloss",
    "dloss",
]

STATE_COLS = [
    "sttl",
    "dttl",
]

FEATURE_COLS = [
    "sbytes",
    "dbytes",
    "spkts",
    "dpkts",
    "sloss",
    "dloss",
    "sload",
    "dload",
    "sttl",
    "dttl",
    "dur",
]


@dataclass
class GraphSnapshot:

    window_idx: int
    t_start: float
    t_end: float

    node_ids: List[str] = field(default_factory=list)

    # edges represented using integer node indices
    edges: List[Tuple[int, int]] = field(default_factory=list)

    

In [ ]:
# ============================================================
# CELL 46 — RAW FEATURE SANITY CHECK
# ============================================================

raw_feature_cols = [
    "sbytes",
    "dbytes",
    "spkts",
    "dpkts",
    "sloss",
    "dloss",
    "sttl",
    "dttl",
    "sload",
    "dload",
    "duration"
]

print("RAW DATASET FEATURE STATISTICS\n")

display(
    df[raw_feature_cols].describe().T
)

RAW DATASET FEATURE STATISTICS



,count,mean,std,min,25%,50%,75%,max
sbytes,200000.0,5.363003e+03,8.118579e+04,0.0,264.000000,1580.000000,3.614000e+03,1.367739e+07
dbytes,200000.0,3.941573e+04,1.705080e+05,0.0,178.000000,2456.000000,1.495200e+04,1.465542e+07
spkts,200000.0,3.621540e+01,8.926358e+01,0.0,2.000000,14.000000,4.800000e+01,1.020000e+04
dpkts,200000.0,4.616381e+01,1.284283e+02,0.0,2.000000,18.000000,4.400000e+01,1.097000e+04
sloss,200000.0,5.837835e+00,3.130294e+01,0.0,0.000000,3.000000,7.000000e+00,5.096000e+03
dloss,200000.0,1.760658e+01,5.991625e+01,0.0,0.000000,4.000000,1.500000e+01,5.483000e+03
sttl,200000.0,5.797389e+01,7.105442e+01,0.0,31.000000,31.000000,3.100000e+01,2.550000e+02
dttl,200000.0,4.011772e+01,5.596726e+01,0.0,29.000000,29.000000,2.900000e+01,2.540000e+02
sload,200000.0,1.815315e+07,9.459088e+07,0.0,68764.980470,544776.125000,1.353954e+06,5.268000e+09
dload,200000.0,2.556135e+06,4.227503e+06,0.0,53162.311523,630963.937500,3.311835e+06,2.958034e+07


In [ ]:
# ============================================================
# CELL 47 — RAW DURATION CHECK
# ============================================================

print(
    "Raw duration:",
    df["duration"].describe()
)

print(
    "\nNon-zero durations:",
    (df["duration"] > 0).sum()
)

print(
    "Zero durations:",
    (df["duration"] == 0).sum()
)

print(
    "\nLargest durations:"
)

display(
    df[
        [
            "src_ip",
            "dst_ip",
            "duration",
            "sttl",
            "dttl"
        ]
    ]
    .sort_values(
        "duration",
        ascending=False
    )
    .head(10)
)

Raw duration: count    200000.000000
mean          0.820052
std           4.107772
min           0.000000
25%           0.001133
50%           0.024994
75%           0.357625
max          59.999527
Name: duration, dtype: float64

Non-zero durations: 198904
Zero durations: 1096

Largest durations:


,src_ip,dst_ip,duration,sttl,dttl
26355,175.45.176.2,149.171.126.19,59.999527,62,252
47289,10.40.170.2,10.40.170.2,59.997078,0,0
47282,10.40.170.2,10.40.170.2,59.997078,0,0
47275,10.40.182.3,10.40.182.3,59.997078,0,0
47276,10.40.182.3,10.40.182.3,59.997078,0,0
76414,175.45.176.2,149.171.126.15,59.994354,254,252
67850,10.40.182.3,10.40.182.3,59.991222,0,0
67854,10.40.182.3,10.40.182.3,59.991222,0,0
67852,10.40.170.2,10.40.170.2,59.991211,0,0
67853,10.40.170.2,10.40.170.2,59.991211,0,0


In [ ]:
# ============================================================
# CELL 48 — PARAMETER BREAKDOWN
# ============================================================

print("ENCODER PARAMETER BREAKDOWN\n")

for name, param in encoder.named_parameters():

    print(
        f"{name:50s}",
        f"{param.numel():>8,}",
        tuple(param.shape)
    )

print("\nTOTAL:")
print(
    f"{count_parameters(encoder):,}"
)

print("\nPROJECTION HEAD BREAKDOWN\n")

for name, param in projection_head.named_parameters():

    print(
        f"{name:50s}",
        f"{param.numel():>8,}",
        tuple(param.shape)
    )

print("\nTOTAL:")
print(
    f"{count_parameters(projection_head):,}"
)

ENCODER PARAMETER BREAKDOWN

structural.gat.att                                      240 (1, 4, 60)
structural.gat.bias                                      60 (60,)
structural.gat.lin_l.weight                           2,640 (240, 11)
structural.gat.lin_l.bias                               240 (240,)
structural.gat.lin_r.weight                           2,640 (240, 11)
structural.gat.lin_r.bias                               240 (240,)
temporal.attn.in_proj_weight                         10,800 (180, 60)
temporal.attn.in_proj_bias                              180 (180,)
temporal.attn.out_proj.weight                         3,600 (60, 60)
temporal.attn.out_proj.bias                              60 (60,)
temporal.norm.weight                                     60 (60,)
temporal.norm.bias                                       60 (60,)

TOTAL:
20,820

PROJECTION HEAD BREAKDOWN

net.0.weight                                          3,600 (60, 60)
net.0.bias                                  

In [ ]:
# ============================================================
# CELL 49 — Corrected CTGAD-X temporal graph slicer
# ============================================================

from pathlib import Path

slicer_path = Path(slicer_path)

corrected_slicer = r'''
"""
Causality-aware temporal graph construction.

CTGAD-X-compatible node features:
    sbytes, dbytes,
    spkts, dpkts,
    sloss, dloss,
    sload, dload,
    sttl, dttl,
    duration

Each flow [ts, ts+dur) contributes only to temporal
windows that it actually overlaps.

Additive quantities are allocated according to temporal
overlap.

Rate features are recomputed from:
    allocated bytes / allocated duration

TTL features are mean-pooled over contributing flows.
"""

from __future__ import annotations

import numpy as np
import pandas as pd

from dataclasses import dataclass, field
from typing import Dict, List, Tuple


ADDITIVE_COLS = [
    "sbytes",
    "dbytes",
    "spkts",
    "dpkts",
    "sloss",
    "dloss",
]


STATE_COLS = [
    "sttl",
    "dttl",
]


FEATURE_COLS = [
    "sbytes",
    "dbytes",
    "spkts",
    "dpkts",
    "sloss",
    "dloss",
    "sload",
    "dload",
    "sttl",
    "dttl",
    "duration",
]


@dataclass
class GraphSnapshot:

    window_idx: int
    t_start: float
    t_end: float

    node_ids: List[str] = field(default_factory=list)

    edges: List[Tuple[int, int]] = field(default_factory=list)

    node_features: Dict[str, np.ndarray] = field(default_factory=dict)


def compute_overlap(
    t_s: float,
    dur: float,
    w_start: float,
    delta_t: float
) -> float:

    t_e = t_s + dur
    w_end = w_start + delta_t

    return max(
        0.0,
        min(t_e, w_end) -
        max(t_s, w_start)
    )


def slice_flows_to_windows(
    flows: pd.DataFrame,
    delta_t: float = 30.0,
    ts_col: str = "ts_start",
    dur_col: str = "duration",
    src_col: str = "src_ip",
    dst_col: str = "dst_ip",
    global_t0: float | None = None,
) -> List[GraphSnapshot]:

    if flows.empty:
        return []

    flows = flows.copy()

    # --------------------------------------------------------
    # Temporal origin
    # --------------------------------------------------------

    if global_t0 is None:
        t_min = float(
            pd.to_numeric(
                flows[ts_col],
                errors="coerce"
            ).min()
        )
    else:
        t_min = float(global_t0)

    # --------------------------------------------------------
    # Clean duration
    # --------------------------------------------------------

    flows[dur_col] = pd.to_numeric(
        flows[dur_col],
        errors="coerce"
    ).fillna(0.0).clip(lower=0.0)

    flows[ts_col] = pd.to_numeric(
        flows[ts_col],
        errors="coerce"
    )

    flows = flows.dropna(
        subset=[ts_col]
    )

    if flows.empty:
        return []

    # --------------------------------------------------------
    # Temporal extent
    # --------------------------------------------------------

    t_max = float(
        (
            flows[ts_col] +
            flows[dur_col]
        ).max()
    )

    n_windows = max(
        1,
        int(
            np.ceil(
                max(0.0, t_max - t_min) /
                delta_t
            )
        ) + 1
    )

    snapshots = [
        GraphSnapshot(
            window_idx=i,
            t_start=t_min + i * delta_t,
            t_end=t_min + (i + 1) * delta_t
        )
        for i in range(n_windows)
    ]

    # --------------------------------------------------------
    # Accumulators
    #
    # key = (window_idx, node_id)
    # --------------------------------------------------------

    accum = {}

    edge_set = {
        i: set()
        for i in range(n_windows)
    }

    # --------------------------------------------------------
    # Flow slicing
    # --------------------------------------------------------

    for _, flow in flows.iterrows():

        absolute_ts = float(
            flow[ts_col]
        )

        # Convert to global-relative time.
        t_s = absolute_ts - t_min

        raw_dur = float(
            flow[dur_col]
        )

        # Zero-duration flows have no positive temporal
        # interval and therefore cannot contribute to a
        # positive-overlap temporal slice.
        if raw_dur <= 0.0:
            continue

        dur = raw_dur
        t_e = t_s + dur

        first_w = max(
            0,
            int(
                np.floor(
                    t_s / delta_t
                )
            )
        )

        last_w = min(
            n_windows - 1,
            int(
                np.floor(
                    (t_e - 1e-12) /
                    delta_t
                )
            )
        )

        src = str(
            flow[src_col]
        )

        dst = str(
            flow[dst_col]
        )

        for w in range(
            first_w,
            last_w + 1
        ):

            w_start = w * delta_t

            overlap = compute_overlap(
                t_s,
                dur,
                w_start,
                delta_t
            )

            if overlap <= 0.0:
                continue

            # ------------------------------------------------
            # Fraction of this flow represented in this
            # temporal window.
            # ------------------------------------------------

            frac = overlap / dur

            edge_set[w].add(
                (src, dst)
            )

            # ------------------------------------------------
            # Both endpoints receive the flow-derived
            # node representation, matching the current
            # CTGAD-X implementation convention.
            # ------------------------------------------------

            for node in (src, dst):

                key = (
                    w,
                    node
                )

                if key not in accum:

                    accum[key] = {
                        c: 0.0
                        for c in ADDITIVE_COLS
                    }

                    accum[key].update({
                        "sttl_sum": 0.0,
                        "dttl_sum": 0.0,
                        "state_count": 0.0,

                        # CRITICAL:
                        # accumulated temporal duration
                        "duration": 0.0,
                    })

                a = accum[key]

                # ------------------------------------------------
                # Allocate additive flow quantities
                # proportionally to overlap.
                # ------------------------------------------------

                for c in ADDITIVE_COLS:

                    value = pd.to_numeric(
                        flow.get(
                            c,
                            0.0
                        ),
                        errors="coerce"
                    )

                    if pd.isna(value):
                        value = 0.0

                    a[c] += (
                        float(value) *
                        frac
                    )

                # ------------------------------------------------
                # CRITICAL DURATION ACCOUNTING
                # ------------------------------------------------

                a["duration"] += overlap

                # ------------------------------------------------
                # TTL state features
                # ------------------------------------------------

                sttl = pd.to_numeric(
                    flow.get(
                        "sttl",
                        0.0
                    ),
                    errors="coerce"
                )

                dttl = pd.to_numeric(
                    flow.get(
                        "dttl",
                        0.0
                    ),
                    errors="coerce"
                )

                if pd.isna(sttl):
                    sttl = 0.0

                if pd.isna(dttl):
                    dttl = 0.0

                a["sttl_sum"] += float(
                    sttl
                )

                a["dttl_sum"] += float(
                    dttl
                )

                a["state_count"] += 1.0

    # ------------------------------------------------------------
    # Construct node features
    # ------------------------------------------------------------

    for (
        w,
        node
    ), a in accum.items():

        snap = snapshots[w]

        if node not in snap.node_ids:
            snap.node_ids.append(node)

        duration = max(
            float(a["duration"]),
            1e-9
        )

        count = max(
            float(a["state_count"]),
            1.0
        )

        sbytes = float(
            a["sbytes"]
        )

        dbytes = float(
            a["dbytes"]
        )

        # --------------------------------------------------------
        # Recompute rates using the ACTUAL allocated duration.
        # --------------------------------------------------------

        sload = (
            sbytes * 8.0
        ) / duration

        dload = (
            dbytes * 8.0
        ) / duration

        feature_vector = np.array([

            sbytes,
            dbytes,

            a["spkts"],
            a["dpkts"],

            a["sloss"],
            a["dloss"],

            sload,
            dload,

            a["sttl_sum"] / count,
            a["dttl_sum"] / count,

            duration,

        ], dtype=np.float32)

        snap.node_features[node] = (
            feature_vector
        )

    # ------------------------------------------------------------
    # Convert node IDs to integer edge indices
    # ------------------------------------------------------------

    for w in range(n_windows):

        snap = snapshots[w]

        id_map = {
            node_id: idx
            for idx, node_id in enumerate(
                snap.node_ids
            )
        }

        snap.edges = [
            (
                id_map[src],
                id_map[dst]
            )
            for src, dst in edge_set[w]
            if (
                src in id_map and
                dst in id_map
            )
        ]

    # ------------------------------------------------------------
    # Remove completely empty windows.
    # ------------------------------------------------------------

    return [
        s
        for s in snapshots
        if len(s.node_ids) > 0
    ]


def snapshots_to_sequences(
    snapshots: List[GraphSnapshot],
    seq_len: int = 5,
    stride: int = 1
):

    snapshots = sorted(
        snapshots,
        key=lambda x: x.window_idx
    )

    sequences = []

    for i in range(
        0,
        len(snapshots) - seq_len + 1,
        stride
    ):

        q = snapshots[
            i:i + seq_len
        ]

        indices = [
            s.window_idx
            for s in q
        ]

        if any(
            indices[j + 1] !=
            indices[j] + 1
            for j in range(
                len(indices) - 1
            )
        ):
            continue

        sequences.append(q)

    return sequences
'''

slicer_path.write_text(
    corrected_slicer
)

print("Corrected slicer written to:")
print(slicer_path)
print("\nKey fixes:")
print("1. duration += overlap")
print("2. sload/dload use allocated duration")
print("3. FEATURE_COLS uses 'duration'")
print("4. zero-duration flows are excluded")
print("5. temporal boundary handling corrected")

Corrected slicer written to:
/content/fedctgad_x_v2/project/fedctgad/data/slicing.py

Key fixes:
1. duration += overlap
2. sload/dload use allocated duration
3. FEATURE_COLS uses 'duration'
4. zero-duration flows are excluded
5. temporal boundary handling corrected


In [ ]:
# ============================================================
# CELL 50 — Load corrected slicer directly from slicer_path
# ============================================================

import importlib.util
import sys
from pathlib import Path

slicer_path = Path(slicer_path).resolve()

print("Slicer path:")
print(slicer_path)

print("\nExists:", slicer_path.exists())

if not slicer_path.exists():
    raise FileNotFoundError(
        f"Slicer file not found: {slicer_path}"
    )

# ------------------------------------------------------------
# Load module directly from the actual file
# ------------------------------------------------------------

spec = importlib.util.spec_from_file_location(
    "ctgad_graph_slicer",
    str(slicer_path)
)

graph_slicer = importlib.util.module_from_spec(
    spec
)

sys.modules["ctgad_graph_slicer"] = graph_slicer

spec.loader.exec_module(
    graph_slicer
)

print("\n✓ Corrected graph slicer loaded successfully.")

print("\nFEATURE_COLS:")
print(
    graph_slicer.FEATURE_COLS
)

Slicer path:
/content/fedctgad_x_v2/project/fedctgad/data/slicing.py

Exists: True

✓ Corrected graph slicer loaded successfully.

FEATURE_COLS:
['sbytes', 'dbytes', 'spkts', 'dpkts', 'sloss', 'dloss', 'sload', 'dload', 'sttl', 'dttl', 'duration']


In [ ]:
# ============================================================
# CELL 51 — Rebuild and validate node features
# ============================================================

snapshots = graph_slicer.slice_flows_to_windows(
    df,
    delta_t=30.0,
    ts_col="ts_start",
    dur_col="duration",
    src_col="src_ip",
    dst_col="dst_ip",
    global_t0=GLOBAL_T0,
)

print("Number of snapshots:", len(snapshots))

all_features = []

for snap in snapshots:
    for node, feat in snap.node_features.items():
        all_features.append(feat)

all_features = np.vstack(
    all_features
)

feature_names = [
    "sbytes",
    "dbytes",
    "spkts",
    "dpkts",
    "sloss",
    "dloss",
    "sload",
    "dload",
    "sttl",
    "dttl",
    "duration",
]

feature_df = pd.DataFrame(
    all_features,
    columns=feature_names
)

print("\nGraph-level feature statistics:")
display(
    feature_df.describe().T
)

Number of snapshots: 255

Graph-level feature statistics:


,count,mean,std,min,25%,50%,75%,max
sbytes,9402.0,2.263882e+05,4.454345e+05,0.269541,7984.879272,200704.000000,3.051045e+05,9.755082e+06
dbytes,9402.0,1.669984e+06,2.425659e+06,0.000000,7001.373535,742013.000000,2.303219e+06,2.436138e+07
spkts,9402.0,1.536276e+03,1.522666e+03,0.009626,64.000000,1558.000000,2.444634e+03,1.088855e+04
dpkts,9402.0,1.957936e+03,2.274759e+03,0.000000,48.000000,1604.575928,2.973527e+03,1.977631e+04
sloss,9402.0,2.476220e+02,2.511584e+02,0.000000,12.926291,270.103348,3.974757e+02,3.555990e+03
dloss,9402.0,7.466050e+02,9.459259e+02,0.000000,12.000000,505.939972,1.097414e+03,8.839713e+03
sload,9402.0,2.930904e+06,5.830488e+07,8.466366,13193.295898,61951.173828,1.198183e+05,2.688000e+09
dload,9402.0,4.711905e+05,5.164053e+05,0.000000,9257.258789,365868.484375,7.574773e+05,4.869914e+06
sttl,9402.0,7.835024e+01,8.223822e+01,0.000000,31.000000,31.000000,1.505000e+02,2.543333e+02
dttl,9402.0,8.018434e+01,9.207976e+01,0.000000,29.000000,29.000000,1.680000e+02,2.525714e+02


In [ ]:
# ============================================================
# CELL 52 — Mandatory feature sanity checks
# ============================================================

print("ZERO DURATION:")
print(
    (feature_df["duration"] == 0).sum()
)

print(
    "Duration min:",
    feature_df["duration"].min()
)

print(
    "Duration median:",
    feature_df["duration"].median()
)

print(
    "Duration max:",
    feature_df["duration"].max()
)

print("\nTTL statistics:")
print(
    feature_df[
        ["sttl", "dttl"]
    ].describe()
)

print("\nNon-zero TTL fractions:")

print(
    "sttl > 0:",
    (
        feature_df["sttl"] > 0
    ).mean()
)

print(
    "dttl > 0:",
    (
        feature_df["dttl"] > 0
    ).mean()
)

print("\nRate statistics:")
print(
    feature_df[
        ["sload", "dload"]
    ].describe()
)

assert (
    feature_df["duration"] > 0
).all(), (
    "ERROR: zero-duration graph nodes remain!"
)

assert np.isfinite(
    feature_df.values
).all(), (
    "ERROR: NaN/Inf detected!"
)

print("\n✓ ALL FEATURE SANITY CHECKS PASSED.")

ZERO DURATION:
0
Duration min: 1.0000001111620804e-06
Duration median: 20.148439407348633
Duration max: 920.0897216796875

TTL statistics:
              sttl         dttl
count  9402.000000  9402.000000
mean     78.350243    80.184341
std      82.238220    92.079758
min       0.000000     0.000000
25%      31.000000    29.000000
50%      31.000000    29.000000
75%     150.500000   168.000000
max     254.333328   252.571426

Non-zero TTL fractions:
sttl > 0: 0.9340565837055945
dttl > 0: 0.8704530950861519

Rate statistics:
              sload         dload
count  9.402000e+03  9.402000e+03
mean   2.930904e+06  4.711905e+05
std    5.830488e+07  5.164053e+05
min    8.466366e+00  0.000000e+00
25%    1.319330e+04  9.257259e+03
50%    6.195117e+04  3.658685e+05
75%    1.198183e+05  7.574773e+05
max    2.688000e+09  4.869914e+06

✓ ALL FEATURE SANITY CHECKS PASSED.


In [ ]:
# ============================================================
# CELL 53 — Locate graph construction implementation
# ============================================================

from pathlib import Path

print("PROJECT ROOT:")
print(PROJECT_ROOT)

print("\nPython files containing graph/temporal/slice logic:\n")

for p in Path(PROJECT_ROOT).rglob("*.py"):
    name = p.name.lower()
    path_str = str(p).lower()

    if any(
        key in name or key in path_str
        for key in [
            "graph",
            "slice",
            "temporal",
            "dataset",
            "data"
        ]
    ):
        print(p)

PROJECT ROOT:
/content/fedctgad_x_v2/project/fedctgad

Python files containing graph/temporal/slice logic:

/content/fedctgad_x_v2/project/fedctgad/data/slicing.py
/content/fedctgad_x_v2/project/fedctgad/data/partition.py
/content/fedctgad_x_v2/project/fedctgad/data/loaders.py


In [ ]:
# ============================================================
# CELL 54 — Search original implementation for CTGAD features
# ============================================================

import re

search_terms = [
    "sbytes",
    "dbytes",
    "spkts",
    "dpkts",
    "sttl",
    "dttl",
    "sload",
    "dload",
    "duration",
    "overlap",
    "delta_t",
]

for p in Path(PROJECT_ROOT).rglob("*.py"):

    try:
        text = p.read_text(
            errors="ignore"
        )
    except Exception:
        continue

    hits = [
        term
        for term in search_terms
        if term.lower() in text.lower()
    ]

    if hits:
        print("\n" + "=" * 80)
        print(p)
        print("Hits:", hits)

        # Show relevant lines
        lines = text.splitlines()

        for i, line in enumerate(lines):

            if any(
                term.lower() in line.lower()
                for term in hits
            ):
                start = max(
                    0,
                    i - 3
                )
                end = min(
                    len(lines),
                    i + 4
                )

                print(
                    f"\n--- lines {start+1}-{end} ---"
                )

                for j in range(
                    start,
                    end
                ):
                    print(
                        f"{j+1:4}: {lines[j]}"
                    )


/content/fedctgad_x_v2/project/fedctgad/data/slicing.py
Hits: ['sbytes', 'dbytes', 'spkts', 'dpkts', 'sttl', 'dttl', 'sload', 'dload', 'duration', 'overlap', 'delta_t']

--- lines 3-9 ---
   3: Causality-aware temporal graph construction.
   4: 
   5: CTGAD-X-compatible node features:
   6:     sbytes, dbytes,
   7:     spkts, dpkts,
   8:     sloss, dloss,
   9:     sload, dload,

--- lines 4-10 ---
   4: 
   5: CTGAD-X-compatible node features:
   6:     sbytes, dbytes,
   7:     spkts, dpkts,
   8:     sloss, dloss,
   9:     sload, dload,
  10:     sttl, dttl,

--- lines 6-12 ---
   6:     sbytes, dbytes,
   7:     spkts, dpkts,
   8:     sloss, dloss,
   9:     sload, dload,
  10:     sttl, dttl,
  11:     duration
  12: 

--- lines 7-13 ---
   7:     spkts, dpkts,
   8:     sloss, dloss,
   9:     sload, dload,
  10:     sttl, dttl,
  11:     duration
  12: 
  13: Each flow [ts, ts+dur) contributes only to temporal

--- lines 8-14 ---
   8:     sloss, dloss,
   9:     sload, dlo

In [ ]:
# ============================================================
# CELL 55 — Inspect actual CTGAD-X model implementation
# ============================================================

from pathlib import Path

print("MODEL FILES:\n")

for p in Path(PROJECT_ROOT).rglob("*.py"):
    name = p.name.lower()

    if (
        "model" in name
        or "encoder" in name
        or "loss" in name
    ):
        print(p)

MODEL FILES:

/content/fedctgad_x_v2/project/fedctgad/model/losses.py
/content/fedctgad_x_v2/project/fedctgad/model/encoder.py


In [ ]:
# ============================================================
# CELL 56 — Locate CTGAD_Encoder implementation
# ============================================================

for p in Path(PROJECT_ROOT).rglob("*.py"):

    try:
        text = p.read_text(
            errors="ignore"
        )
    except Exception:
        continue

    if "class CTGAD_Encoder" in text:

        print("=" * 90)
        print("FOUND ENCODER:")
        print(p)
        print("=" * 90)

        lines = text.splitlines()

        for i, line in enumerate(lines):

            if "class CTGAD_Encoder" in line:

                # Print a generous section around the class.
                start = max(
                    0,
                    i - 5
                )

                end = min(
                    len(lines),
                    i + 180
                )

                for j in range(
                    start,
                    end
                ):
                    print(
                        f"{j+1:4}: {lines[j]}"
                    )

                break

FOUND ENCODER:
/content/fedctgad_x_v2/project/fedctgad/model/encoder.py
  41:         attn_out, _ = self.attn(seq, seq, seq)
  42:         out = self.norm(seq + attn_out)          # residual + layernorm
  43:         return out[:, -1, :]                      # last time step's embedding
  44: 
  45: 
  46: class CTGAD_Encoder(nn.Module):
  47:     """
  48:     Full spatio-temporal encoder. Input: a sequence of L graph snapshots
  49:     (each a torch_geometric Data object with node features + edge_index).
  50:     Output: a single Denc-dim sequence embedding.
  51:     """
  52: 
  53:     def __init__(self, in_dim: int = 11, hidden_dim: int = 60, seq_len: int = 5,
  54:                  struct_heads: int = 4, temporal_heads: int = 4):
  55:         super().__init__()
  56:         self.structural = StructuralEncoder(in_dim, hidden_dim, struct_heads)
  57:         self.temporal = TemporalEncoder(hidden_dim, temporal_heads, seq_len)
  58:         self.hidden_dim = hidden_dim
  59:   

In [ ]:
# ============================================================
# CELL 57 — Locate ProjectionHead implementation
# ============================================================

for p in Path(PROJECT_ROOT).rglob("*.py"):

    try:
        text = p.read_text(
            errors="ignore"
        )
    except Exception:
        continue

    if "class ProjectionHead" in text:

        print("=" * 90)
        print("FOUND PROJECTION HEAD:")
        print(p)
        print("=" * 90)

        lines = text.splitlines()

        for i, line in enumerate(lines):

            if "class ProjectionHead" in line:

                start = max(
                    0,
                    i - 5
                )

                end = min(
                    len(lines),
                    i + 100
                )

                for j in range(
                    start,
                    end
                ):
                    print(
                        f"{j+1:4}: {lines[j]}"
                    )

                break

FOUND PROJECTION HEAD:
/content/fedctgad_x_v2/project/fedctgad/model/encoder.py
  82:         out.index_add_(0, batch_index, h)
  83:         counts.index_add_(0, batch_index, torch.ones(h.size(0), 1, device=h.device, dtype=h.dtype))
  84:         return out / counts.clamp(min=1)
  85: 
  86: 
  87: class ProjectionHead(nn.Module):
  88:     """Used only during contrastive training; discarded at inference (matches CTGAD-X)."""
  89: 
  90:     def __init__(self, in_dim: int = 60, hidden_dim: int = 60, out_dim: int = 80):
  91:         super().__init__()
  92:         self.net = nn.Sequential(
  93:             nn.Linear(in_dim, hidden_dim), nn.ReLU(),
  94:             nn.Linear(hidden_dim, out_dim),
  95:         )
  96: 
  97:     def forward(self, z: torch.Tensor) -> torch.Tensor:
  98:         return self.net(z)


In [ ]:
# ============================================================
# CELL 58 — Print COMPLETE encoder.py
# ============================================================

from pathlib import Path

encoder_path = (
    Path(PROJECT_ROOT)
    / "model"
    / "encoder.py"
)

print("=" * 100)
print("COMPLETE ENCODER SOURCE")
print(encoder_path)
print("=" * 100)

encoder_text = encoder_path.read_text(
    errors="ignore"
)

print(encoder_text)

COMPLETE ENCODER SOURCE
/content/fedctgad_x_v2/project/fedctgad/model/encoder.py
"""
CTGAD_Encoder: GATv2 structural layer -> mean-pool aggregation -> temporal
multi-head self-attention. Architecturally identical to the base CTGAD-X
paper's encoder (kept unmodified on purpose — the research contribution of
this project is in server/aggregate.py, not the encoder itself).

Requires: torch, torch_geometric
    pip install torch torch_geometric
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv
from torch_geometric.data import Data, Batch


class StructuralEncoder(nn.Module):
    """One GATv2 layer per snapshot, applied independently across time steps."""

    def __init__(self, in_dim: int, hidden_dim: int = 60, heads: int = 4):
        super().__init__()
        # concat=False averages head outputs -> keeps output dim == hidden_dim
        self.gat = GATv2Conv(in_dim, hidden_dim, heads=heads, concat=False, dropout=0.1)

    def

In [ ]:
# ============================================================
# CELL 59 — Locate exact architecture statements in CTGAD-X
# ============================================================

# Search the conversation/library files for exact architecture
# terminology used by the paper.

print("""
Search the CTGAD-X source for:

1. 16,320
2. GATv2
3. 60
4. 4 heads
5. Multi-head
6. temporal attention
7. projection
8. 80
9. parameter
""")


Search the CTGAD-X source for:

1. 16,320
2. GATv2
3. 60
4. 4 heads
5. Multi-head
6. temporal attention
7. projection
8. 80
9. parameter



In [ ]:
# ============================================================
# CELL 59 — Training readiness check
# ============================================================

print("Checking training objects...\n")

required = [
    "train_sequences",
    "val_sequences",
    "test_sequences",
    "scaler",
    "CTGAD_Encoder",
    "ProjectionHead",
]

for name in required:
    print(
        f"{name:25s}:",
        "✓" if name in globals() else "MISSING"
    )

print("\nSequence counts:")

if "train_sequences" in globals():
    print("Train:", len(train_sequences))

if "val_sequences" in globals():
    print("Validation:", len(val_sequences))

if "test_sequences" in globals():
    print("Test:", len(test_sequences))

Checking training objects...

train_sequences          : MISSING
val_sequences            : MISSING
test_sequences           : MISSING
scaler                   : ✓
CTGAD_Encoder            : ✓
ProjectionHead           : ✓

Sequence counts:


In [ ]:
# ============================================================
# CELL 59A — Rebuild normal-only temporal graph sequences
# ============================================================

# Use the corrected slicer already loaded as graph_slicer.

normal_df = df[
    df["label"] == 0
].copy()

print("Normal rows:", len(normal_df))

# ------------------------------------------------------------
# IMPORTANT:
# Use the SAME GLOBAL_T0 as the complete traffic stream.
# This preserves the global temporal coordinate system.
# ------------------------------------------------------------

normal_snapshots = graph_slicer.slice_flows_to_windows(
    normal_df,
    delta_t=30.0,
    ts_col="ts_start",
    dur_col="duration",
    src_col="src_ip",
    dst_col="dst_ip",
    global_t0=GLOBAL_T0,
)

print(
    "Normal snapshots:",
    len(normal_snapshots)
)

# ------------------------------------------------------------
# Create contiguous 5-window sequences.
# ------------------------------------------------------------

normal_sequences = graph_slicer.snapshots_to_sequences(
    normal_snapshots,
    seq_len=5,
    stride=1,
)

print(
    "Total normal sequences:",
    len(normal_sequences)
)

Normal rows: 177785
Normal snapshots: 255
Total normal sequences: 251


In [ ]:
# ============================================================
# CELL 59B — Chronological leakage-safe normal split
# ============================================================

n_normal = len(
    normal_sequences
)

train_end = int(
    0.70 * n_normal
)

val_end = int(
    0.80 * n_normal
)

train_sequences = normal_sequences[
    :train_end
]

val_sequences = normal_sequences[
    train_end:val_end
]

test_sequences = normal_sequences[
    val_end:
]

print("NORMAL SEQUENCE SPLIT")
print("=" * 60)

print(
    "Total:",
    n_normal
)

print(
    "Train:",
    len(train_sequences)
)

print(
    "Validation:",
    len(val_sequences)
)

print(
    "Test:",
    len(test_sequences)
)

NORMAL SEQUENCE SPLIT
Total: 251
Train: 175
Validation: 25
Test: 51


In [ ]:
# ============================================================
# CELL 59B — LEAKAGE-SAFE TEMPORAL WINDOW SPLIT
# ============================================================

# normal_snapshots contains the chronological normal-only graph snapshots
n_windows = len(normal_snapshots)

train_cut = int(0.70 * n_windows)
val_cut   = int(0.80 * n_windows)

train_snapshot_idx = list(range(0, train_cut))
val_snapshot_idx   = list(range(train_cut, val_cut))
test_snapshot_idx  = list(range(val_cut, n_windows))

print("Total normal windows:", n_windows)
print("Train windows:", len(train_snapshot_idx))
print("Validation windows:", len(val_snapshot_idx))
print("Test windows:", len(test_snapshot_idx))

# Verify that the WINDOW partitions themselves are disjoint
assert set(train_snapshot_idx).isdisjoint(set(val_snapshot_idx))
assert set(train_snapshot_idx).isdisjoint(set(test_snapshot_idx))
assert set(val_snapshot_idx).isdisjoint(set(test_snapshot_idx))

print("\nWindow-level split is disjoint ✓")

Total normal windows: 255
Train windows: 178
Validation windows: 26
Test windows: 51

Window-level split is disjoint ✓


In [ ]:
# ============================================================
# CELL 59C — BUILD SEQUENCES INSIDE EACH PARTITION
# ============================================================

SEQ_LEN = 5

def make_partition_sequences(snapshots, indices, seq_len=5):
    """
    Build contiguous sequences only within the supplied
    temporal partition.
    """
    partition = [snapshots[i] for i in indices]

    sequences = []

    for i in range(len(partition) - seq_len + 1):
        seq = partition[i:i + seq_len]

        # Keep only genuinely contiguous temporal windows
        window_ids = [s.window_idx for s in seq]

        if all(window_ids[j] + 1 == window_ids[j + 1]
               for j in range(len(window_ids) - 1)):
            sequences.append(seq)

    return sequences


train_sequences = make_partition_sequences(
    normal_snapshots,
    train_snapshot_idx,
    SEQ_LEN
)

val_sequences = make_partition_sequences(
    normal_snapshots,
    val_snapshot_idx,
    SEQ_LEN
)

test_sequences = make_partition_sequences(
    normal_snapshots,
    test_snapshot_idx,
    SEQ_LEN
)

print("Train sequences:", len(train_sequences))
print("Validation sequences:", len(val_sequences))
print("Test sequences:", len(test_sequences))

Train sequences: 174
Validation sequences: 22
Test sequences: 47


In [ ]:
# ============================================================
# CELL 59D — VERIFY NO WINDOW OVERLAP
# ============================================================

def sequence_window_ids(sequences):
    ids = set()

    for seq in sequences:
        for snapshot in seq:
            ids.add(snapshot.window_idx)

    return ids


train_windows = sequence_window_ids(train_sequences)
val_windows   = sequence_window_ids(val_sequences)
test_windows  = sequence_window_ids(test_sequences)

print("Train windows:", len(train_windows))
print("Validation windows:", len(val_windows))
print("Test windows:", len(test_windows))

print("\nTrain ∩ Validation:", len(train_windows & val_windows))
print("Train ∩ Test:", len(train_windows & test_windows))
print("Validation ∩ Test:", len(val_windows & test_windows))

assert train_windows.isdisjoint(val_windows), \
    "Train/Validation leakage detected!"

assert train_windows.isdisjoint(test_windows), \
    "Train/Test leakage detected!"

assert val_windows.isdisjoint(test_windows), \
    "Validation/Test leakage detected!"

print("\n" + "="*60)
print("LEAKAGE CHECK PASSED ✓")
print("="*60)

Train windows: 178
Validation windows: 26
Test windows: 51

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0

LEAKAGE CHECK PASSED ✓


In [ ]:
# ============================================================
# CELL 60 — GRAPH SNAPSHOT → PYTORCH GEOMETRIC DATA
# ============================================================

import torch
from torch_geometric.data import Data

def snapshot_to_pyg(snapshot, scaler):
    """
    Convert one GraphSnapshot into a PyTorch Geometric graph.

    Node features:
        11-dimensional CTGAD-X feature vector

    Features are standardized using the scaler fitted
    ONLY on the training data.
    """

    # --------------------------------------------------------
    # Extract node IDs
    # --------------------------------------------------------
    node_ids = list(snapshot.node_features.keys())

    # --------------------------------------------------------
    # Build feature matrix
    # --------------------------------------------------------
    X = []

    for node_id in node_ids:
        X.append(snapshot.node_features[node_id])

    X = torch.tensor(
        scaler.transform(X),
        dtype=torch.float32
    )

    # --------------------------------------------------------
    # Map original node IDs → consecutive PyG indices
    # --------------------------------------------------------
    node_to_idx = {
        node_id: i
        for i, node_id in enumerate(node_ids)
    }

    # --------------------------------------------------------
    # Build edge index
    # --------------------------------------------------------
    edges = []

    for src, dst in snapshot.edges:
        if src in node_to_idx and dst in node_to_idx:
            edges.append([
                node_to_idx[src],
                node_to_idx[dst]
            ])

    if len(edges) > 0:
        edge_index = torch.tensor(
            edges,
            dtype=torch.long
        ).t().contiguous()
    else:
        edge_index = torch.empty(
            (2, 0),
            dtype=torch.long
        )

    # --------------------------------------------------------
    # PyG graph
    # --------------------------------------------------------
    return Data(
        x=X,
        edge_index=edge_index
    )


print("snapshot_to_pyg() created ✓")

snapshot_to_pyg() created ✓


In [ ]:
# ============================================================
# CELL 61 — CONVERT TEMPORAL SEQUENCES TO PYTORCH GEOMETRIC
# ============================================================

def convert_sequences(sequences, scaler):

    converted = []

    for sequence in sequences:

        temporal_graphs = []

        for snapshot in sequence:
            graph = snapshot_to_pyg(
                snapshot,
                scaler
            )

            temporal_graphs.append(graph)

        converted.append(temporal_graphs)

    return converted


train_graph_sequences = convert_sequences(
    train_sequences,
    scaler
)

val_graph_sequences = convert_sequences(
    val_sequences,
    scaler
)

test_graph_sequences = convert_sequences(
    test_sequences,
    scaler
)

print("Train graph sequences:", len(train_graph_sequences))
print("Validation graph sequences:", len(val_graph_sequences))
print("Test graph sequences:", len(test_graph_sequences))

print("\nExample temporal sequence:")
print("Number of snapshots:", len(train_graph_sequences[0]))

for t, graph in enumerate(train_graph_sequences[0]):
    print(
        f"t={t}: "
        f"nodes={graph.num_nodes}, "
        f"edges={graph.num_edges}, "
        f"features={tuple(graph.x.shape)}"
    )

Train graph sequences: 174
Validation graph sequences: 22
Test graph sequences: 47

Example temporal sequence:
Number of snapshots: 5
t=0: nodes=3, edges=0, features=(3, 11)
t=1: nodes=25, edges=0, features=(25, 11)
t=2: nodes=25, edges=0, features=(25, 11)
t=3: nodes=25, edges=0, features=(25, 11)
t=4: nodes=27, edges=0, features=(27, 11)


In [ ]:
# ============================================================
# CELL 61A — GRAPH CONNECTIVITY AUDIT
# ============================================================

def connectivity_stats(graph_sequences):

    total_snapshots = 0
    zero_edge_snapshots = 0
    total_nodes = 0
    total_edges = 0

    for sequence in graph_sequences:
        for graph in sequence:
            total_snapshots += 1
            total_nodes += graph.num_nodes
            total_edges += graph.num_edges

            if graph.num_edges == 0:
                zero_edge_snapshots += 1

    return {
        "snapshots": total_snapshots,
        "zero_edge_snapshots": zero_edge_snapshots,
        "zero_edge_pct": 100 * zero_edge_snapshots / total_snapshots,
        "total_nodes": total_nodes,
        "total_edges": total_edges,
        "avg_edges_per_snapshot": total_edges / total_snapshots,
    }


for name, sequences in [
    ("Train", train_graph_sequences),
    ("Validation", val_graph_sequences),
    ("Test", test_graph_sequences),
]:

    stats = connectivity_stats(sequences)

    print(f"\n{name}")
    print("-" * 40)
    print("Snapshots:", stats["snapshots"])
    print("Zero-edge snapshots:", stats["zero_edge_snapshots"])
    print("Zero-edge percentage:", f"{stats['zero_edge_pct']:.2f}%")
    print("Total nodes:", stats["total_nodes"])
    print("Total edges:", stats["total_edges"])
    print(
        "Average edges/snapshot:",
        f"{stats['avg_edges_per_snapshot']:.2f}"
    )


Train
----------------------------------------
Snapshots: 870
Zero-edge snapshots: 870
Zero-edge percentage: 100.00%
Total nodes: 23776
Total edges: 0
Average edges/snapshot: 0.00

Validation
----------------------------------------
Snapshots: 110
Zero-edge snapshots: 110
Zero-edge percentage: 100.00%
Total nodes: 3343
Total edges: 0
Average edges/snapshot: 0.00

Test
----------------------------------------
Snapshots: 235
Zero-edge snapshots: 235
Zero-edge percentage: 100.00%
Total nodes: 6108
Total edges: 0
Average edges/snapshot: 0.00


In [ ]:
# ============================================================
# CELL 61B — EDGE STRUCTURE SANITY CHECK
# ============================================================

sample_snapshot = train_sequences[0][0]

print("Snapshot type:", type(sample_snapshot))
print("Number of nodes:", len(sample_snapshot.node_features))
print("Number of edges:", len(sample_snapshot.edges))

print("\nFirst 20 edges:")
for edge in list(sample_snapshot.edges)[:20]:
    print(edge)

Snapshot type: <class 'ctgad_graph_slicer.GraphSnapshot'>
Number of nodes: 3
Number of edges: 2

First 20 edges:
(2, 1)
(0, 1)


In [ ]:
# ============================================================
# CELL 60 — CORRECT GRAPH SNAPSHOT → PYTORCH GEOMETRIC DATA
# ============================================================

import torch
from torch_geometric.data import Data


def snapshot_to_pyg(snapshot, scaler):
    """
    Convert one CTGAD GraphSnapshot into a PyTorch Geometric graph.

    Important:
    GraphSnapshot.edges already stores integer node indices.
    Therefore, edges must NOT be remapped using node IDs.
    """

    # --------------------------------------------------------
    # Preserve the node ordering used by node_features
    # --------------------------------------------------------
    node_ids = list(snapshot.node_features.keys())

    # --------------------------------------------------------
    # Build node feature matrix
    # --------------------------------------------------------
    X = [
        snapshot.node_features[node_id]
        for node_id in node_ids
    ]

    # Training-only fitted scaler
    X = scaler.transform(X)

    X = torch.tensor(
        X,
        dtype=torch.float32
    )

    # --------------------------------------------------------
    # Graph edges
    # --------------------------------------------------------
    edges = list(snapshot.edges)

    if len(edges) > 0:

        edge_index = torch.tensor(
            edges,
            dtype=torch.long
        ).t().contiguous()

    else:

        edge_index = torch.empty(
            (2, 0),
            dtype=torch.long
        )

    # --------------------------------------------------------
    # Sanity check
    # --------------------------------------------------------
    if edge_index.numel() > 0:

        assert edge_index.min().item() >= 0

        assert edge_index.max().item() < len(node_ids), (
            f"Invalid edge index: max={edge_index.max().item()}, "
            f"num_nodes={len(node_ids)}"
        )

    # --------------------------------------------------------
    # PyTorch Geometric graph
    # --------------------------------------------------------
    return Data(
        x=X,
        edge_index=edge_index
    )


print("Corrected snapshot_to_pyg() created ✓")

Corrected snapshot_to_pyg() created ✓


In [ ]:
# ============================================================
# CELL 61 — REBUILD PYTORCH GEOMETRIC SEQUENCES
# ============================================================

def convert_sequences(sequences, scaler):

    converted = []

    for sequence in sequences:

        temporal_graphs = []

        for snapshot in sequence:

            graph = snapshot_to_pyg(
                snapshot,
                scaler
            )

            temporal_graphs.append(graph)

        converted.append(temporal_graphs)

    return converted


train_graph_sequences = convert_sequences(
    train_sequences,
    scaler
)

val_graph_sequences = convert_sequences(
    val_sequences,
    scaler
)

test_graph_sequences = convert_sequences(
    test_sequences,
    scaler
)

print("Train graph sequences:", len(train_graph_sequences))
print("Validation graph sequences:", len(val_graph_sequences))
print("Test graph sequences:", len(test_graph_sequences))

Train graph sequences: 174
Validation graph sequences: 22
Test graph sequences: 47


In [ ]:
# ============================================================
# CELL 61A — VERIFY GRAPH CONNECTIVITY AFTER FIX
# ============================================================

def connectivity_stats(graph_sequences):

    total_snapshots = 0
    zero_edge_snapshots = 0
    total_nodes = 0
    total_edges = 0

    for sequence in graph_sequences:

        for graph in sequence:

            total_snapshots += 1
            total_nodes += graph.num_nodes
            total_edges += graph.num_edges

            if graph.num_edges == 0:
                zero_edge_snapshots += 1

    return {
        "snapshots": total_snapshots,
        "zero_edge_snapshots": zero_edge_snapshots,
        "zero_edge_pct":
            100 * zero_edge_snapshots / total_snapshots,
        "total_nodes": total_nodes,
        "total_edges": total_edges,
        "avg_edges_per_snapshot":
            total_edges / total_snapshots
    }


for name, sequences in [
    ("Train", train_graph_sequences),
    ("Validation", val_graph_sequences),
    ("Test", test_graph_sequences)
]:

    stats = connectivity_stats(sequences)

    print(f"\n{name}")
    print("-" * 40)

    print(
        "Snapshots:",
        stats["snapshots"]
    )

    print(
        "Zero-edge snapshots:",
        stats["zero_edge_snapshots"]
    )

    print(
        "Zero-edge percentage:",
        f"{stats['zero_edge_pct']:.2f}%"
    )

    print(
        "Total nodes:",
        stats["total_nodes"]
    )

    print(
        "Total edges:",
        stats["total_edges"]
    )

    print(
        "Average edges/snapshot:",
        f"{stats['avg_edges_per_snapshot']:.2f}"
    )


Train
----------------------------------------
Snapshots: 870
Zero-edge snapshots: 0
Zero-edge percentage: 0.00%
Total nodes: 23776
Total edges: 92041
Average edges/snapshot: 105.79

Validation
----------------------------------------
Snapshots: 110
Zero-edge snapshots: 0
Zero-edge percentage: 0.00%
Total nodes: 3343
Total edges: 11807
Average edges/snapshot: 107.34

Test
----------------------------------------
Snapshots: 235
Zero-edge snapshots: 0
Zero-edge percentage: 0.00%
Total nodes: 6108
Total edges: 25027
Average edges/snapshot: 106.50


In [ ]:
# ============================================================
# CELL 61B — FINAL EDGE SANITY CHECK
# ============================================================

graph = train_graph_sequences[0][0]

print("Nodes:", graph.num_nodes)
print("Edges:", graph.num_edges)
print("Feature shape:", tuple(graph.x.shape))

print("\nedge_index:")
print(graph.edge_index)

print("\nOriginal GraphSnapshot edges:")
print(list(train_sequences[0][0].edges))

Nodes: 3
Edges: 2
Feature shape: (3, 11)

edge_index:
tensor([[2, 0],
        [1, 1]])

Original GraphSnapshot edges:
[(2, 1), (0, 1)]


In [ ]:
# ============================================================
# CELL 62 — TEMPORAL GRAPH BATCHING
# ============================================================

from torch_geometric.data import Batch


def batch_temporal_sequences(graph_sequences, batch_size):
    """
    Create mini-batches of temporal graph sequences.

    Each sequence contains exactly SEQ_LEN snapshots.

    For each temporal position t, graphs from all sequences
    in the mini-batch are combined into one PyG Batch.
    """

    batches = []

    for start in range(0, len(graph_sequences), batch_size):

        sequence_batch = graph_sequences[
            start:start + batch_size
        ]

        temporal_batches = []

        for t in range(SEQ_LEN):

            graphs_at_t = [
                sequence[t]
                for sequence in sequence_batch
            ]

            temporal_batch = Batch.from_data_list(
                graphs_at_t
            )

            temporal_batches.append(
                temporal_batch
            )

        batches.append(temporal_batches)

    return batches


BATCH_SIZE = 16

train_batches = batch_temporal_sequences(
    train_graph_sequences,
    BATCH_SIZE
)

val_batches = batch_temporal_sequences(
    val_graph_sequences,
    BATCH_SIZE
)

test_batches = batch_temporal_sequences(
    test_graph_sequences,
    BATCH_SIZE
)

print("Train batches:", len(train_batches))
print("Validation batches:", len(val_batches))
print("Test batches:", len(test_batches))

Train batches: 11
Validation batches: 2
Test batches: 3


In [ ]:
# ============================================================
# CELL 62A — BATCH SANITY CHECK
# ============================================================

batch = train_batches[0]

print("Temporal snapshots in batch:", len(batch))

for t, graph_batch in enumerate(batch):

    print(
        f"t={t}: "
        f"graphs={graph_batch.num_graphs}, "
        f"nodes={graph_batch.num_nodes}, "
        f"edges={graph_batch.num_edges}, "
        f"x={tuple(graph_batch.x.shape)}"
    )

    assert graph_batch.x.shape[1] == 11
    assert graph_batch.edge_index.shape[0] == 2

print("\nTemporal batching check passed ✓")

Temporal snapshots in batch: 5
t=0: graphs=16, nodes=396, edges=1572, x=(396, 11)
t=1: graphs=16, nodes=422, edges=1674, x=(422, 11)
t=2: graphs=16, nodes=423, edges=1672, x=(423, 11)
t=3: graphs=16, nodes=427, edges=1671, x=(427, 11)
t=4: graphs=16, nodes=432, edges=1674, x=(432, 11)

Temporal batching check passed ✓


In [ ]:
# ============================================================
# CELL 63 — LOCATE PROJECT LOSS IMPLEMENTATION
# ============================================================

import os

candidates = [
    "/content/fedctgad_x_v2/project/fedctgad/model/losses.py",
    "/content/fedctgad_x_v2/project/model/losses.py",
]

for path in candidates:
    print(path, "→", os.path.exists(path))

/content/fedctgad_x_v2/project/fedctgad/model/losses.py → True
/content/fedctgad_x_v2/project/model/losses.py → False


In [ ]:
# ============================================================
# CELL 63A — ADD PROJECT ROOT TO PYTHON PATH
# ============================================================

import sys

PROJECT_ROOT = "/content/fedctgad_x_v2/project"

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)
print("Exists:", os.path.exists(PROJECT_ROOT))
print("Added to sys.path ✓")

Project root: /content/fedctgad_x_v2/project
Exists: True
Added to sys.path ✓


In [ ]:
# ============================================================
# CELL 63B — IMPORT PROJECT LOSS
# ============================================================

import inspect
import fedctgad.model.losses as losses

print("Imported:", losses.__file__)
print("\nAvailable names:")
print([x for x in dir(losses) if not x.startswith("_")])

Imported: /content/fedctgad_x_v2/project/fedctgad/model/losses.py

Available names:
['F', 'nt_xent_loss', 'torch']


In [ ]:
# ============================================================
# CELL 63C — INSPECT NT-XENT IMPLEMENTATION
# ============================================================

print(inspect.getsource(losses.nt_xent_loss))

def nt_xent_loss(z_i: torch.Tensor, z_j: torch.Tensor, temperature: float = 0.1) -> torch.Tensor:
    """
    z_i, z_j: (batch, proj_dim) — two views of the same batch of sequences.
    Returns scalar loss.
    """
    batch_size = z_i.size(0)
    z = torch.cat([z_i, z_j], dim=0)                      # (2B, D)
    z = F.normalize(z, dim=1)
    sim = torch.matmul(z, z.T) / temperature              # (2B, 2B) cosine sim / tau

    # mask out self-similarity on the diagonal
    mask = torch.eye(2 * batch_size, dtype=torch.bool, device=z.device)
    sim.masked_fill_(mask, float("-inf"))

    # positive pair indices: i <-> i+B and i+B <-> i
    pos_idx = torch.arange(2 * batch_size, device=z.device)
    pos_idx = (pos_idx + batch_size) % (2 * batch_size)

    log_prob = F.log_softmax(sim, dim=1)
    loss = -log_prob[torch.arange(2 * batch_size, device=z.device), pos_idx]
    return loss.mean()



In [ ]:
# ============================================================
# CELL 63D — CHECK FUNCTION SIGNATURE
# ============================================================

print(inspect.signature(losses.nt_xent_loss))

(z_i: torch.Tensor, z_j: torch.Tensor, temperature: float = 0.1) -> torch.Tensor


In [ ]:
# ============================================================
# CELL 64 — INITIALIZE CTGAD-X MODEL
# ============================================================

import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

encoder = CTGAD_Encoder(
    in_dim=11,
    hidden_dim=60,
    seq_len=5,
    struct_heads=4,
    temporal_heads=4
).to(device)

projection = ProjectionHead(
    in_dim=60,
    hidden_dim=60,
    out_dim=80
).to(device)

print("\nEncoder parameters:",
      sum(p.numel() for p in encoder.parameters()))

print("Projection parameters:",
      sum(p.numel() for p in projection.parameters()))

print("Total parameters:",
      sum(p.numel() for p in encoder.parameters()) +
      sum(p.numel() for p in projection.parameters()))

Device: cuda

Encoder parameters: 20820
Projection parameters: 8540
Total parameters: 29360


In [ ]:
# ============================================================
# CELL 64A — CORRECT TEMPORAL ENCODING WRAPPER
# ============================================================

def encode_temporal_batch(
    temporal_batch,
    encoder,
    projection=None
):
    """
    temporal_batch:
        List of PyG Batch objects, one per temporal timestep.

    The CTGAD_Encoder itself performs:
        GATv2
        → graph mean pooling
        → temporal attention

    All graph batches are moved to the same device as the model.
    """

    # Move every temporal graph batch to the model device
    model_device = next(encoder.parameters()).device

    temporal_batch = [
        data_t.to(model_device)
        for data_t in temporal_batch
    ]

    # Use the repository's actual forward()
    z = encoder(temporal_batch)

    if projection is not None:
        p = projection(z)
        return z, p

    return z

In [ ]:
# ============================================================
# CELL 64A — INSPECT ACTUAL CTGAD_ENCODER FORWARD
# ============================================================

import inspect

print(inspect.getsource(CTGAD_Encoder.forward))

    def forward(self, snapshot_sequence: list) -> torch.Tensor:
        """
        snapshot_sequence: list of length L of torch_geometric.data.Data,
        one Data object per time step, batched over the batch dimension
        (i.e. snapshot_sequence[t] is a Batch of B graphs at time t).
        Returns: (batch, hidden_dim) sequence embeddings.
        """
        snap_embeds = []
        for data_t in snapshot_sequence:
            h = self.structural(data_t.x, data_t.edge_index)     # (total_nodes_in_batch, hidden_dim)
            # mean-pool per graph in the batch -> (batch, hidden_dim)
            pooled = self._mean_pool_per_graph(h, data_t.batch)
            snap_embeds.append(pooled)
        seq = torch.stack(snap_embeds, dim=1)   # (batch, L, hidden_dim)
        return self.temporal(seq)



In [ ]:
# ============================================================
# CELL 64B — INSPECT MODEL STRUCTURE
# ============================================================

print(encoder)

CTGAD_Encoder(
  (structural): StructuralEncoder(
    (gat): GATv2Conv(11, 60, heads=4)
  )
  (temporal): TemporalEncoder(
    (attn): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=60, out_features=60, bias=True)
    )
    (norm): LayerNorm((60,), eps=1e-05, elementwise_affine=True)
  )
)


In [ ]:
# ============================================================
# CELL 64B — FINAL FORWARD PASS CHECK
# ============================================================

encoder.eval()
projection.eval()

with torch.no_grad():

    z, p = encode_temporal_batch(
        train_batches[0],
        encoder,
        projection
    )

print("Device:", device)
print("Encoder output shape:", tuple(z.shape))
print("Projection output shape:", tuple(p.shape))
print("Encoder output device:", z.device)
print("Projection output device:", p.device)
print("Encoder finite:", torch.isfinite(z).all().item())
print("Projection finite:", torch.isfinite(p).all().item())

assert z.shape == (16, 60)
assert p.shape == (16, 80)

assert z.device.type == "cuda"
assert p.device.type == "cuda"

assert torch.isfinite(z).all()
assert torch.isfinite(p).all()

print("\nFORWARD PASS PASSED ✓")

Device: cuda
Encoder output shape: (16, 60)
Projection output shape: (16, 80)
Encoder output device: cuda:0
Projection output device: cuda:0
Encoder finite: True
Projection finite: True

FORWARD PASS PASSED ✓


In [ ]:
# ============================================================
# CELL 64C — PROJECT NT-XENT SANITY CHECK
# ============================================================

from fedctgad.model.losses import nt_xent_loss

encoder.train()
projection.train()

z1, p1 = encode_temporal_batch(
    train_batches[0],
    encoder,
    projection
)

z2, p2 = encode_temporal_batch(
    train_batches[0],
    encoder,
    projection
)

loss_test = nt_xent_loss(
    p1,
    p2,
    temperature=0.1
)

print("Projection 1:", tuple(p1.shape))
print("Projection 2:", tuple(p2.shape))
print("NT-Xent loss:", loss_test.item())
print("Loss device:", loss_test.device)

assert loss_test.ndim == 0
assert torch.isfinite(loss_test)

print("\nNT-XENT SANITY CHECK PASSED ✓")

Projection 1: (16, 80)
Projection 2: (16, 80)
NT-Xent loss: 3.0764856338500977
Loss device: cuda:0

NT-XENT SANITY CHECK PASSED ✓


In [ ]:
# ============================================================
# CELL 65 — CTGAD-X TRAINING SETUP
# ============================================================

import torch
from fedctgad.model.losses import nt_xent_loss

# ------------------------------------------------------------
# Training hyperparameters
# ------------------------------------------------------------

EPOCHS = 100
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
TEMPERATURE = 0.1

# ------------------------------------------------------------
# Reinitialize model for a clean training run
# ------------------------------------------------------------

encoder = CTGAD_Encoder(
    in_dim=11,
    hidden_dim=60,
    seq_len=5,
    struct_heads=4,
    temporal_heads=4
).to(device)

projection = ProjectionHead(
    in_dim=60,
    hidden_dim=60,
    out_dim=80
).to(device)

# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer = torch.optim.Adam(
    list(encoder.parameters()) +
    list(projection.parameters()),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print("Device:", device)
print("Epochs:", EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Temperature:", TEMPERATURE)

print("\nEncoder parameters:",
      sum(p.numel() for p in encoder.parameters()))

print("Projection parameters:",
      sum(p.numel() for p in projection.parameters()))

print("Total parameters:",
      sum(p.numel() for p in encoder.parameters()) +
      sum(p.numel() for p in projection.parameters()))

print("\nTraining setup ready ✓")

Device: cuda
Epochs: 100
Learning rate: 0.001
Weight decay: 1e-05
Temperature: 0.1

Encoder parameters: 20820
Projection parameters: 8540
Total parameters: 29360

Training setup ready ✓


In [ ]:
# ============================================================
# CELL 66 — CTGAD-X CONTRASTIVE TRAINING
# ============================================================

import copy
import time

best_val_loss = float("inf")
best_epoch = -1

best_encoder_state = None
best_projection_state = None

train_loss_history = []
val_loss_history = []

print("=" * 70)
print("STARTING CTGAD-X TRAINING")
print("=" * 70)

for epoch in range(1, EPOCHS + 1):

    epoch_start = time.time()

    # ========================================================
    # TRAIN
    # ========================================================

    encoder.train()
    projection.train()

    total_train_loss = 0.0
    train_batches_count = 0

    for temporal_batch in train_batches:

        optimizer.zero_grad()

        # ----------------------------------------------------
        # View 1
        # ----------------------------------------------------

        _, p1 = encode_temporal_batch(
            temporal_batch,
            encoder,
            projection
        )

        # ----------------------------------------------------
        # View 2
        # Independent stochastic forward pass
        # ----------------------------------------------------

        _, p2 = encode_temporal_batch(
            temporal_batch,
            encoder,
            projection
        )

        # ----------------------------------------------------
        # NT-Xent
        # ----------------------------------------------------

        loss = nt_xent_loss(
            p1,
            p2,
            temperature=TEMPERATURE
        )

        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(
            list(encoder.parameters()) +
            list(projection.parameters()),
            max_norm=1.0
        )

        optimizer.step()

        total_train_loss += loss.item()
        train_batches_count += 1

    avg_train_loss = (
        total_train_loss /
        train_batches_count
    )

    # ========================================================
    # VALIDATION
    # ========================================================

    encoder.eval()
    projection.eval()

    total_val_loss = 0.0
    val_batches_count = 0

    with torch.no_grad():

        for temporal_batch in val_batches:

            _, p1 = encode_temporal_batch(
                temporal_batch,
                encoder,
                projection
            )

            _, p2 = encode_temporal_batch(
                temporal_batch,
                encoder,
                projection
            )

            val_loss = nt_xent_loss(
                p1,
                p2,
                temperature=TEMPERATURE
            )

            total_val_loss += val_loss.item()
            val_batches_count += 1

    avg_val_loss = (
        total_val_loss /
        val_batches_count
    )

    # ========================================================
    # RECORD
    # ========================================================

    train_loss_history.append(avg_train_loss)
    val_loss_history.append(avg_val_loss)

    epoch_time = time.time() - epoch_start

    # ========================================================
    # BEST MODEL CHECKPOINT
    # ========================================================

    if avg_val_loss < best_val_loss:

        best_val_loss = avg_val_loss
        best_epoch = epoch

        best_encoder_state = copy.deepcopy(
            encoder.state_dict()
        )

        best_projection_state = copy.deepcopy(
            projection.state_dict()
        )

        marker = " ← BEST"

    else:

        marker = ""

    # ========================================================
    # LOG
    # ========================================================

    print(
        f"Epoch {epoch:03d}/{EPOCHS} | "
        f"Train Loss: {avg_train_loss:.6f} | "
        f"Val Loss: {avg_val_loss:.6f} | "
        f"Time: {epoch_time:.2f}s"
        f"{marker}"
    )

print("\n" + "=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)

print("Best epoch:", best_epoch)
print("Best validation loss:", best_val_loss)

STARTING CTGAD-X TRAINING
Epoch 001/100 | Train Loss: 2.717554 | Val Loss: 1.588600 | Time: 0.86s ← BEST
Epoch 002/100 | Train Loss: 2.230781 | Val Loss: 1.415903 | Time: 0.35s ← BEST
Epoch 003/100 | Train Loss: 1.950707 | Val Loss: 1.140034 | Time: 0.34s ← BEST
Epoch 004/100 | Train Loss: 1.934600 | Val Loss: 1.004429 | Time: 0.33s ← BEST
Epoch 005/100 | Train Loss: 1.918548 | Val Loss: 1.034636 | Time: 0.34s
Epoch 006/100 | Train Loss: 1.862935 | Val Loss: 1.043552 | Time: 0.32s
Epoch 007/100 | Train Loss: 1.736984 | Val Loss: 1.056045 | Time: 0.32s
Epoch 008/100 | Train Loss: 1.698887 | Val Loss: 0.912244 | Time: 0.34s ← BEST
Epoch 009/100 | Train Loss: 1.769155 | Val Loss: 0.979962 | Time: 0.33s
Epoch 010/100 | Train Loss: 1.690557 | Val Loss: 0.901054 | Time: 0.33s ← BEST
Epoch 011/100 | Train Loss: 1.664584 | Val Loss: 0.928672 | Time: 0.34s
Epoch 012/100 | Train Loss: 1.641190 | Val Loss: 0.862298 | Time: 0.33s ← BEST
Epoch 013/100 | Train Loss: 1.501276 | Val Loss: 0.818065 | T

In [ ]:
# ============================================================
# CELL 67 — RESTORE BEST MODEL
# ============================================================

assert best_encoder_state is not None
assert best_projection_state is not None

encoder.load_state_dict(
    best_encoder_state
)

projection.load_state_dict(
    best_projection_state
)

encoder.eval()
projection.eval()

print("Best encoder restored ✓")
print("Best projection restored ✓")
print("Best epoch:", best_epoch)
print("Best validation loss:", best_val_loss)

Best encoder restored ✓
Best projection restored ✓
Best epoch: 100
Best validation loss: 0.34026187658309937


In [ ]:
# ============================================================
# CELL 67 — RESTORE BEST CTGAD-X CHECKPOINT
# ============================================================

assert best_encoder_state is not None
assert best_projection_state is not None

encoder.load_state_dict(best_encoder_state)
projection.load_state_dict(best_projection_state)

encoder.eval()
projection.eval()

print("=" * 60)
print("BEST CHECKPOINT RESTORED")
print("=" * 60)

print("Best epoch:", best_epoch)
print("Best validation loss:", best_val_loss)

print(
    "Encoder parameters:",
    sum(p.numel() for p in encoder.parameters())
)

print(
    "Projection parameters:",
    sum(p.numel() for p in projection.parameters())
)

BEST CHECKPOINT RESTORED
Best epoch: 100
Best validation loss: 0.34026187658309937
Encoder parameters: 20820
Projection parameters: 8540


In [ ]:
# ============================================================
# CELL 68 — FREEZE CTGAD-X ENCODER
# ============================================================

encoder.eval()
projection.eval()

for param in encoder.parameters():
    param.requires_grad = False

for param in projection.parameters():
    param.requires_grad = False


@torch.no_grad()
def get_sequence_embedding(temporal_batch):
    """
    Obtain the frozen 60-dimensional CTGAD-X encoder
    representation for each temporal sequence.
    """

    z = encode_temporal_batch(
        temporal_batch,
        encoder,
        projection=None
    )

    return z


print("Encoder frozen ✓")
print("Projection frozen ✓")

Encoder frozen ✓
Projection frozen ✓


In [ ]:
# ============================================================
# CELL 69 — BUILD NORMAL REFERENCE EMBEDDING BANK
# ============================================================

@torch.no_grad()
def extract_embeddings(batches):

    all_embeddings = []

    for temporal_batch in batches:

        z = get_sequence_embedding(
            temporal_batch
        )

        all_embeddings.append(
            z.detach().cpu()
        )

    return torch.cat(
        all_embeddings,
        dim=0
    )


train_embeddings = extract_embeddings(
    train_batches
)

val_embeddings = extract_embeddings(
    val_batches
)

test_embeddings = extract_embeddings(
    test_batches
)

print("Train embeddings:",
      tuple(train_embeddings.shape))

print("Validation embeddings:",
      tuple(val_embeddings.shape))

print("Test embeddings:",
      tuple(test_embeddings.shape))

Train embeddings: (174, 60)
Validation embeddings: (22, 60)
Test embeddings: (47, 60)


In [ ]:
# ============================================================
# CELL 70 — EMBEDDING SANITY CHECK
# ============================================================

print("Train finite:",
      torch.isfinite(train_embeddings).all().item())

print("Validation finite:",
      torch.isfinite(val_embeddings).all().item())

print("Test finite:",
      torch.isfinite(test_embeddings).all().item())

print("\nEmbedding statistics")
print("-" * 50)

print(
    "Train mean:",
    train_embeddings.mean().item()
)

print(
    "Train std:",
    train_embeddings.std().item()
)

print(
    "Train min:",
    train_embeddings.min().item()
)

print(
    "Train max:",
    train_embeddings.max().item()
)

assert train_embeddings.shape == (174, 60)
assert val_embeddings.shape == (22, 60)
assert test_embeddings.shape == (47, 60)

assert torch.isfinite(train_embeddings).all()
assert torch.isfinite(val_embeddings).all()
assert torch.isfinite(test_embeddings).all()

print("\nEmbedding extraction passed ✓")

Train finite: True
Validation finite: True
Test finite: True

Embedding statistics
--------------------------------------------------
Train mean: 0.006143416278064251
Train std: 0.9944024085998535
Train min: -3.5145139694213867
Train max: 3.946021556854248

Embedding extraction passed ✓


In [ ]:
# ============================================================
# CELL 71 — K-NN COSINE ANOMALY SCORE
# ============================================================

import torch
import torch.nn.functional as F


def knn_cosine_scores(
    query_embeddings,
    reference_embeddings,
    k=5
):
    """
    Compute k-NN cosine-distance anomaly scores.

    Lower score  -> more similar to normal reference
    Higher score -> more anomalous

    Parameters
    ----------
    query_embeddings:
        [N, D]

    reference_embeddings:
        [M, D]

    k:
        Number of nearest normal neighbours.
    """

    # Normalize embeddings
    query = F.normalize(
        query_embeddings,
        dim=1
    )

    reference = F.normalize(
        reference_embeddings,
        dim=1
    )

    # Cosine similarity
    similarity = torch.matmul(
        query,
        reference.T
    )

    # Convert similarity → cosine distance
    distance = 1.0 - similarity

    # Make sure k is valid
    k = min(k, reference.shape[0])

    # k smallest distances = nearest normal neighbours
    nearest_distances, _ = torch.topk(
        distance,
        k=k,
        dim=1,
        largest=False
    )

    # Mean distance to k nearest neighbours
    scores = nearest_distances.mean(
        dim=1
    )

    return scores

In [ ]:
# ============================================================
# CELL 72 — CALCULATE K-NN SCORES
# ============================================================

K = 5

train_scores = knn_cosine_scores(
    train_embeddings,
    train_embeddings,
    k=K
)

val_scores = knn_cosine_scores(
    val_embeddings,
    train_embeddings,
    k=K
)

test_scores = knn_cosine_scores(
    test_embeddings,
    train_embeddings,
    k=K
)

print("K =", K)

print("\nTrain scores:")
print("Shape:", tuple(train_scores.shape))
print("Min:", train_scores.min().item())
print("Max:", train_scores.max().item())
print("Mean:", train_scores.mean().item())
print("Std:", train_scores.std().item())

print("\nValidation scores:")
print("Shape:", tuple(val_scores.shape))
print("Min:", val_scores.min().item())
print("Max:", val_scores.max().item())
print("Mean:", val_scores.mean().item())
print("Std:", val_scores.std().item())

print("\nTest scores:")
print("Shape:", tuple(test_scores.shape))
print("Min:", test_scores.min().item())
print("Max:", test_scores.max().item())
print("Mean:", test_scores.mean().item())
print("Std:", test_scores.std().item())

K = 5

Train scores:
Shape: (174,)
Min: 9.447336196899414e-05
Max: 0.3727816641330719
Mean: 0.0395539253950119
Std: 0.043412771075963974

Validation scores:
Shape: (22,)
Min: 0.007950377650558949
Max: 0.24310605227947235
Mean: 0.09089413285255432
Std: 0.06749209761619568

Test scores:
Shape: (47,)
Min: 0.00013117790513206273
Max: 0.12099634110927582
Mean: 0.043599776923656464
Std: 0.02607046440243721


In [ ]:
# ============================================================
# CELL 73 — K-NN SELF-NEIGHBOUR SANITY CHECK
# ============================================================

# Compute pairwise distances
train_norm = F.normalize(
    train_embeddings,
    dim=1
)

train_similarity = torch.matmul(
    train_norm,
    train_norm.T
)

train_distance = 1.0 - train_similarity

# Ignore self-distance
train_distance_no_self = train_distance.clone()

train_distance_no_self.fill_diagonal_(
    float("inf")
)

nearest_nonself, _ = torch.topk(
    train_distance_no_self,
    k=K,
    dim=1,
    largest=False
)

train_scores_no_self = nearest_nonself.mean(
    dim=1
)

print("Training score including self:")
print(train_scores.mean().item())

print("\nTraining score excluding self:")
print(train_scores_no_self.mean().item())

print("\nNearest-neighbour check passed ✓")

Training score including self:
0.0395539253950119

Training score excluding self:
0.053992271423339844

Nearest-neighbour check passed ✓


In [ ]:
# ============================================================
# CELL 75 — Inspect snapshot metadata + attack labels
# ============================================================

print("Total snapshots:", len(snapshots))

# Basic snapshot structure
s0 = snapshots[0]

print("\nGraphSnapshot fields:")
print(s0.__dict__.keys())

print("\nSnapshot 0:")
print("window_idx:", s0.window_idx)
print("t_start:", s0.t_start)
print("t_end:", s0.t_end)
print("num_nodes:", len(s0.node_ids))
print("num_edges:", len(s0.edges))
print("feature_dim:", len(next(iter(s0.node_features.values()))))

# ------------------------------------------------------------
# Check variables already present in the notebook that may
# contain window-level labels / attack information.
# ------------------------------------------------------------

candidate_names = [
    "window_labels",
    "global_window_labels",
    "window_attack_labels",
    "attack_labels",
    "snapshot_labels",
    "labels",
    "window_label_df",
    "window_stats",
    "window_summary",
    "window_attack_stats"
]

print("\nPossible existing label variables:")
for name in candidate_names:
    if name in globals():
        obj = globals()[name]
        print(f"\n{name}:")
        print("  type =", type(obj))
        try:
            print("  shape/len =", obj.shape if hasattr(obj, "shape") else len(obj))
        except Exception:
            pass
        print("  preview =", obj[:10] if hasattr(obj, "__getitem__") else obj)

# ------------------------------------------------------------
# Check dataframe columns for attack/category information
# ------------------------------------------------------------

print("\nDataframe columns:")
for i, col in enumerate(df.columns):
    print(i, repr(col))

# Show label-related columns specifically
label_cols = [
    c for c in df.columns
    if any(k in c.lower() for k in ["label", "attack", "category", "class"])
]

print("\nLabel/attack-related columns:")
print(label_cols)

if label_cols:
    print("\nUnique values:")
    for col in label_cols:
        print(f"\n{col}:")
        print(df[col].value_counts(dropna=False).head(30))

Total snapshots: 255

GraphSnapshot fields:
dict_keys(['window_idx', 't_start', 't_end', 'node_ids', 'edges', 'node_features'])

Snapshot 0:
window_idx: 0
t_start: 1421927377.0
t_end: 1421927407.0
num_nodes: 3
num_edges: 2
feature_dim: 11

Possible existing label variables:

Dataframe columns:
0 'src_ip'
1 'sport'
2 'dst_ip'
3 'dsport'
4 'proto'
5 'state'
6 'duration'
7 'sbytes'
8 'dbytes'
9 'sttl'
10 'dttl'
11 'sloss'
12 'dloss'
13 'service'
14 'sload'
15 'dload'
16 'spkts'
17 'dpkts'
18 'swin'
19 'dwin'
20 'stcpb'
21 'dtcpb'
22 'smeansz'
23 'dmeansz'
24 'trans_depth'
25 'res_bdy_len'
26 'sjit'
27 'djit'
28 'ts_start'
29 'ltime'
30 'sintpkt'
31 'dintpkt'
32 'tcprtt'
33 'synack'
34 'ackdat'
35 'is_sm_ips_ports'
36 'ct_state_ttl'
37 'ct_flw_http_mthd'
38 'is_ftp_login'
39 'ct_ftp_cmd'
40 'ct_srv_src'
41 'ct_srv_dst'
42 'ct_dst_ltm'
43 'ct_src_ltm'
44 'ct_src_dport_ltm'
45 'ct_dst_sport_ltm'
46 'ct_dst_src_ltm'
47 'attack_cat'
48 'label'
49 't_rel'
50 'window_start_idx'
51 'window_end_id

In [ ]:
# ============================================================
# CELL 76 — Build exact labels for the 255 temporal windows
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Verify window assignments in the dataframe
# ------------------------------------------------------------

assert "window_start_idx" in df.columns
assert "window_end_idx" in df.columns
assert "label" in df.columns
assert "attack_cat" in df.columns

# The slicer created 255 global windows.
NUM_WINDOWS = len(snapshots)

# ------------------------------------------------------------
# 2. Assign every flow to its temporal window(s)
#
# The dataframe already contains the window boundaries
# generated during the temporal preprocessing.
# ------------------------------------------------------------

window_label_rows = []

for w in range(NUM_WINDOWS):

    mask = (
        (df["window_start_idx"] <= w) &
        (df["window_end_idx"] >= w)
    )

    wdf = df.loc[mask]

    if len(wdf) == 0:
        window_label_rows.append({
            "window_idx": w,
            "n_flows": 0,
            "n_attack": 0,
            "attack_fraction": 0.0,
            "is_attack": 0,
            "attack_categories": ""
        })
        continue

    attack_count = int((wdf["label"] == 1).sum())
    total_count = len(wdf)

    cats = (
        wdf.loc[wdf["label"] == 1, "attack_cat"]
        .astype(str)
        .value_counts()
        .to_dict()
    )

    window_label_rows.append({
        "window_idx": w,
        "n_flows": total_count,
        "n_attack": attack_count,
        "attack_fraction": attack_count / total_count,
        "is_attack": int(attack_count > 0),
        "attack_categories": ", ".join(
            f"{k}:{v}" for k, v in cats.items()
        )
    })

window_label_df = pd.DataFrame(window_label_rows)

# ------------------------------------------------------------
# 3. Sanity checks
# ------------------------------------------------------------

print("Window label table shape:", window_label_df.shape)

print("\nFirst 10 windows:")
display(window_label_df.head(10))

print("\nAttack-bearing windows:",
      window_label_df["is_attack"].sum(),
      "/",
      NUM_WINDOWS)

print("\nNormal-only windows:",
      (window_label_df["is_attack"] == 0).sum())

print("\nAttack fraction statistics:")
print(
    window_label_df["attack_fraction"].describe()
)

print("\nMost attack-heavy windows:")
display(
    window_label_df
    .sort_values("attack_fraction", ascending=False)
    .head(15)
)

# ------------------------------------------------------------
# 4. Verify that snapshot indices match dataframe windows
# ------------------------------------------------------------

snapshot_window_ids = [s.window_idx for s in snapshots]

assert snapshot_window_ids == list(range(NUM_WINDOWS)), \
    "Snapshot window indices are not contiguous 0..NUM_WINDOWS-1"

print("\n✓ Snapshot/window alignment verified")

Window label table shape: (255, 6)

First 10 windows:


,window_idx,n_flows,n_attack,attack_fraction,is_attack,attack_categories
0,0,6,0,0.000000,0,
1,1,438,26,0.059361,1,"Exploits:17, Reconnaissance:5, DoS:3, Generic:1"
2,2,1839,32,0.017401,1,"Exploits:18, Reconnaissance:8, Generic:2, DoS:..."
3,3,748,30,0.040107,1,"Exploits:18, Reconnaissance:6, Generic:3, Shel..."
4,4,687,31,0.045124,1,"Exploits:18, Reconnaissance:8, DoS:2, Generic:..."
5,5,691,35,0.050651,1,"Exploits:19, Reconnaissance:6, DoS:5, Shellcod..."
6,6,710,31,0.043662,1,"Exploits:17, Reconnaissance:8, DoS:3, Generic:3"
7,7,670,41,0.061194,1,"Exploits:27, Reconnaissance:7, DoS:4, Fuzzers:..."
8,8,703,54,0.076814,1,"Exploits:32, Reconnaissance:7, DoS:6, Generic:..."
9,9,717,95,0.132497,1,"Fuzzers:64, Exploits:20, Reconnaissance:7, She..."



Attack-bearing windows: 235 / 255

Normal-only windows: 20

Attack fraction statistics:
count    255.000000
mean       0.101940
std        0.076815
min        0.000000
25%        0.052523
50%        0.093872
75%        0.135178
max        0.571028
Name: attack_fraction, dtype: float64

Most attack-heavy windows:


,window_idx,n_flows,n_attack,attack_fraction,is_attack,attack_categories
111,111,1781,1017,0.571028,1,"Fuzzers:273, Exploits:253, DoS:120, Backdoors:..."
110,110,1532,857,0.559399,1,"Fuzzers:272, Exploits:229, DoS:113, Backdoors:..."
112,112,1566,817,0.521711,1,"Fuzzers:209, Exploits:203, Generic:102, DoS:10..."
113,113,1266,525,0.414692,1,"Fuzzers:138, Exploits:124, Generic:89, DoS:57,..."
109,109,1092,411,0.376374,1,"Fuzzers:159, Exploits:108, DoS:46, Backdoors:4..."
114,114,1035,271,0.261836,1,"Generic:103, Fuzzers:68, Exploits:49, DoS:15, ..."
120,120,1004,247,0.246016,1,"Generic:154, Fuzzers:53, DoS:16, Exploits:14, ..."
108,108,879,195,0.221843,1,"Exploits:59, Fuzzers:44, DoS:23, Analysis:22, ..."
158,158,917,197,0.214831,1,"Generic:141, Exploits:26, Fuzzers:18, Reconnai..."
166,166,922,192,0.208243,1,"Generic:115, Fuzzers:51, Exploits:23, Worms:1,..."



✓ Snapshot/window alignment verified


In [ ]:
# ============================================================
# CELL 77 — Inspect existing train/val/test sequence objects
# ============================================================

candidate_names = [
    "train_graph_sequences",
    "val_graph_sequences",
    "test_graph_sequences",
    "train_sequences",
    "val_sequences",
    "test_sequences",
    "train_batches",
    "val_batches",
    "test_batches",
]

print("Existing sequence/batch variables:\n")

for name in candidate_names:
    if name in globals():
        obj = globals()[name]

        print(f"{name}")
        print("  type:", type(obj))

        try:
            print("  length:", len(obj))
        except Exception:
            print("  length: unavailable")

        # Inspect first element
        try:
            first = obj[0]
            print("  first element type:", type(first))
        except Exception:
            pass

        print()

Existing sequence/batch variables:

train_graph_sequences
  type: <class 'list'>
  length: 174
  first element type: <class 'list'>

val_graph_sequences
  type: <class 'list'>
  length: 22
  first element type: <class 'list'>

test_graph_sequences
  type: <class 'list'>
  length: 47
  first element type: <class 'list'>

train_sequences
  type: <class 'list'>
  length: 174
  first element type: <class 'list'>

val_sequences
  type: <class 'list'>
  length: 22
  first element type: <class 'list'>

test_sequences
  type: <class 'list'>
  length: 47
  first element type: <class 'list'>

train_batches
  type: <class 'list'>
  length: 11
  first element type: <class 'list'>

val_batches
  type: <class 'list'>
  length: 2
  first element type: <class 'list'>

test_batches
  type: <class 'list'>
  length: 3
  first element type: <class 'list'>



In [ ]:
# ============================================================
# CELL 78 — Find existing temporal split variables
# ============================================================

print("Variables related to split / partition / window:\n")

keywords = [
    "split",
    "train",
    "val",
    "test",
    "window",
    "normal"
]

for name in sorted(globals().keys()):
    lname = name.lower()

    if any(k in lname for k in keywords):
        # Skip very large/unhelpful objects
        if name.startswith("_"):
            continue

        obj = globals()[name]

        try:
            obj_len = len(obj)
        except Exception:
            obj_len = "N/A"

        print(
            f"{name:40s} | "
            f"type={type(obj).__name__:20s} | "
            f"len={obj_len}"
        )

Variables related to split / partition / window:

NORMAL_MAX                               | type=float                | len=N/A
NUM_WINDOWS                              | type=int                  | len=N/A
avg_train_loss                           | type=float                | len=N/A
avg_val_loss                             | type=float                | len=N/A
bad_normal                               | type=int                  | len=N/A
best_val_loss                            | type=float                | len=N/A
cut_train                                | type=int                  | len=N/A
cut_val                                  | type=int                  | len=N/A
loss_test                                | type=Tensor               | len=N/A
make_sequences_from_window_ids           | type=function             | len=N/A
n_normal                                 | type=int                  | len=N/A
n_train                                  | type=int                  | len=N/A
n_

In [ ]:
# ============================================================
# CELL 79 — Verify the ORIGINAL temporal split
# ============================================================

print("TRAIN")
print("  Number of windows:", len(train_snapshot_idx))
print("  Range:", min(train_snapshot_idx), "to", max(train_snapshot_idx))
print("  First 10:", train_snapshot_idx[:10])
print("  Last 10 :", train_snapshot_idx[-10:])

print("\nVALIDATION")
print("  Number of windows:", len(val_snapshot_idx))
print("  Range:", min(val_snapshot_idx), "to", max(val_snapshot_idx))
print("  First 10:", val_snapshot_idx[:10])
print("  Last 10 :", val_snapshot_idx[-10:])

print("\nTEST")
print("  Number of windows:", len(test_snapshot_idx))
print("  Range:", min(test_snapshot_idx), "to", max(test_snapshot_idx))
print("  First 10:", test_snapshot_idx[:10])
print("  Last 10 :", test_snapshot_idx[-10:])

# ------------------------------------------------------------
# Convert to sets for leakage checks
# ------------------------------------------------------------

train_w = set(train_snapshot_idx)
val_w   = set(val_snapshot_idx)
test_w  = set(test_snapshot_idx)

print("\nOverlap:")
print("Train ∩ Val :", len(train_w & val_w))
print("Train ∩ Test:", len(train_w & test_w))
print("Val ∩ Test  :", len(val_w & test_w))

print("\nChronological:")
print("Train < Val :", max(train_w) < min(val_w))
print("Val < Test  :", max(val_w) < min(test_w))

# ------------------------------------------------------------
# Verify against the existing window-id sets
# ------------------------------------------------------------

assert train_w == train_window_ids
assert val_w == val_window_ids
assert test_w == test_window_ids

# ------------------------------------------------------------
# Verify all 255 windows are accounted for
# ------------------------------------------------------------

all_partition_windows = train_w | val_w | test_w

print("\nTotal unique partition windows:", len(all_partition_windows))

assert len(all_partition_windows) == NUM_WINDOWS
assert all_partition_windows == set(range(NUM_WINDOWS))

print("\n✓ Original temporal partition verified")

TRAIN
  Number of windows: 178
  Range: 0 to 177
  First 10: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
  Last 10 : [168, 169, 170, 171, 172, 173, 174, 175, 176, 177]

VALIDATION
  Number of windows: 26
  Range: 178 to 203
  First 10: [178, 179, 180, 181, 182, 183, 184, 185, 186, 187]
  Last 10 : [194, 195, 196, 197, 198, 199, 200, 201, 202, 203]

TEST
  Number of windows: 51
  Range: 204 to 254
  First 10: [204, 205, 206, 207, 208, 209, 210, 211, 212, 213]
  Last 10 : [245, 246, 247, 248, 249, 250, 251, 252, 253, 254]

Overlap:
Train ∩ Val : 0
Train ∩ Test: 0
Val ∩ Test  : 0

Chronological:
Train < Val : True
Val < Test  : True

Total unique partition windows: 255

✓ Original temporal partition verified


In [ ]:
# ============================================================
# CELL 80 — Inspect sequence-construction function
# ============================================================

import inspect

print(inspect.signature(make_sequences_from_window_ids))

print("\nFunction source:\n")
print(inspect.getsource(make_sequences_from_window_ids))

(window_ids, seq_len)

Function source:

def make_sequences_from_window_ids(
    window_ids,
    seq_len
):

    ids = sorted(window_ids)

    result = []

    for i in range(
        len(ids) - seq_len + 1
    ):

        candidate = ids[
            i:i + seq_len
        ]

        if any(
            candidate[j + 1]
            != candidate[j] + 1
            for j in range(seq_len - 1)
        ):
            continue

        result.append([
            normal_lookup[w]
            for w in candidate
        ])

    return result



In [ ]:
# ============================================================
# CELL 81 — All-traffic temporal sequence constructor
# ============================================================

def make_all_traffic_sequences_from_window_ids(
    window_ids,
    seq_len,
    snapshot_lookup
):
    """
    Construct contiguous temporal graph sequences from the
    supplied all-traffic snapshot lookup.

    This mirrors make_sequences_from_window_ids(), but does
    NOT use normal_lookup.
    """

    ids = sorted(window_ids)

    result = []

    for i in range(len(ids) - seq_len + 1):

        candidate = ids[i:i + seq_len]

        # Require strictly contiguous temporal windows
        if any(
            candidate[j + 1] != candidate[j] + 1
            for j in range(seq_len - 1)
        ):
            continue

        result.append([
            snapshot_lookup[w]
            for w in candidate
        ])

    return result


# ------------------------------------------------------------
# Build lookup for ALL-TRAFFIC snapshots
# ------------------------------------------------------------

all_traffic_lookup = {
    snapshot.window_idx: snapshot
    for snapshot in snapshots
}

# ------------------------------------------------------------
# Basic validation
# ------------------------------------------------------------

assert len(all_traffic_lookup) == NUM_WINDOWS

assert set(all_traffic_lookup.keys()) == set(range(NUM_WINDOWS))

print("All-traffic lookup:")
print("  Number of snapshots:", len(all_traffic_lookup))
print("  Window range:",
      min(all_traffic_lookup.keys()),
      "to",
      max(all_traffic_lookup.keys()))

print("\n✓ All-traffic snapshot lookup verified")

All-traffic lookup:
  Number of snapshots: 255
  Window range: 0 to 254

✓ All-traffic snapshot lookup verified


In [ ]:
# ============================================================
# CELL 82 — Construct ALL-TRAFFIC temporal sequences
# ============================================================

SEQ_LEN = 5

all_train_sequences = make_all_traffic_sequences_from_window_ids(
    train_snapshot_idx,
    SEQ_LEN,
    all_traffic_lookup
)

all_val_sequences = make_all_traffic_sequences_from_window_ids(
    val_snapshot_idx,
    SEQ_LEN,
    all_traffic_lookup
)

all_test_sequences = make_all_traffic_sequences_from_window_ids(
    test_snapshot_idx,
    SEQ_LEN,
    all_traffic_lookup
)

print("ALL-TRAFFIC SEQUENCES")
print("---------------------")
print("Train:", len(all_train_sequences))
print("Val  :", len(all_val_sequences))
print("Test :", len(all_test_sequences))

# ------------------------------------------------------------
# Expected number
# ------------------------------------------------------------

print("\nExpected:")
print("Train:", len(train_snapshot_idx) - SEQ_LEN + 1)
print("Val  :", len(val_snapshot_idx) - SEQ_LEN + 1)
print("Test :", len(test_snapshot_idx) - SEQ_LEN + 1)

# ------------------------------------------------------------
# Inspect first sequences
# ------------------------------------------------------------

print("\nFirst train sequence windows:")
print([
    s.window_idx
    for s in all_train_sequences[0]
])

print("\nFirst validation sequence windows:")
print([
    s.window_idx
    for s in all_val_sequences[0]
])

print("\nFirst test sequence windows:")
print([
    s.window_idx
    for s in all_test_sequences[0]
])

# ------------------------------------------------------------
# Verify all sequences are contiguous
# ------------------------------------------------------------

def verify_sequence_continuity(sequences):

    for seq in sequences:

        ids = [s.window_idx for s in seq]

        if ids != list(range(ids[0], ids[0] + len(ids))):
            return False

    return True


assert verify_sequence_continuity(all_train_sequences)
assert verify_sequence_continuity(all_val_sequences)
assert verify_sequence_continuity(all_test_sequences)

print("\n✓ All-traffic sequences are contiguous")

ALL-TRAFFIC SEQUENCES
---------------------
Train: 174
Val  : 22
Test : 47

Expected:
Train: 174
Val  : 22
Test : 47

First train sequence windows:
[0, 1, 2, 3, 4]

First validation sequence windows:
[178, 179, 180, 181, 182]

First test sequence windows:
[204, 205, 206, 207, 208]

✓ All-traffic sequences are contiguous


In [ ]:
# ============================================================
# CELL 83 — Inspect snapshot_to_pyg and scaler
# ============================================================

import inspect

print("snapshot_to_pyg signature:")
print(inspect.signature(snapshot_to_pyg))

print("\nExisting scaler-related variables:")

for name in sorted(globals().keys()):
    if "scaler" in name.lower():
        obj = globals()[name]
        print(
            f"{name:30s} | "
            f"type={type(obj).__name__}"
        )

print("\nKnown vector/scaling variables:")
for name in [
    "train_vectors",
    "val_vectors",
    "test_vectors",
    "train_scaled",
    "val_scaled",
    "test_scaled"
]:
    if name in globals():
        obj = globals()[name]
        print(
            f"{name:20s} | "
            f"type={type(obj).__name__} | "
            f"shape={getattr(obj, 'shape', 'N/A')}"
        )

snapshot_to_pyg signature:
(snapshot, scaler)

Existing scaler-related variables:
StandardScaler                 | type=type
scaler                         | type=StandardScaler

Known vector/scaling variables:
train_vectors        | type=ndarray | shape=(24845, 11)
val_vectors          | type=ndarray | shape=(3405, 11)
test_vectors         | type=ndarray | shape=(6265, 11)
train_scaled         | type=ndarray | shape=(24845, 11)


In [ ]:
# ============================================================
# CELL 84 — Convert ALL-TRAFFIC sequences to PyG
#             using the TRAINING-FITTED scaler
# ============================================================

def convert_temporal_sequences_to_pyg(
    graph_sequences,
    scaler
):
    """
    Convert temporal GraphSnapshot sequences to PyG Data.

    IMPORTANT:
    The supplied scaler must already be fitted on TRAINING data.
    """

    return [
        [
            snapshot_to_pyg(snapshot, scaler)
            for snapshot in sequence
        ]
        for sequence in graph_sequences
    ]


# ------------------------------------------------------------
# Convert using the SAME scaler used by the trained model
# ------------------------------------------------------------

all_train_graph_sequences = convert_temporal_sequences_to_pyg(
    all_train_sequences,
    scaler
)

all_val_graph_sequences = convert_temporal_sequences_to_pyg(
    all_val_sequences,
    scaler
)

all_test_graph_sequences = convert_temporal_sequences_to_pyg(
    all_test_sequences,
    scaler
)

# ------------------------------------------------------------
# Verify counts
# ------------------------------------------------------------

print("ALL-TRAFFIC PyG SEQUENCES")
print("-------------------------")
print("Train:", len(all_train_graph_sequences))
print("Val  :", len(all_val_graph_sequences))
print("Test :", len(all_test_graph_sequences))

assert len(all_train_graph_sequences) == 174
assert len(all_val_graph_sequences) == 22
assert len(all_test_graph_sequences) == 47

# ------------------------------------------------------------
# Verify temporal length
# ------------------------------------------------------------

assert all(
    len(seq) == SEQ_LEN
    for seq in all_train_graph_sequences
)

assert all(
    len(seq) == SEQ_LEN
    for seq in all_val_graph_sequences
)

assert all(
    len(seq) == SEQ_LEN
    for seq in all_test_graph_sequences
)

# ------------------------------------------------------------
# Inspect first ALL-TRAFFIC test sequence
# ------------------------------------------------------------

print("\nFirst all-traffic TEST sequence:")

for t, graph in enumerate(all_test_graph_sequences[0]):

    print(
        f"t={t}: "
        f"nodes={graph.num_nodes}, "
        f"edges={graph.num_edges}, "
        f"x_shape={tuple(graph.x.shape)}, "
        f"edge_index_shape={tuple(graph.edge_index.shape)}"
    )

# ------------------------------------------------------------
# Feature sanity checks
# ------------------------------------------------------------

assert all(
    graph.x.shape[1] == 11
    for seq in all_test_graph_sequences
    for graph in seq
)

assert all(
    torch.isfinite(graph.x).all()
    for seq in all_test_graph_sequences
    for graph in seq
)

print("\n✓ All-traffic PyG conversion passed")
print("✓ Training scaler reused")
print("✓ Feature dimension = 11")
print("✓ All graph features finite")

ALL-TRAFFIC PyG SEQUENCES
-------------------------
Train: 174
Val  : 22
Test : 47

First all-traffic TEST sequence:
t=0: nodes=39, edges=133, x_shape=(39, 11), edge_index_shape=(2, 133)
t=1: nodes=38, edges=127, x_shape=(38, 11), edge_index_shape=(2, 127)
t=2: nodes=39, edges=134, x_shape=(39, 11), edge_index_shape=(2, 134)
t=3: nodes=39, edges=133, x_shape=(39, 11), edge_index_shape=(2, 133)
t=4: nodes=39, edges=126, x_shape=(39, 11), edge_index_shape=(2, 126)

✓ All-traffic PyG conversion passed
✓ Training scaler reused
✓ Feature dimension = 11
✓ All graph features finite


In [ ]:
# ============================================================
# CELL 85 — Batch ALL-TRAFFIC temporal sequences
# ============================================================

all_train_batches = batch_temporal_sequences(
    all_train_graph_sequences,
    batch_size=16
)

all_val_batches = batch_temporal_sequences(
    all_val_graph_sequences,
    batch_size=16
)

all_test_batches = batch_temporal_sequences(
    all_test_graph_sequences,
    batch_size=16
)

print("ALL-TRAFFIC BATCHES")
print("-------------------")
print("Train:", len(all_train_batches))
print("Val  :", len(all_val_batches))
print("Test :", len(all_test_batches))

# ------------------------------------------------------------
# Inspect first test batch
# ------------------------------------------------------------

print("\nFirst ALL-TRAFFIC TEST batch:")

for t, batch in enumerate(all_test_batches[0]):

    print(
        f"t={t}: "
        f"graphs={batch.num_graphs}, "
        f"nodes={batch.num_nodes}, "
        f"edges={batch.num_edges}, "
        f"x_shape={tuple(batch.x.shape)}"
    )

# ------------------------------------------------------------
# Structural checks
# ------------------------------------------------------------

assert all(
    len(batch_sequence) == SEQ_LEN
    for batch_sequence in all_test_batches
)

assert all(
    batch.num_graphs <= 16
    for batch_sequence in all_test_batches
    for batch in batch_sequence
)

print("\n✓ All-traffic batching passed")

ALL-TRAFFIC BATCHES
-------------------
Train: 11
Val  : 2
Test : 3

First ALL-TRAFFIC TEST batch:
t=0: graphs=16, nodes=618, edges=2044, x_shape=(618, 11)
t=1: graphs=16, nodes=618, edges=2036, x_shape=(618, 11)
t=2: graphs=16, nodes=619, edges=2031, x_shape=(619, 11)
t=3: graphs=16, nodes=619, edges=2022, x_shape=(619, 11)
t=4: graphs=16, nodes=619, edges=2013, x_shape=(619, 11)

✓ All-traffic batching passed


In [ ]:
# ============================================================
# CELL 86 — Extract FROZEN encoder embeddings
#             from ALL-TRAFFIC sequences
# ============================================================

encoder.eval()

# Make absolutely sure the encoder remains frozen
for param in encoder.parameters():
    param.requires_grad = False

all_train_embeddings = []
all_val_embeddings = []
all_test_embeddings = []

with torch.no_grad():

    # --------------------------------------------------------
    # ALL-TRAFFIC TRAIN
    # --------------------------------------------------------
    for temporal_batch in all_train_batches:

        z = encoder([
            data.to(next(encoder.parameters()).device)
            for data in temporal_batch
        ])

        all_train_embeddings.append(
            z.detach().cpu()
        )

    # --------------------------------------------------------
    # ALL-TRAFFIC VALIDATION
    # --------------------------------------------------------
    for temporal_batch in all_val_batches:

        z = encoder([
            data.to(next(encoder.parameters()).device)
            for data in temporal_batch
        ])

        all_val_embeddings.append(
            z.detach().cpu()
        )

    # --------------------------------------------------------
    # ALL-TRAFFIC TEST
    # --------------------------------------------------------
    for temporal_batch in all_test_batches:

        z = encoder([
            data.to(next(encoder.parameters()).device)
            for data in temporal_batch
        ])

        all_test_embeddings.append(
            z.detach().cpu()
        )


all_train_embeddings = torch.cat(
    all_train_embeddings,
    dim=0
)

all_val_embeddings = torch.cat(
    all_val_embeddings,
    dim=0
)

all_test_embeddings = torch.cat(
    all_test_embeddings,
    dim=0
)

# ------------------------------------------------------------
# Verify shapes
# ------------------------------------------------------------

print("ALL-TRAFFIC EMBEDDINGS")
print("----------------------")
print("Train:", tuple(all_train_embeddings.shape))
print("Val  :", tuple(all_val_embeddings.shape))
print("Test :", tuple(all_test_embeddings.shape))

# ------------------------------------------------------------
# Expected dimensions
# ------------------------------------------------------------

assert all_train_embeddings.shape == (174, 60)
assert all_val_embeddings.shape == (22, 60)
assert all_test_embeddings.shape == (47, 60)

# ------------------------------------------------------------
# Finite checks
# ------------------------------------------------------------

assert torch.isfinite(all_train_embeddings).all()
assert torch.isfinite(all_val_embeddings).all()
assert torch.isfinite(all_test_embeddings).all()

print("\nEmbedding statistics:")

for name, emb in [
    ("Train", all_train_embeddings),
    ("Val", all_val_embeddings),
    ("Test", all_test_embeddings)
]:
    print(
        f"{name}: "
        f"mean={emb.mean().item():.6f}, "
        f"std={emb.std().item():.6f}, "
        f"min={emb.min().item():.6f}, "
        f"max={emb.max().item():.6f}"
    )

print("\n✓ Frozen all-traffic embedding extraction passed")

ALL-TRAFFIC EMBEDDINGS
----------------------
Train: (174, 60)
Val  : (22, 60)
Test : (47, 60)

Embedding statistics:
Train: mean=0.009786, std=0.994986, min=-3.360868, max=3.783573
Val: mean=0.012458, std=0.994935, min=-2.570426, max=3.928230
Test: mean=0.011137, std=1.000076, min=-3.021332, max=3.736228

✓ Frozen all-traffic embedding extraction passed


In [ ]:
# ============================================================
# CELL 87 — k-NN anomaly scores for ALL-TRAFFIC evaluation
# ============================================================

K = 5

# IMPORTANT:
# train_embeddings = NORMAL-ONLY training reference bank
# all_val_embeddings / all_test_embeddings contain ALL traffic

eval_train_scores = knn_cosine_scores(
    all_train_embeddings,
    train_embeddings,
    k=K
)

eval_val_scores = knn_cosine_scores(
    all_val_embeddings,
    train_embeddings,
    k=K
)

eval_test_scores = knn_cosine_scores(
    all_test_embeddings,
    train_embeddings,
    k=K
)

print("ALL-TRAFFIC k-NN ANOMALY SCORES")
print("--------------------------------")
print("K =", K)

for name, scores in [
    ("Train", eval_train_scores),
    ("Validation", eval_val_scores),
    ("Test", eval_test_scores)
]:
    print(f"\n{name}:")
    print("  Shape:", tuple(scores.shape))
    print("  Min:", scores.min().item())
    print("  Max:", scores.max().item())
    print("  Mean:", scores.mean().item())
    print("  Std:", scores.std().item())

print("\n✓ All-traffic anomaly scores calculated")

ALL-TRAFFIC k-NN ANOMALY SCORES
--------------------------------
K = 5

Train:
  Shape: (174,)
  Min: 0.032987140119075775
  Max: 0.2352692186832428
  Mean: 0.11174836754798889
  Std: 0.04776260256767273

Validation:
  Shape: (22,)
  Min: 0.04230229929089546
  Max: 0.17717771232128143
  Mean: 0.10242171585559845
  Std: 0.04097311198711395

Test:
  Shape: (47,)
  Min: 0.022115563973784447
  Max: 0.17739813029766083
  Mean: 0.07303090393543243
  Std: 0.04168554022908211

✓ All-traffic anomaly scores calculated


In [ ]:
# ============================================================
# CELL 88 — Create sequence-level labels
# ============================================================

SEQ_LEN = 5

def make_sequence_labels(
    sequence_window_ids,
    window_label_df
):
    """
    Create sequence-level attack labels.

    Label = 1 if ANY attack flow occurs in any of the
    five windows.

    Also return the aggregate attack fraction.
    """

    labels = []
    attack_fractions = []
    categories = []

    label_lookup = (
        window_label_df
        .set_index("window_idx")
        .to_dict("index")
    )

    for seq in sequence_window_ids:

        rows = [
            label_lookup[w]
            for w in seq
        ]

        total_flows = sum(
            row["n_flows"]
            for row in rows
        )

        total_attacks = sum(
            row["n_attack"]
            for row in rows
        )

        fraction = (
            total_attacks / total_flows
            if total_flows > 0
            else 0.0
        )

        labels.append(
            int(total_attacks > 0)
        )

        attack_fractions.append(fraction)

        cats = set()

        for row in rows:
            if row["attack_categories"]:
                for item in row["attack_categories"].split(", "):
                    category = item.split(":")[0]
                    cats.add(category)

        categories.append(
            ", ".join(sorted(cats))
        )

    return (
        np.asarray(labels, dtype=np.int64),
        np.asarray(attack_fractions, dtype=np.float64),
        categories
    )


# ------------------------------------------------------------
# Window IDs for each all-traffic sequence
# ------------------------------------------------------------

all_train_seq_window_ids = [
    [s.window_idx for s in seq]
    for seq in all_train_sequences
]

all_val_seq_window_ids = [
    [s.window_idx for s in seq]
    for seq in all_val_sequences
]

all_test_seq_window_ids = [
    [s.window_idx for s in seq]
    for seq in all_test_sequences
]


# ------------------------------------------------------------
# Labels
# ------------------------------------------------------------

all_train_labels, all_train_attack_fraction, all_train_categories = \
    make_sequence_labels(
        all_train_seq_window_ids,
        window_label_df
    )

all_val_labels, all_val_attack_fraction, all_val_categories = \
    make_sequence_labels(
        all_val_seq_window_ids,
        window_label_df
    )

all_test_labels, all_test_attack_fraction, all_test_categories = \
    make_sequence_labels(
        all_test_seq_window_ids,
        window_label_df
    )


# ------------------------------------------------------------
# Print distributions
# ------------------------------------------------------------

for name, labels, fractions in [
    ("Train", all_train_labels, all_train_attack_fraction),
    ("Validation", all_val_labels, all_val_attack_fraction),
    ("Test", all_test_labels, all_test_attack_fraction)
]:

    print(f"\n{name}")

    print("  Sequences:", len(labels))
    print("  Normal sequences:", int((labels == 0).sum()))
    print("  Attack sequences:", int((labels == 1).sum()))

    print(
        "  Attack fraction: "
        f"min={fractions.min():.6f}, "
        f"mean={fractions.mean():.6f}, "
        f"max={fractions.max():.6f}"
    )

assert len(all_test_labels) == len(eval_test_scores)

print("\n✓ Sequence-level labels created")


Train
  Sequences: 174
  Normal sequences: 0
  Attack sequences: 174
  Attack fraction: min=0.032006, mean=0.116574, max=0.501175

Validation
  Sequences: 22
  Normal sequences: 0
  Attack sequences: 22
  Attack fraction: min=0.084799, mean=0.105920, max=0.139163

Test
  Sequences: 47
  Normal sequences: 15
  Attack sequences: 32
  Attack fraction: min=0.000000, mean=0.054558, max=0.133333

✓ Sequence-level labels created


In [ ]:
# ============================================================
# CELL 89 — Threshold-free detection evaluation
# ============================================================

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

y_test = all_test_labels
test_scores_np = eval_test_scores.detach().cpu().numpy()

# ------------------------------------------------------------
# ROC-AUC
# ------------------------------------------------------------

roc_auc = roc_auc_score(
    y_test,
    test_scores_np
)

# ------------------------------------------------------------
# PR-AUC / Average Precision
# ------------------------------------------------------------

pr_auc = average_precision_score(
    y_test,
    test_scores_np
)

print("TEST DETECTION RESULTS")
print("======================")

print(f"ROC-AUC : {roc_auc:.6f}")
print(f"PR-AUC  : {pr_auc:.6f}")

print("\nTest class distribution:")
print("Normal:", int((y_test == 0).sum()))
print("Attack:", int((y_test == 1).sum()))

print("\n✓ Threshold-free evaluation completed")

TEST DETECTION RESULTS
ROC-AUC : 0.897917
PR-AUC  : 0.955795

Test class distribution:
Normal: 15
Attack: 32

✓ Threshold-free evaluation completed


In [ ]:
# ============================================================
# CELL 90 — Normal vs attack score distributions
# ============================================================

normal_scores = test_scores_np[y_test == 0]
attack_scores = test_scores_np[y_test == 1]

print("NORMAL TEST SEQUENCES")
print("---------------------")
print("Count :", len(normal_scores))
print("Min   :", normal_scores.min())
print("Mean  :", normal_scores.mean())
print("Median:", np.median(normal_scores))
print("Max   :", normal_scores.max())

print("\nATTACK TEST SEQUENCES")
print("---------------------")
print("Count :", len(attack_scores))
print("Min   :", attack_scores.min())
print("Mean  :", attack_scores.mean())
print("Median:", np.median(attack_scores))
print("Max   :", attack_scores.max())

print("\nDifference in means:")
print(
    "Attack - Normal =",
    attack_scores.mean() - normal_scores.mean()
)

# ------------------------------------------------------------
# Sort all test sequences by anomaly score
# ------------------------------------------------------------

test_score_table = pd.DataFrame({
    "sequence_idx": np.arange(len(test_scores_np)),
    "score": test_scores_np,
    "label": y_test,
    "attack_fraction": all_test_attack_fraction,
    "attack_categories": all_test_categories
})

test_score_table = test_score_table.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

print("\nHighest-scoring test sequences:")
display(test_score_table.head(15))

print("\nLowest-scoring test sequences:")
display(test_score_table.tail(15))

NORMAL TEST SEQUENCES
---------------------
Count : 15
Min   : 0.022115564
Mean  : 0.03955043
Median: 0.034151755
Max   : 0.059343945

ATTACK TEST SEQUENCES
---------------------
Count : 32
Min   : 0.03314363
Mean  : 0.08872488
Median: 0.07734893
Max   : 0.17739813

Difference in means:
Attack - Normal = 0.04917445

Highest-scoring test sequences:


,sequence_idx,score,label,attack_fraction,attack_categories
0,2,0.177398,1,0.124479,"DoS, Exploits, Fuzzers, Generic, Reconnaissanc..."
1,6,0.158932,1,0.133333,"Backdoors, DoS, Exploits, Fuzzers, Generic, Re..."
2,21,0.155421,1,0.060469,"DoS, Exploits, Fuzzers, Generic, Reconnaissanc..."
3,15,0.154667,1,0.071135,"DoS, Exploits, Fuzzers, Generic, Reconnaissanc..."
4,17,0.150789,1,0.065982,"DoS, Exploits, Fuzzers, Generic, Reconnaissanc..."
5,3,0.136589,1,0.122816,"Backdoors, DoS, Exploits, Fuzzers, Generic, Re..."
6,7,0.123890,1,0.132838,"Backdoors, DoS, Exploits, Fuzzers, Generic, Re..."
7,9,0.118405,1,0.109575,"Backdoors, DoS, Exploits, Fuzzers, Generic, Re..."
8,8,0.116564,1,0.123667,"Backdoors, DoS, Exploits, Fuzzers, Generic, Re..."
9,4,0.115230,1,0.131722,"Backdoors, DoS, Exploits, Fuzzers, Generic, Re..."



Lowest-scoring test sequences:


,sequence_idx,score,label,attack_fraction,attack_categories
32,38,0.043221,0,0.000000,
33,28,0.042091,1,0.034411,"DoS, Exploits, Fuzzers, Generic, Reconnaissanc..."
34,39,0.039256,0,0.000000,
35,11,0.035724,1,0.087620,"Backdoors, DoS, Exploits, Fuzzers, Generic, Re..."
36,26,0.034727,1,0.058762,"Backdoors, DoS, Exploits, Fuzzers, Generic, Re..."
37,12,0.034684,1,0.079199,"Backdoors, DoS, Exploits, Fuzzers, Generic, Re..."
38,35,0.034152,0,0.000000,
39,45,0.033449,0,0.000000,
40,27,0.033144,1,0.050232,"Backdoors, DoS, Exploits, Fuzzers, Generic, Re..."
41,32,0.032956,0,0.000000,


In [ ]:
# ============================================================
# CELL 91 — Leakage-safe threshold calibration
# ============================================================

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix
)

# ------------------------------------------------------------
# Normal-only training scores with SELF excluded
# ------------------------------------------------------------

normal_train_scores = (
    train_scores_no_self
    .detach()
    .cpu()
    .numpy()
)

# ------------------------------------------------------------
# Calibrate threshold from NORMAL TRAINING ONLY
# ------------------------------------------------------------

threshold_95 = np.percentile(
    normal_train_scores,
    95
)

threshold_99 = np.percentile(
    normal_train_scores,
    99
)

print("NORMAL TRAINING SCORE CALIBRATION")
print("=================================")

print(f"Mean : {normal_train_scores.mean():.6f}")
print(f"Std  : {normal_train_scores.std():.6f}")
print(f"95th percentile: {threshold_95:.6f}")
print(f"99th percentile: {threshold_99:.6f}")

print("\n✓ Thresholds calibrated without using test labels")

NORMAL TRAINING SCORE CALIBRATION
Mean : 0.053992
Std  : 0.056990
95th percentile: 0.141144
99th percentile: 0.312869

✓ Thresholds calibrated without using test labels


In [ ]:
# ============================================================
# CELL 92 — Thresholded TEST evaluation
# ============================================================

test_scores_np = (
    eval_test_scores
    .detach()
    .cpu()
    .numpy()
)

y_test = all_test_labels

def evaluate_threshold(
    scores,
    labels,
    threshold
):
    predictions = (
        scores >= threshold
    ).astype(np.int64)

    tn, fp, fn, tp = confusion_matrix(
        labels,
        predictions,
        labels=[0, 1]
    ).ravel()

    return {
        "threshold": threshold,
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(
            labels,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            labels,
            predictions,
            zero_division=0
        ),
        "f1": f1_score(
            labels,
            predictions,
            zero_division=0
        ),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }


result_95 = evaluate_threshold(
    test_scores_np,
    y_test,
    threshold_95
)

result_99 = evaluate_threshold(
    test_scores_np,
    y_test,
    threshold_99
)

print("TEST RESULTS")
print("============")

print("\n95th-percentile threshold:")
for k, v in result_95.items():
    print(f"{k}: {v}")

print("\n99th-percentile threshold:")
for k, v in result_99.items():
    print(f"{k}: {v}")

TEST RESULTS

95th-percentile threshold:
threshold: 0.141144260764122
accuracy: 0.425531914893617
precision: 1.0
recall: 0.15625
f1: 0.2702702702702703
TN: 15
FP: 0
FN: 27
TP: 5

99th-percentile threshold:
threshold: 0.3128686845302582
accuracy: 0.3191489361702128
precision: 0.0
recall: 0.0
f1: 0.0
TN: 15
FP: 0
FN: 32
TP: 0


In [ ]:
# ============================================================
# CELL 93 — Attack severity vs anomaly score
# ============================================================

analysis_df = test_score_table.copy()

# Only attack sequences
attack_analysis = analysis_df[
    analysis_df["label"] == 1
].copy()

# ------------------------------------------------------------
# Correlation
# ------------------------------------------------------------

score_attack_corr = np.corrcoef(
    attack_analysis["score"],
    attack_analysis["attack_fraction"]
)[0, 1]

print("ATTACK SEVERITY ANALYSIS")
print("========================")

print(
    f"Correlation(score, attack_fraction): "
    f"{score_attack_corr:.6f}"
)

# ------------------------------------------------------------
# Detection at the 95th percentile threshold
# ------------------------------------------------------------

attack_analysis["detected_95"] = (
    attack_analysis["score"] >= threshold_95
)

print("\nAttack sequences:")
print("Total:",
      len(attack_analysis))

print(
    "Detected:",
    int(attack_analysis["detected_95"].sum())
)

print(
    "Missed:",
    int((~attack_analysis["detected_95"]).sum())
)

# ------------------------------------------------------------
# Compare detected vs missed attacks
# ------------------------------------------------------------

print("\nDETECTED ATTACKS")
print("----------------")

display(
    attack_analysis[
        attack_analysis["detected_95"]
    ][
        [
            "sequence_idx",
            "score",
            "attack_fraction",
            "attack_categories"
        ]
    ].sort_values(
        "score",
        ascending=False
    )
)

print("\nMISSED ATTACKS")
print("--------------")

display(
    attack_analysis[
        ~attack_analysis["detected_95"]
    ][
        [
            "sequence_idx",
            "score",
            "attack_fraction",
            "attack_categories"
        ]
    ].sort_values(
        "score",
        ascending=True
    )
)

ATTACK SEVERITY ANALYSIS
Correlation(score, attack_fraction): 0.453017

Attack sequences:
Total: 32
Detected: 5
Missed: 27

DETECTED ATTACKS
----------------


,sequence_idx,score,attack_fraction,attack_categories
0,2,0.177398,0.124479,"DoS, Exploits, Fuzzers, Generic, Reconnaissanc..."
1,6,0.158932,0.133333,"Backdoors, DoS, Exploits, Fuzzers, Generic, Re..."
2,21,0.155421,0.060469,"DoS, Exploits, Fuzzers, Generic, Reconnaissanc..."
3,15,0.154667,0.071135,"DoS, Exploits, Fuzzers, Generic, Reconnaissanc..."
4,17,0.150789,0.065982,"DoS, Exploits, Fuzzers, Generic, Reconnaissanc..."



MISSED ATTACKS
--------------


,sequence_idx,score,attack_fraction,attack_categories
40,27,0.033144,0.050232,"Backdoors, DoS, Exploits, Fuzzers, Generic, Re..."
37,12,0.034684,0.079199,"Backdoors, DoS, Exploits, Fuzzers, Generic, Re..."
36,26,0.034727,0.058762,"Backdoors, DoS, Exploits, Fuzzers, Generic, Re..."
35,11,0.035724,0.087620,"Backdoors, DoS, Exploits, Fuzzers, Generic, Re..."
33,28,0.042091,0.034411,"DoS, Exploits, Fuzzers, Generic, Reconnaissanc..."
30,25,0.049669,0.060447,"Backdoors, DoS, Exploits, Fuzzers, Generic, Re..."
28,29,0.051996,0.019298,"DoS, Exploits, Fuzzers, Generic, Reconnaissance"
27,0,0.052988,0.126576,"Analysis, Backdoors, DoS, Exploits, Fuzzers, G..."
26,30,0.054110,0.009019,Fuzzers
22,22,0.064893,0.061179,"DoS, Exploits, Fuzzers, Generic, Reconnaissanc..."


In [ ]:
# ============================================================
# CELL 94 — Map test sequences back to temporal windows
# ============================================================

test_sequence_analysis = []

for seq_idx, seq in enumerate(all_test_sequences):

    window_ids = [
        snapshot.window_idx
        for snapshot in seq
    ]

    test_sequence_analysis.append({
        "sequence_idx": seq_idx,
        "window_start": window_ids[0],
        "window_end": window_ids[-1],
        "windows": window_ids,
        "score": float(test_scores_np[seq_idx]),
        "label": int(all_test_labels[seq_idx]),
        "attack_fraction": float(
            all_test_attack_fraction[seq_idx]
        ),
        "attack_categories": all_test_categories[seq_idx]
    })

test_sequence_analysis_df = pd.DataFrame(
    test_sequence_analysis
)

# ------------------------------------------------------------
# Missed attacks at the 95th percentile threshold
# ------------------------------------------------------------

missed_attacks_df = test_sequence_analysis_df[
    (test_sequence_analysis_df["label"] == 1) &
    (test_sequence_analysis_df["score"] < threshold_95)
].copy()

print("MISSED ATTACK SEQUENCES")
print("=======================")

print("Count:", len(missed_attacks_df))

display(
    missed_attacks_df[
        [
            "sequence_idx",
            "window_start",
            "window_end",
            "score",
            "attack_fraction",
            "attack_categories"
        ]
    ].sort_values(
        "score"
    )
)

MISSED ATTACK SEQUENCES
Count: 27


,sequence_idx,window_start,window_end,score,attack_fraction,attack_categories
27,27,231,235,0.033144,0.050232,"Backdoors, DoS, Exploits, Fuzzers, Generic, Re..."
12,12,216,220,0.034684,0.079199,"Backdoors, DoS, Exploits, Fuzzers, Generic, Re..."
26,26,230,234,0.034727,0.058762,"Backdoors, DoS, Exploits, Fuzzers, Generic, Re..."
11,11,215,219,0.035724,0.087620,"Backdoors, DoS, Exploits, Fuzzers, Generic, Re..."
28,28,232,236,0.042091,0.034411,"DoS, Exploits, Fuzzers, Generic, Reconnaissanc..."
25,25,229,233,0.049669,0.060447,"Backdoors, DoS, Exploits, Fuzzers, Generic, Re..."
29,29,233,237,0.051996,0.019298,"DoS, Exploits, Fuzzers, Generic, Reconnaissance"
0,0,204,208,0.052988,0.126576,"Analysis, Backdoors, DoS, Exploits, Fuzzers, G..."
30,30,234,238,0.054110,0.009019,Fuzzers
22,22,226,230,0.064893,0.061179,"DoS, Exploits, Fuzzers, Generic, Reconnaissanc..."


In [ ]:
# ============================================================
# CELL 95 — Inspect temporal attack composition of misses
# ============================================================

print("TEMPORAL BREAKDOWN OF MISSED ATTACKS")
print("====================================")

for _, row in missed_attacks_df.sort_values("score").iterrows():

    seq_idx = int(row["sequence_idx"])
    window_ids = row["windows"]

    print("\n" + "=" * 80)
    print(
        f"Sequence {seq_idx} | "
        f"Score={row['score']:.6f} | "
        f"Sequence attack fraction={row['attack_fraction']:.6f}"
    )

    print("Windows:", window_ids)

    display(
        window_label_df[
            window_label_df["window_idx"].isin(window_ids)
        ][
            [
                "window_idx",
                "n_flows",
                "n_attack",
                "attack_fraction",
                "attack_categories"
            ]
        ].sort_values("window_idx")
    )

TEMPORAL BREAKDOWN OF MISSED ATTACKS

Sequence 27 | Score=0.033144 | Sequence attack fraction=0.050232
Windows: [231, 232, 233, 234, 235]


,window_idx,n_flows,n_attack,attack_fraction,attack_categories
231,231,765,63,0.082353,"Fuzzers:27, Exploits:25, Reconnaissance:7, DoS..."
232,232,805,60,0.074534,"Fuzzers:32, Exploits:11, Reconnaissance:8, DoS..."
233,233,773,39,0.050453,"Fuzzers:32, Exploits:4, DoS:1, Generic:1, Reco..."
234,234,768,28,0.036458,Fuzzers:28
235,235,771,5,0.006485,Fuzzers:5



Sequence 12 | Score=0.034684 | Sequence attack fraction=0.079199
Windows: [216, 217, 218, 219, 220]


,window_idx,n_flows,n_attack,attack_fraction,attack_categories
216,216,680,49,0.072059,"Exploits:21, Fuzzers:14, DoS:6, Reconnaissance..."
217,217,687,71,0.103348,"Fuzzers:38, Exploits:12, Reconnaissance:10, Do..."
218,218,712,56,0.078652,"Fuzzers:30, Exploits:17, Reconnaissance:4, DoS..."
219,219,705,59,0.083688,"Exploits:25, Fuzzers:23, Reconnaissance:8, DoS..."
220,220,663,38,0.057315,"Exploits:25, Reconnaissance:7, DoS:3, Generic:..."



Sequence 26 | Score=0.034727 | Sequence attack fraction=0.058762
Windows: [230, 231, 232, 233, 234]


,window_idx,n_flows,n_attack,attack_fraction,attack_categories
230,230,718,35,0.048747,"Exploits:21, Reconnaissance:7, Generic:3, Fuzz..."
231,231,765,63,0.082353,"Fuzzers:27, Exploits:25, Reconnaissance:7, DoS..."
232,232,805,60,0.074534,"Fuzzers:32, Exploits:11, Reconnaissance:8, DoS..."
233,233,773,39,0.050453,"Fuzzers:32, Exploits:4, DoS:1, Generic:1, Reco..."
234,234,768,28,0.036458,Fuzzers:28


... (44 outputs left out)


Sequence 7 | Score=0.123890 | Sequence attack fraction=0.132838
Windows: [211, 212, 213, 214, 215]


,window_idx,n_flows,n_attack,attack_fraction,attack_categories
211,211,777,98,0.126126,"Generic:32, Fuzzers:28, Exploits:23, Reconnais..."
212,212,792,138,0.174242,"Fuzzers:65, Generic:44, Exploits:15, Reconnais..."
213,213,877,116,0.132269,"Generic:60, Fuzzers:25, Exploits:18, DoS:5, Re..."
214,214,835,109,0.130539,"Generic:76, Exploits:20, Reconnaissance:6, DoS..."
215,215,754,75,0.099469,"Generic:42, Exploits:22, Reconnaissance:7, She..."



Sequence 3 | Score=0.136589 | Sequence attack fraction=0.122816
Windows: [207, 208, 209, 210, 211]


,window_idx,n_flows,n_attack,attack_fraction,attack_categories
207,207,801,103,0.128589,"Generic:72, Exploits:16, Reconnaissance:10, Do..."
208,208,857,114,0.133022,"Generic:83, Exploits:20, Reconnaissance:7, Fuz..."
209,209,859,105,0.122235,"Generic:65, Exploits:35, Reconnaissance:4, Fuz..."
210,210,769,79,0.102731,"Generic:48, Exploits:18, Reconnaissance:8, Fuz..."
211,211,777,98,0.126126,"Generic:32, Fuzzers:28, Exploits:23, Reconnais..."


In [ ]:
# ============================================================
# CELL 96 — Attack-category distribution and exposure
# ============================================================

from collections import Counter

print("ATTACK CATEGORY ANALYSIS")
print("========================")

# ------------------------------------------------------------
# Collect attack categories from the actual dataframe
# ------------------------------------------------------------

attack_df = df[df["label"] == 1].copy()

overall_category_counts = (
    attack_df["attack_cat"]
    .value_counts()
)

print("\nAll attack flows in development data:")
display(overall_category_counts.to_frame("count"))

# ------------------------------------------------------------
# Categories present in the TEST partition
# ------------------------------------------------------------

test_window_set = set(test_snapshot_idx)

test_attack_df = df[
    (df["label"] == 1) &
    (
        df["window_start_idx"].isin(test_window_set) |
        df["window_end_idx"].isin(test_window_set)
    )
].copy()

test_category_counts = (
    test_attack_df["attack_cat"]
    .value_counts()
)

print("\nAttack categories in TEST:")
display(test_category_counts.to_frame("count"))

# ------------------------------------------------------------
# Categories present in each partition
# ------------------------------------------------------------

train_window_set = set(train_snapshot_idx)
val_window_set = set(val_snapshot_idx)

train_attack_df = df[
    (df["label"] == 1) &
    (
        df["window_start_idx"].isin(train_window_set) |
        df["window_end_idx"].isin(train_window_set)
    )
]

val_attack_df = df[
    (df["label"] == 1) &
    (
        df["window_start_idx"].isin(val_window_set) |
        df["window_end_idx"].isin(val_window_set)
    )
]

train_categories = set(
    train_attack_df["attack_cat"].dropna().unique()
)

val_categories = set(
    val_attack_df["attack_cat"].dropna().unique()
)

test_categories = set(
    test_attack_df["attack_cat"].dropna().unique()
)

print("\nCategory sets:")
print("Train:", sorted(train_categories))
print("Val  :", sorted(val_categories))
print("Test :", sorted(test_categories))

print("\nTest categories NOT appearing in attack traffic of train:")
print(sorted(test_categories - train_categories))

print("\n✓ Category exposure analysis completed")

ATTACK CATEGORY ANALYSIS

All attack flows in development data:


,count
attack_cat,
Generic,7522
Exploits,5409
Fuzzers,5051
Reconnaissance,1759
DoS,1167
Backdoors,534
Analysis,526
Shellcode,223
Worms,24



Attack categories in TEST:


,count
attack_cat,
Generic,725
Exploits,557
Fuzzers,458
Reconnaissance,228
DoS,77
Shellcode,29
Backdoors,11
Analysis,2
Worms,1



Category sets:
Train: ['Analysis', 'Backdoors', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Reconnaissance', 'Shellcode', 'Worms']
Val  : ['Analysis', 'Backdoors', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Reconnaissance', 'Shellcode', 'Worms']
Test : ['Analysis', 'Backdoors', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Reconnaissance', 'Shellcode', 'Worms']

Test categories NOT appearing in attack traffic of train:
[]

✓ Category exposure analysis completed


In [ ]:
# ============================================================
# CELL 97 — Attack categories vs detector outcome
# ============================================================

# Expand each attack sequence into category rows

category_rows = []

for _, row in attack_analysis.iterrows():

    categories = [
        c.strip()
        for c in str(row["attack_categories"]).split(",")
        if c.strip()
    ]

    for category in categories:

        category_rows.append({
            "sequence_idx": int(row["sequence_idx"]),
            "category": category,
            "score": float(row["score"]),
            "attack_fraction": float(row["attack_fraction"]),
            "detected_95": bool(row["score"] >= threshold_95)
        })

category_df = pd.DataFrame(category_rows)

# ------------------------------------------------------------
# Aggregate
# ------------------------------------------------------------

category_summary = (
    category_df
    .groupby("category")
    .agg(
        sequence_count=("sequence_idx", "nunique"),
        detected_count=("detected_95", "sum"),
        mean_score=("score", "mean"),
        median_score=("score", "median"),
        mean_attack_fraction=("attack_fraction", "mean")
    )
    .reset_index()
)

category_summary["detection_rate_95"] = (
    category_summary["detected_count"] /
    category_summary["sequence_count"]
)

category_summary = category_summary.sort_values(
    "mean_score",
    ascending=True
)

print("CATEGORY-LEVEL DETECTION ANALYSIS")
print("=================================")

display(category_summary)

CATEGORY-LEVEL DETECTION ANALYSIS


,category,sequence_count,detected_count,mean_score,median_score,mean_attack_fraction,detection_rate_95
8,Worms,5,0,0.046351,0.042091,0.053052,0.000000
0,Analysis,2,0,0.062672,0.062672,0.124968,0.000000
1,Backdoors,19,1,0.083034,0.073078,0.096780,0.052632
4,Fuzzers,32,5,0.088725,0.077349,0.080132,0.156250
3,Exploits,30,5,0.090427,0.084161,0.085128,0.166667
5,Generic,30,5,0.090427,0.084161,0.085128,0.166667
6,Reconnaissance,30,5,0.090427,0.084161,0.085128,0.166667
2,DoS,30,5,0.090427,0.084161,0.085128,0.166667
7,Shellcode,29,5,0.091752,0.086703,0.087398,0.172414


In [ ]:
# ============================================================
# CELL 98 — Temporal embedding velocity
# ============================================================

import torch.nn.functional as F

# Normalize embeddings before measuring trajectory movement
test_emb_norm = F.normalize(
    all_test_embeddings,
    p=2,
    dim=1
)

# Consecutive sequence-to-sequence cosine distance
temporal_velocity = 1.0 - torch.sum(
    test_emb_norm[:-1] * test_emb_norm[1:],
    dim=1
)

temporal_velocity_np = (
    temporal_velocity
    .detach()
    .cpu()
    .numpy()
)

# ------------------------------------------------------------
# Align with sequence labels
# ------------------------------------------------------------

# Velocity[i] describes transition:
# sequence i -> sequence i+1
#
# For a conservative label, mark the transition as attack
# if either adjacent sequence contains attack traffic.

velocity_labels = np.maximum(
    all_test_labels[:-1],
    all_test_labels[1:]
)

velocity_attack_fraction = np.maximum(
    all_test_attack_fraction[:-1],
    all_test_attack_fraction[1:]
)

print("TEMPORAL EMBEDDING VELOCITY")
print("===========================")

print("Number of transitions:",
      len(temporal_velocity_np))

print(
    "Min:",
    temporal_velocity_np.min()
)

print(
    "Mean:",
    temporal_velocity_np.mean()
)

print(
    "Max:",
    temporal_velocity_np.max()
)

print(
    "Normal-transition mean:",
    temporal_velocity_np[
        velocity_labels == 0
    ].mean()
)

print(
    "Attack-transition mean:",
    temporal_velocity_np[
        velocity_labels == 1
    ].mean()
)

print("\n✓ Temporal velocity calculated")

TEMPORAL EMBEDDING VELOCITY
Number of transitions: 46
Min: 0.0049907565
Mean: 0.22325936
Max: 1.0562037
Normal-transition mean: 0.080086365
Attack-transition mean: 0.28589755

✓ Temporal velocity calculated


In [ ]:
# ============================================================
# CELL 99 — Evaluate temporal velocity
# ============================================================

velocity_roc_auc = roc_auc_score(
    velocity_labels,
    temporal_velocity_np
)

velocity_pr_auc = average_precision_score(
    velocity_labels,
    temporal_velocity_np
)

velocity_corr = np.corrcoef(
    temporal_velocity_np,
    velocity_attack_fraction
)[0, 1]

print("TEMPORAL VELOCITY RESULTS")
print("=========================")

print(f"ROC-AUC : {velocity_roc_auc:.6f}")
print(f"PR-AUC  : {velocity_pr_auc:.6f}")
print(
    f"Correlation with attack fraction: "
    f"{velocity_corr:.6f}"
)

TEMPORAL VELOCITY RESULTS
ROC-AUC : 0.732143
PR-AUC  : 0.891224
Correlation with attack fraction: 0.326684


In [ ]:
# ============================================================
# CELL 100 — NKVEL: Build normal transition bank
# ============================================================

import numpy as np
import torch
import torch.nn.functional as F

# ------------------------------------------------------------
# Normalize normal training embeddings
# ------------------------------------------------------------

normal_train_emb = F.normalize(
    train_embeddings,
    p=2,
    dim=1
)

# ------------------------------------------------------------
# Consecutive transitions
# ------------------------------------------------------------

normal_transitions = (
    normal_train_emb[1:] - normal_train_emb[:-1]
)

# Normalize transition directions
normal_transitions = F.normalize(
    normal_transitions,
    p=2,
    dim=1
)

print("NKVEL NORMAL TRANSITION BANK")
print("============================")

print(
    "Normal embedding shape:",
    tuple(normal_train_emb.shape)
)

print(
    "Transition bank shape:",
    tuple(normal_transitions.shape)
)

print(
    "Finite:",
    torch.isfinite(normal_transitions).all().item()
)

print(
    "Transition norm mean:",
    normal_transitions.norm(dim=1).mean().item()
)

print(
    "Transition norm std:",
    normal_transitions.norm(dim=1).std().item()
)

print("\n✓ Normal transition bank constructed")

NKVEL NORMAL TRANSITION BANK
Normal embedding shape: (174, 60)
Transition bank shape: (173, 60)
Finite: True
Transition norm mean: 1.0
Transition norm std: 5.3774936503714343e-08

✓ Normal transition bank constructed


In [ ]:
# ============================================================
# CELL 101 — NKVEL evolution novelty
# ============================================================

def transition_novelty(
    embeddings,
    normal_transition_bank,
    k=5
):
    """
    Calculate transition novelty against a normal
    transition reference bank.
    """

    emb = F.normalize(
        embeddings,
        p=2,
        dim=1
    )

    transitions = emb[1:] - emb[:-1]

    # Normalize transition directions
    transitions = F.normalize(
        transitions,
        p=2,
        dim=1
    )

    # Cosine similarity
    similarity = transitions @ normal_transition_bank.T

    # Convert similarity to cosine distance
    distance = 1.0 - similarity

    # k nearest normal transitions
    knn_distances, _ = torch.topk(
        distance,
        k=min(k, normal_transition_bank.shape[0]),
        largest=False,
        dim=1
    )

    novelty = knn_distances.mean(dim=1)

    return novelty


# ------------------------------------------------------------
# Test transition novelty
# ------------------------------------------------------------

test_evolution_novelty = transition_novelty(
    all_test_embeddings,
    normal_transitions,
    k=5
)

test_evolution_np = (
    test_evolution_novelty
    .detach()
    .cpu()
    .numpy()
)

print("NKVEL EVOLUTION NOVELTY")
print("=======================")

print(
    "Transitions:",
    len(test_evolution_np)
)

print(
    "Min:",
    test_evolution_np.min()
)

print(
    "Mean:",
    test_evolution_np.mean()
)

print(
    "Max:",
    test_evolution_np.max()
)

print("\n✓ Evolution novelty calculated")

NKVEL EVOLUTION NOVELTY
Transitions: 46
Min: 0.0716836
Mean: 0.22929138
Max: 0.5468809

✓ Evolution novelty calculated


In [ ]:
# ============================================================
# CELL 102 — Evaluate NKVEL evolution novelty
# ============================================================

# Labels correspond to the transition:
# sequence i -> sequence i+1
#
# Attack if either endpoint sequence is attack-bearing.

nkvel_transition_labels = np.maximum(
    all_test_labels[:-1],
    all_test_labels[1:]
)

evolution_roc_auc = roc_auc_score(
    nkvel_transition_labels,
    test_evolution_np
)

evolution_pr_auc = average_precision_score(
    nkvel_transition_labels,
    test_evolution_np
)

print("NKVEL EVOLUTION-NOVELTY RESULTS")
print("================================")

print(
    f"ROC-AUC : {evolution_roc_auc:.6f}"
)

print(
    f"PR-AUC  : {evolution_pr_auc:.6f}"
)

NKVEL EVOLUTION-NOVELTY RESULTS
ROC-AUC : 0.750000
PR-AUC  : 0.882669


In [ ]:
# ============================================================
# CELL 103A — Find existing test-score variables
# ============================================================

print("Variables containing 'score':")
print([
    name for name in globals()
    if "score" in name.lower()
])

print("\nVariables containing 'test':")
print([
    name for name in globals()
    if "test" in name.lower()
])

Variables containing 'score':
['knn_cosine_scores', 'train_scores', 'val_scores', 'test_scores', 'train_scores_no_self', 'eval_train_scores', 'eval_val_scores', 'eval_test_scores', 'scores', 'roc_auc_score', 'average_precision_score', 'test_scores_np', 'normal_scores', 'attack_scores', 'test_score_table', 'precision_score', 'recall_score', 'f1_score', 'accuracy_score', 'normal_train_scores', 'score_attack_corr']

Variables containing 'test':
['normal_test_seqs', 'test_windows', 'test_window_ids', 'test_w', 'test_vectors', 'test_batches', 'test_sequences', 'test_snapshot_idx', 'test_graph_sequences', 'loss_test', 'test_embeddings', 'test_scores', 'all_test_sequences', 'all_test_graph_sequences', 'all_test_batches', 'all_test_embeddings', 'eval_test_scores', 'all_test_seq_window_ids', 'all_test_labels', 'all_test_attack_fraction', 'all_test_categories', 'y_test', 'test_scores_np', 'test_score_table', 'test_sequence_analysis', 'test_sequence_analysis_df', 'test_window_set', 'test_attack_d

In [ ]:
# ============================================================
# CELL 103A — Verify all-traffic CTGAD-X scores
# ============================================================

print("all_test_embeddings :", all_test_embeddings.shape)

for name in [
    "test_scores",
    "test_scores_np",
    "eval_test_scores",
    "scores"
]:
    if name in globals():
        obj = globals()[name]
        try:
            print(
                f"{name:20s}: shape={np.asarray(obj).shape}"
            )
        except:
            print(
                f"{name:20s}: type={type(obj)}"
            )

all_test_embeddings : torch.Size([47, 60])
test_scores         : shape=(47,)
test_scores_np      : shape=(47,)
eval_test_scores    : shape=(47,)
scores              : shape=(47,)


In [ ]:
# ============================================================
# CELL 103 — NKVEL trajectory dynamics
# ============================================================

import numpy as np
import torch
import torch.nn.functional as F

# ------------------------------------------------------------
# Use ALL-TRAFFIC CTGAD-X test scores
# ------------------------------------------------------------

state_score_np = np.asarray(
    eval_test_scores
).reshape(-1)

# Safety checks
assert len(all_test_embeddings) == 47
assert len(state_score_np) == 47
assert len(all_test_labels) == 47
assert len(all_test_attack_fraction) == 47

# ------------------------------------------------------------
# Normalize test embeddings
# ------------------------------------------------------------

test_z = F.normalize(
    all_test_embeddings,
    p=2,
    dim=1
)

# ------------------------------------------------------------
# 1. Temporal velocity
# ------------------------------------------------------------

velocity = (
    1.0 -
    torch.sum(
        test_z[:-1] * test_z[1:],
        dim=1
    )
)

velocity_np = (
    velocity.detach()
    .cpu()
    .numpy()
)

# ------------------------------------------------------------
# 2. Velocity acceleration
# ------------------------------------------------------------

acceleration_np = np.diff(
    velocity_np
)

# ------------------------------------------------------------
# 3. State-score evolution
# ------------------------------------------------------------

state_velocity_np = np.diff(
    state_score_np
)

# Absolute state-score change
state_change_np = np.abs(
    state_velocity_np
)

print("NKVEL TRAJECTORY DYNAMICS")
print("=========================")

print("Embeddings:", len(test_z))
print("Velocity:", len(velocity_np))
print("Acceleration:", len(acceleration_np))
print("State evolution:", len(state_velocity_np))

print("\nVelocity")
print("  Mean:", velocity_np.mean())
print("  Std :", velocity_np.std())
print("  Max :", velocity_np.max())

print("\nAcceleration")
print("  Mean:", acceleration_np.mean())
print("  Std :", acceleration_np.std())
print("  Max abs:", np.abs(acceleration_np).max())

print("\nState-score evolution")
print("  Mean:", state_velocity_np.mean())
print("  Std :", state_velocity_np.std())
print("  Max abs:", state_change_np.max())

print("\n✓ Trajectory dynamics extracted")

NKVEL TRAJECTORY DYNAMICS
Embeddings: 47
Velocity: 46
Acceleration: 45
State evolution: 46

Velocity
  Mean: 0.22325936
  Std : 0.25552696
  Max : 1.0562037

Acceleration
  Mean: 0.000114713774
  Std : 0.27525693
  Max abs: 1.0088856

State-score evolution
  Mean: -5.1922285e-05
  Std : 0.037052643
  Max abs: 0.10504119

✓ Trajectory dynamics extracted


In [ ]:
# ============================================================
# CELL 104 — Evaluate trajectory dynamics
# ============================================================

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

# ------------------------------------------------------------
# Velocity labels
# ------------------------------------------------------------

velocity_labels = np.maximum(
    all_test_labels[:-1],
    all_test_labels[1:]
)

# ------------------------------------------------------------
# Acceleration labels
# ------------------------------------------------------------

trajectory_labels = np.maximum(
    all_test_labels[:-2],
    np.maximum(
        all_test_labels[1:-1],
        all_test_labels[2:]
    )
)

# ------------------------------------------------------------
# Velocity
# ------------------------------------------------------------

velocity_auc = roc_auc_score(
    velocity_labels,
    velocity_np
)

velocity_pr = average_precision_score(
    velocity_labels,
    velocity_np
)

# ------------------------------------------------------------
# Absolute acceleration
# ------------------------------------------------------------

acceleration_auc = roc_auc_score(
    trajectory_labels,
    np.abs(acceleration_np)
)

acceleration_pr = average_precision_score(
    trajectory_labels,
    np.abs(acceleration_np)
)

# ------------------------------------------------------------
# Absolute state-score evolution
# ------------------------------------------------------------

state_change_auc = roc_auc_score(
    trajectory_labels,
    state_change_np[1:]
)

state_change_pr = average_precision_score(
    trajectory_labels,
    state_change_np[1:]
)

print("NKVEL TRAJECTORY SIGNAL RESULTS")
print("================================")

print(
    f"Velocity           ROC-AUC: {velocity_auc:.6f} | "
    f"PR-AUC: {velocity_pr:.6f}"
)

print(
    f"Acceleration       ROC-AUC: {acceleration_auc:.6f} | "
    f"PR-AUC: {acceleration_pr:.6f}"
)

print(
    f"State-score change ROC-AUC: {state_change_auc:.6f} | "
    f"PR-AUC: {state_change_pr:.6f}"
)

NKVEL TRAJECTORY SIGNAL RESULTS
Velocity           ROC-AUC: 0.732143 | PR-AUC: 0.891224
Acceleration       ROC-AUC: 0.829327 | PR-AUC: 0.927392
State-score change ROC-AUC: 0.735577 | PR-AUC: 0.894527


In [ ]:
# ============================================================
# CELL 106 — State vs evolution relationship
# ============================================================

from scipy.stats import pearsonr, spearmanr

# ------------------------------------------------------------
# Align state score with transition evolution
# ------------------------------------------------------------

state_for_evolution = state_score_np[1:]

evolution_signal = state_change_np

# ------------------------------------------------------------
# Correlations
# ------------------------------------------------------------

pearson_corr, pearson_p = pearsonr(
    state_for_evolution,
    evolution_signal
)

spearman_corr, spearman_p = spearmanr(
    state_for_evolution,
    evolution_signal
)

print("STATE vs EVOLUTION")
print("===================")

print(
    f"Pearson correlation  : {pearson_corr:.6f}"
)

print(
    f"Pearson p-value      : {pearson_p:.6f}"
)

print(
    f"Spearman correlation : {spearman_corr:.6f}"
)

print(
    f"Spearman p-value     : {spearman_p:.6f}"
)

STATE vs EVOLUTION
Pearson correlation  : 0.622130
Pearson p-value      : 0.000004
Spearman correlation : 0.577182
Spearman p-value     : 0.000027


In [ ]:
# ============================================================
# CELL 107 — NKVEL rank-normalized fusion
# ============================================================

from scipy.stats import rankdata

# ------------------------------------------------------------
# Align state score with evolution signal
# ------------------------------------------------------------

state_signal = state_score_np[1:]
evolution_signal = state_change_np

# ------------------------------------------------------------
# Rank normalization
# ------------------------------------------------------------

state_rank = (
    rankdata(state_signal, method="average")
    / len(state_signal)
)

evolution_rank = (
    rankdata(evolution_signal, method="average")
    / len(evolution_signal)
)

# ------------------------------------------------------------
# Equal-weight NKVEL score
# ------------------------------------------------------------

nkvel_score = (
    0.5 * state_rank +
    0.5 * evolution_rank
)

# ------------------------------------------------------------
# Labels
# ------------------------------------------------------------

nkvel_labels = np.maximum(
    all_test_labels[:-1],
    all_test_labels[1:]
)

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

nkvel_roc_auc = roc_auc_score(
    nkvel_labels,
    nkvel_score
)

nkvel_pr_auc = average_precision_score(
    nkvel_labels,
    nkvel_score
)

print("NKVEL V1 — STATE + EVOLUTION")
print("==============================")

print(
    f"ROC-AUC : {nkvel_roc_auc:.6f}"
)

print(
    f"PR-AUC  : {nkvel_pr_auc:.6f}"
)

print("\nComponents:")
print(
    f"State component mean     : {state_rank.mean():.6f}"
)

print(
    f"Evolution component mean : {evolution_rank.mean():.6f}"
)

print("\n✓ Initial NKVEL fusion calculated")

NKVEL V1 — STATE + EVOLUTION
ROC-AUC : 0.863839
PR-AUC  : 0.943228

Components:
State component mean     : 0.510870
Evolution component mean : 0.510870

✓ Initial NKVEL fusion calculated


In [ ]:
# ============================================================
# CELL 108 — Incremental value of evolution
# ============================================================

print("INCREMENTAL NKVEL VALUE")
print("=======================")

print(
    f"CTGAD-X state only      : "
    f"ROC-AUC={roc_auc_score(nkvel_labels, state_signal):.6f} | "
    f"PR-AUC={average_precision_score(nkvel_labels, state_signal):.6f}"
)

print(
    f"Evolution only          : "
    f"ROC-AUC={roc_auc_score(nkvel_labels, evolution_signal):.6f} | "
    f"PR-AUC={average_precision_score(nkvel_labels, evolution_signal):.6f}"
)

print(
    f"NKVEL state + evolution : "
    f"ROC-AUC={nkvel_roc_auc:.6f} | "
    f"PR-AUC={nkvel_pr_auc:.6f}"
)

INCREMENTAL NKVEL VALUE
CTGAD-X state only      : ROC-AUC=0.877232 | PR-AUC=0.951202
Evolution only          : ROC-AUC=0.763393 | PR-AUC=0.898618
NKVEL state + evolution : ROC-AUC=0.863839 | PR-AUC=0.943228


In [ ]:
# ============================================================
# CELL 109 — NKVEL lineage persistence signals
# ============================================================

# State scores for the 47 test sequences
S = np.asarray(
    eval_test_scores
).reshape(-1)

# Signed evolution:
# positive  -> moving farther from normal
# negative  -> moving closer to normal
dS = np.diff(S)

# ------------------------------------------------------------
# 1. Positive outward movement
# ------------------------------------------------------------

outward_movement = np.maximum(
    dS,
    0.0
)

# ------------------------------------------------------------
# 2. Cumulative outward movement
# ------------------------------------------------------------

cumulative_outward = np.cumsum(
    outward_movement
)

# ------------------------------------------------------------
# 3. Consecutive outward movement
# ------------------------------------------------------------

consecutive_outward = np.zeros_like(dS)

run = 0

for i, value in enumerate(dS):

    if value > 0:
        run += 1
    else:
        run = 0

    consecutive_outward[i] = run

# ------------------------------------------------------------
# 4. Rolling 3-step evolution magnitude
# ------------------------------------------------------------

abs_dS = np.abs(dS)

rolling_evolution = np.zeros_like(abs_dS)

for i in range(len(abs_dS)):

    start = max(0, i - 2)

    rolling_evolution[i] = np.mean(
        abs_dS[start:i+1]
    )

print("NKVEL LINEAGE SIGNALS")
print("=====================")

print(
    "Outward movement mean:",
    outward_movement.mean()
)

print(
    "Cumulative outward max:",
    cumulative_outward.max()
)

print(
    "Maximum consecutive outward steps:",
    consecutive_outward.max()
)

print(
    "Rolling evolution mean:",
    rolling_evolution.mean()
)

print("\n✓ Lineage signals calculated")

NKVEL LINEAGE SIGNALS
Outward movement mean: 0.0136767
Cumulative outward max: 0.6291283
Maximum consecutive outward steps: 4.0
Rolling evolution mean: 0.02785696

✓ Lineage signals calculated


In [ ]:
# ============================================================
# CELL 110 — Evaluate lineage persistence
# ============================================================

# Labels aligned with transitions
lineage_labels = np.maximum(
    all_test_labels[:-1],
    all_test_labels[1:]
)

signals = {
    "Outward movement": outward_movement,
    "Cumulative outward": cumulative_outward,
    "Consecutive outward": consecutive_outward,
    "Rolling evolution": rolling_evolution
}

print("NKVEL LINEAGE SIGNAL RESULTS")
print("=============================")

for name, signal in signals.items():

    auc = roc_auc_score(
        lineage_labels,
        signal
    )

    pr = average_precision_score(
        lineage_labels,
        signal
    )

    print(
        f"{name:24s} "
        f"ROC-AUC: {auc:.6f} | "
        f"PR-AUC: {pr:.6f}"
    )

NKVEL LINEAGE SIGNAL RESULTS
Outward movement         ROC-AUC: 0.543527 | PR-AUC: 0.790148
Cumulative outward       ROC-AUC: 0.004464 | PR-AUC: 0.507558
Consecutive outward      ROC-AUC: 0.492188 | PR-AUC: 0.705645
Rolling evolution        ROC-AUC: 0.839286 | PR-AUC: 0.936439


In [ ]:
# ============================================================
# CELL 111 — STRICT 3-STEP NKVEL EVOLUTION
# ============================================================

S = np.asarray(
    eval_test_scores
).reshape(-1)

# State-score changes
dS = np.diff(S)

# Absolute behavioural evolution
abs_dS = np.abs(dS)

# Strict 3-transition rolling evolution
strict_rolling_evolution = np.array([
    np.mean(
        abs_dS[i-2:i+1]
    )
    for i in range(2, len(abs_dS))
])

# Corresponding sequence labels
strict_labels = np.maximum.reduce([
    all_test_labels[:-3],
    all_test_labels[1:-2],
    all_test_labels[2:-1],
    all_test_labels[3:]
])

print("STRICT NKVEL 3-STEP EVOLUTION")
print("==============================")

print(
    "Evolution observations:",
    len(strict_rolling_evolution)
)

print(
    "Labels:",
    len(strict_labels)
)

print(
    "Mean:",
    strict_rolling_evolution.mean()
)

print(
    "Std:",
    strict_rolling_evolution.std()
)

print(
    "Min:",
    strict_rolling_evolution.min()
)

print(
    "Max:",
    strict_rolling_evolution.max()
)

print("\n✓ Strict temporal alignment verified")

STRICT NKVEL 3-STEP EVOLUTION
Evolution observations: 44
Labels: 44
Mean: 0.027269216
Std: 0.017840428
Min: 0.0038828615
Max: 0.067073315

✓ Strict temporal alignment verified


In [ ]:
# ============================================================
# CELL 112 — STRICT NKVEL EVALUATION
# ============================================================

strict_roc_auc = roc_auc_score(
    strict_labels,
    strict_rolling_evolution
)

strict_pr_auc = average_precision_score(
    strict_labels,
    strict_rolling_evolution
)

print("STRICT NKVEL EVOLUTION")
print("======================")

print(
    f"ROC-AUC : {strict_roc_auc:.6f}"
)

print(
    f"PR-AUC  : {strict_pr_auc:.6f}"
)

STRICT NKVEL EVOLUTION
ROC-AUC : 0.843750
PR-AUC  : 0.943758


In [ ]:
# ============================================================
# CELL 113 — NKVEL normal training state scores
# ============================================================

# Normal training embeddings
train_z = F.normalize(
    train_embeddings,
    p=2,
    dim=1
)

# Normal reference bank
reference_z = F.normalize(
    train_embeddings,
    p=2,
    dim=1
)

# Cosine distance matrix
train_distances = (
    1.0 -
    train_z @ reference_z.T
)

# Exclude self-neighbour
train_distances.fill_diagonal_(
    float("inf")
)

# k nearest normal neighbours
k = 5

train_knn_distances, _ = torch.topk(
    train_distances,
    k=k,
    largest=False,
    dim=1
)

normal_train_state_scores = (
    train_knn_distances.mean(dim=1)
    .detach()
    .cpu()
    .numpy()
)

print("NKVEL NORMAL TRAIN STATE SCORES")
print("================================")

print(
    "Shape:",
    normal_train_state_scores.shape
)

print(
    "Mean:",
    normal_train_state_scores.mean()
)

print(
    "Std:",
    normal_train_state_scores.std()
)

print(
    "Min:",
    normal_train_state_scores.min()
)

print(
    "Max:",
    normal_train_state_scores.max()
)

print("\n✓ Leakage-safe normal state scores constructed")

NKVEL NORMAL TRAIN STATE SCORES
Shape: (174,)
Mean: 0.05399227
Std: 0.056989558
Min: 0.0001630187
Max: 0.47378713

✓ Leakage-safe normal state scores constructed


In [ ]:
# ============================================================
# CELL 114 — NKVEL normal evolution reference
# ============================================================

# Absolute state-score evolution
normal_dS = np.abs(
    np.diff(
        normal_train_state_scores
    )
)

# Strict 3-step local evolution
normal_local_evolution = np.array([
    np.mean(
        normal_dS[i-2:i+1]
    )
    for i in range(
        2,
        len(normal_dS)
    )
])

print("NKVEL NORMAL EVOLUTION REFERENCE")
print("=================================")

print(
    "Observations:",
    len(normal_local_evolution)
)

print(
    "Mean:",
    normal_local_evolution.mean()
)

print(
    "Std:",
    normal_local_evolution.std()
)

print(
    "Min:",
    normal_local_evolution.min()
)

print(
    "Max:",
    normal_local_evolution.max()
)

print("\n✓ Normal evolution reference constructed")

NKVEL NORMAL EVOLUTION REFERENCE
Observations: 171
Mean: 0.028100647
Std: 0.02628127
Min: 0.0019511502
Max: 0.19109416

✓ Normal evolution reference constructed


In [ ]:
# ============================================================
# CELL 115 — Leakage-safe NKVEL threshold
# ============================================================

nkvel_evolution_threshold_95 = np.percentile(
    normal_local_evolution,
    95
)

nkvel_evolution_threshold_99 = np.percentile(
    normal_local_evolution,
    99
)

print("NKVEL EVOLUTION THRESHOLDS")
print("===========================")

print(
    f"95th percentile: "
    f"{nkvel_evolution_threshold_95:.6f}"
)

print(
    f"99th percentile: "
    f"{nkvel_evolution_threshold_99:.6f}"
)

NKVEL EVOLUTION THRESHOLDS
95th percentile: 0.076359
99th percentile: 0.117666


In [ ]:
# ============================================================
# CELL 116 — NKVEL validation evolution
# ============================================================

# ------------------------------------------------------------
# Validation embeddings
# ------------------------------------------------------------

val_z = F.normalize(
    all_val_embeddings,
    p=2,
    dim=1
)

# ------------------------------------------------------------
# Validation CTGAD-X state scores
# ------------------------------------------------------------

val_state_scores = np.asarray(
    eval_val_scores
).reshape(-1)

assert len(val_z) == len(val_state_scores)
assert len(val_z) == len(all_test_labels[:0]) + 22  # sanity only

# ------------------------------------------------------------
# Absolute state-score evolution
# ------------------------------------------------------------

val_dS = np.abs(
    np.diff(val_state_scores)
)

# ------------------------------------------------------------
# Strict 3-transition local evolution
# ------------------------------------------------------------

val_local_evolution = np.array([
    np.mean(
        val_dS[i-2:i+1]
    )
    for i in range(
        2,
        len(val_dS)
    )
])

print("NKVEL VALIDATION EVOLUTION")
print("============================")

print(
    "Validation sequences:",
    len(val_state_scores)
)

print(
    "Strict evolution observations:",
    len(val_local_evolution)
)

print(
    "Mean:",
    val_local_evolution.mean()
)

print(
    "Std:",
    val_local_evolution.std()
)

print(
    "Min:",
    val_local_evolution.min()
)

print(
    "Max:",
    val_local_evolution.max()
)

print("\n✓ Validation evolution calculated")

NKVEL VALIDATION EVOLUTION
Validation sequences: 22
Strict evolution observations: 19
Mean: 0.035387184
Std: 0.014548442
Min: 0.013957416
Max: 0.059490547

✓ Validation evolution calculated


In [ ]:
# ============================================================
# CELL 117 — Apply frozen NKVEL thresholds to validation
# ============================================================

val_flag_95 = (
    val_local_evolution
    >= nkvel_evolution_threshold_95
)

val_flag_99 = (
    val_local_evolution
    >= nkvel_evolution_threshold_99
)

print("FROZEN NKVEL VALIDATION CHECK")
print("==============================")

print(
    f"95th threshold: "
    f"{nkvel_evolution_threshold_95:.6f}"
)

print(
    f"99th threshold: "
    f"{nkvel_evolution_threshold_99:.6f}"
)

print("\n95th percentile threshold:")
print(
    f"Flagged: {val_flag_95.sum()} / "
    f"{len(val_flag_95)}"
)

print(
    f"Rate: {val_flag_95.mean():.4f}"
)

print("\n99th percentile threshold:")
print(
    f"Flagged: {val_flag_99.sum()} / "
    f"{len(val_flag_99)}"
)

print(
    f"Rate: {val_flag_99.mean():.4f}"
)

FROZEN NKVEL VALIDATION CHECK
95th threshold: 0.076359
99th threshold: 0.117666

95th percentile threshold:
Flagged: 0 / 19
Rate: 0.0000

99th percentile threshold:
Flagged: 0 / 19
Rate: 0.0000


In [ ]:
# ============================================================
# CELL 118 — FINAL FROZEN NKVEL TEST EVOLUTION
# ============================================================

# ------------------------------------------------------------
# Frozen CTGAD-X test state scores
# ------------------------------------------------------------

S_test = np.asarray(
    eval_test_scores
).reshape(-1)

assert len(S_test) == 47

# ------------------------------------------------------------
# Absolute state evolution
# ------------------------------------------------------------

test_dS = np.abs(
    np.diff(S_test)
)

# ------------------------------------------------------------
# STRICT 3-STEP EVOLUTION
# ------------------------------------------------------------

test_local_evolution = np.array([
    np.mean(
        test_dS[i-2:i+1]
    )
    for i in range(
        2,
        len(test_dS)
    )
])

# ------------------------------------------------------------
# Corresponding labels
# ------------------------------------------------------------

test_nkvel_labels = np.maximum.reduce([
    all_test_labels[:-3],
    all_test_labels[1:-2],
    all_test_labels[2:-1],
    all_test_labels[3:]
])

assert len(test_local_evolution) == 44
assert len(test_nkvel_labels) == 44

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

final_evolution_roc = roc_auc_score(
    test_nkvel_labels,
    test_local_evolution
)

final_evolution_pr = average_precision_score(
    test_nkvel_labels,
    test_local_evolution
)

print("FINAL FROZEN NKVEL EVOLUTION")
print("=============================")

print(
    f"Observations : {len(test_local_evolution)}"
)

print(
    f"ROC-AUC      : {final_evolution_roc:.6f}"
)

print(
    f"PR-AUC       : {final_evolution_pr:.6f}"
)

print("\nFrozen thresholds")

print(
    f"95th percentile: "
    f"{nkvel_evolution_threshold_95:.6f}"
)

print(
    f"99th percentile: "
    f"{nkvel_evolution_threshold_99:.6f}"
)

print("\nThreshold detections")

print(
    "95th:",
    int(
        np.sum(
            test_local_evolution
            >= nkvel_evolution_threshold_95
        )
    ),
    "/",
    len(test_local_evolution)
)

print(
    "99th:",
    int(
        np.sum(
            test_local_evolution
            >= nkvel_evolution_threshold_99
        )
    ),
    "/",
    len(test_local_evolution)
)

FINAL FROZEN NKVEL EVOLUTION
Observations : 44
ROC-AUC      : 0.843750
PR-AUC       : 0.943758

Frozen thresholds
95th percentile: 0.076359
99th percentile: 0.117666

Threshold detections
95th: 0 / 44
99th: 0 / 44


In [ ]:
# ============================================================
# CELL 119 — CTGAD-X vs FROZEN NKVEL
# ============================================================

# ------------------------------------------------------------
# CTGAD-X state score aligned to same 44 observations
# ------------------------------------------------------------

test_state_aligned = S_test[3:]

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

ctgad_roc = roc_auc_score(
    test_nkvel_labels,
    test_state_aligned
)

ctgad_pr = average_precision_score(
    test_nkvel_labels,
    test_state_aligned
)

print("CTGAD-X vs NKVEL")
print("=================")

print(
    f"CTGAD-X State      "
    f"ROC-AUC: {ctgad_roc:.6f} | "
    f"PR-AUC: {ctgad_pr:.6f}"
)

print(
    f"NKVEL Evolution    "
    f"ROC-AUC: {final_evolution_roc:.6f} | "
    f"PR-AUC: {final_evolution_pr:.6f}"
)

print("\nImprovement")

print(
    f"ROC-AUC Δ: "
    f"{final_evolution_roc - ctgad_roc:+.6f}"
)

print(
    f"PR-AUC Δ: "
    f"{final_evolution_pr - ctgad_pr:+.6f}"
)

CTGAD-X vs NKVEL
CTGAD-X State      ROC-AUC: 0.802083 | PR-AUC: 0.932590
NKVEL Evolution    ROC-AUC: 0.843750 | PR-AUC: 0.943758

Improvement
ROC-AUC Δ: +0.041667
PR-AUC Δ: +0.011168


In [ ]:
# ============================================================
# CELL 120A — Find existing dataframe variables
# ============================================================

import pandas as pd

print("DataFrame variables in the current notebook:")
print("==============================================")

for name, obj in globals().items():

    if isinstance(obj, pd.DataFrame):

        print(
            f"{name:35s} "
            f"shape={obj.shape}"
        )

DataFrame variables in the current notebook:
raw_sample                          shape=(5, 49)
df                                  shape=(200000, 52)
normal_df                           shape=(177785, 52)
feature_df                          shape=(9402, 11)
wdf                                 shape=(690, 52)
window_label_df                     shape=(255, 6)
test_score_table                    shape=(47, 5)
analysis_df                         shape=(47, 5)
attack_analysis                     shape=(32, 6)
test_sequence_analysis_df           shape=(47, 8)
missed_attacks_df                   shape=(27, 8)
attack_df                           shape=(22215, 52)
test_attack_df                      shape=(2088, 52)
train_attack_df                     shape=(17919, 52)
val_attack_df                       shape=(2218, 52)
category_df                         shape=(207, 5)
category_summary                    shape=(9, 7)


In [ ]:
# ============================================================
# CELL 120 — Attack-family distribution across temporal splits
# ============================================================

# Use the existing 200k development dataframe
df_dev = df.copy()

# ------------------------------------------------------------
# Build window → temporal split mapping
# ------------------------------------------------------------

train_window_set = set(
    train_snapshot_idx
)

val_window_set = set(
    val_snapshot_idx
)

test_window_set = set(
    test_snapshot_idx
)


def get_temporal_split(window_idx):

    if window_idx in train_window_set:
        return "train"

    elif window_idx in val_window_set:
        return "validation"

    elif window_idx in test_window_set:
        return "test"

    return "unknown"


# ------------------------------------------------------------
# Assign each flow to its temporal split
# ------------------------------------------------------------

df_dev["temporal_split"] = (
    df_dev["window_start_idx"]
    .apply(get_temporal_split)
)

# ------------------------------------------------------------
# Attack-only data
# ------------------------------------------------------------

attack_df_dev = df_dev[
    df_dev["label"] == 1
].copy()

# ------------------------------------------------------------
# Distribution
# ------------------------------------------------------------

family_distribution = pd.crosstab(
    attack_df_dev["attack_cat"],
    attack_df_dev["temporal_split"]
)

print("ATTACK-FAMILY TEMPORAL DISTRIBUTION")
print("====================================")

print(
    family_distribution
)

print("\nFamilies present in each split:")
print("--------------------------------")

for split in [
    "train",
    "validation",
    "test"
]:

    families = sorted(
        attack_df_dev.loc[
            attack_df_dev["temporal_split"] == split,
            "attack_cat"
        ]
        .dropna()
        .unique()
    )

    print(
        f"{split:12s}: {families}"
    )

ATTACK-FAMILY TEMPORAL DISTRIBUTION
temporal_split  test  train  validation
attack_cat                             
Analysis           2    468          56
Backdoors          9    516           9
DoS               77   1006          84
Exploits         553   4352         504
Fuzzers          456   4217         378
Generic          725   5846         951
Reconnaissance   226   1321         212
Shellcode         29    172          22
Worms              1     21           2

Families present in each split:
--------------------------------
train       : ['Analysis', 'Backdoors', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Reconnaissance', 'Shellcode', 'Worms']
validation  : ['Analysis', 'Backdoors', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Reconnaissance', 'Shellcode', 'Worms']
test        : ['Analysis', 'Backdoors', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Reconnaissance', 'Shellcode', 'Worms']


In [ ]:
# ============================================================
# CELL 121 — Inspect existing graph/model pipeline functions
# ============================================================

import inspect

keywords = [
    "graph",
    "snapshot",
    "sequence",
    "pyg",
    "train",
    "encoder"
]

print("RELEVANT FUNCTIONS")
print("==================")

# Take a static snapshot first
global_items = list(globals().items())

for name, obj in global_items:

    if callable(obj) and not name.startswith("_"):

        name_lower = name.lower()

        if any(
            keyword in name_lower
            for keyword in keywords
        ):

            try:
                print(
                    f"\n{name}"
                )

                print(
                    f"  signature: "
                    f"{inspect.signature(obj)}"
                )

            except Exception:
                print(
                    f"\n{name}"
                )

RELEVANT FUNCTIONS

snapshots_to_sequences
  signature: (snapshots: 'List[GraphSnapshot]', seq_len: 'int' = 5, stride: 'int' = 1)

snapshot_to_data
  signature: (snap, scaler)

sequence_to_data
  signature: (sequence, scaler)

batch_sequences
  signature: (sequences, scaler)

CTGAD_Encoder
  signature: (in_dim: int = 11, hidden_dim: int = 60, seq_len: int = 5, struct_heads: int = 4, temporal_heads: int = 4)

sequence_window_set
  signature: (sequences)

make_sequences_from_window_ids
  signature: (window_ids, seq_len)

encoder
  signature: (*args, **kwargs)

make_partition_sequences
  signature: (snapshots, indices, seq_len=5)

sequence_window_ids
  signature: (sequences)

snapshot_to_pyg
  signature: (snapshot, scaler)

convert_sequences
  signature: (sequences, scaler)

graph
  signature: (*args: str) -> Iterable

batch_temporal_sequences
  signature: (graph_sequences, batch_size)

graph_batch
  signature: (*args: str) -> Iterable

get_sequence_embedding
  signature: (temporal_batch)

In [ ]:
# ============================================================
# CELL 122 — Inspect existing snapshot construction
# ============================================================

print("EXISTING SNAPSHOT OBJECT")
print("========================")

print(
    "Number of snapshots:",
    len(snapshots)
)

print(
    "Snapshot type:",
    type(snapshots[0])
)

print(
    "First snapshot:"
)

print(
    snapshots[0]
)

print("\nGraphSnapshot fields:")
print(
    snapshots[0].__dict__.keys()
)

EXISTING SNAPSHOT OBJECT
Number of snapshots: 255
Snapshot type: <class 'ctgad_graph_slicer.GraphSnapshot'>
First snapshot:
GraphSnapshot(window_idx=0, t_start=1421927377.0, t_end=1421927407.0, node_ids=['10.40.182.1', '224.0.0.5', '10.40.85.1'], edges=[(2, 1), (0, 1)], node_features={'10.40.182.1': array([460.76004  ,   0.       ,   7.1993756,   0.       ,   0.       ,
         0.       ,  61.43467  ,   0.       ,   1.       ,   0.       ,
        60.       ], dtype=float32), '224.0.0.5': array([921.52   ,   0.     ,  14.39875,   0.     ,   0.     ,   0.     ,
        61.43467,   0.     ,   1.     ,   0.     , 120.     ],
      dtype=float32), '10.40.85.1': array([460.76    ,   0.      ,   7.199375,   0.      ,   0.      ,
         0.      ,  61.434666,   0.      ,   1.      ,   0.      ,
        60.      ], dtype=float32)})

GraphSnapshot fields:
dict_keys(['window_idx', 't_start', 't_end', 'node_ids', 'edges', 'node_features'])


In [ ]:
# ============================================================
# CELL 123 — Find snapshot construction function
# ============================================================

import inspect

# Find functions whose source mentions GraphSnapshot
for name, obj in list(globals().items()):

    if callable(obj):

        try:
            source = inspect.getsource(obj)

            if (
                "GraphSnapshot" in source
                or "snapshots" in name.lower()
            ):
                print("\n" + "=" * 70)
                print(name)
                print("=" * 70)
                print(source[:6000])

        except (
            TypeError,
            OSError,
            IOError
        ):
            pass


snapshots_to_sequences
                        "sttl",


snapshot_to_pyg
def snapshot_to_pyg(snapshot, scaler):
    """
    Convert one CTGAD GraphSnapshot into a PyTorch Geometric graph.

    Important:
    GraphSnapshot.edges already stores integer node indices.
    Therefore, edges must NOT be remapped using node IDs.
    """

    # --------------------------------------------------------
    # Preserve the node ordering used by node_features
    # --------------------------------------------------------
    node_ids = list(snapshot.node_features.keys())

    # --------------------------------------------------------
    # Build node feature matrix
    # --------------------------------------------------------
    X = [
        snapshot.node_features[node_id]
        for node_id in node_ids
    ]

    # Training-only fitted scaler
    X = scaler.transform(X)

    X = torch.tensor(
        X,
        dtype=torch.float32
    )

    # --------------------------------------------------

In [ ]:
# ============================================================
# CELL 124 — Inspect graph slicer module
# ============================================================

import ctgad_graph_slicer
import inspect

print("MODULE:")
print(ctgad_graph_slicer)

print("\nMODULE FILE:")
print(inspect.getfile(ctgad_graph_slicer))

print("\nPUBLIC CALLABLES:")
print("=================")

for name, obj in vars(ctgad_graph_slicer).items():

    if callable(obj) and not name.startswith("_"):

        try:
            print(
                f"{name:<40} "
                f"{inspect.signature(obj)}"
            )
        except Exception:
            print(name)

MODULE:
<module 'ctgad_graph_slicer' from '/content/fedctgad_x_v2/project/fedctgad/data/slicing.py'>

MODULE FILE:
/content/fedctgad_x_v2/project/fedctgad/data/slicing.py

PUBLIC CALLABLES:
dataclass                                (cls=None, /, *, init=True, repr=True, eq=True, order=False, unsafe_hash=False, frozen=False, match_args=True, kw_only=False, slots=False, weakref_slot=False)
field                                    (*, default=<dataclasses._MISSING_TYPE object at 0x7f751f69aa50>, default_factory=<dataclasses._MISSING_TYPE object at 0x7f751f69aa50>, init=True, repr=True, hash=None, compare=True, metadata=None, kw_only=<dataclasses._MISSING_TYPE object at 0x7f751f69aa50>)
Dict                                     (*args, **kwargs)
List                                     (*args, **kwargs)
Tuple                                    (*args, **kwargs)
GraphSnapshot                            (window_idx: 'int', t_start: 'float', t_end: 'float', node_ids: 'List[str]' = <factory>, ed

In [ ]:
# ============================================================
# CELL 125 — Inspect likely graph slicing functions
# ============================================================

import inspect
import ctgad_graph_slicer

for name, obj in vars(ctgad_graph_slicer).items():

    if callable(obj) and not name.startswith("_"):

        try:
            source = inspect.getsource(obj)

            keywords = [
                "overlap",
                "window",
                "GraphSnapshot",
                "node_features",
                "t_start",
                "t_end"
            ]

            if any(k in source for k in keywords):

                print("\n" + "=" * 90)
                print(name)
                print("=" * 90)
                print(source[:12000])

        except Exception:
            pass


GraphSnapshot
@dataclass
class GraphSnapshot:

    window_idx: int
    t_start: float
    t_end: float

    node_ids: List[str] = field(default_factory=list)

    edges: List[Tuple[int, int]] = field(default_factory=list)

    node_features: Dict[str, np.ndarray] = field(default_factory=dict)


compute_overlap
def compute_overlap(
    t_s: float,
    dur: float,
    w_start: float,
    delta_t: float
) -> float:

    t_e = t_s + dur
    w_end = w_start + delta_t

    return max(
        0.0,
        min(t_e, w_end) -
        max(t_s, w_start)
    )


slice_flows_to_windows
def slice_flows_to_windows(
    flows: pd.DataFrame,
    delta_t: float = 30.0,
    ts_col: str = "ts_start",
    dur_col: str = "duration",
    src_col: str = "src_ip",
    dst_col: str = "dst_ip",
    global_t0: float | None = None,
) -> List[GraphSnapshot]:

    if flows.empty:
        return []

    flows = flows.copy()

    # --------------------------------------------------------
    # Temporal origin
    # -

In [ ]:
# ============================================================
# CELL 126 — Recover original temporal origin
# ============================================================

ORIGINAL_GLOBAL_T0 = float(snapshots[0].t_start)

print("Original global_t0:", ORIGINAL_GLOBAL_T0)
print("First snapshot start:", snapshots[0].t_start)
print("First snapshot end:", snapshots[0].t_end)

assert snapshots[0].t_start == ORIGINAL_GLOBAL_T0

print("\nTemporal origin recovered successfully.")

Original global_t0: 1421927377.0
First snapshot start: 1421927377.0
First snapshot end: 1421927407.0

Temporal origin recovered successfully.


In [ ]:
# ============================================================
# CELL 127 — Verify original 30-second window grid
# ============================================================

WINDOW_SIZE = 30.0

window_starts = np.array([
    s.t_start for s in snapshots
], dtype=float)

window_ends = np.array([
    s.t_end for s in snapshots
], dtype=float)

window_indices = np.array([
    s.window_idx for s in snapshots
], dtype=int)

print("Number of snapshots:", len(snapshots))
print("Window size:", WINDOW_SIZE)

print("\nFirst 5 windows:")
for i in range(min(5, len(snapshots))):
    print(
        f"window={window_indices[i]:3d} | "
        f"{window_starts[i]:.1f} -> {window_ends[i]:.1f}"
    )

print("\nLast window:")
print(
    f"window={window_indices[-1]} | "
    f"{window_starts[-1]:.1f} -> {window_ends[-1]:.1f}"
)

# Check 30-second spacing
start_diffs = np.diff(window_starts)

print("\nUnique start differences:", np.unique(start_diffs))

assert np.allclose(start_diffs, WINDOW_SIZE)
assert np.allclose(
    window_ends - window_starts,
    WINDOW_SIZE
)

print("\nWindow grid verified.")

Number of snapshots: 255
Window size: 30.0

First 5 windows:
window=  0 | 1421927377.0 -> 1421927407.0
window=  1 | 1421927407.0 -> 1421927437.0
window=  2 | 1421927437.0 -> 1421927467.0
window=  3 | 1421927467.0 -> 1421927497.0
window=  4 | 1421927497.0 -> 1421927527.0

Last window:
window=254 | 1421934997.0 -> 1421935027.0

Unique start differences: [30.]

Window grid verified.


In [ ]:
# ============================================================
# CELL 128 — Identify attack/category columns
# ============================================================

print("DataFrame shape:", df.shape)

print("\nColumns containing attack/category/label information:")
print("=" * 70)

for col in df.columns:

    col_lower = str(col).lower()

    if any(
        key in col_lower
        for key in [
            "attack",
            "category",
            "label",
            "class"
        ]
    ):
        print(
            f"{col:<35} "
            f"dtype={df[col].dtype}"
        )

DataFrame shape: (200000, 52)

Columns containing attack/category/label information:
attack_cat                          dtype=object
label                               dtype=int64


In [ ]:
# ============================================================
# CELL 129 — Inspect attack category distribution
# ============================================================

CATEGORY_COL = "attack_cat"

print("Using category column:", CATEGORY_COL)

print("\nUnique values:")
print(
    df[CATEGORY_COL]
    .value_counts(dropna=False)
)

print("\nNumber of unique values:",
      df[CATEGORY_COL].nunique(dropna=False))

Using category column: attack_cat

Unique values:
attack_cat
Normal            177785
Generic             7522
Exploits            5409
Fuzzers             5051
Reconnaissance      1759
DoS                 1167
Backdoors            534
Analysis             526
Shellcode            223
Worms                 24
Name: count, dtype: int64

Number of unique values: 10


In [ ]:
# ============================================================
# CELL 130 — Inspect attack categories + label relationship
# ============================================================

print("Attack category values:")
print(
    sorted(
        df[CATEGORY_COL]
        .dropna()
        .astype(str)
        .unique()
    )
)

print("\nCategory × label:")
print(
    pd.crosstab(
        df[CATEGORY_COL].fillna("NaN"),
        df["label"]
    )
)

Attack category values:
['Analysis', 'Backdoors', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Normal', 'Reconnaissance', 'Shellcode', 'Worms']

Category × label:
label                0     1
attack_cat                  
Analysis             0   526
Backdoors            0   534
DoS                  0  1167
Exploits             0  5409
Fuzzers              0  5051
Generic              0  7522
Normal          177785     0
Reconnaissance       0  1759
Shellcode            0   223
Worms                0    24


In [ ]:
# ============================================================
# CELL 131 — Define LOFO configuration
# ============================================================

HELD_OUT_FAMILY = "Shellcode"

CATEGORY_COL = "attack_cat"
LABEL_COL = "label"

print("Held-out attack family:", HELD_OUT_FAMILY)
print("Category column:", CATEGORY_COL)
print("Label column:", LABEL_COL)

Held-out attack family: Shellcode
Category column: attack_cat
Label column: label


In [ ]:
# ============================================================
# CELL 132 — Chronological LOFO dataframe split
# ============================================================

train_start = float(snapshots[0].t_start)
train_end = float(snapshots[177].t_end)

val_start = float(snapshots[178].t_start)
val_end = float(snapshots[203].t_end)

test_start = float(snapshots[204].t_start)
test_end = float(snapshots[254].t_end)

print("TRAIN:")
print(train_start, "->", train_end)

print("\nVALIDATION:")
print(val_start, "->", val_end)

print("\nTEST:")
print(test_start, "->", test_end)

TRAIN:
1421927377.0 -> 1421932717.0

VALIDATION:
1421932717.0 -> 1421933497.0

TEST:
1421933497.0 -> 1421935027.0


In [ ]:
# ============================================================
# CELL 133 — Identify temporal columns
# ============================================================

print("Temporal columns:")
print("=" * 70)

for col in df.columns:

    name = str(col).lower()

    if any(
        key in name
        for key in [
            "time",
            "ts",
            "start",
            "duration",
            "dur"
        ]
    ):
        print(
            f"{col:<35} dtype={df[col].dtype}"
        )

Temporal columns:
duration                            dtype=float64
spkts                               dtype=int64
dpkts                               dtype=int64
ts_start                            dtype=int64
ltime                               dtype=int64
is_sm_ips_ports                     dtype=int64
window_start_idx                    dtype=int64


In [ ]:
# ============================================================
# CELL 134 — Build chronological dataframe partitions
# ============================================================

TS_COL = "ts_start"

# Ensure numeric timestamp
df[TS_COL] = pd.to_numeric(
    df[TS_COL],
    errors="coerce"
)

assert df[TS_COL].notna().all()

# Existing snapshot-derived temporal boundaries
TRAIN_START = train_start
TRAIN_END   = train_end

VAL_START = val_start
VAL_END   = val_end

TEST_START = test_start
TEST_END   = test_end

# ------------------------------------------------------------
# Chronological partitions
# ------------------------------------------------------------

train_base_df = df[
    (df[TS_COL] >= TRAIN_START) &
    (df[TS_COL] <  TRAIN_END)
].copy()

val_base_df = df[
    (df[TS_COL] >= VAL_START) &
    (df[TS_COL] <  VAL_END)
].copy()

test_base_df = df[
    (df[TS_COL] >= TEST_START) &
    (df[TS_COL] <  TEST_END)
].copy()

print("TRAIN:", train_base_df.shape)
print("VAL  :", val_base_df.shape)
print("TEST :", test_base_df.shape)

print("\nTotal partitioned rows:",
      len(train_base_df) +
      len(val_base_df) +
      len(test_base_df))

print("Original df rows:", len(df))

TRAIN: (143045, 52)
VAL  : (20382, 52)
TEST : (36573, 52)

Total partitioned rows: 200000
Original df rows: 200000


In [ ]:
# ============================================================
# CELL 135 — Verify temporal partition integrity
# ============================================================

print("TRAIN range:")
print(
    train_base_df[TS_COL].min(),
    "->",
    train_base_df[TS_COL].max()
)

print("\nVAL range:")
print(
    val_base_df[TS_COL].min(),
    "->",
    val_base_df[TS_COL].max()
)

print("\nTEST range:")
print(
    test_base_df[TS_COL].min(),
    "->",
    test_base_df[TS_COL].max()
)

# No temporal overlap
assert train_base_df[TS_COL].max() < val_base_df[TS_COL].min()
assert val_base_df[TS_COL].max() < test_base_df[TS_COL].min()

print("\n✓ Chronological separation verified.")

TRAIN range:
1421927377 -> 1421932716

VAL range:
1421932717 -> 1421933496

TEST range:
1421933497 -> 1421935026

✓ Chronological separation verified.


In [ ]:
# ============================================================
# CELL 136 — Create genuine LOFO training dataframe
# ============================================================

train_lofo_df = train_base_df[
    train_base_df[CATEGORY_COL] != HELD_OUT_FAMILY
].copy()

print("Held-out family:", HELD_OUT_FAMILY)

print("\nOriginal training distribution:")
print(
    train_base_df[CATEGORY_COL]
    .value_counts()
)

print("\nLOFO training distribution:")
print(
    train_lofo_df[CATEGORY_COL]
    .value_counts()
)

Held-out family: Shellcode

Original training distribution:
attack_cat
Normal            125126
Generic             5846
Exploits            4352
Fuzzers             4217
Reconnaissance      1321
DoS                 1006
Backdoors            516
Analysis             468
Shellcode            172
Worms                 21
Name: count, dtype: int64

LOFO training distribution:
attack_cat
Normal            125126
Generic             5846
Exploits            4352
Fuzzers             4217
Reconnaissance      1321
DoS                 1006
Backdoors            516
Analysis             468
Worms                 21
Name: count, dtype: int64


In [ ]:
# ============================================================
# CELL 137 — Hard LOFO leakage checks
# ============================================================

# 1. No held-out family in training
held_out_remaining = (
    train_lofo_df[CATEGORY_COL] == HELD_OUT_FAMILY
).sum()

assert held_out_remaining == 0

# 2. Normal traffic must remain
normal_train_count = (
    train_lofo_df[CATEGORY_COL] == "Normal"
).sum()

assert normal_train_count > 0

# 3. Training still contains attacks
attack_train_count = (
    train_lofo_df[LABEL_COL] == 1
).sum()

assert attack_train_count > 0

# 4. Validation/test must retain held-out family
val_heldout_count = (
    val_base_df[CATEGORY_COL] == HELD_OUT_FAMILY
).sum()

test_heldout_count = (
    test_base_df[CATEGORY_COL] == HELD_OUT_FAMILY
).sum()

print("Held-out family in TRAIN:", held_out_remaining)
print("Held-out family in VAL  :", val_heldout_count)
print("Held-out family in TEST :", test_heldout_count)

print("\nNormal rows remaining in LOFO TRAIN:",
      normal_train_count)

print("Attack rows remaining in LOFO TRAIN:",
      attack_train_count)

assert val_heldout_count > 0
assert test_heldout_count > 0

print("\n✓ LOFO family exclusion verified.")
print("✓ Held-out family remains available for evaluation.")

Held-out family in TRAIN: 0
Held-out family in VAL  : 22
Held-out family in TEST : 29

Normal rows remaining in LOFO TRAIN: 125126
Attack rows remaining in LOFO TRAIN: 17747

✓ LOFO family exclusion verified.
✓ Held-out family remains available for evaluation.


In [ ]:
# ============================================================
# CELL 138 — Verify attack families available for training
# ============================================================

train_attack_categories = sorted(
    train_lofo_df.loc[
        train_lofo_df[LABEL_COL] == 1,
        CATEGORY_COL
    ].unique()
)

print("Attack families in LOFO TRAIN:")
for category in train_attack_categories:
    count = (
        train_lofo_df[CATEGORY_COL] == category
    ).sum()
    print(f"{category:<20} {count}")

print("\nHeld-out:", HELD_OUT_FAMILY)

assert HELD_OUT_FAMILY not in train_attack_categories

print("\n✓ Held-out family is absent from training attacks.")

Attack families in LOFO TRAIN:
Analysis             468
Backdoors            516
DoS                  1006
Exploits             4352
Fuzzers              4217
Generic              5846
Reconnaissance       1321
Worms                21

Held-out: Shellcode

✓ Held-out family is absent from training attacks.


In [ ]:
# ============================================================
# CELL 139 — Audit flows crossing partition boundaries
# ============================================================

# Flow end time
train_base_df["_ts_end"] = (
    train_base_df[TS_COL].astype(float) +
    train_base_df["duration"].astype(float).clip(lower=0.0)
)

val_base_df["_ts_end"] = (
    val_base_df[TS_COL].astype(float) +
    val_base_df["duration"].astype(float).clip(lower=0.0)
)

test_base_df["_ts_end"] = (
    test_base_df[TS_COL].astype(float) +
    test_base_df["duration"].astype(float).clip(lower=0.0)
)

# A flow beginning in TRAIN and extending beyond TRAIN_END
train_cross_val = train_base_df[
    train_base_df["_ts_end"] > TRAIN_END
].copy()

# A flow beginning in VAL and extending beyond VAL_END
val_cross_test = val_base_df[
    val_base_df["_ts_end"] > VAL_END
].copy()

print("TRAIN flows crossing TRAIN_END:", len(train_cross_val))
print("VAL flows crossing VAL_END:", len(val_cross_test))

print("\nMaximum TRAIN flow end:")
print(train_base_df["_ts_end"].max())

print("TRAIN_END:")
print(TRAIN_END)

print("\nMaximum VAL flow end:")
print(val_base_df["_ts_end"].max())

print("VAL_END:")
print(VAL_END)

TRAIN flows crossing TRAIN_END: 5
VAL flows crossing VAL_END: 18

Maximum TRAIN flow end:
1421932745.982857
TRAIN_END:
1421932717.0

Maximum VAL flow end:
1421933543.947433
VAL_END:
1421933497.0


In [ ]:
# ============================================================
# CELL 140 — Boundary-crossing family audit
# ============================================================

print("TRAIN → VAL crossing flows by attack category:")
print(
    train_cross_val[CATEGORY_COL]
    .value_counts(dropna=False)
)

print("\nVAL → TEST crossing flows by attack category:")
print(
    val_cross_test[CATEGORY_COL]
    .value_counts(dropna=False)
)

shellcode_cross_train = (
    train_cross_val[CATEGORY_COL] == HELD_OUT_FAMILY
).sum()

print(
    "\nHeld-out Shellcode flows crossing TRAIN boundary:",
    shellcode_cross_train
)

TRAIN → VAL crossing flows by attack category:
attack_cat
Normal    5
Name: count, dtype: int64

VAL → TEST crossing flows by attack category:
attack_cat
Normal            8
Exploits          4
Fuzzers           2
Backdoors         2
Reconnaissance    2
Name: count, dtype: int64

Held-out Shellcode flows crossing TRAIN boundary: 0


In [ ]:
# ============================================================
# CELL 141 — Clean temporary audit columns
# ============================================================

train_base_df.drop(
    columns=["_ts_end"],
    inplace=True
)

val_base_df.drop(
    columns=["_ts_end"],
    inplace=True
)

test_base_df.drop(
    columns=["_ts_end"],
    inplace=True
)

print("Temporary boundary-audit columns removed.")

Temporary boundary-audit columns removed.


In [ ]:
# ============================================================
# CELL 142 — Build LOFO training graph snapshots
# ============================================================

from ctgad_graph_slicer import slice_flows_to_windows

lofo_train_snapshots = slice_flows_to_windows(
    flows=train_lofo_df,
    delta_t=30.0,
    ts_col="ts_start",
    dur_col="duration",
    src_col="src_ip",
    dst_col="dst_ip",
    global_t0=ORIGINAL_GLOBAL_T0
)

print("LOFO training snapshots:", len(lofo_train_snapshots))

print("\nFirst snapshot:")
print(lofo_train_snapshots[0])

print("\nLast snapshot:")
print(lofo_train_snapshots[-1])

LOFO training snapshots: 179

First snapshot:
GraphSnapshot(window_idx=0, t_start=1421927377.0, t_end=1421927407.0, node_ids=['10.40.182.1', '224.0.0.5', '10.40.85.1'], edges=[(2, 1), (0, 1)], node_features={'10.40.182.1': array([460.76004  ,   0.       ,   7.1993756,   0.       ,   0.       ,
         0.       ,  61.43467  ,   0.       ,   1.       ,   0.       ,
        60.       ], dtype=float32), '224.0.0.5': array([921.52   ,   0.     ,  14.39875,   0.     ,   0.     ,   0.     ,
        61.43467,   0.     ,   1.     ,   0.     , 120.     ],
      dtype=float32), '10.40.85.1': array([460.76    ,   0.      ,   7.199375,   0.      ,   0.      ,
         0.      ,  61.434666,   0.      ,   1.      ,   0.      ,
        60.      ], dtype=float32)})

Last snapshot:
GraphSnapshot(window_idx=178, t_start=1421932717.0, t_end=1421932747.0, node_ids=['10.40.182.1', '10.40.182.3', '10.40.85.30', '10.40.85.1', '59.166.0.3', '149.171.126.1'], edges=[(4, 5), (2, 3), (0, 1)], node_features={'10.

In [ ]:
# ============================================================
# CELL 143 — Verify LOFO training graph integrity
# ============================================================

print("LOFO GRAPH INTEGRITY")
print("=" * 70)

print("Number of snapshots:", len(lofo_train_snapshots))

window_ids = [
    s.window_idx
    for s in lofo_train_snapshots
]

print("First window:", window_ids[0])
print("Last window:", window_ids[-1])

print("Unique windows:", len(set(window_ids)))

# Window IDs must remain chronological
assert window_ids == sorted(window_ids)

# No duplicate window IDs
assert len(window_ids) == len(set(window_ids))

# Every snapshot must contain nodes
assert all(
    len(s.node_ids) > 0
    for s in lofo_train_snapshots
)

# Every node must have an 11-D feature vector
feature_dims = set()

for snapshot in lofo_train_snapshots:
    for node, feature in snapshot.node_features.items():
        feature_dims.add(len(feature))

print("Feature dimensions:", feature_dims)

assert feature_dims == {11}

print("\n✓ LOFO graph structure verified.")

LOFO GRAPH INTEGRITY
Number of snapshots: 179
First window: 0
Last window: 178
Unique windows: 179
Feature dimensions: {11}

✓ LOFO graph structure verified.


In [ ]:
# ============================================================
# CELL 144 — Verify graph provenance
# ============================================================

print("Graph source rows:", len(train_lofo_df))

print(
    "Shellcode rows in graph source:",
    (
        train_lofo_df[CATEGORY_COL] == HELD_OUT_FAMILY
    ).sum()
)

assert (
    train_lofo_df[CATEGORY_COL] == HELD_OUT_FAMILY
).sum() == 0

print("\n✓ Graph was constructed exclusively from LOFO training data.")
print("✓ Held-out Shellcode cannot contribute to these snapshots.")

Graph source rows: 142873
Shellcode rows in graph source: 0

✓ Graph was constructed exclusively from LOFO training data.
✓ Held-out Shellcode cannot contribute to these snapshots.


In [ ]:
# ============================================================
# CELL 145 — Inspect LOFO graph temporal coverage
# ============================================================

lofo_window_ids = np.array([
    s.window_idx
    for s in lofo_train_snapshots
])

print("LOFO training window coverage:")
print(
    lofo_window_ids.min(),
    "->",
    lofo_window_ids.max()
)

print("\nExpected chronological TRAIN window range:")
print(
    train_snapshot_idx[0],
    "->",
    train_snapshot_idx[-1]
)

missing_train_windows = sorted(
    set(train_snapshot_idx) -
    set(lofo_window_ids)
)

print("\nTraining windows absent after LOFO filtering:")
print(missing_train_windows)

print(
    "\nNumber of non-empty LOFO training windows:",
    len(lofo_window_ids)
)

print(
    "Number of original TRAIN windows:",
    len(train_snapshot_idx)
)

LOFO training window coverage:
0 -> 178

Expected chronological TRAIN window range:
0 -> 177

Training windows absent after LOFO filtering:
[]

Number of non-empty LOFO training windows: 179
Number of original TRAIN windows: 178


In [ ]:
# ============================================================
# CELL 146 — Restrict LOFO graph to original TRAIN windows
# ============================================================

LOFO_TRAIN_END_WINDOW = 177

lofo_train_snapshots = [
    s
    for s in lofo_train_snapshots
    if s.window_idx <= LOFO_TRAIN_END_WINDOW
]

print("LOFO training snapshots after temporal restriction:",
      len(lofo_train_snapshots))

print(
    "Window range:",
    lofo_train_snapshots[0].window_idx,
    "->",
    lofo_train_snapshots[-1].window_idx
)

print(
    "Expected:",
    train_snapshot_idx[0],
    "->",
    train_snapshot_idx[-1]
)

assert len(lofo_train_snapshots) == len(train_snapshot_idx)

assert [
    s.window_idx for s in lofo_train_snapshots
] == list(train_snapshot_idx)

print("\n✓ LOFO training graph exactly matches TRAIN window range.")

LOFO training snapshots after temporal restriction: 178
Window range: 0 -> 177
Expected: 0 -> 177

✓ LOFO training graph exactly matches TRAIN window range.


In [ ]:
# ============================================================
# CELL 147 — Verify boundary-window handling
# ============================================================

last_train_snapshot = lofo_train_snapshots[-1]

print("Last LOFO training snapshot:")
print(last_train_snapshot)

print("\nWindow index:", last_train_snapshot.window_idx)
print("Start:", last_train_snapshot.t_start)
print("End:", last_train_snapshot.t_end)

print("\nExpected TRAIN boundary:")
print("TRAIN_END:", TRAIN_END)

assert last_train_snapshot.window_idx == 177
assert last_train_snapshot.t_end == TRAIN_END

# Ensure validation window is not present
assert all(
    s.window_idx < 178
    for s in lofo_train_snapshots
)

print("\n✓ Validation window 178 excluded from LOFO training.")
print("✓ Legitimate overlap contribution inside window 177 retained.")

Last LOFO training snapshot:
GraphSnapshot(window_idx=177, t_start=1421932687.0, t_end=1421932717.0, node_ids=['10.40.182.1', '224.0.0.5', '10.40.85.1', '175.45.176.1', '149.171.126.11', '59.166.0.1', '149.171.126.0', '175.45.176.3', '149.171.126.15', '149.171.126.4', '59.166.0.2', '59.166.0.3', '149.171.126.7', '59.166.0.0', '149.171.126.1', '59.166.0.6', '149.171.126.8', '149.171.126.9', '149.171.126.18', '149.171.126.2', '59.166.0.5', '59.166.0.4', '149.171.126.5', '175.45.176.0', '149.171.126.13', '149.171.126.3', '149.171.126.19', '149.171.126.6', '59.166.0.9', '59.166.0.7', '59.166.0.8', '175.45.176.2', '149.171.126.17', '149.171.126.10', '10.40.182.3', '149.171.126.16', '149.171.126.14', '149.171.126.12', '10.40.85.30'], edges=[(5, 9), (30, 25), (29, 19), (21, 19), (30, 16), (11, 19), (31, 33), (30, 27), (23, 18), (29, 25), (28, 9), (15, 17), (21, 25), (31, 24), (30, 14), (31, 18), (11, 25), (20, 9), (13, 17), (13, 6), (11, 16), (10, 17), (23, 4), (23, 8), (18, 3), (11, 14), (10

In [ ]:
# ============================================================
# CELL 148 — LOFO training graph statistics
# ============================================================

lofo_node_instances = sum(
    len(s.node_ids)
    for s in lofo_train_snapshots
)

lofo_edges = sum(
    len(s.edges)
    for s in lofo_train_snapshots
)

lofo_feature_values = np.concatenate([
    np.concatenate([
        np.asarray(feature, dtype=np.float32)
        for feature in s.node_features.values()
    ])
    for s in lofo_train_snapshots
])

print("LOFO TRAIN GRAPH")
print("=" * 70)

print("Snapshots:", len(lofo_train_snapshots))
print("Node instances:", lofo_node_instances)
print("Edges:", lofo_edges)

print(
    "Feature values:",
    lofo_feature_values.shape
)

print(
    "Finite:",
    np.isfinite(lofo_feature_values).all()
)

print(
    "Feature min:",
    np.min(lofo_feature_values)
)

print(
    "Feature max:",
    np.max(lofo_feature_values)
)

assert np.isfinite(lofo_feature_values).all()

print("\n✓ All LOFO graph features are finite.")

LOFO TRAIN GRAPH
Snapshots: 178
Node instances: 6723
Edges: 22384
Feature values: (73953,)
Finite: True
Feature min: 0.0
Feature max: 2410857200.0

✓ All LOFO graph features are finite.


In [ ]:
# ============================================================
# CELL 149 — Reusable LOFO CTGAD-X + NKVEL experiment
# ============================================================

import copy
import numpy as np
import pandas as pd
import torch

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_fscore_support,
    accuracy_score,
)

from ctgad_graph_slicer import slice_flows_to_windows


# ------------------------------------------------------------
# Fixed experimental configuration
# ------------------------------------------------------------

LOFO_FAMILIES = [
    "Generic",
    "Exploits",
    "Fuzzers",
    "Reconnaissance",
    "DoS",
    "Backdoors",
    "Analysis",
    "Shellcode",
]

SEQ_LEN = 5
WINDOW_SIZE = 30.0
K_NEIGHBORS = 5

TRAIN_WINDOWS = list(range(0, 178))
VAL_WINDOWS   = list(range(178, 204))
TEST_WINDOWS  = list(range(204, 255))


# ------------------------------------------------------------
# Helper: graph construction
# ------------------------------------------------------------

def build_lofo_snapshots(
    flows,
    global_t0,
    allowed_window_ids=None,
):

    snaps = slice_flows_to_windows(
        flows=flows,
        delta_t=WINDOW_SIZE,
        ts_col="ts_start",
        dur_col="duration",
        src_col="src_ip",
        dst_col="dst_ip",
        global_t0=global_t0,
    )

    if allowed_window_ids is not None:

        allowed = set(allowed_window_ids)

        snaps = [
            s for s in snaps
            if s.window_idx in allowed
        ]

    return snaps


# ------------------------------------------------------------
# Helper: create all-traffic sequences
# ------------------------------------------------------------

def build_sequences(
    snapshots,
    window_ids,
    seq_len=SEQ_LEN,
):

    lookup = {
        s.window_idx: s
        for s in snapshots
    }

    sequences = []

    ids = sorted(
        set(window_ids) &
        set(lookup.keys())
    )

    for start in range(
        len(ids) - seq_len + 1
    ):

        candidate = ids[
            start:start + seq_len
        ]

        # Require truly consecutive windows.
        if any(
            candidate[i + 1] != candidate[i] + 1
            for i in range(seq_len - 1)
        ):
            continue

        sequences.append([
            lookup[w]
            for w in candidate
        ])

    return sequences


# ------------------------------------------------------------
# Helper: sequence labels
# ------------------------------------------------------------

def sequence_metadata(
    sequences,
    traffic_df,
    category_col=CATEGORY_COL,
    label_col=LABEL_COL,
):

    metadata = []

    # Fast temporal lookup
    work = traffic_df[
        [
            "ts_start",
            "duration",
            category_col,
            label_col,
        ]
    ].copy()

    work["ts_end"] = (
        work["ts_start"].astype(float)
        +
        work["duration"].astype(float).clip(lower=0.0)
    )

    for seq in sequences:

        start_t = float(seq[0].t_start)
        end_t   = float(seq[-1].t_end)

        # Flow overlaps sequence interval.
        overlap_mask = (
            (work["ts_start"] < end_t) &
            (work["ts_end"] > start_t)
        )

        flows = work.loc[overlap_mask]

        attack_flows = flows[
            flows[label_col] == 1
        ]

        label = int(
            len(attack_flows) > 0
        )

        if len(attack_flows) > 0:

            categories = sorted(
                attack_flows[
                    category_col
                ]
                .astype(str)
                .unique()
                .tolist()
            )

        else:
            categories = []

        metadata.append({
            "label": label,
            "attack_fraction": (
                len(attack_flows) /
                max(len(flows), 1)
            ),
            "categories": categories,
            "window_start": seq[0].window_idx,
            "window_end": seq[-1].window_idx,
        })

    return metadata


# ------------------------------------------------------------
# Helper: embeddings
# ------------------------------------------------------------

@torch.no_grad()
def embed_sequences(
    graph_sequences,
    fitted_scaler,
    fitted_encoder,
):

    batches = convert_temporal_sequences_to_pyg(
        graph_sequences,
        fitted_scaler,
    )

    embeddings = []

    fitted_encoder.eval()

    for sequence in batches:

        temporal_batch = sequence

        z = get_sequence_embedding(
            temporal_batch
        )

        if torch.is_tensor(z):

            z = z.detach().cpu().numpy()

        z = np.asarray(z)

        if z.ndim > 1:
            z = z.squeeze(0)

        embeddings.append(z)

    return np.asarray(
        embeddings,
        dtype=np.float32
    )


# ------------------------------------------------------------
# Helper: KNN normal-state score
# ------------------------------------------------------------

def state_scores(
    query_embeddings,
    reference_embeddings,
    k=K_NEIGHBORS,
):

    q = np.asarray(
        query_embeddings,
        dtype=np.float32
    )

    r = np.asarray(
        reference_embeddings,
        dtype=np.float32
    )

    q = q / (
        np.linalg.norm(
            q,
            axis=1,
            keepdims=True
        ) + 1e-12
    )

    r = r / (
        np.linalg.norm(
            r,
            axis=1,
            keepdims=True
        ) + 1e-12
    )

    distances = 1.0 - (
        q @ r.T
    )

    k_eff = min(k, r.shape[0])

    nearest = np.partition(
        distances,
        kth=k_eff - 1,
        axis=1
    )[:, :k_eff]

    return nearest.mean(axis=1)


# ------------------------------------------------------------
# Main LOFO function
# ------------------------------------------------------------

def run_lofo_fold(
    held_out_family,
    source_df=df,
    verbose=True,
):

    print("\n" + "=" * 80)
    print("LOFO FOLD:", held_out_family)
    print("=" * 80)

    # ========================================================
    # 1. Chronological data
    # ========================================================

    train_base = source_df[
        (source_df["ts_start"] >= TRAIN_START) &
        (source_df["ts_start"] < TRAIN_END)
    ].copy()

    val_base = source_df[
        (source_df["ts_start"] >= VAL_START) &
        (source_df["ts_start"] < VAL_END)
    ].copy()

    test_base = source_df[
        (source_df["ts_start"] >= TEST_START) &
        (source_df["ts_start"] < TEST_END)
    ].copy()

    # Remove held-out family ONLY from training.
    train_fold = train_base[
        train_base[CATEGORY_COL] != held_out_family
    ].copy()

    # Hard LOFO assertion.
    assert not (
        train_fold[CATEGORY_COL] == held_out_family
    ).any()

    # ========================================================
    # 2. Build training graph
    # ========================================================

    train_snaps = build_lofo_snapshots(
        train_fold,
        ORIGINAL_GLOBAL_T0,
        TRAIN_WINDOWS,
    )

    # Require complete chronological training graph.
    train_ids = [
        s.window_idx
        for s in train_snaps
    ]

    assert train_ids == TRAIN_WINDOWS, (
        f"Training windows changed: "
        f"{train_ids[:5]} ... {train_ids[-5:]}"
    )

    # ========================================================
    # 3. Build validation/test graphs
    #
    # IMPORTANT:
    # Validation/test retain ALL families.
    # ========================================================

    val_snaps = build_lofo_snapshots(
        val_base,
        ORIGINAL_GLOBAL_T0,
        VAL_WINDOWS,
    )

    test_snaps = build_lofo_snapshots(
        test_base,
        ORIGINAL_GLOBAL_T0,
        TEST_WINDOWS,
    )

    # ========================================================
    # 4. Sequences
    # ========================================================

    train_seq = build_sequences(
        train_snaps,
        TRAIN_WINDOWS,
    )

    val_seq = build_sequences(
        val_snaps,
        VAL_WINDOWS,
    )

    test_seq = build_sequences(
        test_snaps,
        TEST_WINDOWS,
    )

    assert len(train_seq) > 0
    assert len(val_seq) > 0
    assert len(test_seq) > 0

    # ========================================================
    # 5. Metadata
    # ========================================================

    train_meta = sequence_metadata(
        train_seq,
        train_fold,
    )

    val_meta = sequence_metadata(
        val_seq,
        val_base,
    )

    test_meta = sequence_metadata(
        test_seq,
        test_base,
    )

    # ========================================================
    # 6. Fresh TRAIN-only scaler
    # ========================================================

    train_vectors = np.vstack([
        np.vstack([
            np.asarray(
                s.node_features[node],
                dtype=np.float32
            )
            for node in s.node_features
        ])
        for s in train_snaps
    ])

    fold_scaler = StandardScaler()

    fold_scaler.fit(
        train_vectors
    )

    # ========================================================
    # 7. Fresh CTGAD-X encoder
    # ========================================================

    fold_encoder = CTGAD_Encoder(
        in_dim=11,
        hidden_dim=60,
        seq_len=SEQ_LEN,
        struct_heads=4,
        temporal_heads=4,
    ).to(device)

    # --------------------------------------------------------
    # IMPORTANT:
    # Reuse the already established CTGAD-X training routine
    # rather than silently changing the objective.
    # --------------------------------------------------------

    train_pyg = convert_temporal_sequences_to_pyg(
        train_seq,
        fold_scaler,
    )

    val_pyg = convert_temporal_sequences_to_pyg(
        val_seq,
        fold_scaler,
    )

    # Return the fold objects here for the training stage.
    # The actual encoder training is handled below using
    # the same NT-Xent procedure as the original notebook.

    if verbose:
        print("Train sequences:", len(train_seq))
        print("Val sequences:", len(val_seq))
        print("Test sequences:", len(test_seq))

        print(
            "Held-out family in train:",
            (
                train_fold[CATEGORY_COL] ==
                held_out_family
            ).sum()
        )

        print(
            "Held-out family in test:",
            (
                test_base[CATEGORY_COL] ==
                held_out_family
            ).sum()
        )

    return {
        "family": held_out_family,
        "train_df": train_fold,
        "val_df": val_base,
        "test_df": test_base,

        "train_snapshots": train_snaps,
        "val_snapshots": val_snaps,
        "test_snapshots": test_snaps,

        "train_sequences": train_seq,
        "val_sequences": val_seq,
        "test_sequences": test_seq,

        "train_meta": train_meta,
        "val_meta": val_meta,
        "test_meta": test_meta,

        "scaler": fold_scaler,
        "encoder": fold_encoder,

        "train_pyg": train_pyg,
        "val_pyg": val_pyg,
    }


print("LOFO experiment engine created.")

LOFO experiment engine created.


In [ ]:
shellcode_fold = run_lofo_fold(
    "Shellcode",
    verbose=True
)


LOFO FOLD: Shellcode
Train sequences: 174
Val sequences: 22
Test sequences: 47
Held-out family in train: 0
Held-out family in test: 29


In [ ]:
# ============================================================
# CELL 150 — Inspect CTGAD-X encoder interface
# ============================================================

import inspect

print("CTGAD_Encoder signature:")
print(inspect.signature(CTGAD_Encoder))

print("\nForward signature:")
print(inspect.signature(CTGAD_Encoder.forward))

print("\nCurrent device:", device)

print("\nExisting training-related callables:")
for name in [
    "graph_batch",
    "get_sequence_embedding",
    "encoder",
    "train_loader",
    "val_loader",
    "optimizer",
    "criterion",
]:
    if name in globals():
        obj = globals()[name]
        try:
            print(f"{name}: {type(obj)} | {inspect.signature(obj) if callable(obj) else 'not callable'}")
        except Exception:
            print(f"{name}: {type(obj)}")

CTGAD_Encoder signature:
(in_dim: int = 11, hidden_dim: int = 60, seq_len: int = 5, struct_heads: int = 4, temporal_heads: int = 4)

Forward signature:
(self, snapshot_sequence: list) -> torch.Tensor

Current device: cuda

Existing training-related callables:
graph_batch: <class 'abc.DataBatch'> | (*args: str) -> Iterable
get_sequence_embedding: <class 'function'> | (temporal_batch)
encoder: <class 'model.encoder.CTGAD_Encoder'> | (*args, **kwargs)
optimizer: <class 'torch.optim.adam.Adam'> | not callable


In [ ]:
# ============================================================
# CELL 151 — Recover original CTGAD-X contrastive training code
# ============================================================

import inspect

keywords = [
    "NT-Xent",
    "nt_xent",
    "contrastive",
    "temperature",
    "dropout",
    "optimizer.step",
    "clip_grad_norm"
]

found = []

for name, obj in list(globals().items()):

    if not callable(obj):
        continue

    try:
        source = inspect.getsource(obj)
    except (TypeError, OSError, IOError):
        continue

    score = sum(
        key.lower() in source.lower()
        for key in keywords
    )

    if score >= 2:
        found.append((score, name, source))

for score, name, source in sorted(
    found,
    key=lambda x: (-x[0], x[1])
):

    print("\n" + "=" * 90)
    print(f"{name}  | keyword matches: {score}")
    print("=" * 90)
    print(source[:10000])


FederatedClient  | keyword matches: 5
class FederatedClient:
    def __init__(self, client_id: int, encoder: "CTGAD_Encoder", proj_head: "ProjectionHead",
                 lr: float = 1e-3, temperature: float = 0.1, k_neighbors: int = 15):
        import torch  # deferred: only FederatedClient needs the torch runtime
        self._torch = torch
        self.client_id = client_id
        self.encoder = encoder
        self.proj_head = proj_head
        self.temperature = temperature
        self.k = k_neighbors
        self.optimizer = torch.optim.AdamW(
            list(self.encoder.parameters()) + list(self.proj_head.parameters()), lr=lr
        )
        self.reference_bank: Optional[np.ndarray] = None      # local, never transmitted
        self.prev_calibration_embeddings: Optional[np.ndarray] = None

    def set_global_weights(self, global_encoder_state_dict: dict):
        self.encoder.load_state_dict(copy.deepcopy(global_encoder_state_dict))

    def local_train_epoch(self, seq

In [ ]:
# ============================================================
# CELL 152 — Inspect exact CTGAD-X projection head
# ============================================================

import inspect

print("ProjectionHead objects found:")
print("=" * 70)

for name, obj in list(globals().items()):

    if name.lower() == "projectionhead":

        print("Name:", name)
        print("Class:", obj)
        print("Signature:", inspect.signature(obj))

        try:
            print("\nSource:")
            print(inspect.getsource(obj))
        except Exception as e:
            print("Could not retrieve source:", e)

ProjectionHead objects found:
Name: ProjectionHead
Class: <class 'model.encoder.ProjectionHead'>
Signature: (in_dim: int = 60, hidden_dim: int = 60, out_dim: int = 80)

Source:
class ProjectionHead(nn.Module):
    """Used only during contrastive training; discarded at inference (matches CTGAD-X)."""

    def __init__(self, in_dim: int = 60, hidden_dim: int = 60, out_dim: int = 80):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z)



In [ ]:
# ============================================================
# CELL 153 — Fresh CTGAD-X training for Shellcode LOFO
# CORRECT PyG BATCH VERSION
# ============================================================

import copy
import torch
from torch_geometric.data import Batch

from model.encoder import ProjectionHead
from model.losses import nt_xent_loss


# ------------------------------------------------------------
# Fresh CTGAD-X model
# ------------------------------------------------------------

fold_encoder = CTGAD_Encoder(
    in_dim=11,
    hidden_dim=60,
    seq_len=SEQ_LEN,
    struct_heads=4,
    temporal_heads=4,
).to(device)

fold_proj = ProjectionHead(
    in_dim=60,
    hidden_dim=60,
    out_dim=80
).to(device)


# ------------------------------------------------------------
# Original CTGAD-X training configuration
# ------------------------------------------------------------

FOLD_EPOCHS = 100
FOLD_LR = 1e-3
FOLD_WEIGHT_DECAY = 1e-5
FOLD_TEMPERATURE = 0.1
FOLD_GRAD_CLIP = 1.0
FOLD_BATCH_SIZE = 16


fold_optimizer = torch.optim.AdamW(
    list(fold_encoder.parameters()) +
    list(fold_proj.parameters()),
    lr=FOLD_LR,
    weight_decay=FOLD_WEIGHT_DECAY
)


# ------------------------------------------------------------
# Convert sequence -> list of PyG Data objects
# ------------------------------------------------------------

def sequence_to_pyg(
    sequence,
    scaler
):

    return [
        snapshot_to_pyg(
            snapshot,
            scaler
        )
        for snapshot in sequence
    ]


# ------------------------------------------------------------
# IMPORTANT:
# The encoder's GAT layer requires `batch`.
#
# Therefore each temporal timestep containing multiple
# sequences must be represented as one PyG Batch.
# ------------------------------------------------------------

def make_temporal_batch(
    sequences,
    scaler,
    timestep
):

    graphs = [
        snapshot_to_pyg(
            sequence[timestep],
            scaler
        )
        for sequence in sequences
    ]

    return Batch.from_data_list(
        graphs
    ).to(device)


# ------------------------------------------------------------
# Encode a batch of temporal sequences
# ------------------------------------------------------------

def encode_sequence_batch(
    encoder_model,
    sequences,
    scaler
):

    temporal_graphs = []

    for t in range(SEQ_LEN):

        temporal_graphs.append(
            make_temporal_batch(
                sequences,
                scaler,
                t
            )
        )

    # CTGAD_Encoder expects:
    #
    # [
    #   PyG Batch at timestep 0,
    #   PyG Batch at timestep 1,
    #   ...
    # ]
    #
    return encoder_model(
        temporal_graphs
    )


# ------------------------------------------------------------
# Make ordinary sequence chunks
# ------------------------------------------------------------

def chunk_sequences(
    sequences,
    batch_size
):

    return [
        sequences[i:i + batch_size]
        for i in range(
            0,
            len(sequences),
            batch_size
        )
    ]


train_sequence_batches = chunk_sequences(
    shellcode_fold["train_sequences"],
    FOLD_BATCH_SIZE
)

val_sequence_batches = chunk_sequences(
    shellcode_fold["val_sequences"],
    FOLD_BATCH_SIZE
)


# ------------------------------------------------------------
# Device / batching sanity check
# ------------------------------------------------------------

test_batch = train_sequence_batches[0]

test_temporal_batch = make_temporal_batch(
    test_batch,
    shellcode_fold["scaler"],
    timestep=0
)

print(
    "PyG Batch type:",
    type(test_temporal_batch)
)

print(
    "x device:",
    test_temporal_batch.x.device
)

print(
    "edge_index device:",
    test_temporal_batch.edge_index.device
)

print(
    "batch device:",
    test_temporal_batch.batch.device
)

print(
    "Number of graphs:",
    test_temporal_batch.num_graphs
)

assert test_temporal_batch.batch is not None
assert test_temporal_batch.num_graphs == len(test_batch)


# ------------------------------------------------------------
# One training epoch
# ------------------------------------------------------------

def train_one_fold_epoch(
    encoder_model,
    projection_model,
    sequence_batches,
    optimizer
):

    encoder_model.train()
    projection_model.train()

    total_loss = 0.0

    for sequences in sequence_batches:

        optimizer.zero_grad()

        # ----------------------------------------------------
        # View 1
        # ----------------------------------------------------

        h1 = encode_sequence_batch(
            encoder_model,
            sequences,
            shellcode_fold["scaler"]
        )

        # ----------------------------------------------------
        # View 2
        #
        # Encoder remains in train mode, so GAT/attention
        # dropout produces an independent stochastic view.
        # ----------------------------------------------------

        h2 = encode_sequence_batch(
            encoder_model,
            sequences,
            shellcode_fold["scaler"]
        )

        z1 = projection_model(h1)
        z2 = projection_model(h2)

        loss = nt_xent_loss(
            z1,
            z2,
            FOLD_TEMPERATURE
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            list(encoder_model.parameters()) +
            list(projection_model.parameters()),
            FOLD_GRAD_CLIP
        )

        optimizer.step()

        total_loss += loss.item()

    return total_loss / max(
        len(sequence_batches),
        1
    )


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

@torch.no_grad()
def validation_loss(
    encoder_model,
    projection_model,
    sequence_batches
):

    encoder_model.eval()
    projection_model.eval()

    total_loss = 0.0

    for sequences in sequence_batches:

        h1 = encode_sequence_batch(
            encoder_model,
            sequences,
            shellcode_fold["scaler"]
        )

        h2 = encode_sequence_batch(
            encoder_model,
            sequences,
            shellcode_fold["scaler"]
        )

        z1 = projection_model(h1)
        z2 = projection_model(h2)

        loss = nt_xent_loss(
            z1,
            z2,
            FOLD_TEMPERATURE
        )

        total_loss += loss.item()

    return total_loss / max(
        len(sequence_batches),
        1
    )


# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

best_val_loss = float("inf")

best_encoder_state = None
best_proj_state = None
best_epoch = None

history = []

for epoch in range(
    1,
    FOLD_EPOCHS + 1
):

    train_loss = train_one_fold_epoch(
        fold_encoder,
        fold_proj,
        train_sequence_batches,
        fold_optimizer
    )

    val_loss = validation_loss(
        fold_encoder,
        fold_proj,
        val_sequence_batches
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss
    })

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        best_epoch = epoch

        best_encoder_state = copy.deepcopy(
            fold_encoder.state_dict()
        )

        best_proj_state = copy.deepcopy(
            fold_proj.state_dict()
        )

    if (
        epoch == 1
        or epoch % 10 == 0
        or epoch == FOLD_EPOCHS
    ):

        print(
            f"Epoch {epoch:03d} | "
            f"Train Loss: {train_loss:.6f} | "
            f"Val Loss: {val_loss:.6f}"
        )


# ------------------------------------------------------------
# Restore best model
# ------------------------------------------------------------

fold_encoder.load_state_dict(
    best_encoder_state
)

fold_proj.load_state_dict(
    best_proj_state
)

fold_encoder.eval()
fold_proj.eval()


print("\n" + "=" * 70)
print("SHELLCODE LOFO TRAINING COMPLETE")
print("=" * 70)

print("Best epoch:", best_epoch)
print(
    "Best validation loss:",
    best_val_loss
)

print(
    "Encoder parameters:",
    sum(
        p.numel()
        for p in fold_encoder.parameters()
    )
)

print(
    "Projection parameters:",
    sum(
        p.numel()
        for p in fold_proj.parameters()
    )
)

PyG Batch type: <class 'abc.DataBatch'>
x device: cuda:0
edge_index device: cuda:0
batch device: cuda:0
Number of graphs: 16
Epoch 001 | Train Loss: 2.020921 | Val Loss: 0.911903
Epoch 010 | Train Loss: 0.131231 | Val Loss: 0.250379
Epoch 020 | Train Loss: 0.050487 | Val Loss: 0.228062
Epoch 030 | Train Loss: 0.012519 | Val Loss: 0.315458
Epoch 040 | Train Loss: 0.010503 | Val Loss: 0.247358
Epoch 050 | Train Loss: 0.004749 | Val Loss: 0.259383
Epoch 060 | Train Loss: 0.003344 | Val Loss: 0.202816
Epoch 070 | Train Loss: 0.003062 | Val Loss: 0.229646
Epoch 080 | Train Loss: 0.002732 | Val Loss: 0.208456
Epoch 090 | Train Loss: 0.002856 | Val Loss: 0.159351
Epoch 100 | Train Loss: 0.004323 | Val Loss: 0.130849

SHELLCODE LOFO TRAINING COMPLETE
Best epoch: 100
Best validation loss: 0.13084883242845535
Encoder parameters: 20820
Projection parameters: 8540


In [ ]:
print(shellcode_fold.keys())

dict_keys(['family', 'train_df', 'val_df', 'test_df', 'train_snapshots', 'val_snapshots', 'test_snapshots', 'train_sequences', 'val_sequences', 'test_sequences', 'train_meta', 'val_meta', 'test_meta', 'scaler', 'encoder', 'train_pyg', 'val_pyg'])


In [ ]:
# ============================================================
# CHECK — NORMAL WINDOWS AVAILABLE IN SHELLCODE LOFO TRAIN
# ============================================================

train_snaps = shellcode_fold["train_snapshots"]

normal_windows = []

for i, snap in enumerate(train_snaps):
    cats = set()

    if hasattr(snap, "categories"):
        cats = set(snap.categories)
    elif isinstance(snap, dict):
        cats = set(snap.get("categories", []))

    # A window is normal if it contains no attack category
    if len(cats) == 0:
        normal_windows.append(i)

print("Total training snapshots:", len(train_snaps))
print("Normal training snapshots:", len(normal_windows))
print("First normal windows:", normal_windows[:20])

Total training snapshots: 178
Normal training snapshots: 178
First normal windows: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]


In [ ]:
import inspect

print(inspect.signature(slice_flows_to_windows))

(flows: 'pd.DataFrame', delta_t: 'float' = 30.0, ts_col: 'str' = 'ts_start', dur_col: 'str' = 'duration', src_col: 'str' = 'src_ip', dst_col: 'str' = 'dst_ip', global_t0: 'float | None' = None) -> 'List[GraphSnapshot]'


In [ ]:
# ============================================================
# SHELLCODE LOFO — TRUE NORMAL-ONLY TRAINING SEQUENCES
# ============================================================

global_t0 = 1421927377.0

normal_train_df = shellcode_fold["train_df"][
    shellcode_fold["train_df"]["label"] == 0
].copy()

print("LOFO training flows :", len(shellcode_fold["train_df"]))
print("Normal training flows:", len(normal_train_df))
print(
    "Attack training flows:",
    len(shellcode_fold["train_df"]) - len(normal_train_df)
)

# Same causal slicer and exact original temporal origin
normal_train_snapshots = slice_flows_to_windows(
    normal_train_df,
    delta_t=30.0,
    ts_col="ts_start",
    dur_col="duration",
    src_col="src_ip",
    dst_col="dst_ip",
    global_t0=global_t0
)

# Keep ONLY chronological training windows
normal_train_snapshots = [
    s for s in normal_train_snapshots
    if s.window_idx <= 177
]

print("\nNormal-only training snapshots:",
      len(normal_train_snapshots))

# Build contiguous 5-window sequences
normal_train_sequences = snapshots_to_sequences(
    normal_train_snapshots,
    seq_len=SEQ_LEN
)

print(
    "Normal-only training sequences:",
    len(normal_train_sequences)
)

assert len(normal_train_sequences) > 0

print(
    "First sequence length:",
    len(normal_train_sequences[0])
)

print("\nTRUE NORMAL-ONLY TRAINING DATA READY")

LOFO training flows : 142873
Normal training flows: 125126
Attack training flows: 17747

Normal-only training snapshots: 178
Normal-only training sequences: 174
First sequence length: 5

TRUE NORMAL-ONLY TRAINING DATA READY


In [ ]:
print("normal_train_snapshots exists:",
      "normal_train_snapshots" in globals())

if "normal_train_snapshots" in globals():
    print("Type:", type(normal_train_snapshots))
    print("Count:", len(normal_train_snapshots))
    if len(normal_train_snapshots) > 0:
        print("First snapshot type:", type(normal_train_snapshots[0]))
        print("First snapshot:", normal_train_snapshots[0])

normal_train_snapshots exists: True
Type: <class 'list'>
Count: 178
First snapshot type: <class 'ctgad_graph_slicer.GraphSnapshot'>
First snapshot: GraphSnapshot(window_idx=0, t_start=1421927377.0, t_end=1421927407.0, node_ids=['10.40.182.1', '224.0.0.5', '10.40.85.1'], edges=[(2, 1), (0, 1)], node_features={'10.40.182.1': array([460.76004  ,   0.       ,   7.1993756,   0.       ,   0.       ,
         0.       ,  61.43467  ,   0.       ,   1.       ,   0.       ,
        60.       ], dtype=float32), '224.0.0.5': array([921.52   ,   0.     ,  14.39875,   0.     ,   0.     ,   0.     ,
        61.43467,   0.     ,   1.     ,   0.     , 120.     ],
      dtype=float32), '10.40.85.1': array([460.76    ,   0.      ,   7.199375,   0.      ,   0.      ,
         0.      ,  61.434666,   0.      ,   1.      ,   0.      ,
        60.      ], dtype=float32)})


In [ ]:
# ============================================================
# BUILD TRUE NORMAL-ONLY REFERENCE SEQUENCES
# ============================================================

normal_train_sequences = snapshots_to_sequences(
    normal_train_snapshots,
    seq_len=SEQ_LEN
)

print("Normal-only snapshots :", len(normal_train_snapshots))
print("Normal-only sequences :", len(normal_train_sequences))

assert len(normal_train_sequences) > 0
assert all(len(seq) == SEQ_LEN for seq in normal_train_sequences)

print("Sequence length:", len(normal_train_sequences[0]))
print("READY FOR NORMAL REFERENCE EMBEDDINGS")

Normal-only snapshots : 178
Normal-only sequences : 174
Sequence length: 5
READY FOR NORMAL REFERENCE EMBEDDINGS


In [ ]:
# ============================================================
# CELL 155 — SHELLCODE LOFO FINAL EVALUATION
# Clean normal-only reference bank + NKVEL
# ============================================================

import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score

# ------------------------------------------------------------
# 1. Embed clean normal-only reference sequences
# ------------------------------------------------------------

normal_ref_emb = get_fold_embeddings(
    normal_train_sequences,
    shellcode_fold["scaler"]
)

print("Normal reference embeddings:", normal_ref_emb.shape)


# ------------------------------------------------------------
# 2. Existing frozen Shellcode test embeddings
# ------------------------------------------------------------

# test_emb was already generated by CELL 154
print("Shellcode test embeddings :", test_emb.shape)


# ------------------------------------------------------------
# 3. Convert test metadata to binary labels
# ------------------------------------------------------------

# IMPORTANT:
# test_meta label=1 means the 5-window sequence contains attack traffic.
# For the held-out-family evaluation, we use the sequence-level binary label.

test_labels = np.array([
    int(item["label"])
    for item in test_meta
])

print("\nTest labels:", np.bincount(test_labels))


# ------------------------------------------------------------
# 4. CTGAD-X state score
# ------------------------------------------------------------

def cosine_distance_to_bank(embeddings, reference):
    e = embeddings / (
        np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-12
    )

    r = reference / (
        np.linalg.norm(reference, axis=1, keepdims=True) + 1e-12
    )

    similarity = e @ r.T

    # k-NN K=5, matching the original CTGAD-X evaluation
    k = min(5, similarity.shape[1])

    nearest = np.partition(
        similarity,
        -k,
        axis=1
    )[:, -k:]

    return 1.0 - np.mean(nearest, axis=1)


test_state = cosine_distance_to_bank(
    test_emb,
    normal_ref_emb
)


# ------------------------------------------------------------
# 5. NKVEL local evolution
# ------------------------------------------------------------

velocity = np.abs(
    test_state[1:] - test_state[:-1]
)

test_evolution = (
    velocity[:-2]
    + velocity[1:-1]
    + velocity[2:]
) / 3.0

# Evolution score corresponds to t=3,...,T-1
evolution_labels = test_labels[3:]


# ------------------------------------------------------------
# 6. Metrics
# ------------------------------------------------------------

state_auc = roc_auc_score(
    test_labels,
    test_state
)

state_pr = average_precision_score(
    test_labels,
    test_state
)

evolution_auc = roc_auc_score(
    evolution_labels,
    test_evolution
)

evolution_pr = average_precision_score(
    evolution_labels,
    test_evolution
)


# ------------------------------------------------------------
# 7. Final result
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("SHELLCODE-HOLDOUT LOFO — FINAL RESULTS")
print("=" * 72)

print(f"CTGAD-X State ROC-AUC   : {state_auc:.6f}")
print(f"CTGAD-X State PR-AUC    : {state_pr:.6f}")

print(f"NKVEL Evolution ROC-AUC : {evolution_auc:.6f}")
print(f"NKVEL Evolution PR-AUC  : {evolution_pr:.6f}")

print(
    f"\nROC-AUC improvement     : "
    f"{evolution_auc - state_auc:+.6f}"
)

print(
    f"PR-AUC improvement      : "
    f"{evolution_pr - state_pr:+.6f}"
)

print("\nHeld-out family         : Shellcode")
print("Test sequences          :", len(test_labels))
print("Attack sequences        :", int(test_labels.sum()))
print("Normal sequences        :", int((test_labels == 0).sum()))
print("Evolution observations  :", len(test_evolution))

NameError: name 'get_fold_embeddings' is not defined

In [ ]:
# ============================================================
# CELL 156 — FINAL SHELLCODE LOFO TRAINING
# NORMAL-ONLY CONTRASTIVE LEARNING
# ============================================================

import copy
import numpy as np
import torch
from torch_geometric.data import Batch

from model.encoder import CTGAD_Encoder
from model.encoder import ProjectionHead
from model.losses import nt_xent_loss

# ------------------------------------------------------------
# 1. Get Shellcode fold
# ------------------------------------------------------------

# shellcode_fold should already exist from the restored notebook.
assert "shellcode_fold" in globals()

# ------------------------------------------------------------
# 2. Build TRUE normal-only training sequences
# ------------------------------------------------------------

normal_train_df = shellcode_fold["train_df"][
    shellcode_fold["train_df"]["label"] == 0
].copy()

global_t0 = 1421927377.0

normal_train_snapshots = slice_flows_to_windows(
    normal_train_df,
    delta_t=30.0,
    ts_col="ts_start",
    dur_col="duration",
    src_col="src_ip",
    dst_col="dst_ip",
    global_t0=global_t0
)

normal_train_snapshots = [
    s for s in normal_train_snapshots
    if s.window_idx <= 177
]

normal_train_sequences = snapshots_to_sequences(
    normal_train_snapshots,
    seq_len=SEQ_LEN
)

print("Normal training flows :", len(normal_train_df))
print("Normal snapshots      :", len(normal_train_snapshots))
print("Normal sequences      :", len(normal_train_sequences))

assert len(normal_train_sequences) > 0


# ------------------------------------------------------------
# 3. Fresh scaler fitted ONLY on normal training graph
# ------------------------------------------------------------

from sklearn.preprocessing import StandardScaler

normal_train_features = []

for snapshot in normal_train_snapshots:
    for node_id in snapshot.node_ids:
        normal_train_features.append(
            snapshot.node_features[node_id]
        )

normal_train_features = np.asarray(
    normal_train_features,
    dtype=np.float32
)

normal_scaler = StandardScaler()
normal_scaler.fit(normal_train_features)

print(
    "Scaler fitted on normal training features:",
    normal_train_features.shape
)


# ------------------------------------------------------------
# 4. Fresh encoder + projection head
# ------------------------------------------------------------

final_encoder = CTGAD_Encoder(
    in_dim=11,
    hidden_dim=60,
    seq_len=SEQ_LEN,
    struct_heads=4,
    temporal_heads=4
).to(device)

final_projection = ProjectionHead(
    in_dim=60,
    hidden_dim=60,
    out_dim=80
).to(device)


# ------------------------------------------------------------
# 5. Temporal PyG batching
# ------------------------------------------------------------

FINAL_BATCH_SIZE = 16

def make_normal_temporal_batch(
    sequences,
    timestep,
    scaler
):
    graphs = [
        snapshot_to_pyg(
            sequence[timestep],
            scaler
        )
        for sequence in sequences
    ]

    return Batch.from_data_list(graphs).to(device)


def encode_normal_batch(
    sequences,
    scaler
):
    temporal_graphs = []

    for t in range(SEQ_LEN):
        temporal_graphs.append(
            make_normal_temporal_batch(
                sequences,
                t,
                scaler
            )
        )

    return final_encoder(temporal_graphs)


normal_sequence_batches = [
    normal_train_sequences[i:i + FINAL_BATCH_SIZE]
    for i in range(
        0,
        len(normal_train_sequences),
        FINAL_BATCH_SIZE
    )
]


# ------------------------------------------------------------
# 6. Contrastive training
# ------------------------------------------------------------

FINAL_EPOCHS = 100
FINAL_LR = 1e-3
FINAL_WEIGHT_DECAY = 1e-5
FINAL_TEMPERATURE = 0.1
FINAL_GRAD_CLIP = 1.0

optimizer = torch.optim.AdamW(
    list(final_encoder.parameters()) +
    list(final_projection.parameters()),
    lr=FINAL_LR,
    weight_decay=FINAL_WEIGHT_DECAY
)


def train_normal_epoch():

    final_encoder.train()
    final_projection.train()

    total_loss = 0.0

    for sequences in normal_sequence_batches:

        optimizer.zero_grad()

        h1 = encode_normal_batch(
            sequences,
            normal_scaler
        )

        h2 = encode_normal_batch(
            sequences,
            normal_scaler
        )

        z1 = final_projection(h1)
        z2 = final_projection(h2)

        loss = nt_xent_loss(
            z1,
            z2,
            FINAL_TEMPERATURE
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            list(final_encoder.parameters()) +
            list(final_projection.parameters()),
            FINAL_GRAD_CLIP
        )

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(normal_sequence_batches)


best_loss = float("inf")
best_encoder_state = None
best_projection_state = None
best_epoch = None

for epoch in range(1, FINAL_EPOCHS + 1):

    loss = train_normal_epoch()

    if loss < best_loss:
        best_loss = loss
        best_epoch = epoch
        best_encoder_state = copy.deepcopy(
            final_encoder.state_dict()
        )
        best_projection_state = copy.deepcopy(
            final_projection.state_dict()
        )

    if (
        epoch == 1
        or epoch % 10 == 0
        or epoch == FINAL_EPOCHS
    ):
        print(
            f"Epoch {epoch:03d} | "
            f"Normal NT-Xent Loss: {loss:.6f}"
        )


# Restore best encoder
final_encoder.load_state_dict(
    best_encoder_state
)

final_projection.load_state_dict(
    best_projection_state
)

final_encoder.eval()
final_projection.eval()

print("\n" + "=" * 72)
print("PROPER NORMAL-ONLY SHELLCODE LOFO TRAINING COMPLETE")
print("=" * 72)

print("Best epoch:", best_epoch)
print("Best loss :", best_loss)

print(
    "Encoder parameters:",
    sum(p.numel() for p in final_encoder.parameters())
)

print(
    "Projection parameters:",
    sum(p.numel() for p in final_projection.parameters())
)

Normal training flows : 125126
Normal snapshots      : 178
Normal sequences      : 174
Scaler fitted on normal training features: (4845, 11)
Epoch 001 | Normal NT-Xent Loss: 1.783028
Epoch 010 | Normal NT-Xent Loss: 0.148358
Epoch 020 | Normal NT-Xent Loss: 0.054901
Epoch 030 | Normal NT-Xent Loss: 0.029040
Epoch 040 | Normal NT-Xent Loss: 0.011392
Epoch 050 | Normal NT-Xent Loss: 0.006394
Epoch 060 | Normal NT-Xent Loss: 0.004573
Epoch 070 | Normal NT-Xent Loss: 0.004110
Epoch 080 | Normal NT-Xent Loss: 0.002955
Epoch 090 | Normal NT-Xent Loss: 0.006247
Epoch 100 | Normal NT-Xent Loss: 0.008922

PROPER NORMAL-ONLY SHELLCODE LOFO TRAINING COMPLETE
Best epoch: 80
Best loss : 0.00295464650050483
Encoder parameters: 20820
Projection parameters: 8540


In [ ]:
# ============================================================
# CELL 157 — FINAL SHELLCODE LOFO EVALUATION
# ============================================================

import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score

# ------------------------------------------------------------
# 1. Frozen embedding function
# ------------------------------------------------------------

@torch.no_grad()
def get_final_embeddings(sequences, scaler):

    final_encoder.eval()

    outputs = []

    for i in range(
        0,
        len(sequences),
        FINAL_BATCH_SIZE
    ):

        batch_sequences = sequences[
            i:i + FINAL_BATCH_SIZE
        ]

        temporal_graphs = []

        for t in range(SEQ_LEN):
            temporal_graphs.append(
                make_temporal_batch(
                    batch_sequences,
                    scaler,
                    timestep=t
                )
            )

        embedding = final_encoder(
            temporal_graphs
        )

        outputs.append(
            embedding.cpu().numpy()
        )

    return np.concatenate(
        outputs,
        axis=0
    )


# ------------------------------------------------------------
# 2. Clean normal reference embeddings
# ------------------------------------------------------------

normal_reference_embeddings = get_final_embeddings(
    normal_train_sequences,
    normal_scaler
)

print(
    "Normal reference:",
    normal_reference_embeddings.shape
)


# ------------------------------------------------------------
# 3. Shellcode test embeddings
# ------------------------------------------------------------

test_sequences = shellcode_fold["test_sequences"]

test_embeddings = get_final_embeddings(
    test_sequences,
    normal_scaler
)

print(
    "Shellcode test:",
    test_embeddings.shape
)


# ------------------------------------------------------------
# 4. Test labels
# ------------------------------------------------------------

test_meta = shellcode_fold["test_meta"]

test_labels = np.array([
    int(item["label"])
    for item in test_meta
])

print(
    "Test labels:",
    np.bincount(test_labels)
)


# ------------------------------------------------------------
# 5. CTGAD-X KNN cosine state score
# ------------------------------------------------------------

def knn_cosine_distance(
    queries,
    reference,
    k=5
):

    queries = queries / (
        np.linalg.norm(
            queries,
            axis=1,
            keepdims=True
        ) + 1e-12
    )

    reference = reference / (
        np.linalg.norm(
            reference,
            axis=1,
            keepdims=True
        ) + 1e-12
    )

    similarity = queries @ reference.T

    k = min(k, similarity.shape[1])

    nearest = np.partition(
        similarity,
        -k,
        axis=1
    )[:, -k:]

    return 1.0 - np.mean(
        nearest,
        axis=1
    )


state_scores = knn_cosine_distance(
    test_embeddings,
    normal_reference_embeddings,
    k=5
)


# ------------------------------------------------------------
# 6. NKVEL local evolution score
# ------------------------------------------------------------

velocity = np.abs(
    state_scores[1:] -
    state_scores[:-1]
)

evolution_scores = (
    velocity[:-2] +
    velocity[1:-1] +
    velocity[2:]
) / 3.0

evolution_labels = test_labels[3:]


# ------------------------------------------------------------
# 7. Metrics
# ------------------------------------------------------------

state_roc = roc_auc_score(
    test_labels,
    state_scores
)

state_pr = average_precision_score(
    test_labels,
    state_scores
)

evolution_roc = roc_auc_score(
    evolution_labels,
    evolution_scores
)

evolution_pr = average_precision_score(
    evolution_labels,
    evolution_scores
)


# ------------------------------------------------------------
# 8. Final result
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("SHELLCODE-HOLDOUT — NORMAL-ONLY CTGAD-X + NKVEL")
print("=" * 72)

print(
    f"CTGAD-X State ROC-AUC   : {state_roc:.6f}"
)

print(
    f"CTGAD-X State PR-AUC    : {state_pr:.6f}"
)

print(
    f"NKVEL Evolution ROC-AUC : {evolution_roc:.6f}"
)

print(
    f"NKVEL Evolution PR-AUC  : {evolution_pr:.6f}"
)

print(
    f"\nROC-AUC Δ : "
    f"{evolution_roc - state_roc:+.6f}"
)

print(
    f"PR-AUC Δ  : "
    f"{evolution_pr - state_pr:+.6f}"
)

print("\nHeld-out family :", "Shellcode")
print("Test sequences  :", len(test_labels))
print("Normal          :", int((test_labels == 0).sum()))
print("Attack          :", int(test_labels.sum()))
print(
    "Evolution obs   :",
    len(evolution_scores)
)

Normal reference: (174, 60)
Shellcode test: (47, 60)
Test labels: [15 32]

SHELLCODE-HOLDOUT — NORMAL-ONLY CTGAD-X + NKVEL
CTGAD-X State ROC-AUC   : 0.977083
CTGAD-X State PR-AUC    : 0.989811
NKVEL Evolution ROC-AUC : 0.664368
NKVEL Evolution PR-AUC  : 0.851568

ROC-AUC Δ : -0.312716
PR-AUC Δ  : -0.138242

Held-out family : Shellcode
Test sequences  : 47
Normal          : 15
Attack          : 32
Evolution obs   : 44


In [ ]:
# ============================================================
# CELL 158 — SAGE
# State-Aware Graph Evolution
# ============================================================

import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score

# ------------------------------------------------------------
# 1. Normal temporal evolution reference
# ------------------------------------------------------------

# Consecutive embedding movement in NORMAL behaviour
normal_velocity = np.linalg.norm(
    normal_reference_embeddings[1:]
    - normal_reference_embeddings[:-1],
    axis=1
)

# Robust normal evolution statistics
normal_vel_median = np.median(normal_velocity)

normal_vel_mad = np.median(
    np.abs(
        normal_velocity -
        normal_vel_median
    )
)

normal_vel_scale = (
    1.4826 * normal_vel_mad + 1e-8
)

print("Normal evolution median:",
      normal_vel_median)

print("Normal evolution robust scale:",
      normal_vel_scale)


# ------------------------------------------------------------
# 2. Test embedding evolution
# ------------------------------------------------------------

test_velocity = np.linalg.norm(
    test_embeddings[1:]
    - test_embeddings[:-1],
    axis=1
)

# Robust temporal surprise
temporal_surprise = np.maximum(
    0.0,
    (
        test_velocity -
        normal_vel_median
    ) / normal_vel_scale
)

# Align with current state score
state_aligned = state_scores[1:]


# ------------------------------------------------------------
# 3. State normalization using NORMAL reference
# ------------------------------------------------------------

normal_state_scores = knn_cosine_distance(
    normal_reference_embeddings,
    normal_reference_embeddings,
    k=5
)

state_median = np.median(
    normal_state_scores
)

state_mad = np.median(
    np.abs(
        normal_state_scores -
        state_median
    )
)

state_scale = (
    1.4826 * state_mad + 1e-8
)

state_surprise = np.maximum(
    0.0,
    (
        state_aligned -
        state_median
    ) / state_scale
)


# ------------------------------------------------------------
# 4. SAGE gating
# ------------------------------------------------------------

# Evolution matters more when the system is already
# outside the normal behavioural manifold.

state_gate = (
    state_surprise /
    (1.0 + state_surprise)
)

sage_score = (
    state_surprise
    +
    state_gate * temporal_surprise
)


# ------------------------------------------------------------
# 5. Labels
# ------------------------------------------------------------

sage_labels = test_labels[1:]


# ------------------------------------------------------------
# 6. Evaluate
# ------------------------------------------------------------

sage_roc = roc_auc_score(
    sage_labels,
    sage_score
)

sage_pr = average_precision_score(
    sage_labels,
    sage_score
)


# ------------------------------------------------------------
# 7. Also evaluate pure temporal surprise
# ------------------------------------------------------------

temporal_roc = roc_auc_score(
    sage_labels,
    temporal_surprise
)

temporal_pr = average_precision_score(
    sage_labels,
    temporal_surprise
)


# ------------------------------------------------------------
# 8. Final comparison
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("SAGE — STATE-AWARE GRAPH EVOLUTION")
print("=" * 72)

print(
    f"CTGAD-X State ROC-AUC : "
    f"{state_roc:.6f}"
)

print(
    f"CTGAD-X State PR-AUC  : "
    f"{state_pr:.6f}"
)

print(
    f"Temporal Surprise ROC-AUC : "
    f"{temporal_roc:.6f}"
)

print(
    f"Temporal Surprise PR-AUC  : "
    f"{temporal_pr:.6f}"
)

print(
    f"SAGE ROC-AUC : "
    f"{sage_roc:.6f}"
)

print(
    f"SAGE PR-AUC  : "
    f"{sage_pr:.6f}"
)

print(
    "\nSAGE Δ ROC-AUC vs CTGAD-X:",
    f"{sage_roc - state_roc:+.6f}"
)

print(
    "SAGE Δ PR-AUC vs CTGAD-X:",
    f"{sage_pr - state_pr:+.6f}"
)

print("\nHeld-out family: Shellcode")

Normal evolution median: 5.5491095
Normal evolution robust scale: 2.1278477

SAGE — STATE-AWARE GRAPH EVOLUTION
CTGAD-X State ROC-AUC : 0.977083
CTGAD-X State PR-AUC  : 0.989811
Temporal Surprise ROC-AUC : 0.496774
Temporal Surprise PR-AUC  : 0.712756
SAGE ROC-AUC : 0.978495
SAGE PR-AUC  : 0.989754

SAGE Δ ROC-AUC vs CTGAD-X: +0.001411
SAGE Δ PR-AUC vs CTGAD-X: -0.000057

Held-out family: Shellcode


In [ ]:
# ============================================================
# CELL 159 — COUNTERFACTUAL TEMPORAL CONSISTENCY
# ============================================================

import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score

# ------------------------------------------------------------
# 1. Normal temporal transition model
# ------------------------------------------------------------

# z(t+1) ≈ z(t) + [z(t) - z(t-1)]
#
# This is a constant-velocity counterfactual:
# "Where should normal behaviour naturally move next?"

normal_prev = normal_reference_embeddings[:-2]
normal_curr = normal_reference_embeddings[1:-1]
normal_next = normal_reference_embeddings[2:]

normal_velocity = (
    normal_curr - normal_prev
)

normal_predicted_next = (
    normal_curr + normal_velocity
)

normal_transition_error = np.linalg.norm(
    normal_next - normal_predicted_next,
    axis=1
)

# Robust normal transition distribution
transition_median = np.median(
    normal_transition_error
)

transition_mad = np.median(
    np.abs(
        normal_transition_error -
        transition_median
    )
)

transition_scale = (
    1.4826 * transition_mad + 1e-8
)

print(
    "Normal transition median:",
    transition_median
)

print(
    "Normal transition scale:",
    transition_scale
)


# ------------------------------------------------------------
# 2. Counterfactual prediction on TEST
# ------------------------------------------------------------

test_prev = test_embeddings[:-2]
test_curr = test_embeddings[1:-1]
test_actual_next = test_embeddings[2:]

test_velocity = (
    test_curr - test_prev
)

test_predicted_next = (
    test_curr + test_velocity
)

counterfactual_error = np.linalg.norm(
    test_actual_next -
    test_predicted_next,
    axis=1
)


# ------------------------------------------------------------
# 3. Robust counterfactual surprise
# ------------------------------------------------------------

ctc_score = np.maximum(
    0.0,
    (
        counterfactual_error -
        transition_median
    ) / transition_scale
)


# ------------------------------------------------------------
# 4. Align labels
# ------------------------------------------------------------

ctc_labels = test_labels[2:]


# ------------------------------------------------------------
# 5. Evaluate CTC
# ------------------------------------------------------------

ctc_roc = roc_auc_score(
    ctc_labels,
    ctc_score
)

ctc_pr = average_precision_score(
    ctc_labels,
    ctc_score
)


# ------------------------------------------------------------
# 6. State + CTC
# ------------------------------------------------------------

state_for_ctc = state_scores[2:]

# Rank-normalize both signals
def rank_normalize(x):
    order = np.argsort(np.argsort(x))
    return order / max(len(x) - 1, 1)

state_rank = rank_normalize(
    state_for_ctc
)

ctc_rank = rank_normalize(
    ctc_score
)

# Fixed equal-weight fusion
ctc_fusion = (
    0.5 * state_rank +
    0.5 * ctc_rank
)

fusion_roc = roc_auc_score(
    ctc_labels,
    ctc_fusion
)

fusion_pr = average_precision_score(
    ctc_labels,
    ctc_fusion
)


# ------------------------------------------------------------
# 7. Results
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("COUNTERFACTUAL TEMPORAL CONSISTENCY")
print("=" * 72)

print(
    f"CTGAD-X State ROC-AUC : "
    f"{state_roc:.6f}"
)

print(
    f"CTGAD-X State PR-AUC  : "
    f"{state_pr:.6f}"
)

print(
    f"CTC ROC-AUC            : "
    f"{ctc_roc:.6f}"
)

print(
    f"CTC PR-AUC             : "
    f"{ctc_pr:.6f}"
)

print(
    f"State + CTC ROC-AUC    : "
    f"{fusion_roc:.6f}"
)

print(
    f"State + CTC PR-AUC     : "
    f"{fusion_pr:.6f}"
)

print(
    "\nFusion Δ ROC-AUC:",
    f"{fusion_roc - state_roc:+.6f}"
)

print(
    "Fusion Δ PR-AUC :",
    f"{fusion_pr - state_pr:+.6f}"
)

print("\nHeld-out family: Shellcode")

Normal transition median: 9.043188
Normal transition scale: 3.5464911

COUNTERFACTUAL TEMPORAL CONSISTENCY
CTGAD-X State ROC-AUC : 0.977083
CTGAD-X State PR-AUC  : 0.989811
CTC ROC-AUC            : 0.503333
CTC PR-AUC             : 0.679726
State + CTC ROC-AUC    : 0.666667
State + CTC PR-AUC     : 0.832702

Fusion Δ ROC-AUC: -0.310417
Fusion Δ PR-AUC : -0.157109

Held-out family: Shellcode


In [ ]:
# ============================================================
# CELL 159 — FULL UNSEEN-FAMILY EVALUATION
# Frozen normal-only CTGAD-X + Local Normal-Manifold Score
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

# ------------------------------------------------------------
# PRIMARY LOFO FAMILIES
# ------------------------------------------------------------

LOFO_FAMILIES = [
    "Generic",
    "Exploits",
    "Fuzzers",
    "Reconnaissance",
    "DoS",
    "Backdoors",
    "Analysis",
    "Shellcode"
]

K_NEIGHBORS = 5


# ------------------------------------------------------------
# 1. Local normal-manifold score
# ------------------------------------------------------------

def manifold_scores(
    query_embeddings,
    reference_embeddings,
    k=5
):
    """
    Returns two fixed scores:

    state_score:
        1 - mean cosine similarity to k nearest normal neighbours

    density_score:
        mean cosine distance to the k nearest normal neighbours

    Here density_score intentionally uses the same k-neighbour
    neighbourhood but is retained separately for the final audit.
    """

    q = query_embeddings / (
        np.linalg.norm(
            query_embeddings,
            axis=1,
            keepdims=True
        ) + 1e-12
    )

    r = reference_embeddings / (
        np.linalg.norm(
            reference_embeddings,
            axis=1,
            keepdims=True
        ) + 1e-12
    )

    similarity = q @ r.T

    k = min(k, similarity.shape[1])

    nearest = np.partition(
        similarity,
        -k,
        axis=1
    )[:, -k:]

    state = 1.0 - np.mean(
        nearest,
        axis=1
    )

    # Local consistency:
    # variance of similarity among nearest normal neighbours.
    # An isolated point has less consistent neighbourhood support.
    neighbour_std = np.std(
        nearest,
        axis=1
    )

    return state, neighbour_std


# ------------------------------------------------------------
# 2. Build test sequence family labels
# ------------------------------------------------------------

test_meta_all = shellcode_fold["test_meta"]

assert len(test_meta_all) == len(
    shellcode_fold["test_sequences"]
)

family_results = []


# ------------------------------------------------------------
# 3. Evaluate every held-out family
# ------------------------------------------------------------

for family in LOFO_FAMILIES:

    selected_indices = []

    for i, item in enumerate(test_meta_all):

        categories = set(
            item.get("categories", [])
        )

        # Normal sequence
        is_normal = (
            int(item.get("label", 0)) == 0
        )

        # Sequence containing this specific family
        contains_family = (
            family in categories
        )

        if is_normal or contains_family:
            selected_indices.append(i)

    selected_indices = np.asarray(
        selected_indices,
        dtype=int
    )

    family_embeddings = test_embeddings[
        selected_indices
    ]

    family_labels = np.array([
        0 if int(test_meta_all[i]["label"]) == 0
        else 1
        for i in selected_indices
    ])

    # IMPORTANT:
    # If a positive sequence contains the family, it is positive.
    # Normal sequences are the only negatives.
    # Other attack families are never treated as negatives.

    n_normal = int(
        (family_labels == 0).sum()
    )

    n_attack = int(
        (family_labels == 1).sum()
    )

    if n_normal < 2 or n_attack < 2:
        print(
            f"{family}: insufficient samples"
        )
        continue

    # --------------------------------------------------------
    # Scores
    # --------------------------------------------------------

    state, neighbourhood_variability = (
        manifold_scores(
            family_embeddings,
            normal_reference_embeddings,
            k=K_NEIGHBORS
        )
    )

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    state_roc = roc_auc_score(
        family_labels,
        state
    )

    state_pr = average_precision_score(
        family_labels,
        state
    )

    # Higher neighbour variability is NOT automatically
    # anomalous. Convert it into a fixed anomaly-oriented
    # score by multiplying with state.
    manifold_score = (
        state *
        (
            1.0 +
            neighbourhood_variability
        )
    )

    manifold_roc = roc_auc_score(
        family_labels,
        manifold_score
    )

    manifold_pr = average_precision_score(
        family_labels,
        manifold_score
    )

    family_results.append({
        "Family": family,
        "Normal": n_normal,
        "Attack": n_attack,
        "CTGAD-X ROC-AUC": state_roc,
        "CTGAD-X PR-AUC": state_pr,
        "Manifold ROC-AUC": manifold_roc,
        "Manifold PR-AUC": manifold_pr
    })

    print(
        f"{family:16s} | "
        f"N={n_normal:2d} "
        f"A={n_attack:2d} | "
        f"CTGAD ROC={state_roc:.4f} "
        f"PR={state_pr:.4f} | "
        f"MAN ROC={manifold_roc:.4f} "
        f"PR={manifold_pr:.4f}"
    )


# ------------------------------------------------------------
# 4. Final table
# ------------------------------------------------------------

family_results_df = pd.DataFrame(
    family_results
)

print("\n" + "=" * 90)
print("UNSEEN ATTACK-FAMILY RESULTS")
print("=" * 90)

display(
    family_results_df.round(6)
)


# ------------------------------------------------------------
# 5. Macro averages
# ------------------------------------------------------------

print("\nMACRO AVERAGES")

print(
    "CTGAD-X ROC-AUC:",
    family_results_df[
        "CTGAD-X ROC-AUC"
    ].mean()
)

print(
    "CTGAD-X PR-AUC:",
    family_results_df[
        "CTGAD-X PR-AUC"
    ].mean()
)

print(
    "Manifold ROC-AUC:",
    family_results_df[
        "Manifold ROC-AUC"
    ].mean()
)

print(
    "Manifold PR-AUC:",
    family_results_df[
        "Manifold PR-AUC"
    ].mean()
)

Generic          | N=15 A=30 | CTGAD ROC=0.9911 PR=0.9957 | MAN ROC=0.9911 PR=0.9957
Exploits         | N=15 A=30 | CTGAD ROC=0.9911 PR=0.9957 | MAN ROC=0.9911 PR=0.9957
Fuzzers          | N=15 A=32 | CTGAD ROC=0.9771 PR=0.9898 | MAN ROC=0.9813 PR=0.9914
Reconnaissance   | N=15 A=30 | CTGAD ROC=0.9911 PR=0.9957 | MAN ROC=0.9911 PR=0.9957
DoS              | N=15 A=30 | CTGAD ROC=0.9911 PR=0.9957 | MAN ROC=0.9911 PR=0.9957
Backdoors        | N=15 A=19 | CTGAD ROC=0.9965 PR=0.9974 | MAN ROC=0.9965 PR=0.9974
Analysis         | N=15 A= 2 | CTGAD ROC=1.0000 PR=1.0000 | MAN ROC=1.0000 PR=1.0000
Shellcode        | N=15 A=29 | CTGAD ROC=0.9954 PR=0.9977 | MAN ROC=0.9954 PR=0.9977

UNSEEN ATTACK-FAMILY RESULTS


,Family,Normal,Attack,CTGAD-X ROC-AUC,CTGAD-X PR-AUC,Manifold ROC-AUC,Manifold PR-AUC
0,Generic,15,30,0.991111,0.995656,0.991111,0.995656
1,Exploits,15,30,0.991111,0.995656,0.991111,0.995656
2,Fuzzers,15,32,0.977083,0.989811,0.981250,0.991355
3,Reconnaissance,15,30,0.991111,0.995656,0.991111,0.995656
4,DoS,15,30,0.991111,0.995656,0.991111,0.995656
5,Backdoors,15,19,0.996491,0.997368,0.996491,0.997368
6,Analysis,15,2,1.000000,1.000000,1.000000,1.000000
7,Shellcode,15,29,0.995402,0.997662,0.995402,0.997662



MACRO AVERAGES
CTGAD-X ROC-AUC: 0.9916776630873161
CTGAD-X PR-AUC: 0.9959331457444379
Manifold ROC-AUC: 0.9921984964206494
Manifold PR-AUC: 0.9961261959374881


In [ ]:
# ============================================================
# CELL 160 — CORRECT UNSEEN-FAMILY LOFO EVALUATION
#
# Same frozen NORMAL-ONLY encoder
# Each attack family evaluated independently
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

LOFO_FAMILIES = [
    "Generic",
    "Exploits",
    "Fuzzers",
    "Reconnaissance",
    "DoS",
    "Backdoors",
    "Analysis",
    "Shellcode"
]

K = 5

# ------------------------------------------------------------
# 1. Use the COMMON chronological test set
# ------------------------------------------------------------

# shellcode_fold["test_sequences"] is the complete test period,
# not Shellcode-only data.
#
# Its graph content contains all attack families.
# We only use shellcode_fold because its test split is the
# common chronological test interval 204--254.

COMMON_TEST_SEQUENCES = shellcode_fold["test_sequences"]
COMMON_TEST_META = shellcode_fold["test_meta"]

assert len(COMMON_TEST_SEQUENCES) == len(COMMON_TEST_META)

print("Common test sequences:", len(COMMON_TEST_SEQUENCES))


# ------------------------------------------------------------
# 2. Embed common test set ONCE
# ------------------------------------------------------------

common_test_embeddings = get_final_embeddings(
    COMMON_TEST_SEQUENCES,
    normal_scaler
)

print(
    "Common test embeddings:",
    common_test_embeddings.shape
)


# ------------------------------------------------------------
# 3. Compute CTGAD-X state score ONCE
# ------------------------------------------------------------

common_state_scores = knn_cosine_distance(
    common_test_embeddings,
    normal_reference_embeddings,
    k=K
)


# ------------------------------------------------------------
# 4. Evaluate each family independently
# ------------------------------------------------------------

lofo_results = []

for family in LOFO_FAMILIES:

    selected_indices = []

    for i, meta in enumerate(COMMON_TEST_META):

        label = int(meta["label"])
        categories = set(meta.get("categories", []))

        # Negative = genuinely normal sequence
        is_normal = (label == 0)

        # Positive = sequence containing this family
        is_target_attack = (
            label == 1 and
            family in categories
        )

        # IMPORTANT:
        # Other attack families are excluded from this
        # family-specific evaluation.
        if is_normal or is_target_attack:
            selected_indices.append(i)

    selected_indices = np.asarray(
        selected_indices,
        dtype=int
    )

    y = np.array([
        0 if COMMON_TEST_META[i]["label"] == 0
        else 1
        for i in selected_indices
    ])

    scores = common_state_scores[
        selected_indices
    ]

    n_normal = int((y == 0).sum())
    n_attack = int((y == 1).sum())

    if n_normal == 0 or n_attack == 0:
        continue

    roc = roc_auc_score(
        y,
        scores
    )

    pr = average_precision_score(
        y,
        scores
    )

    lofo_results.append({
        "Family": family,
        "Normal": n_normal,
        "Attack": n_attack,
        "ROC-AUC": roc,
        "PR-AUC": pr
    })

    print(
        f"{family:16s} | "
        f"N={n_normal:2d} | "
        f"A={n_attack:2d} | "
        f"ROC-AUC={roc:.6f} | "
        f"PR-AUC={pr:.6f}"
    )


# ------------------------------------------------------------
# 5. Final LOFO table
# ------------------------------------------------------------

lofo_df = pd.DataFrame(
    lofo_results
)

print("\n" + "=" * 72)
print("VALID UNSEEN-FAMILY EVALUATION")
print("=" * 72)

display(
    lofo_df.round(6)
)


# ------------------------------------------------------------
# 6. Macro-average
# ------------------------------------------------------------

macro_roc = lofo_df["ROC-AUC"].mean()
macro_pr = lofo_df["PR-AUC"].mean()

print("\n" + "=" * 72)
print("MACRO LOFO RESULTS")
print("=" * 72)

print(
    f"Macro ROC-AUC : {macro_roc:.6f}"
)

print(
    f"Macro PR-AUC  : {macro_pr:.6f}"
)

Common test sequences: 47
Common test embeddings: (47, 60)
Generic          | N=15 | A=30 | ROC-AUC=0.991111 | PR-AUC=0.995656
Exploits         | N=15 | A=30 | ROC-AUC=0.991111 | PR-AUC=0.995656
Fuzzers          | N=15 | A=32 | ROC-AUC=0.977083 | PR-AUC=0.989811
Reconnaissance   | N=15 | A=30 | ROC-AUC=0.991111 | PR-AUC=0.995656
DoS              | N=15 | A=30 | ROC-AUC=0.991111 | PR-AUC=0.995656
Backdoors        | N=15 | A=19 | ROC-AUC=0.996491 | PR-AUC=0.997368
Analysis         | N=15 | A= 2 | ROC-AUC=1.000000 | PR-AUC=1.000000
Shellcode        | N=15 | A=29 | ROC-AUC=0.995402 | PR-AUC=0.997662

VALID UNSEEN-FAMILY EVALUATION


,Family,Normal,Attack,ROC-AUC,PR-AUC
0,Generic,15,30,0.991111,0.995656
1,Exploits,15,30,0.991111,0.995656
2,Fuzzers,15,32,0.977083,0.989811
3,Reconnaissance,15,30,0.991111,0.995656
4,DoS,15,30,0.991111,0.995656
5,Backdoors,15,19,0.996491,0.997368
6,Analysis,15,2,1.000000,1.000000
7,Shellcode,15,29,0.995402,0.997662



MACRO LOFO RESULTS
Macro ROC-AUC : 0.991678
Macro PR-AUC  : 0.995933


In [ ]:
# ============================================================
# CELL 161 — NORMAL MANIFOLD GEOMETRY
# ============================================================

import numpy as np
from sklearn.covariance import LedoitWolf
from sklearn.metrics import roc_auc_score, average_precision_score

# ------------------------------------------------------------
# 1. Fit normal embedding distribution
# ------------------------------------------------------------

normal_mu = np.mean(
    normal_reference_embeddings,
    axis=0
)

# Shrinkage covariance is stable for 60-D embeddings
lw = LedoitWolf()
lw.fit(normal_reference_embeddings)

covariance = lw.covariance_
precision = lw.precision_

print("Normal reference:", normal_reference_embeddings.shape)
print("Embedding dimension:", normal_reference_embeddings.shape[1])


# ------------------------------------------------------------
# 2. Mahalanobis distance
# ------------------------------------------------------------

def mahalanobis_score(
    embeddings,
    mean_vector,
    precision_matrix
):

    delta = embeddings - mean_vector

    return np.sqrt(
        np.maximum(
            0.0,
            np.einsum(
                "ij,jk,ik->i",
                delta,
                precision_matrix,
                delta
            )
        )
    )


normal_mahal = mahalanobis_score(
    normal_reference_embeddings,
    normal_mu,
    precision
)

test_mahal = mahalanobis_score(
    test_embeddings,
    normal_mu,
    precision
)


# ------------------------------------------------------------
# 3. Evaluate on the untouched Shellcode test set
# ------------------------------------------------------------

mahal_roc = roc_auc_score(
    test_labels,
    test_mahal
)

mahal_pr = average_precision_score(
    test_labels,
    test_mahal
)


# ------------------------------------------------------------
# 4. Fixed rank fusion with CTGAD-X
# ------------------------------------------------------------

def rank_normalize(x):

    order = np.argsort(
        np.argsort(x)
    )

    return order / max(
        len(x) - 1,
        1
    )


state_rank = rank_normalize(
    state_scores
)

mahal_rank = rank_normalize(
    test_mahal
)

# Fixed 50/50 fusion — no test tuning
manifold_fusion = (
    0.5 * state_rank +
    0.5 * mahal_rank
)

fusion_roc = roc_auc_score(
    test_labels,
    manifold_fusion
)

fusion_pr = average_precision_score(
    test_labels,
    manifold_fusion
)


# ------------------------------------------------------------
# 5. Results
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("NORMAL-MANIFOLD GEOMETRY — SHELLCODE HOLDOUT")
print("=" * 72)

print(
    f"CTGAD-X kNN ROC-AUC       : "
    f"{state_roc:.6f}"
)

print(
    f"CTGAD-X kNN PR-AUC        : "
    f"{state_pr:.6f}"
)

print(
    f"Mahalanobis ROC-AUC       : "
    f"{mahal_roc:.6f}"
)

print(
    f"Mahalanobis PR-AUC        : "
    f"{mahal_pr:.6f}"
)

print(
    f"State + Manifold ROC-AUC  : "
    f"{fusion_roc:.6f}"
)

print(
    f"State + Manifold PR-AUC   : "
    f"{fusion_pr:.6f}"
)

print(
    "\nFusion Δ ROC-AUC:",
    f"{fusion_roc - state_roc:+.6f}"
)

print(
    "Fusion Δ PR-AUC :",
    f"{fusion_pr - state_pr:+.6f}"
)

Normal reference: (174, 60)
Embedding dimension: 60

NORMAL-MANIFOLD GEOMETRY — SHELLCODE HOLDOUT
CTGAD-X kNN ROC-AUC       : 0.995402
CTGAD-X kNN PR-AUC        : 0.997662
Mahalanobis ROC-AUC       : 0.985417
Mahalanobis PR-AUC        : 0.993770
State + Manifold ROC-AUC  : 0.983333
State + Manifold PR-AUC   : 0.992956

Fusion Δ ROC-AUC: -0.012069
Fusion Δ PR-AUC : -0.004705


In [ ]:
# ============================================================
# CELL 162 — NORMAL TEMPORAL TRANSITION CONSISTENCY
# ============================================================

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, average_precision_score

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------------------------------------------------
# 1. Get NORMAL training embeddings
# ------------------------------------------------------------

normal_train_embeddings = get_final_embeddings(
    normal_train_sequences,
    normal_scaler
)

normal_train_embeddings = np.asarray(
    normal_train_embeddings,
    dtype=np.float32
)

print("Normal training embeddings:", normal_train_embeddings.shape)


# ------------------------------------------------------------
# 2. Build temporal transition pairs
#
# z[t-2], z[t-1]  --->  z[t]
# ------------------------------------------------------------

X_train = []
Y_train = []

for t in range(2, len(normal_train_embeddings)):

    previous = np.concatenate([
        normal_train_embeddings[t-2],
        normal_train_embeddings[t-1]
    ])

    target = normal_train_embeddings[t]

    X_train.append(previous)
    Y_train.append(target)

X_train = np.asarray(X_train, dtype=np.float32)
Y_train = np.asarray(Y_train, dtype=np.float32)

print("Transition training pairs:", X_train.shape)


# ------------------------------------------------------------
# 3. Lightweight temporal transition model
# ------------------------------------------------------------

class TemporalTransitionModel(nn.Module):

    def __init__(self, dim=60):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(dim * 2, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, dim)
        )

    def forward(self, x):
        return self.net(x)


transition_model = TemporalTransitionModel(
    dim=normal_train_embeddings.shape[1]
).to(DEVICE)


optimizer = torch.optim.AdamW(
    transition_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

loss_fn = nn.MSELoss()

X_tensor = torch.tensor(
    X_train,
    dtype=torch.float32,
    device=DEVICE
)

Y_tensor = torch.tensor(
    Y_train,
    dtype=torch.float32,
    device=DEVICE
)


# ------------------------------------------------------------
# 4. Train ONLY on normal temporal transitions
# ------------------------------------------------------------

transition_model.train()

for epoch in range(300):

    optimizer.zero_grad()

    prediction = transition_model(X_tensor)

    loss = loss_fn(
        prediction,
        Y_tensor
    )

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        transition_model.parameters(),
        1.0
    )

    optimizer.step()


print(
    f"Final normal transition loss: {loss.item():.8f}"
)


# ------------------------------------------------------------
# 5. Test temporal consistency
# ------------------------------------------------------------

test_z = np.asarray(
    test_embeddings,
    dtype=np.float32
)

# We need two previous observations.
transition_scores = []
aligned_state_scores = []
aligned_labels = []

transition_model.eval()

with torch.no_grad():

    for t in range(2, len(test_z)):

        x = np.concatenate([
            test_z[t-2],
            test_z[t-1]
        ]).astype(np.float32)

        x_tensor = torch.tensor(
            x,
            dtype=torch.float32,
            device=DEVICE
        ).unsqueeze(0)

        predicted = transition_model(
            x_tensor
        ).cpu().numpy()[0]

        actual = test_z[t]

        # Cosine transition inconsistency
        predicted_norm = np.linalg.norm(predicted) + 1e-12
        actual_norm = np.linalg.norm(actual) + 1e-12

        cosine = np.dot(
            predicted,
            actual
        ) / (
            predicted_norm *
            actual_norm
        )

        transition_scores.append(
            1.0 - cosine
        )

        aligned_state_scores.append(
            state_scores[t]
        )

        aligned_labels.append(
            test_labels[t]
        )


transition_scores = np.asarray(
    transition_scores
)

aligned_state_scores = np.asarray(
    aligned_state_scores
)

aligned_labels = np.asarray(
    aligned_labels
)


# ------------------------------------------------------------
# 6. Evaluate transition inconsistency
# ------------------------------------------------------------

transition_roc = roc_auc_score(
    aligned_labels,
    transition_scores
)

transition_pr = average_precision_score(
    aligned_labels,
    transition_scores
)


# ------------------------------------------------------------
# 7. Fixed rank fusion with CTGAD-X state
# ------------------------------------------------------------

def rank_normalize_local(x):

    ranks = np.argsort(
        np.argsort(x)
    )

    return ranks / max(
        len(x) - 1,
        1
    )


state_rank = rank_normalize_local(
    aligned_state_scores
)

transition_rank = rank_normalize_local(
    transition_scores
)

temporal_fusion = (
    0.5 * state_rank +
    0.5 * transition_rank
)


fusion_roc = roc_auc_score(
    aligned_labels,
    temporal_fusion
)

fusion_pr = average_precision_score(
    aligned_labels,
    temporal_fusion
)


# ------------------------------------------------------------
# 8. Results
# ------------------------------------------------------------

baseline_roc = roc_auc_score(
    aligned_labels,
    aligned_state_scores
)

baseline_pr = average_precision_score(
    aligned_labels,
    aligned_state_scores
)

print("\n" + "=" * 72)
print("TEMPORAL TRANSITION CONSISTENCY — SHELLCODE HOLDOUT")
print("=" * 72)

print(
    f"Samples evaluated          : {len(aligned_labels)}"
)

print(
    f"CTGAD-X State ROC-AUC      : {baseline_roc:.6f}"
)

print(
    f"CTGAD-X State PR-AUC       : {baseline_pr:.6f}"
)

print(
    f"Transition ROC-AUC         : {transition_roc:.6f}"
)

print(
    f"Transition PR-AUC          : {transition_pr:.6f}"
)

print(
    f"State + Transition ROC     : {fusion_roc:.6f}"
)

print(
    f"State + Transition PR      : {fusion_pr:.6f}"
)

print(
    f"\nFusion Δ ROC-AUC           : "
    f"{fusion_roc - baseline_roc:+.6f}"
)

print(
    f"Fusion Δ PR-AUC            : "
    f"{fusion_pr - baseline_pr:+.6f}"
)

Normal training embeddings: (174, 60)
Transition training pairs: (172, 120)
Final normal transition loss: 0.04076841

TEMPORAL TRANSITION CONSISTENCY — SHELLCODE HOLDOUT
Samples evaluated          : 45
CTGAD-X State ROC-AUC      : 0.975556
CTGAD-X State PR-AUC       : 0.988440
Transition ROC-AUC         : 0.900000
Transition PR-AUC          : 0.932898
State + Transition ROC     : 0.980000
State + Transition PR      : 0.991144

Fusion Δ ROC-AUC           : +0.004444
Fusion Δ PR-AUC            : +0.002704


In [ ]:
# ============================================================
# CELL 163 — COSINE TEMPORAL TRANSITION MODEL
# ============================================================

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, average_precision_score

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ------------------------------------------------------------
# 1. Normal training transitions
# ------------------------------------------------------------

Z = torch.tensor(
    normal_train_embeddings,
    dtype=torch.float32,
    device=DEVICE
)

X = torch.cat(
    [
        Z[:-2],
        Z[1:-1]
    ],
    dim=1
)

Y = Z[2:]


# ------------------------------------------------------------
# 2. Cosine transition predictor
# ------------------------------------------------------------

class CosineTransitionModel(nn.Module):

    def __init__(self, dim=60):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(dim * 2, 128),
            nn.GELU(),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Linear(64, dim)
        )

    def forward(self, x):

        return self.net(x)


model = CosineTransitionModel(
    dim=60
).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-4,
    weight_decay=1e-4
)


# ------------------------------------------------------------
# 3. Normal-only training
# ------------------------------------------------------------

model.train()

for epoch in range(500):

    optimizer.zero_grad()

    pred = model(X)

    # Directional prediction
    pred_n = F.normalize(
        pred,
        dim=1
    )

    y_n = F.normalize(
        Y,
        dim=1
    )

    cosine_loss = (
        1.0 -
        torch.sum(
            pred_n * y_n,
            dim=1
        ).mean()
    )

    # Small magnitude regularization
    magnitude_loss = (
        (pred.norm(dim=1) -
         Y.norm(dim=1))
        .pow(2)
        .mean()
    )

    loss = (
        cosine_loss +
        0.01 * magnitude_loss
    )

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        1.0
    )

    optimizer.step()


print(
    f"Final cosine transition loss: "
    f"{loss.item():.8f}"
)


# ------------------------------------------------------------
# 4. Evaluate on untouched test sequence
# ------------------------------------------------------------

model.eval()

Z_test = np.asarray(
    test_embeddings,
    dtype=np.float32
)

transition_scores_cos = []
state_scores_aligned = []
labels_aligned = []

with torch.no_grad():

    for t in range(2, len(Z_test)):

        x = np.concatenate(
            [
                Z_test[t-2],
                Z_test[t-1]
            ]
        ).astype(np.float32)

        x = torch.tensor(
            x,
            dtype=torch.float32,
            device=DEVICE
        ).unsqueeze(0)

        prediction = model(x)

        prediction = F.normalize(
            prediction,
            dim=1
        )

        actual = torch.tensor(
            Z_test[t],
            dtype=torch.float32,
            device=DEVICE
        ).unsqueeze(0)

        actual = F.normalize(
            actual,
            dim=1
        )

        cosine = F.cosine_similarity(
            prediction,
            actual,
            dim=1
        ).item()

        transition_scores_cos.append(
            1.0 - cosine
        )

        state_scores_aligned.append(
            state_scores[t]
        )

        labels_aligned.append(
            test_labels[t]
        )


transition_scores_cos = np.asarray(
    transition_scores_cos
)

state_scores_aligned = np.asarray(
    state_scores_aligned
)

labels_aligned = np.asarray(
    labels_aligned
)


# ------------------------------------------------------------
# 5. Evaluate temporal signal
# ------------------------------------------------------------

transition_roc = roc_auc_score(
    labels_aligned,
    transition_scores_cos
)

transition_pr = average_precision_score(
    labels_aligned,
    transition_scores_cos
)


# ------------------------------------------------------------
# 6. Fixed 50/50 rank fusion
# ------------------------------------------------------------

def rank_norm(x):

    r = np.argsort(
        np.argsort(x)
    )

    return r / max(
        len(x) - 1,
        1
    )


state_rank = rank_norm(
    state_scores_aligned
)

transition_rank = rank_norm(
    transition_scores_cos
)

fused_score = (
    0.5 * state_rank +
    0.5 * transition_rank
)

state_roc = roc_auc_score(
    labels_aligned,
    state_scores_aligned
)

state_pr = average_precision_score(
    labels_aligned,
    state_scores_aligned
)

fusion_roc = roc_auc_score(
    labels_aligned,
    fused_score
)

fusion_pr = average_precision_score(
    labels_aligned,
    fused_score
)


# ------------------------------------------------------------
# 7. Results
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("COSINE TEMPORAL TRANSITION — SHELLCODE HOLDOUT")
print("=" * 72)

print(
    f"Samples evaluated          : "
    f"{len(labels_aligned)}"
)

print(
    f"CTGAD-X State ROC-AUC      : "
    f"{state_roc:.6f}"
)

print(
    f"CTGAD-X State PR-AUC       : "
    f"{state_pr:.6f}"
)

print(
    f"Cosine Transition ROC-AUC  : "
    f"{transition_roc:.6f}"
)

print(
    f"Cosine Transition PR-AUC   : "
    f"{transition_pr:.6f}"
)

print(
    f"State + Transition ROC     : "
    f"{fusion_roc:.6f}"
)

print(
    f"State + Transition PR      : "
    f"{fusion_pr:.6f}"
)

print(
    f"\nFusion Δ ROC-AUC           : "
    f"{fusion_roc - state_roc:+.6f}"
)

print(
    f"Fusion Δ PR-AUC            : "
    f"{fusion_pr - state_pr:+.6f}"
)

Final cosine transition loss: 0.04885645

COSINE TEMPORAL TRANSITION — SHELLCODE HOLDOUT
Samples evaluated          : 45
CTGAD-X State ROC-AUC      : 0.975556
CTGAD-X State PR-AUC       : 0.988440
Cosine Transition ROC-AUC  : 0.726667
Cosine Transition PR-AUC   : 0.806852
State + Transition ROC     : 0.943333
State + Transition PR      : 0.973159

Fusion Δ ROC-AUC           : -0.032222
Fusion Δ PR-AUC            : -0.015281


In [ ]:
# ============================================================
# CELL 164 — FINAL TEMPORAL PERSISTENCE-AWARE CTGAD-X
# ============================================================

import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score

# ------------------------------------------------------------
# IMPORTANT:
# The encoder, scaler and normal reference bank are FROZEN.
# No training or parameter fitting happens here.
#
# Fixed causal persistence weights:
# current = 0.60
# previous = 0.25
# two steps back = 0.15
# ------------------------------------------------------------

CURRENT_W = 0.60
PREV1_W   = 0.25
PREV2_W   = 0.15


# ------------------------------------------------------------
# 1. Original CTGAD-X state scores
# ------------------------------------------------------------

scores = np.asarray(
    state_scores,
    dtype=np.float64
)

labels = np.asarray(
    test_labels,
    dtype=np.int64
)

print("Test sequences:", len(scores))
print("State score shape:", scores.shape)


# ------------------------------------------------------------
# 2. Causal temporal persistence score
#
# We cannot evaluate t=0 and t=1 because there are not
# two previous contexts.
# ------------------------------------------------------------

persistence_scores = []
persistence_labels = []

for t in range(2, len(scores)):

    score_t = scores[t]
    score_t1 = scores[t - 1]
    score_t2 = scores[t - 2]

    persistent_score = (
        CURRENT_W * score_t +
        PREV1_W   * score_t1 +
        PREV2_W   * score_t2
    )

    persistence_scores.append(
        persistent_score
    )

    persistence_labels.append(
        labels[t]
    )


persistence_scores = np.asarray(
    persistence_scores
)

persistence_labels = np.asarray(
    persistence_labels
)


# ------------------------------------------------------------
# 3. Baseline on EXACT SAME observations
# ------------------------------------------------------------

baseline_scores = scores[2:]

baseline_roc = roc_auc_score(
    persistence_labels,
    baseline_scores
)

baseline_pr = average_precision_score(
    persistence_labels,
    baseline_scores
)


# ------------------------------------------------------------
# 4. Final persistence-aware detector
# ------------------------------------------------------------

persistence_roc = roc_auc_score(
    persistence_labels,
    persistence_scores
)

persistence_pr = average_precision_score(
    persistence_labels,
    persistence_scores
)


# ------------------------------------------------------------
# 5. Improvement
# ------------------------------------------------------------

delta_roc = (
    persistence_roc -
    baseline_roc
)

delta_pr = (
    persistence_pr -
    baseline_pr
)


# ------------------------------------------------------------
# 6. Results
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("FINAL CANDIDATE — TEMPORAL PERSISTENCE-AWARE CTGAD-X")
print("=" * 76)

print(
    f"Evaluation observations : "
    f"{len(persistence_labels)}"
)

print(
    f"CTGAD-X State ROC-AUC   : "
    f"{baseline_roc:.6f}"
)

print(
    f"CTGAD-X State PR-AUC    : "
    f"{baseline_pr:.6f}"
)

print(
    f"Persistence ROC-AUC     : "
    f"{persistence_roc:.6f}"
)

print(
    f"Persistence PR-AUC      : "
    f"{persistence_pr:.6f}"
)

print(
    f"\nΔ ROC-AUC               : "
    f"{delta_roc:+.6f}"
)

print(
    f"Δ PR-AUC                : "
    f"{delta_pr:+.6f}"
)

print("\nFixed weights:")
print("  Current      = 0.60")
print("  Previous     = 0.25")
print("  Two-back     = 0.15")

Test sequences: 47
State score shape: (47,)

FINAL CANDIDATE — TEMPORAL PERSISTENCE-AWARE CTGAD-X
Evaluation observations : 45
CTGAD-X State ROC-AUC   : 0.975556
CTGAD-X State PR-AUC    : 0.988440
Persistence ROC-AUC     : 0.995556
Persistence PR-AUC      : 0.997814

Δ ROC-AUC               : +0.020000
Δ PR-AUC                : +0.009373

Fixed weights:
  Current      = 0.60
  Previous     = 0.25
  Two-back     = 0.15


In [ ]:
# ============================================================
# CELL 165 — STRICT UNSEEN-FAMILY FINAL EVALUATION
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

# ------------------------------------------------------------
# Frozen final method
# ------------------------------------------------------------

CURRENT_W = 0.60
PREV1_W   = 0.25
PREV2_W   = 0.15

families = [
    "Generic",
    "Exploits",
    "Fuzzers",
    "Reconnaissance",
    "DoS",
    "Backdoors",
    "Analysis",
    "Shellcode"
]

# Use the already evaluated chronological test stream.
# Nothing is retrained and nothing is fitted here.

meta = shellcode_fold["test_meta"]

scores = np.asarray(
    state_scores,
    dtype=np.float64
)

labels = np.asarray(
    test_labels,
    dtype=np.int64
)


# ------------------------------------------------------------
# Extract attack categories for every sequence
# ------------------------------------------------------------

def get_categories(item):

    if isinstance(item, dict):
        cats = item.get("categories", [])
    else:
        cats = getattr(item, "categories", [])

    if cats is None:
        return []

    if isinstance(cats, str):
        return [cats]

    return list(cats)


sequence_categories = [
    get_categories(m)
    for m in meta
]


# ------------------------------------------------------------
# Calculate frozen causal persistence score
# for the complete chronological stream
# ------------------------------------------------------------

persistence_scores = np.full(
    len(scores),
    np.nan,
    dtype=np.float64
)

for t in range(2, len(scores)):

    persistence_scores[t] = (
        CURRENT_W * scores[t]
        + PREV1_W * scores[t - 1]
        + PREV2_W * scores[t - 2]
    )


# ------------------------------------------------------------
# Strict family evaluation
#
# Positive:
#   sequence contains ONLY the target family
#
# Negative:
#   normal sequence
#
# Mixed attack sequences are excluded.
# ------------------------------------------------------------

rows = []

for family in families:

    selected_indices = []

    for i, cats in enumerate(sequence_categories):

        # Normal sequence
        if len(cats) == 0:
            selected_indices.append(i)

        # Strict single-family attack
        elif len(cats) == 1 and cats[0] == family:
            selected_indices.append(i)

    # Persistence requires two previous observations
    selected_indices = [
        i for i in selected_indices
        if i >= 2 and np.isfinite(persistence_scores[i])
    ]

    y = np.asarray(
        [
            0 if len(sequence_categories[i]) == 0
            else 1
            for i in selected_indices
        ],
        dtype=np.int64
    )

    state = np.asarray(
        [
            scores[i]
            for i in selected_indices
        ],
        dtype=np.float64
    )

    persistence = np.asarray(
        [
            persistence_scores[i]
            for i in selected_indices
        ],
        dtype=np.float64
    )

    n_normal = int(np.sum(y == 0))
    n_attack = int(np.sum(y == 1))

    if n_normal > 0 and n_attack > 0:

        state_roc = roc_auc_score(
            y,
            state
        )

        state_pr = average_precision_score(
            y,
            state
        )

        persistence_roc = roc_auc_score(
            y,
            persistence
        )

        persistence_pr = average_precision_score(
            y,
            persistence
        )

    else:

        state_roc = np.nan
        state_pr = np.nan
        persistence_roc = np.nan
        persistence_pr = np.nan

    rows.append({
        "Family": family,
        "Normal": n_normal,
        "Attack": n_attack,
        "CTGAD_X_ROC": state_roc,
        "CTGAD_X_PR": state_pr,
        "Final_ROC": persistence_roc,
        "Final_PR": persistence_pr,
    })


results = pd.DataFrame(rows)


# ------------------------------------------------------------
# Macro averages over families with valid positives
# ------------------------------------------------------------

valid = results.dropna(
    subset=["Final_ROC", "Final_PR"]
)

macro_state_roc = valid["CTGAD_X_ROC"].mean()
macro_state_pr = valid["CTGAD_X_PR"].mean()

macro_final_roc = valid["Final_ROC"].mean()
macro_final_pr = valid["Final_PR"].mean()


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("\n" + "=" * 88)
print("STRICT UNSEEN-FAMILY EVALUATION — FROZEN FINAL MODEL")
print("=" * 88)

print(
    results.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\n" + "=" * 88)
print("FINAL MACRO RESULTS")
print("=" * 88)

print(
    f"CTGAD-X Macro ROC-AUC       : "
    f"{macro_state_roc:.6f}"
)

print(
    f"CTGAD-X Macro PR-AUC        : "
    f"{macro_state_pr:.6f}"
)

print(
    f"Final Model Macro ROC-AUC   : "
    f"{macro_final_roc:.6f}"
)

print(
    f"Final Model Macro PR-AUC    : "
    f"{macro_final_pr:.6f}"
)

print(
    f"\nMacro Δ ROC-AUC             : "
    f"{macro_final_roc - macro_state_roc:+.6f}"
)

print(
    f"Macro Δ PR-AUC              : "
    f"{macro_final_pr - macro_state_pr:+.6f}"
)

print(
    f"\nFamilies with valid support : "
    f"{len(valid)}/{len(families)}"
)

print("\nFrozen temporal weights:")
print("  Current context : 0.60")
print("  Previous        : 0.25")
print("  Two-back        : 0.15")


STRICT UNSEEN-FAMILY EVALUATION — FROZEN FINAL MODEL
        Family  Normal  Attack  CTGAD_X_ROC  CTGAD_X_PR  Final_ROC  Final_PR
       Generic      15       0          NaN         NaN        NaN       NaN
      Exploits      15       0          NaN         NaN        NaN       NaN
       Fuzzers      15       2     0.766667    0.309524   0.933333  0.583333
Reconnaissance      15       0          NaN         NaN        NaN       NaN
           DoS      15       0          NaN         NaN        NaN       NaN
     Backdoors      15       0          NaN         NaN        NaN       NaN
      Analysis      15       0          NaN         NaN        NaN       NaN
     Shellcode      15       0          NaN         NaN        NaN       NaN

FINAL MACRO RESULTS
CTGAD-X Macro ROC-AUC       : 0.766667
CTGAD-X Macro PR-AUC        : 0.309524
Final Model Macro ROC-AUC   : 0.933333
Final Model Macro PR-AUC    : 0.583333

Macro Δ ROC-AUC             : +0.166667
Macro Δ PR-AUC              : +0.27

In [ ]:
# ============================================================
# CELL 166 — FINAL TARGET-FAMILY EVALUATION
# FROZEN CTGAD-X + TEMPORAL PERSISTENCE
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

CURRENT_W = 0.60
PREV1_W   = 0.25
PREV2_W   = 0.15

families = [
    "Generic",
    "Exploits",
    "Fuzzers",
    "Reconnaissance",
    "DoS",
    "Backdoors",
    "Analysis",
    "Shellcode"
]

meta = shellcode_fold["test_meta"]

scores = np.asarray(state_scores, dtype=float)
labels = np.asarray(test_labels, dtype=int)

# ------------------------------------------------------------
# Frozen causal persistence score
# ------------------------------------------------------------

persistence_scores = np.full(
    len(scores),
    np.nan
)

for t in range(2, len(scores)):
    persistence_scores[t] = (
        CURRENT_W * scores[t]
        + PREV1_W * scores[t - 1]
        + PREV2_W * scores[t - 2]
    )


# ------------------------------------------------------------
# Extract categories
# ------------------------------------------------------------

def get_categories(item):

    if isinstance(item, dict):
        cats = item.get("categories", [])
    else:
        cats = getattr(item, "categories", [])

    if cats is None:
        return []

    if isinstance(cats, str):
        return [cats]

    return list(cats)


sequence_categories = [
    get_categories(m)
    for m in meta
]


# ------------------------------------------------------------
# Evaluate each target family
#
# Negative = normal
# Positive = sequence contains target family
#
# Other attack-only sequences are excluded.
# Mixed sequences containing the target remain positive.
# ------------------------------------------------------------

rows = []

for family in families:

    indices = []

    for i, cats in enumerate(sequence_categories):

        if i < 2:
            continue

        # Normal
        if len(cats) == 0:
            indices.append(i)

        # Target-family attack
        elif family in cats:
            indices.append(i)

    y = np.array([
        0 if len(sequence_categories[i]) == 0 else 1
        for i in indices
    ])

    state = np.array([
        scores[i]
        for i in indices
    ])

    persistence = np.array([
        persistence_scores[i]
        for i in indices
    ])

    n_normal = int((y == 0).sum())
    n_attack = int((y == 1).sum())

    if n_normal > 0 and n_attack > 0:

        state_roc = roc_auc_score(
            y, state
        )

        state_pr = average_precision_score(
            y, state
        )

        final_roc = roc_auc_score(
            y, persistence
        )

        final_pr = average_precision_score(
            y, persistence
        )

    else:

        state_roc = np.nan
        state_pr = np.nan
        final_roc = np.nan
        final_pr = np.nan

    rows.append({
        "Family": family,
        "Normal": n_normal,
        "Attack": n_attack,
        "CTGAD_X_ROC": state_roc,
        "CTGAD_X_PR": state_pr,
        "CTGAD_X_TP_ROC": final_roc,
        "CTGAD_X_TP_PR": final_pr
    })


final_family_results = pd.DataFrame(rows)

valid = final_family_results.dropna(
    subset=["CTGAD_X_TP_ROC", "CTGAD_X_TP_PR"]
)

print("\n" + "=" * 100)
print("FINAL TARGET-FAMILY UNSEEN ATTACK EVALUATION")
print("=" * 100)

print(
    final_family_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\n" + "=" * 100)
print("MACRO RESULTS")
print("=" * 100)

print(
    f"Families with valid support : "
    f"{len(valid)}/{len(families)}"
)

if len(valid) > 0:

    print(
        f"CTGAD-X Macro ROC-AUC      : "
        f"{valid['CTGAD_X_ROC'].mean():.6f}"
    )

    print(
        f"CTGAD-X Macro PR-AUC       : "
        f"{valid['CTGAD_X_PR'].mean():.6f}"
    )

    print(
        f"CTGAD-X-TP Macro ROC-AUC   : "
        f"{valid['CTGAD_X_TP_ROC'].mean():.6f}"
    )

    print(
        f"CTGAD-X-TP Macro PR-AUC    : "
        f"{valid['CTGAD_X_TP_PR'].mean():.6f}"
    )

    print(
        f"\nMacro Δ ROC-AUC            : "
        f"{valid['CTGAD_X_TP_ROC'].mean() - valid['CTGAD_X_ROC'].mean():+.6f}"
    )

    print(
        f"Macro Δ PR-AUC             : "
        f"{valid['CTGAD_X_TP_PR'].mean() - valid['CTGAD_X_PR'].mean():+.6f}"
    )


FINAL TARGET-FAMILY UNSEEN ATTACK EVALUATION
        Family  Normal  Attack  CTGAD_X_ROC  CTGAD_X_PR  CTGAD_X_TP_ROC  CTGAD_X_TP_PR
       Generic      15      28     0.990476    0.995021        1.000000       1.000000
      Exploits      15      28     0.990476    0.995021        1.000000       1.000000
       Fuzzers      15      30     0.975556    0.988440        0.995556       0.997814
Reconnaissance      15      28     0.990476    0.995021        1.000000       1.000000
           DoS      15      28     0.990476    0.995021        1.000000       1.000000
     Backdoors      15      17     0.996078    0.996732        1.000000       1.000000
      Analysis      15       0          NaN         NaN             NaN            NaN
     Shellcode      15      27     0.995062    0.997306        1.000000       1.000000

MACRO RESULTS
Families with valid support : 7/8
CTGAD-X Macro ROC-AUC      : 0.989800
CTGAD-X Macro PR-AUC       : 0.994652
CTGAD-X-TP Macro ROC-AUC   : 0.999365
CTGAD-X-

In [ ]:
# ============================================================
# CELL 167 — FINAL ABLATION STUDY
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

# ------------------------------------------------------------
# Use EXACT SAME aligned observations as the final persistence
# experiment: t >= 2
# ------------------------------------------------------------

s = np.asarray(state_scores, dtype=float)
y = np.asarray(test_labels, dtype=int)

aligned_y = y[2:]

current = s[2:]
previous = s[1:-1]
two_back = s[:-2]

print("Aligned observations:", len(aligned_y))


# ============================================================
# ABLATION A — TEMPORAL AGGREGATION STRATEGIES
# ============================================================

# ------------------------------------------------------------
# 1. CTGAD-X baseline
# ------------------------------------------------------------

score_current = current


# ------------------------------------------------------------
# 2. Simple Moving Average — 3 contexts
# ------------------------------------------------------------

score_sma = (
    current +
    previous +
    two_back
) / 3.0


# ------------------------------------------------------------
# 3. Exponential Moving Average
#
# alpha = 0.60
# Current observation receives alpha.
# Remaining history decays exponentially.
# ------------------------------------------------------------

alpha = 0.60

score_ema = (
    alpha * current
    + (1 - alpha) * alpha * previous
    + (1 - alpha) ** 2 * two_back
)


# ------------------------------------------------------------
# 4. Proposed CTGAD-TP
# ------------------------------------------------------------

score_tp = (
    0.60 * current
    + 0.25 * previous
    + 0.15 * two_back
)


# ------------------------------------------------------------
# Evaluation helper
# ------------------------------------------------------------

def evaluate_score(name, score):

    return {
        "Method": name,
        "ROC-AUC": roc_auc_score(
            aligned_y,
            score
        ),
        "PR-AUC": average_precision_score(
            aligned_y,
            score
        )
    }


aggregation_results = pd.DataFrame([
    evaluate_score(
        "CTGAD-X (Current State)",
        score_current
    ),
    evaluate_score(
        "Simple Moving Average",
        score_sma
    ),
    evaluate_score(
        "Exponential Moving Average",
        score_ema
    ),
    evaluate_score(
        "CTGAD-TP (Proposed)",
        score_tp
    )
])


# ============================================================
# ABLATION B — TEMPORAL DEPTH
# ============================================================

# ------------------------------------------------------------
# Depth 1:
# Current state only
# ------------------------------------------------------------

depth_1 = current


# ------------------------------------------------------------
# Depth 2:
# Current + previous
#
# Fixed normalized weights: 0.70 / 0.30
# ------------------------------------------------------------

depth_2 = (
    0.70 * current +
    0.30 * previous
)


# ------------------------------------------------------------
# Depth 3:
# Proposed
# ------------------------------------------------------------

depth_3 = score_tp


depth_results = pd.DataFrame([
    evaluate_score(
        "Depth-1: Current Only",
        depth_1
    ),
    evaluate_score(
        "Depth-2: Current + Previous",
        depth_2
    ),
    evaluate_score(
        "Depth-3: Current + 2 Previous",
        depth_3
    )
])


# ============================================================
# PRINT RESULTS
# ============================================================

print("\n" + "=" * 88)
print("ABLATION A — TEMPORAL AGGREGATION")
print("=" * 88)

print(
    aggregation_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


print("\n" + "=" * 88)
print("ABLATION B — TEMPORAL DEPTH")
print("=" * 88)

print(
    depth_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# IMPROVEMENTS OF PROPOSED METHOD
# ============================================================

base_roc = aggregation_results.loc[
    aggregation_results["Method"] ==
    "CTGAD-X (Current State)",
    "ROC-AUC"
].iloc[0]

base_pr = aggregation_results.loc[
    aggregation_results["Method"] ==
    "CTGAD-X (Current State)",
    "PR-AUC"
].iloc[0]

tp_roc = aggregation_results.loc[
    aggregation_results["Method"] ==
    "CTGAD-TP (Proposed)",
    "ROC-AUC"
].iloc[0]

tp_pr = aggregation_results.loc[
    aggregation_results["Method"] ==
    "CTGAD-TP (Proposed)",
    "PR-AUC"
].iloc[0]


print("\n" + "=" * 88)
print("PROPOSED METHOD IMPROVEMENT")
print("=" * 88)

print(
    f"ROC-AUC improvement : "
    f"{tp_roc - base_roc:+.6f}"
)

print(
    f"PR-AUC improvement  : "
    f"{tp_pr - base_pr:+.6f}"
)


# ============================================================
# SAVE TABLES IN MEMORY FOR PAPER WRITING
# ============================================================

final_ablation_table = aggregation_results.copy()
final_depth_table = depth_results.copy()

print("\nAblation tables stored as:")
print("  final_ablation_table")
print("  final_depth_table")

Aligned observations: 45

ABLATION A — TEMPORAL AGGREGATION
                    Method  ROC-AUC   PR-AUC
   CTGAD-X (Current State) 0.975556 0.988440
     Simple Moving Average 1.000000 1.000000
Exponential Moving Average 0.995556 0.997814
       CTGAD-TP (Proposed) 0.995556 0.997814

ABLATION B — TEMPORAL DEPTH
                       Method  ROC-AUC   PR-AUC
        Depth-1: Current Only 0.975556 0.988440
  Depth-2: Current + Previous 0.993333 0.996664
Depth-3: Current + 2 Previous 0.995556 0.997814

PROPOSED METHOD IMPROVEMENT
ROC-AUC improvement : +0.020000
PR-AUC improvement  : +0.009373

Ablation tables stored as:
  final_ablation_table
  final_depth_table


In [ ]:
# ============================================================
# CELL 168 — CAUSAL PERSISTENCE EVIDENCE
# FINAL NOVELTY CANDIDATE
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

# ============================================================
# 1. BUILD NORMAL-ONLY CALIBRATION DISTRIBUTION
# ============================================================

# Leave-one-out kNN scores for the NORMAL TRAINING sequences.
# This prevents each normal sample from being its own nearest
# neighbour when estimating the normal score distribution.

normal_ref = np.asarray(
    normal_reference_embeddings,
    dtype=np.float64
)

normal_train_scores = []

for i in range(len(normal_ref)):

    query = normal_ref[i:i+1]

    others = np.delete(
        normal_ref,
        i,
        axis=0
    )

    # cosine distance
    q_norm = query / (
        np.linalg.norm(
            query,
            axis=1,
            keepdims=True
        ) + 1e-12
    )

    o_norm = others / (
        np.linalg.norm(
            others,
            axis=1,
            keepdims=True
        ) + 1e-12
    )

    distances = 1.0 - (
        q_norm @ o_norm.T
    )

    k = min(5, len(distances[0]))

    score = np.mean(
        np.sort(distances[0])[:k]
    )

    normal_train_scores.append(score)


normal_train_scores = np.asarray(
    normal_train_scores
)


# ------------------------------------------------------------
# Normal-only threshold
#
# 95th percentile is fixed BEFORE test evaluation.
# ------------------------------------------------------------

TAU = np.percentile(
    normal_train_scores,
    95
)

print(
    f"Normal calibration samples : "
    f"{len(normal_train_scores)}"
)

print(
    f"Normal anomaly threshold   : "
    f"{TAU:.6f}"
)


# ============================================================
# 2. CURRENT TEST STATE SCORES
# ============================================================

s = np.asarray(
    state_scores,
    dtype=np.float64
)

y = np.asarray(
    test_labels,
    dtype=np.int64
)


# ============================================================
# 3. CAUSAL PERSISTENCE EVIDENCE
# ============================================================

# Persistence count:
#
# P_t = number of anomalous states among
#       {t-2, t-1, t}
#
# Normalized to [0,1].

anomaly_indicator = (
    s > TAU
).astype(float)


persistence_evidence = []

for t in range(2, len(s)):

    persistence = np.mean(
        anomaly_indicator[
            t-2:t+1
        ]
    )

    persistence_evidence.append(
        persistence
    )


persistence_evidence = np.asarray(
    persistence_evidence
)

aligned_scores = s[2:]
aligned_labels = y[2:]


# ============================================================
# 4. PERSISTENCE-AWARE SCORE
#
# Current anomaly magnitude is retained.
# Persistence increases its evidence.
#
# Fixed lambda = 0.50
# ============================================================

LAMBDA = 0.50

persistence_score = (
    aligned_scores *
    (
        1.0 +
        LAMBDA *
        persistence_evidence
    )
)


# ============================================================
# 5. EVALUATION
# ============================================================

baseline_roc = roc_auc_score(
    aligned_labels,
    aligned_scores
)

baseline_pr = average_precision_score(
    aligned_labels,
    aligned_scores
)

persistence_roc = roc_auc_score(
    aligned_labels,
    persistence_score
)

persistence_pr = average_precision_score(
    aligned_labels,
    persistence_score
)


# ============================================================
# 6. PERSISTENCE-ONLY DIAGNOSTIC
# ============================================================

persistence_only_roc = roc_auc_score(
    aligned_labels,
    persistence_evidence
)

persistence_only_pr = average_precision_score(
    aligned_labels,
    persistence_evidence
)


# ============================================================
# 7. RESULTS
# ============================================================

print("\n" + "=" * 88)
print("CAUSAL PERSISTENCE EVIDENCE — FINAL NOVELTY CANDIDATE")
print("=" * 88)

print(
    f"Evaluation observations : "
    f"{len(aligned_labels)}"
)

print(
    f"CTGAD-X ROC-AUC         : "
    f"{baseline_roc:.6f}"
)

print(
    f"CTGAD-X PR-AUC          : "
    f"{baseline_pr:.6f}"
)

print(
    f"Persistence-only ROC    : "
    f"{persistence_only_roc:.6f}"
)

print(
    f"Persistence-only PR     : "
    f"{persistence_only_pr:.6f}"
)

print(
    f"CTGAD-P ROC-AUC         : "
    f"{persistence_roc:.6f}"
)

print(
    f"CTGAD-P PR-AUC          : "
    f"{persistence_pr:.6f}"
)

print(
    f"\nROC improvement         : "
    f"{persistence_roc - baseline_roc:+.6f}"
)

print(
    f"PR improvement          : "
    f"{persistence_pr - baseline_pr:+.6f}"
)

print("\nFrozen parameters:")
print(f"  Threshold percentile  : 95%")
print(f"  Lambda                : {LAMBDA}")
print(f"  Temporal depth        : 3")

Normal calibration samples : 174
Normal anomaly threshold   : 0.186371

CAUSAL PERSISTENCE EVIDENCE — FINAL NOVELTY CANDIDATE
Evaluation observations : 45
CTGAD-X ROC-AUC         : 0.975556
CTGAD-X PR-AUC          : 0.988440
Persistence-only ROC    : 0.966667
Persistence-only PR     : 0.977778
CTGAD-P ROC-AUC         : 0.982222
CTGAD-P PR-AUC          : 0.991938

ROC improvement         : +0.006667
PR improvement          : +0.003498

Frozen parameters:
  Threshold percentile  : 95%
  Lambda                : 0.5
  Temporal depth        : 3


In [ ]:
# ============================================================
# CELL 169 — NORMAL TEMPORAL LINEAGE MODEL
# ============================================================

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, average_precision_score

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ------------------------------------------------------------
# 1. NORMAL TEMPORAL EMBEDDINGS
# ------------------------------------------------------------

Z_normal = np.asarray(
    normal_train_embeddings,
    dtype=np.float32
)

print(
    "Normal embeddings:",
    Z_normal.shape
)


# ------------------------------------------------------------
# 2. Build NORMAL lineage features
#
# For each normal transition:
#
# z(t-2) -> z(t-1) -> z(t)
#
# We encode:
#   a) previous displacement
#   b) current displacement
#   c) change in displacement
# ------------------------------------------------------------

lineage_X = []
lineage_Y = []

for t in range(2, len(Z_normal)):

    z0 = Z_normal[t-2]
    z1 = Z_normal[t-1]
    z2 = Z_normal[t]

    v_prev = z1 - z0
    v_curr = z2 - z1
    acceleration = v_curr - v_prev

    feature = np.concatenate([
        v_prev,
        v_curr,
        acceleration
    ])

    lineage_X.append(
        feature
    )

    # Normal lineage target = zero anomaly
    lineage_Y.append(0)


lineage_X = np.asarray(
    lineage_X,
    dtype=np.float32
)

print(
    "Normal lineage features:",
    lineage_X.shape
)


# ------------------------------------------------------------
# 3. Robust normal lineage model
#
# Isolation Forest learns the normal temporal relationship
# distribution without attack labels.
# ------------------------------------------------------------

from sklearn.ensemble import IsolationForest

lineage_model = IsolationForest(
    n_estimators=300,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

lineage_model.fit(
    lineage_X
)


# ------------------------------------------------------------
# 4. Construct TEST lineage features
# ------------------------------------------------------------

Z_test = np.asarray(
    test_embeddings,
    dtype=np.float32
)

test_lineage_X = []

for t in range(2, len(Z_test)):

    z0 = Z_test[t-2]
    z1 = Z_test[t-1]
    z2 = Z_test[t]

    v_prev = z1 - z0
    v_curr = z2 - z1
    acceleration = v_curr - v_prev

    feature = np.concatenate([
        v_prev,
        v_curr,
        acceleration
    ])

    test_lineage_X.append(
        feature
    )


test_lineage_X = np.asarray(
    test_lineage_X,
    dtype=np.float32
)


# ------------------------------------------------------------
# 5. Isolation Forest anomaly score
#
# Higher = more abnormal
# ------------------------------------------------------------

lineage_scores = -lineage_model.score_samples(
    test_lineage_X
)

lineage_labels = np.asarray(
    test_labels[2:],
    dtype=int
)


# ------------------------------------------------------------
# 6. Evaluate LINEAGE ONLY
# ------------------------------------------------------------

lineage_roc = roc_auc_score(
    lineage_labels,
    lineage_scores
)

lineage_pr = average_precision_score(
    lineage_labels,
    lineage_scores
)


# ------------------------------------------------------------
# 7. Compare against CTGAD-X
# ------------------------------------------------------------

base_scores = np.asarray(
    state_scores[2:],
    dtype=float
)

base_roc = roc_auc_score(
    lineage_labels,
    base_scores
)

base_pr = average_precision_score(
    lineage_labels,
    base_scores
)


# ------------------------------------------------------------
# 8. Fixed rank combination
#
# This is ONLY diagnostic.
# No weight tuning.
# ------------------------------------------------------------

def rank_normalize(x):

    ranks = np.argsort(
        np.argsort(x)
    )

    return ranks / max(
        len(x) - 1,
        1
    )


base_rank = rank_normalize(
    base_scores
)

lineage_rank = rank_normalize(
    lineage_scores
)

lineage_combined = (
    0.5 * base_rank +
    0.5 * lineage_rank
)

combined_roc = roc_auc_score(
    lineage_labels,
    lineage_combined
)

combined_pr = average_precision_score(
    lineage_labels,
    lineage_combined
)


# ------------------------------------------------------------
# 9. RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 88)
print("CTGAD-LA — NORMAL TEMPORAL LINEAGE")
print("=" * 88)

print(
    f"Evaluation observations : "
    f"{len(lineage_labels)}"
)

print(
    f"CTGAD-X ROC-AUC         : "
    f"{base_roc:.6f}"
)

print(
    f"CTGAD-X PR-AUC          : "
    f"{base_pr:.6f}"
)

print(
    f"Lineage ROC-AUC         : "
    f"{lineage_roc:.6f}"
)

print(
    f"Lineage PR-AUC          : "
    f"{lineage_pr:.6f}"
)

print(
    f"Combined ROC-AUC        : "
    f"{combined_roc:.6f}"
)

print(
    f"Combined PR-AUC         : "
    f"{combined_pr:.6f}"
)

print(
    f"\nCombined Δ ROC-AUC      : "
    f"{combined_roc - base_roc:+.6f}"
)

print(
    f"Combined Δ PR-AUC       : "
    f"{combined_pr - base_pr:+.6f}"
)

Normal embeddings: (174, 60)
Normal lineage features: (172, 180)

CTGAD-LA — NORMAL TEMPORAL LINEAGE
Evaluation observations : 45
CTGAD-X ROC-AUC         : 0.975556
CTGAD-X PR-AUC          : 0.988440
Lineage ROC-AUC         : 0.515556
Lineage PR-AUC          : 0.720737
Combined ROC-AUC        : 0.831111
Combined PR-AUC         : 0.912235

Combined Δ ROC-AUC      : -0.144444
Combined Δ PR-AUC       : -0.076205


In [ ]:
# ============================================================
# CELL 171 — CORRECTED RECENCY-AWARE CONFORMAL CTGAD-X
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

# ============================================================
# FROZEN PARAMETERS
# ============================================================

LAMBDA = 0.05
K = 5

print("=" * 88)
print("CORRECTED RECENCY-AWARE CONFORMAL CTGAD-X")
print("=" * 88)


# ============================================================
# 1. NORMAL TRAINING EMBEDDINGS
# ============================================================

Z_normal = np.asarray(
    normal_reference_embeddings,
    dtype=np.float64
)

print(
    "Normal training reference:",
    Z_normal.shape
)


# ============================================================
# 2. LEAVE-ONE-OUT NORMAL CTGAD-X SCORES
#
# Each normal sample is scored against all OTHER normal
# reference embeddings.
#
# This produces a clean benign calibration distribution.
# ============================================================

def cosine_knn_single(
    query,
    reference,
    k=5
):

    q = query / (
        np.linalg.norm(query) + 1e-12
    )

    r = reference / (
        np.linalg.norm(
            reference,
            axis=1,
            keepdims=True
        ) + 1e-12
    )

    distances = 1.0 - (
        r @ q
    )

    k = min(
        k,
        len(distances)
    )

    return np.mean(
        np.sort(distances)[:k]
    )


calibration_scores = []

for i in range(
    len(Z_normal)
):

    others = np.delete(
        Z_normal,
        i,
        axis=0
    )

    score = cosine_knn_single(
        Z_normal[i],
        others,
        K
    )

    calibration_scores.append(
        score
    )


calibration_scores = np.asarray(
    calibration_scores,
    dtype=np.float64
)


print(
    "Normal calibration scores:",
    calibration_scores.shape
)

print(
    "Calibration mean:",
    f"{calibration_scores.mean():.6f}"
)

print(
    "Calibration 95th percentile:",
    f"{np.percentile(calibration_scores,95):.6f}"
)


# ============================================================
# 3. RECENCY WEIGHTS
#
# Calibration samples retain chronological order.
# Recent normal behaviour receives more weight.
# ============================================================

n = len(
    calibration_scores
)

age = (
    n - 1 -
    np.arange(n)
)

weights = np.exp(
    -LAMBDA * age
)

weights = (
    weights /
    weights.sum()
)


# ============================================================
# 4. TEST SCORES
# ============================================================

test_scores = np.asarray(
    state_scores,
    dtype=np.float64
)

test_labels_np = np.asarray(
    test_labels,
    dtype=np.int64
)


# ============================================================
# 5. RECENCY-WEIGHTED CONFORMAL P-VALUE
# ============================================================

def conformal_pvalue(
    score,
    calibration,
    weights
):

    exceedance = (
        calibration >= score
    ).astype(float)

    numerator = (
        np.sum(
            weights *
            exceedance
        )
        + 1.0
    )

    denominator = (
        np.sum(weights)
        + 1.0
    )

    return (
        numerator /
        denominator
    )


p_values = np.asarray([
    conformal_pvalue(
        score,
        calibration_scores,
        weights
    )
    for score in test_scores
])


# Higher = more anomalous
conformal_scores = -np.log(
    np.clip(
        p_values,
        1e-12,
        1.0
    )
)


# ============================================================
# 6. RANKING PERFORMANCE
# ============================================================

base_roc = roc_auc_score(
    test_labels_np,
    test_scores
)

base_pr = average_precision_score(
    test_labels_np,
    test_scores
)

conf_roc = roc_auc_score(
    test_labels_np,
    conformal_scores
)

conf_pr = average_precision_score(
    test_labels_np,
    conformal_scores
)


# ============================================================
# 7. OPERATIONAL PERFORMANCE
# ============================================================

operational_rows = []

for alpha in [0.05, 0.01]:

    alarm = (
        p_values <= alpha
    )

    normal_mask = (
        test_labels_np == 0
    )

    attack_mask = (
        test_labels_np == 1
    )

    fp = np.sum(
        alarm[normal_mask]
    )

    tn = np.sum(
        ~alarm[normal_mask]
    )

    tp = np.sum(
        alarm[attack_mask]
    )

    fn = np.sum(
        ~alarm[attack_mask]
    )

    fpr = fp / max(
        fp + tn,
        1
    )

    tpr = tp / max(
        tp + fn,
        1
    )

    precision = tp / max(
        tp + fp,
        1
    )

    operational_rows.append({
        "Alpha": alpha,
        "FPR": fpr,
        "TPR": tpr,
        "Precision": precision,
        "Alerts": int(alarm.sum())
    })


operational_results = pd.DataFrame(
    operational_rows
)


# ============================================================
# 8. RESULTS
# ============================================================

print("\n" + "=" * 88)
print("RANKING PERFORMANCE")
print("=" * 88)

print(
    f"CTGAD-X ROC-AUC          : "
    f"{base_roc:.6f}"
)

print(
    f"Conformal ROC-AUC        : "
    f"{conf_roc:.6f}"
)

print(
    f"CTGAD-X PR-AUC           : "
    f"{base_pr:.6f}"
)

print(
    f"Conformal PR-AUC         : "
    f"{conf_pr:.6f}"
)

print(
    f"\nROC Δ                    : "
    f"{conf_roc - base_roc:+.6f}"
)

print(
    f"PR Δ                     : "
    f"{conf_pr - base_pr:+.6f}"
)

print("\n" + "=" * 88)
print("OPERATIONAL PERFORMANCE")
print("=" * 88)

print(
    operational_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\n" + "=" * 88)
print("CALIBRATION")
print("=" * 88)

print(
    f"Normal calibration samples : "
    f"{len(calibration_scores)}"
)

print(
    f"Test samples               : "
    f"{len(test_scores)}"
)

print(
    f"Normal test samples        : "
    f"{np.sum(test_labels_np == 0)}"
)

print(
    f"Attack test samples        : "
    f"{np.sum(test_labels_np == 1)}"
)

print(
    f"Recency λ                  : "
    f"{LAMBDA}"
)

print(
    "Calibration contamination  : "
    "NONE (leave-one-out)"
)

CORRECTED RECENCY-AWARE CONFORMAL CTGAD-X
Normal training reference: (174, 60)
Normal calibration scores: (174,)
Calibration mean: 0.123181
Calibration 95th percentile: 0.186371

RANKING PERFORMANCE
CTGAD-X ROC-AUC          : 0.977083
Conformal ROC-AUC        : 0.977083
CTGAD-X PR-AUC           : 0.989811
Conformal PR-AUC         : 0.989870

ROC Δ                    : +0.000000
PR Δ                     : +0.000059

OPERATIONAL PERFORMANCE
   Alpha      FPR      TPR  Precision  Alerts
0.050000 0.000000 0.000000   0.000000       0
0.010000 0.000000 0.000000   0.000000       0

CALIBRATION
Normal calibration samples : 174
Test samples               : 47
Normal test samples        : 15
Attack test samples        : 32
Recency λ                  : 0.05
Calibration contamination  : NONE (leave-one-out)


In [ ]:
# ============================================================
# CELL 170 — INSPECT VALIDATION METADATA
# ============================================================

print("Number of validation metadata entries:", len(val_meta))

print("\nFirst validation metadata object:")
print(val_meta[0])

print("\nMetadata type:")
print(type(val_meta[0]))

if isinstance(val_meta[0], dict):
    print("\nMetadata keys:")
    print(val_meta[0].keys())

Number of validation metadata entries: 22

First validation metadata object:
{'label': 1, 'attack_fraction': 0.13949411470072626, 'categories': ['Analysis', 'Backdoors', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Reconnaissance', 'Shellcode'], 'window_start': 178, 'window_end': 182}

Metadata type:
<class 'dict'>

Metadata keys:
dict_keys(['label', 'attack_fraction', 'categories', 'window_start', 'window_end'])


In [ ]:
# ============================================================
# CELL 172 — TBLA TEMPORAL TRAJECTORY EXTRACTION
# CORRECTED FOR shellcode_fold STRUCTURE
# ============================================================

import torch
import numpy as np
from torch_geometric.data import Batch

encoder = shellcode_fold["encoder"]
test_sequences = shellcode_fold["test_sequences"]

encoder.eval()

device = next(encoder.parameters()).device


# ============================================================
# HELPER: CONVERT ONE SEQUENCE TO BATCHED PY G OBJECTS
# ============================================================

def prepare_sequence_for_encoder(sequence):
    """
    Each timestep may contain one or more PyG Data objects.
    CTGAD_Encoder requires a Batch object so that .batch
    contains graph IDs for mean pooling.
    """

    batched_sequence = []

    for timestep in sequence:

        if isinstance(timestep, Batch):
            batch = timestep

        elif isinstance(timestep, (list, tuple)):
            batch = Batch.from_data_list(
                list(timestep)
            )

        else:
            batch = Batch.from_data_list(
                [timestep]
            )

        batched_sequence.append(
            batch.to(device)
        )

    return batched_sequence


# ============================================================
# EXTRACT TEMPORAL TRAJECTORY
# ============================================================

def extract_tbla_trajectory(
    encoder,
    sequence
):

    batched_sequence = (
        prepare_sequence_for_encoder(
            sequence
        )
    )

    with torch.no_grad():

        snapshot_embeddings = []

        # ----------------------------------------------------
        # 1. GATv2 structural representation
        # ----------------------------------------------------

        for graph_batch in batched_sequence:

            h = encoder.gat(
                graph_batch.x,
                graph_batch.edge_index
            )

            h = encoder.elu(h)

            # Mean pooling over nodes
            pooled = (
                encoder._mean_pool_per_graph(
                    h,
                    graph_batch.batch
                )
            )

            # One graph per timestep
            snapshot_embeddings.append(
                pooled.squeeze(0)
            )

        # [T, D]
        temporal_input = torch.stack(
            snapshot_embeddings,
            dim=0
        )

        # ----------------------------------------------------
        # 2. Temporal attention
        # ----------------------------------------------------

        temporal_input_batched = (
            temporal_input.unsqueeze(1)
        )

        temporal_output = (
            encoder.temporal_attention(
                temporal_input_batched
            )
        )

        # [T, D]
        temporal_output = (
            temporal_output.squeeze(1)
        )

        # Final CTGAD-X embedding
        final_embedding = (
            temporal_output[-1]
        )

    return (
        temporal_output.cpu().numpy(),
        final_embedding.cpu().numpy()
    )


# ============================================================
# EXTRACT ALL TEST TRAJECTORIES
# ============================================================

tbla_trajectories = []
tbla_final_embeddings = []

for seq in test_sequences:

    trajectory, final_embedding = (
        extract_tbla_trajectory(
            encoder,
            seq
        )
    )

    tbla_trajectories.append(
        trajectory
    )

    tbla_final_embeddings.append(
        final_embedding
    )


tbla_trajectories = np.asarray(
    tbla_trajectories
)

tbla_final_embeddings = np.asarray(
    tbla_final_embeddings
)


# ============================================================
# SANITY CHECK
# ============================================================

print("=" * 88)
print("TBLA TEMPORAL TRAJECTORY EXTRACTION")
print("=" * 88)

print(
    "Number of test sequences:",
    len(tbla_trajectories)
)

print(
    "Trajectory tensor shape:",
    tbla_trajectories.shape
)

print(
    "Expected:",
    "(47, 5, 60)"
)

print(
    "Final embeddings shape:",
    tbla_final_embeddings.shape
)

print(
    "Expected:",
    "(47, 60)"
)

print(
    "Finite trajectories:",
    np.isfinite(
        tbla_trajectories
    ).all()
)

print(
    "Finite final embeddings:",
    np.isfinite(
        tbla_final_embeddings
    ).all()
)


# ============================================================
# COMPARE WITH EXISTING CTGAD-X EMBEDDINGS
# ============================================================

if "test_embeddings" in globals():

    existing_embeddings = np.asarray(
        test_embeddings
    )

    diff = np.max(
        np.abs(
            tbla_final_embeddings -
            existing_embeddings
        )
    )

    print(
        "Max difference vs existing "
        "CTGAD-X embeddings:",
        f"{diff:.10f}"
    )

AttributeError: 'GraphSnapshotBatch' object has no attribute 'stores_as'

In [ ]:
# ============================================================
# CELL 173 — INSPECT CTGAD-X GraphSnapshotBatch
# ============================================================

seq0 = shellcode_fold["test_sequences"][0]

print("=" * 88)
print("CTGAD-X TEST SEQUENCE STRUCTURE")
print("=" * 88)

print("Sequence type:")
print(type(seq0))

print("\nSequence length:")
print(len(seq0))

for i, item in enumerate(seq0):

    print("\n" + "-" * 70)
    print(f"TIMESTEP {i}")
    print("-" * 70)

    print("Type:")
    print(type(item))

    print("\nAttributes:")
    print([
        x for x in dir(item)
        if not x.startswith("_")
    ])

    print("\n__dict__:")
    if hasattr(item, "__dict__"):
        print(item.__dict__)
    else:
        print("No __dict__")

    if i >= 0:
        break

CTGAD-X TEST SEQUENCE STRUCTURE
Sequence type:
<class 'list'>

Sequence length:
5

----------------------------------------------------------------------
TIMESTEP 0
----------------------------------------------------------------------
Type:
<class 'ctgad_graph_slicer.GraphSnapshot'>

Attributes:
['edges', 'node_features', 'node_ids', 't_end', 't_start', 'window_idx']

__dict__:
{'window_idx': 204, 't_start': 1421933497.0, 't_end': 1421933527.0, 'node_ids': ['59.166.0.2', '149.171.126.1', '59.166.0.8', '149.171.126.8', '10.40.182.1', '224.0.0.5', '59.166.0.6', '149.171.126.4', '175.45.176.1', '149.171.126.18', '59.166.0.7', '149.171.126.3', '149.171.126.6', '149.171.126.5', '59.166.0.4', '59.166.0.3', '59.166.0.5', '10.40.85.1', '175.45.176.0', '149.171.126.14', '149.171.126.9', '175.45.176.3', '149.171.126.10', '149.171.126.7', '59.166.0.0', '149.171.126.19', '149.171.126.0', '59.166.0.1', '149.171.126.13', '149.171.126.2', '59.166.0.9', '149.171.126.11', '149.171.126.17', '175.45.176

In [ ]:
# ============================================================
# CELL 174 — INSPECT EXACT TRAINED CTGAD-X ENCODER
# ============================================================

import inspect

encoder = shellcode_fold["encoder"]

print("=" * 88)
print("EXACT CTGAD-X ENCODER INSPECTION")
print("=" * 88)

print("\nEncoder class:")
print(type(encoder))

print("\nEncoder attributes/modules:")
print([
    name for name in dir(encoder)
    if not name.startswith("_")
])

print("\n" + "-" * 88)
print("NAMED MODULES")
print("-" * 88)

for name, module in encoder.named_modules():
    print(
        f"{name:40s} -> {type(module).__name__}"
    )

print("\n" + "-" * 88)
print("FORWARD SOURCE")
print("-" * 88)

try:
    print(
        inspect.getsource(
            encoder.forward
        )
    )
except Exception as e:
    print(
        "Could not retrieve forward source:",
        repr(e)
    )

EXACT CTGAD-X ENCODER INSPECTION

Encoder class:
<class 'model.encoder.CTGAD_Encoder'>

Encoder attributes/modules:
['T_destination', 'add_module', 'apply', 'bfloat16', 'buffers', 'call_super_init', 'children', 'compile', 'cpu', 'cuda', 'double', 'dump_patches', 'eval', 'extra_repr', 'float', 'forward', 'get_buffer', 'get_extra_state', 'get_parameter', 'get_submodule', 'half', 'hidden_dim', 'ipu', 'load_state_dict', 'modules', 'mtia', 'named_buffers', 'named_children', 'named_modules', 'named_parameters', 'parameters', 'register_backward_hook', 'register_buffer', 'register_forward_hook', 'register_forward_pre_hook', 'register_full_backward_hook', 'register_full_backward_pre_hook', 'register_load_state_dict_post_hook', 'register_load_state_dict_pre_hook', 'register_module', 'register_parameter', 'register_state_dict_post_hook', 'register_state_dict_pre_hook', 'requires_grad_', 'seq_len', 'set_extra_state', 'set_submodule', 'share_memory', 'state_dict', 'structural', 'temporal', 'to', 't

In [ ]:
# ============================================================
# CELL 175 — EXACT CTGAD-X TEMPORAL STATE EXTRACTION
# ============================================================

import torch
import numpy as np
from torch_geometric.data import Data, Batch

encoder = shellcode_fold["encoder"]
test_sequences = shellcode_fold["test_sequences"]

encoder.eval()

device = next(
    encoder.parameters()
).device


# ============================================================
# GRAPH SNAPSHOT → PYTORCH GEOMETRIC DATA
# ============================================================

def graphsnapshot_to_data(
    snapshot,
    scaler
):
    """
    Reconstruct the exact PyG graph representation used
    by CTGAD-X from a GraphSnapshot.
    """

    # Node ordering is fixed by snapshot.node_ids
    node_ids = snapshot.node_ids

    X = np.vstack([
        snapshot.node_features[node_id]
        for node_id in node_ids
    ]).astype(
        np.float32
    )

    # Apply the already-fitted LOFO training scaler
    X = scaler.transform(
        X
    ).astype(
        np.float32
    )

    # Edges
    edge_index = torch.tensor(
        np.asarray(
            snapshot.edges,
            dtype=np.int64
        ).T,
        dtype=torch.long
    )

    x = torch.tensor(
        X,
        dtype=torch.float32
    )

    return Data(
        x=x,
        edge_index=edge_index
    )


# ============================================================
# USE THE FROZEN LOFO SCALER
# ============================================================

scaler = shellcode_fold["scaler"]


# ============================================================
# CONVERT ONE TEST SEQUENCE
# ============================================================

def convert_raw_sequence(
    sequence
):

    data_sequence = []

    for snapshot in sequence:

        data = graphsnapshot_to_data(
            snapshot,
            scaler
        )

        data_sequence.append(
            data
        )

    return data_sequence


# ============================================================
# EXACT STRUCTURAL STATE EXTRACTION
# ============================================================

def extract_structural_trajectory(
    encoder,
    sequence
):

    data_sequence = (
        convert_raw_sequence(
            sequence
        )
    )

    with torch.no_grad():

        states = []

        for data in data_sequence:

            data = data.to(
                device
            )

            # EXACT encoder.structural(...)
            h = encoder.structural(
                data.x,
                data.edge_index
            )

            # EXACT mean pooling logic used by encoder.forward()
            batch_index = torch.zeros(
                h.size(0),
                dtype=torch.long,
                device=device
            )

            pooled = (
                encoder._mean_pool_per_graph(
                    h,
                    batch_index
                )
            )

            states.append(
                pooled.squeeze(0)
            )

        # [5, 60]
        structural_states = torch.stack(
            states,
            dim=0
        )

    return (
        structural_states.cpu().numpy()
    )


# ============================================================
# EXTRACT TEST TRAJECTORIES
# ============================================================

tbla_structural_trajectories = []

for seq in test_sequences:

    trajectory = (
        extract_structural_trajectory(
            encoder,
            seq
        )
    )

    tbla_structural_trajectories.append(
        trajectory
    )


tbla_structural_trajectories = np.asarray(
    tbla_structural_trajectories
)


# ============================================================
# SANITY CHECK
# ============================================================

print("=" * 88)
print("EXACT CTGAD-X STRUCTURAL TRAJECTORIES")
print("=" * 88)

print(
    "Trajectory shape:",
    tbla_structural_trajectories.shape
)

print(
    "Expected:",
    "(47, 5, 60)"
)

print(
    "Finite:",
    np.isfinite(
        tbla_structural_trajectories
    ).all()
)

print(
    "Minimum:",
    tbla_structural_trajectories.min()
)

print(
    "Maximum:",
    tbla_structural_trajectories.max()
)

EXACT CTGAD-X STRUCTURAL TRAJECTORIES
Trajectory shape: (47, 5, 60)
Expected: (47, 5, 60)
Finite: True
Minimum: -0.31257898
Maximum: 0.38312835


In [ ]:
# ============================================================
# CELL 176A — FIND THE EXISTING NORMAL-ONLY TRAINING OBJECTS
# ============================================================

print("=" * 88)
print("AVAILABLE NORMAL / REFERENCE OBJECTS")
print("=" * 88)

for key, value in shellcode_fold.items():

    if isinstance(value, np.ndarray):
        print(
            f"{key:30s} ndarray {value.shape}"
        )

    elif isinstance(value, list):
        print(
            f"{key:30s} list    len={len(value)}"
        )

    else:
        print(
            f"{key:30s} {type(value).__name__}"
        )

print("\n" + "-" * 88)

# Search notebook globals for likely normal-training objects
candidate_names = [
    name for name in globals()
    if any(
        token in name.lower()
        for token in [
            "normal",
            "reference",
            "ref",
            "train_embed"
        ]
    )
]

for name in sorted(candidate_names):

    try:
        obj = globals()[name]

        if isinstance(obj, np.ndarray):
            print(
                f"{name:45s} -> ndarray {obj.shape}"
            )

        elif isinstance(obj, list):
            print(
                f"{name:45s} -> list len={len(obj)}"
            )

    except Exception:
        pass

AVAILABLE NORMAL / REFERENCE OBJECTS
family                         str
train_df                       DataFrame
val_df                         DataFrame
test_df                        DataFrame
train_snapshots                list    len=178
val_snapshots                  list    len=26
test_snapshots                 list    len=51
train_sequences                list    len=174
val_sequences                  list    len=22
test_sequences                 list    len=47
train_meta                     list    len=174
val_meta                       list    len=22
test_meta                      list    len=47
scaler                         StandardScaler
encoder                        CTGAD_Encoder
train_pyg                      list    len=174
val_pyg                        list    len=22

----------------------------------------------------------------------------------------
Z_normal                                      -> ndarray (174, 60)
normal_curr                                   -

In [ ]:
# ============================================================
# CELL 176B — TBLA ONE-SHOT EVALUATION
# USING EXACT CTGAD-X NORMAL TRAINING SEQUENCES
# ============================================================

import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score

encoder = shellcode_fold["encoder"]
encoder.eval()

# ------------------------------------------------------------
# 1. TEST TRAJECTORIES
# ------------------------------------------------------------

X_test_traj = np.asarray(
    tbla_structural_trajectories,
    dtype=np.float32
)

# ------------------------------------------------------------
# 2. TEST LABELS
# ------------------------------------------------------------

y_test = np.asarray([
    int(m["label"])
    for m in shellcode_fold["test_meta"]
])

print("=" * 88)
print("TBLA ONE-SHOT EXPERIMENT")
print("=" * 88)

print(
    "Test trajectories:",
    X_test_traj.shape
)

print(
    "Test labels:",
    y_test.shape,
    "| normal:",
    int((y_test == 0).sum()),
    "| attack:",
    int((y_test == 1).sum())
)

# ------------------------------------------------------------
# 3. EXACT NORMAL TRAINING SEQUENCES
# ------------------------------------------------------------

normal_train_seqs_exact = normal_train_seqs

print(
    "\nNormal training sequences:",
    len(normal_train_seqs_exact)
)

# ------------------------------------------------------------
# 4. EXTRACT NORMAL TEMPORAL TRAJECTORIES
# ------------------------------------------------------------

normal_train_traj = []

for seq in normal_train_seqs_exact:

    traj = extract_structural_trajectory(
        encoder,
        seq
    )

    normal_train_traj.append(
        traj
    )

normal_train_traj = np.asarray(
    normal_train_traj,
    dtype=np.float32
)

print(
    "Normal trajectories:",
    normal_train_traj.shape
)

# Expected:
# (174, 5, 60)

# ------------------------------------------------------------
# 5. TEMPORAL DISPLACEMENT
# ------------------------------------------------------------

normal_disp = np.linalg.norm(
    normal_train_traj[:, 1:, :] -
    normal_train_traj[:, :-1, :],
    axis=2
)

test_disp = np.linalg.norm(
    X_test_traj[:, 1:, :] -
    X_test_traj[:, :-1, :],
    axis=2
)

# Shapes:
# normal_disp = (174, 4)
# test_disp   = (47, 4)

# ------------------------------------------------------------
# 6. TEMPORAL ACCELERATION
# ------------------------------------------------------------

normal_acc = np.abs(
    normal_disp[:, 1:] -
    normal_disp[:, :-1]
)

test_acc = np.abs(
    test_disp[:, 1:] -
    test_disp[:, :-1]
)

# ------------------------------------------------------------
# 7. ROBUST NORMAL DYNAMICS CALIBRATION
# ------------------------------------------------------------

disp_median = np.median(
    normal_disp
)

disp_mad = (
    np.median(
        np.abs(
            normal_disp -
            disp_median
        )
    )
    + 1e-8
)

acc_median = np.median(
    normal_acc
)

acc_mad = (
    np.median(
        np.abs(
            normal_acc -
            acc_median
        )
    )
    + 1e-8
)

# ------------------------------------------------------------
# 8. TEST TEMPORAL SURPRISE
# ------------------------------------------------------------

disp_z = (
    test_disp -
    disp_median
) / (
    1.4826 * disp_mad
)

acc_z = (
    test_acc -
    acc_median
) / (
    1.4826 * acc_mad
)

# Only deviations above normal
disp_surprise = np.maximum(
    disp_z,
    0
)

acc_surprise = np.maximum(
    acc_z,
    0
)

# ------------------------------------------------------------
# 9. TBLA SCORE
# ------------------------------------------------------------

tbla_score = (
    np.mean(
        disp_surprise,
        axis=1
    )
    +
    np.mean(
        acc_surprise,
        axis=1
    )
)

# ------------------------------------------------------------
# 10. EARLIEST DIVERGENCE
# ------------------------------------------------------------

earliest_divergence = np.full(
    len(test_disp),
    5.0
)

for i in range(
    len(test_disp)
):

    divergent_steps = np.where(
        disp_z[i] > 0
    )[0]

    if len(divergent_steps) > 0:

        # displacement index 0 corresponds
        # to transition t0 -> t1
        earliest_divergence[i] = (
            divergent_steps[0] + 1
        )

# ------------------------------------------------------------
# 11. EVALUATION
# ------------------------------------------------------------

tbla_roc = roc_auc_score(
    y_test,
    tbla_score
)

tbla_pr = average_precision_score(
    y_test,
    tbla_score
)

# ------------------------------------------------------------
# 12. COMPARE WITH FROZEN CTGAD-X
# ------------------------------------------------------------

ctgad_roc = 0.977083
ctgad_pr = 0.989811

print("\n" + "=" * 88)
print("RESULT")
print("=" * 88)

print(
    f"TBLA ROC-AUC : {tbla_roc:.6f}"
)

print(
    f"TBLA PR-AUC  : {tbla_pr:.6f}"
)

print("\nFrozen CTGAD-X:")

print(
    f"ROC-AUC      : {ctgad_roc:.6f}"
)

print(
    f"PR-AUC       : {ctgad_pr:.6f}"
)

print("\nTBLA improvement:")

print(
    f"ROC Δ        : "
    f"{tbla_roc - ctgad_roc:+.6f}"
)

print(
    f"PR Δ         : "
    f"{tbla_pr - ctgad_pr:+.6f}"
)

# ------------------------------------------------------------
# 13. LINEAGE STATISTICS
# ------------------------------------------------------------

print("\nTemporal lineage statistics:")

print(
    f"Path length mean      : "
    f"{test_disp.sum(axis=1).mean():.6f}"
)

print(
    f"Path length std       : "
    f"{test_disp.sum(axis=1).std():.6f}"
)

print(
    f"Max displacement mean : "
    f"{test_disp.max(axis=1).mean():.6f}"
)

print(
    f"Max acceleration mean : "
    f"{test_acc.max(axis=1).mean():.6f}"
)

print(
    f"Earliest divergence   : "
    f"{earliest_divergence.mean():.3f}"
)

print("=" * 88)

TBLA ONE-SHOT EXPERIMENT
Test trajectories: (47, 5, 60)
Test labels: (47,) | normal: 15 | attack: 32

Normal training sequences: 174
Normal trajectories: (174, 5, 60)

RESULT
TBLA ROC-AUC : 0.719792
TBLA PR-AUC  : 0.870559

Frozen CTGAD-X:
ROC-AUC      : 0.977083
PR-AUC       : 0.989811

TBLA improvement:
ROC Δ        : -0.257291
PR Δ         : -0.119252

Temporal lineage statistics:
Path length mean      : 0.966997
Path length std       : 0.249320
Max displacement mean : 0.364296
Max acceleration mean : 0.186719
Earliest divergence   : 1.915


In [ ]:
# ============================================================
# BREG CELL 1 — INSPECT GRAPH SNAPSHOT STRUCTURE
# ============================================================

import numpy as np
import pandas as pd

train_snapshots = shellcode_fold["train_snapshots"]
test_snapshots  = shellcode_fold["test_snapshots"]

print("=" * 88)
print("BREG — GRAPH SNAPSHOT INSPECTION")
print("=" * 88)

print("\nTrain snapshots:", len(train_snapshots))
print("Test snapshots :", len(test_snapshots))

s = train_snapshots[0]

print("\nSnapshot type:")
print(type(s))

print("\nSnapshot attributes:")
print(s.__dict__.keys())

print("\nWindow:")
print(
    "window_idx =",
    s.window_idx
)

print(
    "t_start    =",
    s.t_start
)

print(
    "t_end      =",
    s.t_end
)

print("\nNodes:")
print(
    "count =",
    len(s.node_ids)
)

print(
    "first 5 =",
    s.node_ids[:5]
)

print("\nEdges:")
print(
    "count =",
    len(s.edges)
)

print(
    "first 10 =",
    s.edges[:10]
)

print("\nNode feature dimensions:")

first_node = s.node_ids[0]

print(
    "node =",
    first_node
)

print(
    "feature shape =",
    np.asarray(
        s.node_features[first_node]
    ).shape
)

print(
    "feature =",
    s.node_features[first_node]
)

print("=" * 88)

BREG — GRAPH SNAPSHOT INSPECTION

Train snapshots: 178
Test snapshots : 51

Snapshot type:
<class 'ctgad_graph_slicer.GraphSnapshot'>

Snapshot attributes:
dict_keys(['window_idx', 't_start', 't_end', 'node_ids', 'edges', 'node_features'])

Window:
window_idx = 0
t_start    = 1421927377.0
t_end      = 1421927407.0

Nodes:
count = 3
first 5 = ['10.40.182.1', '224.0.0.5', '10.40.85.1']

Edges:
count = 2
first 10 = [(2, 1), (0, 1)]

Node feature dimensions:
node = 10.40.182.1
feature shape = (11,)
feature = [460.76004     0.          7.1993756   0.          0.          0.
  61.43467     0.          1.          0.         60.       ]


In [ ]:
# ============================================================
# BREG CELL 2 — INSPECT FLOW-LEVEL RELATIONAL INFORMATION
# ============================================================

import numpy as np
import pandas as pd

df_breg = shellcode_fold["train_df"]

print("=" * 88)
print("BREG — FLOW-LEVEL RELATIONSHIP INSPECTION")
print("=" * 88)

print("\nDataset shape:")
print(df_breg.shape)

print("\nColumns:")
for i, col in enumerate(df_breg.columns):
    print(
        f"{i:3d} : {col}"
    )

print("\n" + "-" * 88)
print("KEY FLOW COLUMNS")
print("-" * 88)

candidate_cols = [
    "ts_start",
    "duration",
    "src_ip",
    "dst_ip",
    "sbytes",
    "dbytes",
    "spkts",
    "dpkts",
    "sload",
    "dload",
    "sttl",
    "dttl",
    "label",
    "attack_cat"
]

for col in candidate_cols:

    if col in df_breg.columns:

        print(
            f"{col:20s} | "
            f"dtype={df_breg[col].dtype} | "
            f"non-null={df_breg[col].notna().sum()}"
        )

print("\n" + "-" * 88)
print("SAMPLE RELATIONSHIPS")
print("-" * 88)

available = [
    c for c in candidate_cols
    if c in df_breg.columns
]

print(
    df_breg[available]
    .head(10)
    .to_string(index=False)
)

print("\n" + "-" * 88)
print("UNIQUE DIRECTED RELATIONSHIPS")
print("-" * 88)

if (
    "src_ip" in df_breg.columns
    and
    "dst_ip" in df_breg.columns
):

    pair_count = (
        df_breg[
            ["src_ip", "dst_ip"]
        ]
        .drop_duplicates()
        .shape[0]
    )

    print(
        "Unique src → dst pairs:",
        pair_count
    )

    print(
        "\nTop relationships by flow count:"
    )

    pair_counts = (
        df_breg
        .groupby(
            ["src_ip", "dst_ip"]
        )
        .size()
        .sort_values(
            ascending=False
        )
        .head(20)
    )

    print(
        pair_counts.to_string()
    )

print("=" * 88)

BREG — FLOW-LEVEL RELATIONSHIP INSPECTION

Dataset shape:
(142873, 52)

Columns:
  0 : src_ip
  1 : sport
  2 : dst_ip
  3 : dsport
  4 : proto
  5 : state
  6 : duration
  7 : sbytes
  8 : dbytes
  9 : sttl
 10 : dttl
 11 : sloss
 12 : dloss
 13 : service
 14 : sload
 15 : dload
 16 : spkts
 17 : dpkts
 18 : swin
 19 : dwin
 20 : stcpb
 21 : dtcpb
 22 : smeansz
 23 : dmeansz
 24 : trans_depth
 25 : res_bdy_len
 26 : sjit
 27 : djit
 28 : ts_start
 29 : ltime
 30 : sintpkt
 31 : dintpkt
 32 : tcprtt
 33 : synack
 34 : ackdat
 35 : is_sm_ips_ports
 36 : ct_state_ttl
 37 : ct_flw_http_mthd
 38 : is_ftp_login
 39 : ct_ftp_cmd
 40 : ct_srv_src
 41 : ct_srv_dst
 42 : ct_dst_ltm
 43 : ct_src_ltm
 44 : ct_src_dport_ltm
 45 : ct_dst_sport_ltm
 46 : ct_dst_src_ltm
 47 : attack_cat
 48 : label
 49 : t_rel
 50 : window_start_idx
 51 : window_end_idx

----------------------------------------------------------------------------------------
KEY FLOW COLUMNS
------------------------------------------

In [ ]:
# ============================================================
# BREG CELL 3 — CONSTRUCT RELATIONAL EVOLUTION GRAPH
# ============================================================

import numpy as np
import pandas as pd

print("=" * 88)
print("BREG — RELATIONAL EVOLUTION GRAPH CONSTRUCTION")
print("=" * 88)

# ------------------------------------------------------------
# SOURCE DATA
# ------------------------------------------------------------

df = shellcode_fold["train_df"].copy()

# ------------------------------------------------------------
# FEATURES USED FOR RELATIONSHIP STATE
#
# IMPORTANT:
# label / attack_cat are NOT used.
# ------------------------------------------------------------

ADD_FEATURES = [
    "sbytes",
    "dbytes",
    "spkts",
    "dpkts",
    "duration"
]

MEAN_FEATURES = [
    "sload",
    "dload",
    "sttl",
    "dttl"
]

GROUP_COLS = [
    "window_start_idx",
    "src_ip",
    "dst_ip"
]

required = (
    GROUP_COLS +
    ADD_FEATURES +
    MEAN_FEATURES
)

missing = [
    c for c in required
    if c not in df.columns
]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

# ------------------------------------------------------------
# REMOVE INVALID WINDOW IDS
# ------------------------------------------------------------

df = df[
    df["window_start_idx"].notna()
].copy()

df["window_start_idx"] = (
    df["window_start_idx"]
    .astype(int)
)

# ------------------------------------------------------------
# AGGREGATE RELATIONSHIP STATE
# ------------------------------------------------------------

agg_add = (
    df
    .groupby(
        GROUP_COLS,
        sort=True
    )[ADD_FEATURES]
    .sum()
    .reset_index()
)

agg_mean = (
    df
    .groupby(
        GROUP_COLS,
        sort=True
    )[MEAN_FEATURES]
    .mean()
    .reset_index()
)

relation_state = agg_add.merge(
    agg_mean,
    on=GROUP_COLS,
    how="left"
)

# Flow count is itself a relational feature
flow_count = (
    df
    .groupby(
        GROUP_COLS,
        sort=True
    )
    .size()
    .rename("flow_count")
    .reset_index()
)

relation_state = relation_state.merge(
    flow_count,
    on=GROUP_COLS,
    how="left"
)

# ------------------------------------------------------------
# ORDER FEATURES
# ------------------------------------------------------------

RELATION_FEATURES = [
    "flow_count",
    "sbytes",
    "dbytes",
    "spkts",
    "dpkts",
    "duration",
    "sload",
    "dload",
    "sttl",
    "dttl"
]

relation_state = relation_state[
    GROUP_COLS +
    RELATION_FEATURES
].copy()

# ------------------------------------------------------------
# BASIC SANITY
# ------------------------------------------------------------

print(
    "\nRelationship-state table:",
    relation_state.shape
)

print(
    "Unique windows:",
    relation_state[
        "window_start_idx"
    ].nunique()
)

print(
    "Unique directed relationships:",
    relation_state[
        ["src_ip", "dst_ip"]
    ]
    .drop_duplicates()
    .shape[0]
)

# ------------------------------------------------------------
# TRANSITION CONSTRUCTION
# ------------------------------------------------------------

# Dictionary:
# window -> {(src,dst): feature_vector}

window_relation_maps = {}

for window_idx, group in relation_state.groupby(
    "window_start_idx",
    sort=True
):

    relation_map = {}

    for row in group.itertuples(
        index=False
    ):

        key = (
            row.src_ip,
            row.dst_ip
        )

        relation_map[key] = np.asarray(
            [
                getattr(row, f)
                for f in RELATION_FEATURES
            ],
            dtype=np.float64
        )

    window_relation_maps[
        int(window_idx)
    ] = relation_map

# ------------------------------------------------------------
# EVOLUTION EVENTS
# ------------------------------------------------------------

breg_events = []

sorted_windows = sorted(
    window_relation_maps.keys()
)

for t in sorted_windows[1:]:

    prev_map = window_relation_maps[
        t - 1
    ]

    curr_map = window_relation_maps[
        t
    ]

    prev_edges = set(
        prev_map.keys()
    )

    curr_edges = set(
        curr_map.keys()
    )

    # ----------------------------------------
    # BIRTH
    # ----------------------------------------

    for edge in (
        curr_edges - prev_edges
    ):

        breg_events.append({
            "window": t,
            "src_ip": edge[0],
            "dst_ip": edge[1],
            "event_type": "BIRTH",
            "previous_exists": 0,
            "current_exists": 1,
            "previous_state": None,
            "current_state": curr_map[edge]
        })

    # ----------------------------------------
    # DEATH
    # ----------------------------------------

    for edge in (
        prev_edges - curr_edges
    ):

        breg_events.append({
            "window": t,
            "src_ip": edge[0],
            "dst_ip": edge[1],
            "event_type": "DEATH",
            "previous_exists": 1,
            "current_exists": 0,
            "previous_state": prev_map[edge],
            "current_state": None
        })

    # ----------------------------------------
    # PERSISTENCE
    # ----------------------------------------

    for edge in (
        prev_edges & curr_edges
    ):

        prev_state = prev_map[
            edge
        ]

        curr_state = curr_map[
            edge
        ]

        breg_events.append({
            "window": t,
            "src_ip": edge[0],
            "dst_ip": edge[1],
            "event_type": "PERSIST",
            "previous_exists": 1,
            "current_exists": 1,
            "previous_state": prev_state,
            "current_state": curr_state
        })

# ------------------------------------------------------------
# EVENT TABLE
# ------------------------------------------------------------

breg_events_df = pd.DataFrame(
    breg_events
)

print("\n" + "-" * 88)
print("RELATIONAL EVOLUTION EVENTS")
print("-" * 88)

print(
    breg_events_df[
        "event_type"
    ]
    .value_counts()
)

# ------------------------------------------------------------
# UNIQUE RELATIONSHIP EVOLUTION
# ------------------------------------------------------------

print("\n" + "-" * 88)
print("UNIQUE RELATIONSHIP TRANSITIONS")
print("-" * 88)

transition_counts = (
    breg_events_df
    .groupby(
        [
            "event_type"
        ]
    )
    .size()
    .sort_values(
        ascending=False
    )
)

print(
    transition_counts
)

# ------------------------------------------------------------
# SAMPLE BIRTHS
# ------------------------------------------------------------

print("\n" + "-" * 88)
print("SAMPLE BIRTH EVENTS")
print("-" * 88)

births = breg_events_df[
    breg_events_df[
        "event_type"
    ] == "BIRTH"
].head(10)

if len(births):

    print(
        births[
            [
                "window",
                "src_ip",
                "dst_ip",
                "event_type"
            ]
        ].to_string(
            index=False
        )
    )

# ------------------------------------------------------------
# SAMPLE DEATHS
# ------------------------------------------------------------

print("\n" + "-" * 88)
print("SAMPLE DEATH EVENTS")
print("-" * 88)

deaths = breg_events_df[
    breg_events_df[
        "event_type"
    ] == "DEATH"
].head(10)

if len(deaths):

    print(
        deaths[
            [
                "window",
                "src_ip",
                "dst_ip",
                "event_type"
            ]
        ].to_string(
            index=False
        )
    )

# ------------------------------------------------------------
# SAMPLE PERSISTENT RELATIONSHIP CHANGES
# ------------------------------------------------------------

print("\n" + "-" * 88)
print("SAMPLE PERSISTENT RELATIONSHIPS")
print("-" * 88)

persistent = breg_events_df[
    breg_events_df[
        "event_type"
    ] == "PERSIST"
].head(5)

for _, row in persistent.iterrows():

    print(
        f"\nWindow {row['window']}: "
        f"{row['src_ip']} -> {row['dst_ip']}"
    )

    print(
        "Previous:",
        np.round(
            row["previous_state"],
            4
        )
    )

    print(
        "Current :",
        np.round(
            row["current_state"],
            4
        )
    )

# ------------------------------------------------------------
# SAVE OBJECTS FOR NEXT STEP
# ------------------------------------------------------------

breg_relation_state = relation_state
breg_window_relation_maps = window_relation_maps
breg_events = breg_events_df

print("\n" + "=" * 88)
print("BREG GRAPH CONSTRUCTION COMPLETE")
print("=" * 88)

BREG — RELATIONAL EVOLUTION GRAPH CONSTRUCTION

Relationship-state table: (22325, 13)
Unique windows: 178
Unique directed relationships: 260

----------------------------------------------------------------------------------------
RELATIONAL EVOLUTION EVENTS
----------------------------------------------------------------------------------------
event_type
PERSIST    19511
BIRTH       2811
DEATH       2689
Name: count, dtype: int64

----------------------------------------------------------------------------------------
UNIQUE RELATIONSHIP TRANSITIONS
----------------------------------------------------------------------------------------
event_type
PERSIST    19511
BIRTH       2811
DEATH       2689
dtype: int64

----------------------------------------------------------------------------------------
SAMPLE BIRTH EVENTS
----------------------------------------------------------------------------------------
 window       src_ip         dst_ip event_type
      1   59.166.0.6  149.171.12

In [ ]:
# ============================================================
# BREG CELL 4 — RELATIONAL TRANSITION DYNAMICS
# ============================================================

import numpy as np
import pandas as pd

print("=" * 88)
print("BREG — RELATIONAL TRANSITION DYNAMICS")
print("=" * 88)

# ------------------------------------------------------------
# FEATURES
# ------------------------------------------------------------

REL_FEATURES = [
    "flow_count",
    "sbytes",
    "dbytes",
    "spkts",
    "dpkts",
    "duration",
    "sload",
    "dload",
    "sttl",
    "dttl"
]

# ------------------------------------------------------------
# BUILD PERSISTENT-EDGE TRANSITION VECTORS
# ------------------------------------------------------------

transition_records = []

for _, row in breg_events[
    breg_events["event_type"] == "PERSIST"
].iterrows():

    prev = np.asarray(
        row["previous_state"],
        dtype=np.float64
    )

    curr = np.asarray(
        row["current_state"],
        dtype=np.float64
    )

    # --------------------------------------------------------
    # Signed log-ratio:
    #
    # log((current + 1)/(previous + 1))
    #
    # Positive  -> increase
    # Negative  -> decrease
    # Near zero -> stable
    # --------------------------------------------------------

    log_ratio = np.log1p(
        curr
    ) - np.log1p(
        prev
    )

    # --------------------------------------------------------
    # Relative L1 magnitude
    # --------------------------------------------------------

    relative_change = (
        np.abs(
            curr - prev
        )
        /
        (
            np.abs(prev) + 1.0
        )
    )

    # --------------------------------------------------------
    # Transition magnitude
    # --------------------------------------------------------

    transition_magnitude = np.linalg.norm(
        log_ratio
    )

    transition_records.append({
        "window": int(row["window"]),
        "src_ip": row["src_ip"],
        "dst_ip": row["dst_ip"],
        "transition_magnitude": transition_magnitude,
        "mean_abs_log_change": np.mean(
            np.abs(log_ratio)
        ),
        "max_abs_log_change": np.max(
            np.abs(log_ratio)
        ),
        "flow_count_change": log_ratio[0],
        "sbytes_change": log_ratio[1],
        "dbytes_change": log_ratio[2],
        "spkts_change": log_ratio[3],
        "dpkts_change": log_ratio[4],
        "duration_change": log_ratio[5],
        "sload_change": log_ratio[6],
        "dload_change": log_ratio[7],
        "sttl_change": log_ratio[8],
        "dttl_change": log_ratio[9]
    })

transition_df = pd.DataFrame(
    transition_records
)

print(
    "\nPersistent transitions:",
    len(transition_df)
)

# ------------------------------------------------------------
# GLOBAL DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "-" * 88)
print("TRANSITION MAGNITUDE")
print("-" * 88)

print(
    transition_df[
        "transition_magnitude"
    ].describe(
        percentiles=[
            .50,
            .75,
            .90,
            .95,
            .99
        ]
    ).to_string()
)

# ------------------------------------------------------------
# TOP TRANSITIONS
# ------------------------------------------------------------

print("\n" + "-" * 88)
print("LARGEST RELATIONAL CHANGES")
print("-" * 88)

top = (
    transition_df
    .sort_values(
        "transition_magnitude",
        ascending=False
    )
    .head(20)
)

print(
    top[
        [
            "window",
            "src_ip",
            "dst_ip",
            "transition_magnitude",
            "flow_count_change",
            "sbytes_change",
            "dbytes_change",
            "spkts_change",
            "dpkts_change"
        ]
    ].to_string(
        index=False
    )
)

# ------------------------------------------------------------
# WINDOW-LEVEL RELATIONAL DYNAMICS
# ------------------------------------------------------------

window_transition_stats = (
    transition_df
    .groupby("window")
    .agg(
        persistent_relationships=(
            "transition_magnitude",
            "count"
        ),
        mean_transition=(
            "transition_magnitude",
            "mean"
        ),
        max_transition=(
            "transition_magnitude",
            "max"
        ),
        transition_std=(
            "transition_magnitude",
            "std"
        )
    )
    .fillna(0)
)

# ------------------------------------------------------------
# ATTACK STATUS FROM ORIGINAL DATA
#
# ONLY USED FOR DESCRIPTIVE ANALYSIS.
# NEVER USED AS A BREG FEATURE.
# ------------------------------------------------------------

window_attack_status = (
    df.groupby(
        "window_start_idx"
    )["label"]
    .max()
    .rename("contains_attack")
)

window_transition_stats = (
    window_transition_stats
    .join(
        window_attack_status,
        how="left"
    )
)

window_transition_stats[
    "contains_attack"
] = (
    window_transition_stats[
        "contains_attack"
    ]
    .fillna(0)
    .astype(int)
)

print("\n" + "-" * 88)
print("WINDOW-LEVEL DYNAMICS")
print("-" * 88)

for label, group in (
    window_transition_stats
    .groupby("contains_attack")
):

    name = (
        "ATTACK-CONTAINING"
        if label == 1
        else "NORMAL"
    )

    print(
        f"\n{name}"
    )

    print(
        group[
            [
                "persistent_relationships",
                "mean_transition",
                "max_transition",
                "transition_std"
            ]
        ]
        .describe()
        .to_string()
    )

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

breg_transition_df = transition_df
breg_window_transition_stats = (
    window_transition_stats
)

print("\n" + "=" * 88)
print("CELL COMPLETE")
print("=" * 88)

BREG — RELATIONAL TRANSITION DYNAMICS

Persistent transitions: 19511

----------------------------------------------------------------------------------------
TRANSITION MAGNITUDE
----------------------------------------------------------------------------------------
count    19511.000000
mean         3.946032
std          2.915145
min          0.000000
50%          3.155025
75%          4.784478
90%          7.316148
95%          9.325324
99%         16.490502
max         26.427778

----------------------------------------------------------------------------------------
LARGEST RELATIONAL CHANGES
----------------------------------------------------------------------------------------
 window       src_ip         dst_ip  transition_magnitude  flow_count_change  sbytes_change  dbytes_change  spkts_change  dpkts_change
     62 175.45.176.2 149.171.126.17             26.427778           0.405465       2.634456      15.186529      5.026071      7.985484
     22 175.45.176.2 149.171.126.15

In [ ]:
# ============================================================
# BREG — CELL 5
# TRANSITION-INTERACTION GRAPH
# ============================================================

from collections import defaultdict
import numpy as np
import pandas as pd

print("=" * 88)
print("BREG — TRANSITION-INTERACTION GRAPH")
print("=" * 88)

# ------------------------------------------------------------
# 1. Use transition table from Cell 4
# ------------------------------------------------------------

tdf = transition_df.copy()

print(f"\nTransition table shape: {tdf.shape}")
print(f"Columns: {list(tdf.columns)}")

# Cell 4 already contains only persistent transitions,
# so DO NOT filter on a nonexistent event_type column.

tdf = tdf.reset_index(drop=True)
tdf["transition_id"] = np.arange(len(tdf))

print(f"Transition nodes: {len(tdf):,}")

# ------------------------------------------------------------
# 2. Group transitions by window
# ------------------------------------------------------------

window_transitions = defaultdict(list)

for row in tdf.itertuples(index=False):

    window_transitions[int(row.window)].append(
        int(row.transition_id)
    )

# ------------------------------------------------------------
# 3. Build transition-interaction edges
# ------------------------------------------------------------

interaction_rows = []

for window, tids in window_transitions.items():

    if len(tids) < 2:
        continue

    # -----------------------------------------------
    # Group by source
    # -----------------------------------------------

    by_src = defaultdict(list)

    # Group by destination
    by_dst = defaultdict(list)

    for tid in tids:

        row = tdf.iloc[tid]

        by_src[row.src_ip].append(tid)
        by_dst[row.dst_ip].append(tid)

    # -----------------------------------------------
    # SHARED SOURCE
    # A -> B
    # A -> C
    # -----------------------------------------------

    for src, group in by_src.items():

        if len(group) > 1:

            for i in range(len(group)):

                for j in range(i + 1, len(group)):

                    a = group[i]
                    b = group[j]

                    interaction_rows.append({
                        "window": window,
                        "a": min(a, b),
                        "b": max(a, b),
                        "interaction": "SHARED_SOURCE"
                    })

    # -----------------------------------------------
    # SHARED DESTINATION
    # A -> C
    # B -> C
    # -----------------------------------------------

    for dst, group in by_dst.items():

        if len(group) > 1:

            for i in range(len(group)):

                for j in range(i + 1, len(group)):

                    a = group[i]
                    b = group[j]

                    interaction_rows.append({
                        "window": window,
                        "a": min(a, b),
                        "b": max(a, b),
                        "interaction": "SHARED_DESTINATION"
                    })

    # -----------------------------------------------
    # CHAIN
    # A -> B
    # B -> C
    # -----------------------------------------------

    for tid_a in tids:

        a = tdf.iloc[tid_a]

        for tid_b in tids:

            if tid_a == tid_b:
                continue

            b = tdf.iloc[tid_b]

            if (
                a.dst_ip == b.src_ip
                and a.src_ip != b.dst_ip
            ):

                interaction_rows.append({
                    "window": window,
                    "a": tid_a,
                    "b": tid_b,
                    "interaction": "CHAIN"
                })

    # -----------------------------------------------
    # REVERSE
    # A -> B
    # B -> A
    # -----------------------------------------------

    pair_lookup = {}

    for tid in tids:

        r = tdf.iloc[tid]

        pair_lookup[
            (r.src_ip, r.dst_ip)
        ] = tid

    for tid in tids:

        r = tdf.iloc[tid]

        reverse_tid = pair_lookup.get(
            (r.dst_ip, r.src_ip)
        )

        if (
            reverse_tid is not None
            and reverse_tid != tid
        ):

            interaction_rows.append({
                "window": window,
                "a": min(tid, reverse_tid),
                "b": max(tid, reverse_tid),
                "interaction": "REVERSE"
            })

# ------------------------------------------------------------
# 4. Create interaction dataframe
# ------------------------------------------------------------

if interaction_rows:

    interaction_df = pd.DataFrame(
        interaction_rows
    )

    interaction_df = (
        interaction_df
        .drop_duplicates(
            subset=[
                "window",
                "a",
                "b",
                "interaction"
            ]
        )
        .reset_index(drop=True)
    )

else:

    interaction_df = pd.DataFrame(
        columns=[
            "window",
            "a",
            "b",
            "interaction"
        ]
    )

print()
print("-" * 88)
print("INTERACTION GRAPH")
print("-" * 88)

print(
    f"Interaction edges: {len(interaction_df):,}"
)

if len(interaction_df):

    print("\nInteraction types:")
    print(
        interaction_df["interaction"]
        .value_counts()
        .to_string()
    )

# ------------------------------------------------------------
# 5. Window-level graph statistics
# ------------------------------------------------------------

window_stats = []

for window, tids in window_transitions.items():

    n_nodes = len(tids)

    if len(interaction_df):

        w_edges = interaction_df[
            interaction_df["window"] == window
        ]

    else:

        w_edges = pd.DataFrame()

    n_edges = len(w_edges)

    if n_nodes > 1:

        possible = n_nodes * (n_nodes - 1) / 2

        density = n_edges / possible

    else:

        density = 0.0

    # -----------------------------------------------
    # Degree statistics
    # -----------------------------------------------

    degree = defaultdict(int)

    for row in w_edges.itertuples(index=False):

        degree[row.a] += 1
        degree[row.b] += 1

    if degree:

        degree_values = np.array(
            list(degree.values()),
            dtype=float
        )

        mean_degree = degree_values.mean()
        max_degree = degree_values.max()
        degree_std = degree_values.std()

    else:

        mean_degree = 0.0
        max_degree = 0.0
        degree_std = 0.0

    window_stats.append({

        "window": window,

        "transition_nodes": n_nodes,

        "interaction_edges": n_edges,

        "interaction_density": density,

        "mean_transition_degree": mean_degree,

        "max_transition_degree": max_degree,

        "transition_degree_std": degree_std
    })

transition_graph_stats = pd.DataFrame(
    window_stats
)

# ------------------------------------------------------------
# 6. Attack status — descriptive analysis ONLY
# ------------------------------------------------------------

window_attack_status = (
    df.groupby("window_start_idx")["label"]
      .max()
      .rename("contains_attack")
)

transition_graph_stats = transition_graph_stats.merge(
    window_attack_status,
    left_on="window",
    right_index=True,
    how="left"
)

transition_graph_stats["contains_attack"] = (
    transition_graph_stats["contains_attack"]
    .fillna(0)
    .astype(int)
)

# ------------------------------------------------------------
# 7. Compare normal vs attack-containing windows
# ------------------------------------------------------------

print()
print("-" * 88)
print("WINDOW-LEVEL TRANSITION-GRAPH STATISTICS")
print("-" * 88)

statistics_columns = [
    "transition_nodes",
    "interaction_edges",
    "interaction_density",
    "mean_transition_degree",
    "max_transition_degree",
    "transition_degree_std"
]

for label, name in [
    (0, "NORMAL"),
    (1, "ATTACK-CONTAINING")
]:

    subset = transition_graph_stats[
        transition_graph_stats["contains_attack"] == label
    ]

    print()
    print(name)

    if len(subset):

        print(
            subset[statistics_columns]
            .describe()
            .round(4)
            .to_string()
        )

# ------------------------------------------------------------
# 8. Interaction composition
# ------------------------------------------------------------

print()
print("-" * 88)
print("INTERACTION COMPOSITION")
print("-" * 88)

if len(interaction_df):

    interaction_with_label = interaction_df.merge(
        transition_graph_stats[
            ["window", "contains_attack"]
        ].drop_duplicates(),
        on="window",
        how="left"
    )

    composition = pd.crosstab(
        interaction_with_label["contains_attack"],
        interaction_with_label["interaction"],
        normalize="index"
    ) * 100

    print(
        composition
        .round(2)
        .to_string()
    )

print()
print("=" * 88)
print("CELL COMPLETE")
print("=" * 88)

BREG — TRANSITION-INTERACTION GRAPH

Transition table shape: (19511, 16)
Columns: ['window', 'src_ip', 'dst_ip', 'transition_magnitude', 'mean_abs_log_change', 'max_abs_log_change', 'flow_count_change', 'sbytes_change', 'dbytes_change', 'spkts_change', 'dpkts_change', 'duration_change', 'sload_change', 'dload_change', 'sttl_change', 'dttl_change']
Transition nodes: 19,511

----------------------------------------------------------------------------------------
INTERACTION GRAPH
----------------------------------------------------------------------------------------
Interaction edges: 157,901

Interaction types:
interaction
SHARED_SOURCE         79578
SHARED_DESTINATION    77823
CHAIN                   420
REVERSE                  80

----------------------------------------------------------------------------------------
WINDOW-LEVEL TRANSITION-GRAPH STATISTICS
----------------------------------------------------------------------------------------

NORMAL

ATTACK-CONTAINING
       tra

In [ ]:
# ============================================================
# BREG — CELL 6
# NORMAL-ONLY RELATIONAL EVOLUTION GRAPH
# ============================================================

from collections import defaultdict
import numpy as np
import pandas as pd

print("=" * 88)
print("BREG — NORMAL-ONLY RELATIONAL EVOLUTION GRAPH")
print("=" * 88)

# ------------------------------------------------------------
# 1. Extract NORMAL traffic from the Shellcode LOFO training set
# ------------------------------------------------------------

train_df_breg_normal = shellcode_fold["train_df"].copy()

train_df_breg_normal = train_df_breg_normal[
    train_df_breg_normal["label"] == 0
].copy()

print("\nNormal training flows:")
print(f"Shape: {train_df_breg_normal.shape}")

print(
    "Unique windows:",
    train_df_breg_normal["window_start_idx"].nunique()
)

print(
    "Window range:",
    train_df_breg_normal["window_start_idx"].min(),
    "->",
    train_df_breg_normal["window_start_idx"].max()
)

# ------------------------------------------------------------
# 2. Define relational state
# ------------------------------------------------------------

REL_FEATURES = [
    "flow_count",
    "sbytes",
    "dbytes",
    "spkts",
    "dpkts",
    "duration",
    "sload",
    "dload",
    "sttl",
    "dttl"
]

# These are derived from original flow-level fields.
# No label or attack category is used.

agg_dict = {
    "sbytes": "sum",
    "dbytes": "sum",
    "spkts": "sum",
    "dpkts": "sum",
    "duration": "sum",
    "sload": "mean",
    "dload": "mean",
    "sttl": "mean",
    "dttl": "mean"
}

# ------------------------------------------------------------
# 3. Construct relationship state for every normal window
# ------------------------------------------------------------

normal_relation_state = (
    train_df_breg_normal
    .groupby(
        [
            "window_start_idx",
            "src_ip",
            "dst_ip"
        ],
        dropna=False
    )
    .agg(agg_dict)
    .reset_index()
)

# Add flow count
flow_counts = (
    train_df_breg_normal
    .groupby(
        [
            "window_start_idx",
            "src_ip",
            "dst_ip"
        ],
        dropna=False
    )
    .size()
    .rename("flow_count")
    .reset_index()
)

normal_relation_state = normal_relation_state.drop(
    columns=["flow_count"],
    errors="ignore"
)

normal_relation_state = normal_relation_state.merge(
    flow_counts,
    on=[
        "window_start_idx",
        "src_ip",
        "dst_ip"
    ],
    how="left"
)

# ------------------------------------------------------------
# 4. Reorder columns
# ------------------------------------------------------------

normal_relation_state = normal_relation_state[
    [
        "window_start_idx",
        "src_ip",
        "dst_ip"
    ] + REL_FEATURES
]

# ------------------------------------------------------------
# 5. Build window -> relationship map
# ------------------------------------------------------------

normal_window_relation_maps = {}

for window, group in normal_relation_state.groupby(
    "window_start_idx"
):

    relation_map = {}

    for row in group.itertuples(index=False):

        key = (
            row.src_ip,
            row.dst_ip
        )

        relation_map[key] = np.array(
            [
                getattr(row, f)
                for f in REL_FEATURES
            ],
            dtype=np.float64
        )

    normal_window_relation_maps[int(window)] = relation_map

# ------------------------------------------------------------
# 6. Construct persistent relational transitions
# ------------------------------------------------------------

normal_transition_rows = []

sorted_windows = sorted(
    normal_window_relation_maps.keys()
)

for prev_window, curr_window in zip(
    sorted_windows[:-1],
    sorted_windows[1:]
):

    prev_map = normal_window_relation_maps[
        prev_window
    ]

    curr_map = normal_window_relation_maps[
        curr_window
    ]

    # Relationships existing in both windows
    persistent = (
        set(prev_map.keys())
        &
        set(curr_map.keys())
    )

    for src_ip, dst_ip in persistent:

        prev = prev_map[
            (src_ip, dst_ip)
        ]

        curr = curr_map[
            (src_ip, dst_ip)
        ]

        # Signed log-ratio
        prev_log = np.log1p(
            np.maximum(prev, 0)
        )

        curr_log = np.log1p(
            np.maximum(curr, 0)
        )

        delta = curr_log - prev_log

        magnitude = np.linalg.norm(
            delta
        )

        normal_transition_rows.append({

            "window": curr_window,

            "src_ip": src_ip,

            "dst_ip": dst_ip,

            "transition_magnitude": magnitude,

            "flow_count_change": delta[0],

            "sbytes_change": delta[1],

            "dbytes_change": delta[2],

            "spkts_change": delta[3],

            "dpkts_change": delta[4],

            "duration_change": delta[5],

            "sload_change": delta[6],

            "dload_change": delta[7],

            "sttl_change": delta[8],

            "dttl_change": delta[9]
        })

normal_transition_df = pd.DataFrame(
    normal_transition_rows
)

print()
print("-" * 88)
print("NORMAL RELATIONAL TRANSITIONS")
print("-" * 88)

print(
    "Persistent transitions:",
    len(normal_transition_df)
)

if len(normal_transition_df):

    print(
        "\nTransition magnitude:"
    )

    print(
        normal_transition_df[
            "transition_magnitude"
        ]
        .describe()
        .round(4)
        .to_string()
    )

# ------------------------------------------------------------
# 7. Construct transition-interaction graph
# ------------------------------------------------------------

if len(normal_transition_df):

    normal_transition_df = (
        normal_transition_df
        .reset_index(drop=True)
    )

    normal_transition_df[
        "transition_id"
    ] = np.arange(
        len(normal_transition_df)
    )

    window_transitions = defaultdict(list)

    for row in normal_transition_df.itertuples(
        index=False
    ):

        window_transitions[
            int(row.window)
        ].append(
            int(row.transition_id)
        )

    interaction_rows = []

    for window, tids in window_transitions.items():

        if len(tids) < 2:
            continue

        by_src = defaultdict(list)
        by_dst = defaultdict(list)

        for tid in tids:

            row = normal_transition_df.iloc[tid]

            by_src[row.src_ip].append(tid)
            by_dst[row.dst_ip].append(tid)

        # Shared source
        for group in by_src.values():

            if len(group) > 1:

                for i in range(len(group)):

                    for j in range(i + 1, len(group)):

                        a = group[i]
                        b = group[j]

                        interaction_rows.append({

                            "window": window,
                            "a": min(a, b),
                            "b": max(a, b),
                            "interaction":
                                "SHARED_SOURCE"
                        })

        # Shared destination
        for group in by_dst.values():

            if len(group) > 1:

                for i in range(len(group)):

                    for j in range(i + 1, len(group)):

                        a = group[i]
                        b = group[j]

                        interaction_rows.append({

                            "window": window,
                            "a": min(a, b),
                            "b": max(a, b),
                            "interaction":
                                "SHARED_DESTINATION"
                        })

        # Chain
        for tid_a in tids:

            a = normal_transition_df.iloc[
                tid_a
            ]

            for tid_b in tids:

                if tid_a == tid_b:
                    continue

                b = normal_transition_df.iloc[
                    tid_b
                ]

                if (
                    a.dst_ip == b.src_ip
                    and
                    a.src_ip != b.dst_ip
                ):

                    interaction_rows.append({

                        "window": window,
                        "a": tid_a,
                        "b": tid_b,
                        "interaction":
                            "CHAIN"
                    })

        # Reverse
        pair_lookup = {}

        for tid in tids:

            r = normal_transition_df.iloc[
                tid
            ]

            pair_lookup[
                (r.src_ip, r.dst_ip)
            ] = tid

        for tid in tids:

            r = normal_transition_df.iloc[
                tid
            ]

            reverse_tid = pair_lookup.get(
                (r.dst_ip, r.src_ip)
            )

            if reverse_tid is not None:

                if reverse_tid != tid:

                    interaction_rows.append({

                        "window": window,
                        "a": min(
                            tid,
                            reverse_tid
                        ),
                        "b": max(
                            tid,
                            reverse_tid
                        ),
                        "interaction":
                            "REVERSE"
                    })

    normal_interaction_df = pd.DataFrame(
        interaction_rows
    )

    if len(normal_interaction_df):

        normal_interaction_df = (
            normal_interaction_df
            .drop_duplicates(
                subset=[
                    "window",
                    "a",
                    "b",
                    "interaction"
                ]
            )
            .reset_index(drop=True)
        )

else:

    normal_interaction_df = pd.DataFrame()

# ------------------------------------------------------------
# 8. Final diagnostics
# ------------------------------------------------------------

print()
print("-" * 88)
print("NORMAL TRANSITION-INTERACTION GRAPH")
print("-" * 88)

print(
    "Transition nodes:",
    len(normal_transition_df)
)

print(
    "Interaction edges:",
    len(normal_interaction_df)
)

if len(normal_interaction_df):

    print(
        "\nInteraction types:"
    )

    print(
        normal_interaction_df[
            "interaction"
        ]
        .value_counts()
        .to_string()
    )

print()
print("=" * 88)
print("CELL COMPLETE")
print("=" * 88)

BREG — NORMAL-ONLY RELATIONAL EVOLUTION GRAPH

Normal training flows:
Shape: (125126, 52)
Unique windows: 178
Window range: 0 -> 177

----------------------------------------------------------------------------------------
NORMAL RELATIONAL TRANSITIONS
----------------------------------------------------------------------------------------
Persistent transitions: 17546

Transition magnitude:
count    17546.0000
mean         3.5318
std          2.1081
min          0.0589
25%          2.0688
50%          3.0324
75%          4.4121
max         23.4983

----------------------------------------------------------------------------------------
NORMAL TRANSITION-INTERACTION GRAPH
----------------------------------------------------------------------------------------
Transition nodes: 17546
Interaction edges: 154021

Interaction types:
interaction
SHARED_DESTINATION    76898
SHARED_SOURCE         76895
CHAIN                   216
REVERSE                  12

CELL COMPLETE


In [ ]:
# ============================================================
# BREG — CELL 7
# HELD-OUT TEST RELATIONAL TRANSITION GRAPH
# ============================================================

print("=" * 88)
print("BREG — HELD-OUT TEST RELATIONAL TRANSITION GRAPH")
print("=" * 88)

# ------------------------------------------------------------
# 1. Get held-out Shellcode test traffic
# ------------------------------------------------------------

test_df_breg = shellcode_fold["test_df"].copy()

print("\nTest flows:")
print("Shape:", test_df_breg.shape)

print(
    "Window range:",
    test_df_breg["window_start_idx"].min(),
    "->",
    test_df_breg["window_start_idx"].max()
)

print(
    "Unique windows:",
    test_df_breg["window_start_idx"].nunique()
)

# ------------------------------------------------------------
# 2. Build relational state
# ------------------------------------------------------------

test_relation_state = (
    test_df_breg
    .groupby(
        [
            "window_start_idx",
            "src_ip",
            "dst_ip"
        ],
        dropna=False
    )
    .agg({
        "sbytes": "sum",
        "dbytes": "sum",
        "spkts": "sum",
        "dpkts": "sum",
        "duration": "sum",
        "sload": "mean",
        "dload": "mean",
        "sttl": "mean",
        "dttl": "mean"
    })
    .reset_index()
)

test_flow_counts = (
    test_df_breg
    .groupby(
        [
            "window_start_idx",
            "src_ip",
            "dst_ip"
        ],
        dropna=False
    )
    .size()
    .rename("flow_count")
    .reset_index()
)

test_relation_state = test_relation_state.merge(
    test_flow_counts,
    on=[
        "window_start_idx",
        "src_ip",
        "dst_ip"
    ],
    how="left"
)

test_relation_state = test_relation_state[
    [
        "window_start_idx",
        "src_ip",
        "dst_ip"
    ] + REL_FEATURES
]

# ------------------------------------------------------------
# 3. Window → relationship map
# ------------------------------------------------------------

test_window_relation_maps = {}

for window, group in test_relation_state.groupby(
    "window_start_idx"
):

    relation_map = {}

    for row in group.itertuples(index=False):

        relation_map[
            (row.src_ip, row.dst_ip)
        ] = np.array(
            [
                getattr(row, f)
                for f in REL_FEATURES
            ],
            dtype=np.float64
        )

    test_window_relation_maps[int(window)] = relation_map

# ------------------------------------------------------------
# 4. Persistent relational transitions
# ------------------------------------------------------------

test_transition_rows = []

sorted_windows = sorted(
    test_window_relation_maps.keys()
)

for prev_window, curr_window in zip(
    sorted_windows[:-1],
    sorted_windows[1:]
):

    prev_map = test_window_relation_maps[
        prev_window
    ]

    curr_map = test_window_relation_maps[
        curr_window
    ]

    persistent = (
        set(prev_map.keys())
        &
        set(curr_map.keys())
    )

    for src_ip, dst_ip in persistent:

        prev = prev_map[
            (src_ip, dst_ip)
        ]

        curr = curr_map[
            (src_ip, dst_ip)
        ]

        prev_log = np.log1p(
            np.maximum(prev, 0)
        )

        curr_log = np.log1p(
            np.maximum(curr, 0)
        )

        delta = curr_log - prev_log

        magnitude = np.linalg.norm(
            delta
        )

        test_transition_rows.append({

            "window": curr_window,

            "src_ip": src_ip,

            "dst_ip": dst_ip,

            "transition_magnitude": magnitude,

            "flow_count_change": delta[0],

            "sbytes_change": delta[1],

            "dbytes_change": delta[2],

            "spkts_change": delta[3],

            "dpkts_change": delta[4],

            "duration_change": delta[5],

            "sload_change": delta[6],

            "dload_change": delta[7],

            "sttl_change": delta[8],

            "dttl_change": delta[9]
        })

test_transition_df = pd.DataFrame(
    test_transition_rows
)

print()
print("-" * 88)
print("TEST RELATIONAL TRANSITIONS")
print("-" * 88)

print(
    "Persistent transitions:",
    len(test_transition_df)
)

if len(test_transition_df):

    print(
        test_transition_df[
            "transition_magnitude"
        ]
        .describe()
        .round(4)
        .to_string()
    )

# ------------------------------------------------------------
# 5. Build transition-interaction graph
# ------------------------------------------------------------

if len(test_transition_df):

    test_transition_df = (
        test_transition_df
        .reset_index(drop=True)
    )

    test_transition_df[
        "transition_id"
    ] = np.arange(
        len(test_transition_df)
    )

    test_window_transitions = defaultdict(list)

    for row in test_transition_df.itertuples(
        index=False
    ):

        test_window_transitions[
            int(row.window)
        ].append(
            int(row.transition_id)
        )

    test_interaction_rows = []

    for window, tids in test_window_transitions.items():

        if len(tids) < 2:
            continue

        by_src = defaultdict(list)
        by_dst = defaultdict(list)

        for tid in tids:

            row = test_transition_df.iloc[tid]

            by_src[row.src_ip].append(tid)
            by_dst[row.dst_ip].append(tid)

        # Shared source
        for group in by_src.values():

            for i in range(len(group)):

                for j in range(i + 1, len(group)):

                    a = group[i]
                    b = group[j]

                    test_interaction_rows.append({
                        "window": window,
                        "a": min(a, b),
                        "b": max(a, b),
                        "interaction": "SHARED_SOURCE"
                    })

        # Shared destination
        for group in by_dst.values():

            for i in range(len(group)):

                for j in range(i + 1, len(group)):

                    a = group[i]
                    b = group[j]

                    test_interaction_rows.append({
                        "window": window,
                        "a": min(a, b),
                        "b": max(a, b),
                        "interaction": "SHARED_DESTINATION"
                    })

        # Chain
        for tid_a in tids:

            a = test_transition_df.iloc[tid_a]

            for tid_b in tids:

                if tid_a == tid_b:
                    continue

                b = test_transition_df.iloc[tid_b]

                if (
                    a.dst_ip == b.src_ip
                    and
                    a.src_ip != b.dst_ip
                ):

                    test_interaction_rows.append({
                        "window": window,
                        "a": tid_a,
                        "b": tid_b,
                        "interaction": "CHAIN"
                    })

        # Reverse
        pair_lookup = {}

        for tid in tids:

            r = test_transition_df.iloc[tid]

            pair_lookup[
                (r.src_ip, r.dst_ip)
            ] = tid

        for tid in tids:

            r = test_transition_df.iloc[tid]

            reverse_tid = pair_lookup.get(
                (r.dst_ip, r.src_ip)
            )

            if (
                reverse_tid is not None
                and reverse_tid != tid
            ):

                test_interaction_rows.append({
                    "window": window,
                    "a": min(tid, reverse_tid),
                    "b": max(tid, reverse_tid),
                    "interaction": "REVERSE"
                })

    test_interaction_df = pd.DataFrame(
        test_interaction_rows
    )

    if len(test_interaction_df):

        test_interaction_df = (
            test_interaction_df
            .drop_duplicates(
                subset=[
                    "window",
                    "a",
                    "b",
                    "interaction"
                ]
            )
            .reset_index(drop=True)
        )

else:

    test_interaction_df = pd.DataFrame()

# ------------------------------------------------------------
# 6. Window-level statistics
# ------------------------------------------------------------

test_graph_stats = []

for window, tids in test_window_transitions.items():

    n_nodes = len(tids)

    w_edges = (
        test_interaction_df[
            test_interaction_df["window"] == window
        ]
        if len(test_interaction_df)
        else pd.DataFrame()
    )

    n_edges = len(w_edges)

    possible = (
        n_nodes * (n_nodes - 1) / 2
        if n_nodes > 1
        else 0
    )

    density = (
        n_edges / possible
        if possible > 0
        else 0
    )

    degree = defaultdict(int)

    for row in w_edges.itertuples(index=False):

        degree[row.a] += 1
        degree[row.b] += 1

    if degree:

        values = np.array(
            list(degree.values()),
            dtype=float
        )

        mean_degree = values.mean()
        max_degree = values.max()
        degree_std = values.std()

    else:

        mean_degree = 0
        max_degree = 0
        degree_std = 0

    test_graph_stats.append({

        "window": window,

        "transition_nodes": n_nodes,

        "interaction_edges": n_edges,

        "interaction_density": density,

        "mean_transition_degree": mean_degree,

        "max_transition_degree": max_degree,

        "transition_degree_std": degree_std
    })

test_graph_stats = pd.DataFrame(
    test_graph_stats
)

# ------------------------------------------------------------
# 7. Add labels ONLY for evaluation
# ------------------------------------------------------------

test_window_labels = (
    test_df_breg
    .groupby("window_start_idx")["label"]
    .max()
    .rename("contains_attack")
)

test_graph_stats = test_graph_stats.merge(
    test_window_labels,
    left_on="window",
    right_index=True,
    how="left"
)

test_graph_stats["contains_attack"] = (
    test_graph_stats["contains_attack"]
    .fillna(0)
    .astype(int)
)

# ------------------------------------------------------------
# 8. Compare TEST normal vs attack windows
# ------------------------------------------------------------

print()
print("-" * 88)
print("TEST TRANSITION-GRAPH STATISTICS")
print("-" * 88)

for label, name in [
    (0, "NORMAL TEST WINDOWS"),
    (1, "ATTACK TEST WINDOWS")
]:

    subset = test_graph_stats[
        test_graph_stats["contains_attack"] == label
    ]

    print()
    print(name)
    print(
        "Windows:",
        len(subset)
    )

    if len(subset):

        print(
            subset[
                [
                    "transition_nodes",
                    "interaction_edges",
                    "interaction_density",
                    "mean_transition_degree",
                    "max_transition_degree",
                    "transition_degree_std"
                ]
            ]
            .describe()
            .round(4)
            .to_string()
        )

print()
print("=" * 88)
print("CELL COMPLETE")
print("=" * 88)

BREG — HELD-OUT TEST RELATIONAL TRANSITION GRAPH

Test flows:
Shape: (36573, 52)
Window range: 204 -> 254
Unique windows: 51

----------------------------------------------------------------------------------------
TEST RELATIONAL TRANSITIONS
----------------------------------------------------------------------------------------
Persistent transitions: 5316
count    5316.0000
mean        3.7061
std         2.6558
min         0.0000
25%         2.0119
50%         2.9585
75%         4.5013
max        22.6008

----------------------------------------------------------------------------------------
TEST TRANSITION-GRAPH STATISTICS
----------------------------------------------------------------------------------------

NORMAL TEST WINDOWS
Windows: 19
       transition_nodes  interaction_edges  interaction_density  mean_transition_degree  max_transition_degree  transition_degree_std
count           19.0000            19.0000              19.0000                 19.0000                19.00